<a href="https://colab.research.google.com/github/hsiuwenliu/Beautiful-Visualization-with-python/blob/master/My_Review_Comments_NSC_114_ANalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Mount Google Drive

First, execute the following cell to mount your Google Drive. You will be prompted to authorize Colab to access your Google Drive files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Once your Drive is mounted, you can access your files. For example, if you have a CSV file named `my_data.csv` in the root of your Google Drive, you can load it into a pandas DataFrame like this:

In [ ]:
"""
import pandas as pd

# Replace 'my_data.csv' with the actual path to your file in Google Drive
file_path = "/content/drive/MyDrive/2023研究/114國科會/google maps reviews scraping result US_more/restaurant_data_combine/2026_01_05_11_20_combine_restaurant_reviews_qualified_共0則餐廳評論.xlsx"

df = pd.read_excel(file_path)
print(f"Successfully loaded data from {file_path}. First 5 rows:")
display(df.head())

"""

In [ ]:
"""
# Excel 轉 SQLite（修正版 v6 - 處理大小寫重複）
import pandas as pd
import sqlite3
import os
from openpyxl import load_workbook
from collections import Counter

def remove_duplicate_columns_case_insensitive(headers):
    """移除重複欄位（不區分大小寫），只保留第一個出現的"""
    seen = set()
    unique_indices = []
    removed = []

    for i, col in enumerate(headers):
        col_lower = col.lower()  # 🔧 轉小寫比較
        if col_lower not in seen:
            seen.add(col_lower)
            unique_indices.append(i)
        else:
            removed.append((i, col))

    return unique_indices, removed


def excel_to_sqlite(excel_path, db_path=None, table_name='reviews', chunk_size=5000):
    """將 Excel 檔案轉換為 SQLite 資料庫"""

    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"找不到檔案: {excel_path}")

    if db_path is None:
        db_path = excel_path.replace('.xlsx', '.db').replace('.xls', '.db')

    file_size_mb = os.path.getsize(excel_path) / (1024 * 1024)
    print(f"📂 來源檔案: {excel_path}")
    print(f"💾 目標資料庫: {db_path}")
    print(f"📊 檔案大小: {file_size_mb:.2f} MB")

    conn = sqlite3.connect(db_path)

    try:
        print(f"📖 使用 openpyxl read_only 模式開啟檔案...")

        wb = load_workbook(excel_path, read_only=True, data_only=True)
        ws = wb.active

        rows = ws.iter_rows()
        header_row = next(rows)

        # 清理欄位名稱
        headers_raw = [str(cell.value).replace(' ', '_').replace('-', '_') if cell.value else f'col_{i}'
                       for i, cell in enumerate(header_row)]

        # 🔍 檢查大小寫重複
        headers_lower = [h.lower() for h in headers_raw]
        duplicates = [item for item, count in Counter(headers_lower).items() if count > 1]
        if duplicates:
            print(f"\n   🔍 發現重複欄位名稱（不區分大小寫）:")
            for dup in duplicates:
                indices = [(i, headers_raw[i]) for i, h in enumerate(headers_lower) if h == dup]
                print(f"      '{dup}' 出現在:")
                for idx, original_name in indices:
                    print(f"         - 第 {idx+1} 欄: '{original_name}'")

        # 🔧 用不區分大小寫的方式找出不重複的欄位索引
        unique_indices, removed = remove_duplicate_columns_case_insensitive(headers_raw)

        # 只保留不重複的欄位名稱
        headers = [headers_raw[i] for i in unique_indices]

        # 在最前面加上 Total_ID 欄位
        headers = ['Total_ID'] + headers

        print(f"\n   原始欄位數: {len(headers_raw)}")
        print(f"   刪除重複後: {len(unique_indices)}")
        print(f"   最終欄位數: {len(headers)} (含 Total_ID)")

        if removed:
            print(f"\n   🗑️ 已刪除 {len(removed)} 個重複欄位:")
            for idx, col_name in removed:
                print(f"      - 第 {idx+1} 欄: '{col_name}'")

        print(f"\n   分塊大小: {chunk_size:,} 行")

        total_rows = 0
        chunk_num = 0
        chunk_data = []
        current_id = 1

        for row in rows:
            # 只取不重複的欄位值
            row_values = [row[i].value for i in unique_indices]

            # 在最前面加上 Total_ID
            row_values = [current_id] + row_values
            current_id += 1

            chunk_data.append(row_values)

            if len(chunk_data) >= chunk_size:
                chunk_num += 1
                df_chunk = pd.DataFrame(chunk_data, columns=headers)

                if chunk_num == 1:
                    df_chunk.to_sql(table_name, conn, if_exists='replace', index=False)
                else:
                    df_chunk.to_sql(table_name, conn, if_exists='append', index=False)

                total_rows += len(chunk_data)
                print(f"   ✅ 第 {chunk_num} 塊完成 (累計: {total_rows:,} 行)")

                chunk_data = []

                if chunk_num % 5 == 0:
                    conn.commit()

        # 處理剩餘資料
        if chunk_data:
            chunk_num += 1
            df_chunk = pd.DataFrame(chunk_data, columns=headers)

            if chunk_num == 1:
                df_chunk.to_sql(table_name, conn, if_exists='replace', index=False)
            else:
                df_chunk.to_sql(table_name, conn, if_exists='append', index=False)

            total_rows += len(chunk_data)
            print(f"   ✅ 第 {chunk_num} 塊完成 (累計: {total_rows:,} 行)")

        conn.commit()
        wb.close()

        # 建立索引
        print("\n🔧 建立索引...")
        cursor = conn.cursor()

        index_cols = ['Total_ID', 'Data_ID', 'Sid', 'Pid', '城市', '商家名稱', '評論分數']
        for col in index_cols:
            if col in headers:
                try:
                    cursor.execute(f'CREATE INDEX IF NOT EXISTS idx_{table_name}_{col} ON {table_name}([{col}])')
                    print(f"   ✅ 建立索引: {col}")
                except Exception as e:
                    print(f"   ⚠️ 索引 {col} 建立失敗: {e}")
        conn.commit()

        print("\n" + "="*50)
        print("✅ 轉換完成!")
        print(f"   📊 總行數: {total_rows:,}")
        print(f"   📋 欄位數: {len(headers)}")
        print(f"   🔢 Total_ID 範圍: 1 ~ {total_rows}")
        print(f"   💾 資料庫: {db_path}")

        db_size_mb = os.path.getsize(db_path) / (1024 * 1024)
        print(f"   📦 資料庫大小: {db_size_mb:.2f} MB")

        return db_path

    finally:
        conn.close()

def query_database(db_path, sql_query):
    """執行 SQL 查詢"""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql(sql_query, conn)
    conn.close()
    return df


def get_table_info(db_path, table_name='reviews'):
    """取得資料表資訊"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 取得行數
    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    row_count = cursor.fetchone()[0]

    # 取得欄位資訊
    cursor.execute(f"PRAGMA table_info({table_name})")
    columns = cursor.fetchall()

    conn.close()

    print(f"\n📊 資料表 '{table_name}' 資訊:")
    print(f"   總行數: {row_count:,}")
    print(f"   欄位數: {len(columns)}")
    print(f"\n   欄位列表:")
    for col in columns:
        print(f"      {col[1]} ({col[2]})")

    return row_count, columns


# ============ 執行轉換 ============
# This part should be removed from the definition cell, as it's an example of usage.
# We will call excel_to_sqlite and get_table_info in separate steps if the user desires.

# excel_file = "/content/drive/MyDrive/2023研究/114國科會/google maps reviews scraping result US_more/restaurant_data_combine/2026_01_05_11_20_combine_restaurant_reviews_qualified_共0則餐廳評論.xlsx"

# db_path = excel_to_sqlite(
#     excel_path=excel_file,
#     db_path="/content/drive/MyDrive/2023研究/114國科會/restaurant_reviews.db",
#     table_name="reviews_qualified",
#     chunk_size=5000
# )

# # 測試查詢
# print("\n📝 測試查詢前 5 筆:")
# df_test = query_database(db_path, "SELECT Total_ID, Pid, Sid, 商家名稱, 評論分數 FROM reviews_qualified LIMIT 5")
# display(df_test)

"""

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sqlite3
import pandas as pd

db_path="/content/drive/MyDrive/2023研究/114國科會/restaurant_reviews.db"

def query_database(db_path, sql_query):
    """執行 SQL 查詢"""
    conn = sqlite3.connect(db_path)
    df = pd.read_sql(sql_query, conn)
    conn.close()
    return df

# 測試查詢
print("\n📝 測試查詢前 5 筆:")
df_test = query_database(db_path, "SELECT Total_ID, Pid, Sid, 商家名稱, 評論分數 FROM reviews_qualified LIMIT 5")
display(df_test)

In [ ]:
# db_path = "/content/drive/MyDrive/2023研究/114國科會/restaurant_reviews.db"
# table_name = "reviews_qualified"
# # get_table_info(db_path, table_name)

In [ ]:
import sqlite3
import pandas as pd

db_path = "/content/drive/MyDrive/2023研究/114國科會/restaurant_reviews.db"
table_name = "reviews_qualified"

# Connect to the database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

try:
    # Use the user-specified Data_ID
    random_data_id = '0x8640bf2545715cb3:0xdc725b8264ec0cbe'

    print(f"選取的 Data_ID: {random_data_id}\n")

    # Query all variables for the selected Data_ID
    sql_query = f"SELECT * FROM {table_name} WHERE Data_ID = '{random_data_id}'"
    df_selected_data = pd.read_sql(sql_query, conn)

    # Display the results
    print(f"顯示 Data_ID '{random_data_id}' 的所有變數與資料:")
    display(df_selected_data)

finally:
    # Close the connection
    conn.close()

In [ ]:
import pandas as pd
from google.colab import files

# Ensure df_selected_data exists from previous execution
if 'df_selected_data' in locals() and isinstance(df_selected_data, pd.DataFrame):
    output_excel_path = "selected_restaurant_data.xlsx"
    df_selected_data.to_excel(output_excel_path, index=False)

    print(f"資料已成功儲存至 '{output_excel_path}'")
    print("請點擊下方連結下載檔案:")
    files.download(output_excel_path)
else:
    print("錯誤: 找不到 'df_selected_data' DataFrame。請先執行隨機選擇 Data_ID 的儲存格。")

In [ ]:
# ====================================================================
# PyABSA 模型備份與載入 - Colab 簡化版
# ====================================================================

# ============================================================
# 📌 步驟 1：首次使用 - 備份模型到 Google Drive（只需執行一次）
# ============================================================
"""
# === 首次備份 ===
def backup_to_drive():
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    import shutil

    # 載入模型
    from pyabsa import ATEPCCheckpointManager
    aspect_extractor = ATEPCCheckpointManager.get_aspect_extractor(
        checkpoint='english',
        auto_device=True
    )

    # 備份
    backup_path = "/content/drive/MyDrive/2023研究/114國科會/pyabsa_backup"
    os.makedirs(backup_path, exist_ok=True)

    # 快取目錄
    cache_dirs = {
        "/root/.cache/huggingface": "huggingface",
        "/root/.cache/pyabsa": "pyabsa",
        "/content/checkpoints": "checkpoints",
    }

    for src, name in cache_dirs.items():
        if os.path.exists(src):
            dest = os.path.join(backup_path, name)
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(src, dest)
            print(f"✅ 已備份: {name}")

    print(f"🎉 備份完成！路徑: {backup_path}")
    return aspect_extractor

"""

# === 從備份載入 ===
def load_from_drive():
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    import shutil

    backup_path = "/content/drive/MyDrive/2023研究/114國科會/pyabsa_backup"

    # 還原快取
    restore_dirs = {
        "huggingface": "/root/.cache/huggingface",
        "pyabsa": "/root/.cache/pyabsa",
        "checkpoints": "/content/checkpoints",
    }

    for name, dest in restore_dirs.items():
        src = os.path.join(backup_path, name)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            if os.path.exists(dest):
                shutil.rmtree(dest)
            shutil.copytree(src, dest)
            print(f"✅ 已還原: {name}")

    # 載入模型
    from pyabsa import ATEPCCheckpointManager
    aspect_extractor = ATEPCCheckpointManager.get_aspect_extractor(
        checkpoint='english',
        auto_device=True
    )

    print("🎉 模型載入成功！")
    return aspect_extractor

In [ ]:
# Aspect Term Extract & Sentiment Inference
# You can inference from a list of setences or a DatasetItem from PyABSA
examples = ['Staff was very rude but food was delicious']
# examples = ['Camera quality is very good but battery drains fast']
inference_source = examples
atepc_result = aspect_extractor.extract_aspect(inference_source=inference_source,  #
                          pred_sentiment=True,  # Predict the sentiment of extracted aspect terms
                          )
atepc_result

In [ ]:
examples = [
    'Camera quality is very good but battery drains fast',
    'The screen is beautiful but the price is too high',
    'Service was excellent and the atmosphere was cozy'
]

atepc_result = aspect_extractor.extract_aspect(
    inference_source=examples,
    pred_sentiment=True,
)

for result in atepc_result:
    print(f"句子: {result['sentence']}")
    print(f"方面: {result['aspect']}")
    print(f"情感: {result['sentiment']}")
    print("---")

In [ ]:
"""

# !pip install pyabsa

# User request: filter df_selected_data by '評論分數' and use '評論內容' for examples
# Assuming a filter for '評論分數' is desired, e.g., == 5.
# You can change the filter condition as needed.

# Import pandas if not already imported in this scope, though it's likely available.
import pandas as pd

# Check if df_selected_data exists and is not empty
if 'df_selected_data' in locals() and not df_selected_data.empty:
    # Filter by 評論分數, assuming a score of 5 for demonstration
    # User requested '==' but no value was provided, using 5 as an example.
    # Make sure '評論分數' is a numeric type for comparison
    df_selected_data['評論分數'] = pd.to_numeric(df_selected_data['評論分數'], errors='coerce')
    filtered_df = df_selected_data[df_selected_data['評論分數'] == 5] # Example filter

    if not filtered_df.empty:
        # Extract '評論內容' and convert to a list, dropping any NaN values
        new_examples = filtered_df['評論內容'].dropna().tolist()

        if new_examples:
            print(f"提取了 {len(new_examples)} 則評論內容，評論分數為 5 的範例。")
            print("正在進行方面級情感分析...")
            # Ensure aspect_extractor is available (it was defined in u8FKFHKgbm73)
            # This assumes u8FKFHKgbm73 has been executed.
            if 'aspect_extractor' in locals():
                atepc_result_filtered = aspect_extractor.extract_aspect(inference_source=new_examples,
                                                                       pred_sentiment=True)

                print("\n篩選後的評論情感分析結果:")
                for i, result in enumerate(atepc_result_filtered):
                    print(f"評論 {i+1}:")
                    print(f"  句子: {result['sentence']}")
                    print(f"  方面: {result['aspect']}")
                    print(f"  情感: {result['sentiment']}")
                    print("---")
            else:
                print("錯誤: 'aspect_extractor' 未定義。請確保已執行 PyABSA 模型初始化的儲存格。")
        else:
            print("在篩選後的資料中找不到 '評論內容'。")
    else:
        print("在選取的店家資料中，找不到評論分數為 5 的評論。請檢查您的篩選條件。")
else:
    print("錯誤: 找不到 'df_selected_data' DataFrame 或其為空。請先執行隨機選擇 Data_ID 的儲存格。")

"""

In [ ]:
"""

import pandas as pd
from collections import defaultdict

# Assuming atepc_result_filtered is available from previous execution
if 'atepc_result_filtered' in locals() and atepc_result_filtered:
    # Dictionary to store aspect counts and sentiment counts
    aspect_sentiment_counts = defaultdict(lambda: {'Positive': 0, 'Negative': 0, 'Neutral': 0, 'Total': 0})

    for review_result in atepc_result_filtered:
        aspects = review_result['aspect']
        sentiments = review_result['sentiment']

        for i in range(len(aspects)):
            aspect = aspects[i]
            sentiment = sentiments[i]

            aspect_sentiment_counts[aspect][sentiment] += 1
            aspect_sentiment_counts[aspect]['Total'] += 1

    # Convert the aggregated data to a DataFrame for easier analysis and display
    df_analysis = pd.DataFrame.from_dict(aspect_sentiment_counts, orient='index')
    df_analysis = df_analysis.fillna(0).astype(int) # Fill NaN with 0 and convert to integer
    df_analysis = df_analysis.sort_values(by='Total', ascending=False)

    print("\n=== 5 星評論的方面級情感分析摘要 ===\n")
    print("以下表格顯示了在 5 星評論中提及的各個方面，以及這些方面的情感分佈：")
    display(df_analysis)

    # Further analysis: Top 10 most positive aspects
    print("\n=== 前 10 個最常被正面提及的方面 ===")
    df_positive_aspects = df_analysis[df_analysis['Positive'] > 0].copy()
    df_positive_aspects['Positive_Ratio'] = df_positive_aspects['Positive'] / df_positive_aspects['Total']
    display(df_positive_aspects.sort_values(by='Positive', ascending=False).head(10)[['Positive', 'Total', 'Positive_Ratio']])

    # Further analysis: Any significantly negative aspects in 5-star reviews?
    print("\n=== 在 5 星評論中被提及為負面的方面 (僅顯示有負面評論的方面) ===")
    df_negative_aspects = df_analysis[df_analysis['Negative'] > 0].copy()
    df_negative_aspects['Negative_Ratio'] = df_negative_aspects['Negative'] / df_negative_aspects['Total']
    display(df_negative_aspects.sort_values(by='Negative', ascending=False)[['Negative', 'Total', 'Negative_Ratio']])

else:
    print("錯誤: 找不到 'atepc_result_filtered' 變數或其為空。請確保已執行 5 星評論篩選和方面情感分析的儲存格。")

"""

### 定義情緒分析函式

以下函式將會根據指定的評論分數，篩選出相關評論，進行方面級情感分析，並輸出結果摘要。

In [ ]:
import pandas as pd
from collections import defaultdict

def perform_sentiment_analysis_for_score(df_data, score, aspect_extractor):
    """
    根據指定的評論分數，對評論內容進行方面級情感分析，並輸出結果摘要。

    Args:
        df_data (pd.DataFrame): 包含評論資料的 DataFrame (例如 df_selected_data)。
        score (int): 要篩選的評論分數 (例如 5)。
        aspect_extractor: PyABSA 的方面情感提取器實例。
    """
    print(f"\n{'='*60}\n正在處理評論分數為 {score} 的評論...\n{'='*60}\n")

    # 確保 '評論分數' 是數值類型
    df_data['評論分數'] = pd.to_numeric(df_data['評論分數'], errors='coerce')
    filtered_df = df_data[df_data['評論分數'] == score]

    if filtered_df.empty:
        print(f"在資料中找不到評論分數為 {score} 的評論。")
        return

    new_examples = filtered_df['評論內容'].dropna().tolist()

    if not new_examples:
        print(f"評論分數為 {score} 的評論中沒有可用的 '評論內容'。")
        return

    print(f"提取了 {len(new_examples)} 則評論內容，評論分數為 {score} 的範例。")
    print("正在進行方面級情感分析...")

    atepc_result_filtered = aspect_extractor.extract_aspect(inference_source=new_examples, pred_sentiment=True)

    aspect_sentiment_counts = defaultdict(lambda: {'Positive': 0, 'Negative': 0, 'Neutral': 0, 'Total': 0})

    for review_result in atepc_result_filtered:
        aspects = review_result['aspect']
        sentiments = review_result['sentiment']

        for i in range(len(aspects)):
            aspect = aspects[i]
            sentiment = sentiments[i]

            aspect_sentiment_counts[aspect][sentiment] += 1
            aspect_sentiment_counts[aspect]['Total'] += 1

    df_analysis = pd.DataFrame.from_dict(aspect_sentiment_counts, orient='index')
    df_analysis = df_analysis.fillna(0).astype(int)
    df_analysis = df_analysis.sort_values(by='Total', ascending=False)

    print(f"\n=== 評論分數為 {score} 的方面級情感分析摘要 ===\n")
    print("以下表格顯示了在這些評論中提及的各個方面，以及這些方面的情感分佈：")
    display(df_analysis)

    # Top 10 most positive aspects
    print(f"\n=== 評論分數為 {score} 的前 10 個最常被正面提及的方面 ===")
    df_positive_aspects = df_analysis[df_analysis['Positive'] > 0].copy()
    if not df_positive_aspects.empty:
        df_positive_aspects['Positive_Ratio'] = df_positive_aspects['Positive'] / df_positive_aspects['Total']
        display(df_positive_aspects.sort_values(by='Positive', ascending=False).head(10)[['Positive', 'Total', 'Positive_Ratio']])
    else:
        print("沒有正面提及的方面。")

    # Any significantly negative aspects in these reviews?
    print(f"\n=== 評論分數為 {score} 的評論中被提及為負面的方面 (僅顯示有負面評論的方面) ===")
    df_negative_aspects = df_analysis[df_analysis['Negative'] > 0].copy()
    if not df_negative_aspects.empty:
        df_negative_aspects['Negative_Ratio'] = df_negative_aspects['Negative'] / df_negative_aspects['Total']
        display(df_negative_aspects.sort_values(by='Negative', ascending=False)[['Negative', 'Total', 'Negative_Ratio']])
    else:
        print("沒有負面提及的方面。")


### 評論分數為 5 的分析

In [ ]:
# 確保 df_selected_data 和 aspect_extractor 已經定義
if 'df_selected_data' in locals() and 'aspect_extractor' in locals():
    perform_sentiment_analysis_for_score(df_selected_data.copy(), 5, aspect_extractor)
else:
    print("錯誤: 'df_selected_data' 或 'aspect_extractor' 未定義。請確保之前的儲存格已執行。")

### 評論分數為 4 的分析

In [ ]:
if 'df_selected_data' in locals() and 'aspect_extractor' in locals():
    perform_sentiment_analysis_for_score(df_selected_data.copy(), 4, aspect_extractor)
else:
    print("錯誤: 'df_selected_data' 或 'aspect_extractor' 未定義。請確保之前的儲存格已執行。")

### 評論分數為 3 的分析

In [ ]:
if 'df_selected_data' in locals() and 'aspect_extractor' in locals():
    perform_sentiment_analysis_for_score(df_selected_data.copy(), 3, aspect_extractor)
else:
    print("錯誤: 'df_selected_data' 或 'aspect_extractor' 未定義。請確保之前的儲存格已執行。")

### 評論分數為 2 的分析

In [ ]:
if 'df_selected_data' in locals() and 'aspect_extractor' in locals():
    perform_sentiment_analysis_for_score(df_selected_data.copy(), 2, aspect_extractor)
else:
    print("錯誤: 'df_selected_data' 或 'aspect_extractor' 未定義。請確保之前的儲存格已執行。")

### 評論分數為 1 的分析

In [ ]:
if 'df_selected_data' in locals() and 'aspect_extractor' in locals():
    perform_sentiment_analysis_for_score(df_selected_data.copy(), 1, aspect_extractor)
else:
    print("錯誤: 'df_selected_data' 或 'aspect_extractor' 未定義。請確保之前的儲存格已執行。")

### 定義多分數範圍情緒分析函式

以下函式將會根據指定的評論分數列表，篩選出相關評論，進行方面級情感分析，並輸出結果摘要。這個函式可以處理多個評論分數的情感分析。

In [ ]:
import pandas as pd
from collections import defaultdict

def perform_sentiment_analysis_for_score_range(df_data, scores_list, aspect_extractor):
    """
    根據指定的評論分數列表，對評論內容進行方面級情感分析，並輸出結果摘要。

    Args:
        df_data (pd.DataFrame): 包含評論資料的 DataFrame (例如 df_selected_data)。
        scores_list (list): 要篩選的評論分數列表 (例如 [1, 2, 3])。
        aspect_extractor: PyABSA 的方面情感提取器實例。
    """
    scores_str = ', '.join(map(str, scores_list))
    print(f"\n{'='*60}\n正在處理評論分數為 {scores_str} 的評論...\n{'='*60}\n")

    # 確保 '評論分數' 是數值類型
    df_data['評論分數'] = pd.to_numeric(df_data['評論分數'], errors='coerce')
    filtered_df = df_data[df_data['評論分數'].isin(scores_list)]

    if filtered_df.empty:
        print(f"在資料中找不到評論分數為 {scores_str} 的評論。")
        return

    new_examples = filtered_df['評論內容'].dropna().tolist()

    if not new_examples:
        print(f"評論分數為 {scores_str} 的評論中沒有可用的 '評論內容'。")
        return

    print(f"提取了 {len(new_examples)} 則評論內容，評論分數為 {scores_str} 的範例。")
    print("正在進行方面級情感分析...")

    atepc_result_filtered = aspect_extractor.extract_aspect(inference_source=new_examples, pred_sentiment=True)

    aspect_sentiment_counts = defaultdict(lambda: {'Positive': 0, 'Negative': 0, 'Neutral': 0, 'Total': 0})

    for review_result in atepc_result_filtered:
        aspects = review_result['aspect']
        sentiments = review_result['sentiment']

        for i in range(len(aspects)):
            aspect = aspects[i]
            sentiment = sentiments[i]

            aspect_sentiment_counts[aspect][sentiment] += 1
            aspect_sentiment_counts[aspect]['Total'] += 1

    df_analysis = pd.DataFrame.from_dict(aspect_sentiment_counts, orient='index')
    df_analysis = df_analysis.fillna(0).astype(int)
    df_analysis = df_analysis.sort_values(by='Total', ascending=False)

    print(f"\n=== 評論分數為 {scores_str} 的方面級情感分析摘要 ===\n")
    print("以下表格顯示了在這些評論中提及的各個方面，以及這些方面的情感分佈：")
    display(df_analysis)

    # Top 10 most positive aspects
    print(f"\n=== 評論分數為 {scores_str} 的前 10 個最常被正面提及的方面 ===")
    df_positive_aspects = df_analysis[df_analysis['Positive'] > 0].copy()
    if not df_positive_aspects.empty:
        df_positive_aspects['Positive_Ratio'] = df_positive_aspects['Positive'] / df_positive_aspects['Total']
        display(df_positive_aspects.sort_values(by='Positive', ascending=False).head(10)[['Positive', 'Total', 'Positive_Ratio']])
    else:
        print("沒有正面提及的方面。")

    # Any significantly negative aspects in these reviews?
    print(f"\n=== 評論分數為 {scores_str} 的評論中被提及為負面的方面 (僅顯示有負面評論的方面) ===")
    df_negative_aspects = df_analysis[df_analysis['Negative'] > 0].copy()
    if not df_negative_aspects.empty:
        df_negative_aspects['Negative_Ratio'] = df_negative_aspects['Negative'] / df_negative_aspects['Total']
        display(df_negative_aspects.sort_values(by='Negative', ascending=False)[['Negative', 'Total', 'Negative_Ratio']])
    else:
        print("沒有負面提及的方面。")


### 評論分數為 1 到 3 的綜合分析

In [ ]:
# 確保 df_selected_data 和 aspect_extractor 已經定義
if 'df_selected_data' in locals() and 'aspect_extractor' in locals():
    perform_sentiment_analysis_for_score_range(df_selected_data.copy(), [1, 2, 3], aspect_extractor)
else:
    print("錯誤: 'df_selected_data' 或 'aspect_extractor' 未定義。請確保之前的儲存格已執行。")

### 提取特定方面和情緒的評論意見

以下函式將會從方面級情感分析結果中，提取針對特定方面 (例如 `food` 或 `service`) 且帶有指定情緒 (例如 `Negative` 或 `Positive`) 的原始評論句子。

### 使用 Aspect Sentiment Triplet Extraction (ASTE) 進行情感三元組提取

這個方法會從句子中提取方面詞 (Aspect Term)、情感詞 (Opinion Term) 以及它們之間的情緒 (Sentiment)，以三元組 `(aspect, opinion, sentiment)` 的形式呈現。這能提供更細緻的情感分析結果。

In [ ]:
"""

from pyabsa import AspectSentimentTripletExtraction as ASTE

# Load a pre-trained ASTE model (e.g., multilingual)
# This might download the model if not already cached
triplet_extractor = ASTE.AspectSentimentTripletExtractor("multilingual")

# Perform prediction on a sentence
text = "The food was delicious, but the service was slow."
print(f"分析句子: '{text}'")
result = triplet_extractor.predict(text)

print("\n情感三元組提取結果:")
for res in result:
    print(res)

"""

In [ ]:
# # Perform prediction on a sentence
# text = "The food was delicious, but the service was slow."
# result = triplet_extractor.predict(text)
# print(result)

In [ ]:
# This the best
# 寫 115 NSC 時的第一版


import warnings
warnings.filterwarnings('ignore')
from collections import Counter

# ============================================
# is_adjective_or_adverb 函數（保持不變）
# ============================================

def is_adjective_or_adverb(word):
    """判斷是否為形容詞或副詞"""

    positive_adjectives = {
        'beautiful', 'excellent', 'great', 'good', 'amazing', 'wonderful',
        'fantastic', 'outstanding', 'impressive', 'stunning', 'gorgeous',
        'perfect', 'superb', 'fabulous', 'marvelous', 'magnificent',
        'delicious', 'tasty', 'delightful', 'comfortable', 'cozy',
        'friendly', 'helpful', 'efficient', 'smooth', 'easy',
        'fast', 'quick', 'reliable', 'stable', 'secure',
        'spacious', 'large', 'big', 'bright', 'clean', 'modern',
        'innovative', 'creative', 'unique', 'special', 'exceptional',
        'high-quality', 'premium', 'luxurious', 'elegant', 'stylish',
        'soft', 'warm', 'cool', 'fresh', 'clear', 'sharp', 'vivid',
        'dim', 'bright', 'light', 'dark', 'crisp', 'smooth',
        'powerful', 'strong', 'solid', 'durable', 'sturdy',
        'romantic', 'peaceful', 'quiet', 'calm', 'relaxing',
        'tender', 'juicy', 'crispy', 'creamy', 'rich',
        'lightweight', 'portable', 'convenient', 'accessible',
        'nice', 'fine', 'lovely', 'charming', 'attractive',
        'pleasant', 'enjoyable', 'satisfying', 'adequate',
        'reasonable', 'decent', 'acceptable', 'maintained', 'tested','interesting'
    }

    negative_adjectives = {
        'terrible', 'bad', 'poor', 'awful', 'horrible', 'disappointing',
        'frustrating', 'annoying', 'uncomfortable', 'inconvenient',
        'slow', 'sluggish', 'laggy', 'unstable', 'unreliable',
        'expensive', 'overpriced', 'costly', 'pricey', 'high',
        'small', 'tiny', 'cramped', 'limited', 'insufficient',
        'short', 'brief', 'inadequate', 'lacking', 'missing',
        'noisy', 'loud', 'distracting', 'disturbing',
        'dirty', 'messy', 'cluttered', 'outdated', 'old', 'obsolete',
        'complicated', 'confusing', 'difficult', 'hard', 'complex',
        'broken', 'damaged', 'faulty', 'defective', 'flawed',
        'cold', 'hot', 'harsh', 'rough', 'stiff',
        'weak', 'fragile', 'unstable', 'insecure',
        'rude', 'unfriendly', 'unhelpful', 'careless',
        'sensitive', 'buggy', 'glitchy', 'sticky'
    }

    intensifiers = {
        'very', 'extremely', 'really', 'quite', 'too', 'so',
        'incredibly', 'amazingly', 'exceptionally', 'remarkably',
        'absolutely', 'totally', 'completely', 'utterly', 'entirely',
        'highly', 'fairly', 'rather', 'somewhat', 'pretty',
        'disappointingly', 'surprisingly', 'unusually', 'well', 'thoroughly'
    }

    word_lower = word.lower().strip('.,!?;:')

    if word_lower in positive_adjectives:
        return True, 'positive_adj'
    if word_lower in negative_adjectives:
        return True, 'negative_adj'
    if word_lower in intensifiers:
        return True, 'intensifier'

    adj_suffixes = ['ful', 'less', 'ous', 'ive', 'able', 'ible', 'al', 'ic', 'y', 'ing', 'ed']
    adv_suffixes = ['ly']

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    for suffix in adv_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adv'

    return False, 'none'


# ============================================
# 輔助函數
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """找到句子邊界"""
    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in ['.', '!', '?']:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in ['.', '!', '?']:
            sentence_end = i
            break

    return sentence_start, sentence_end


def detect_be_verb(tokens, start_idx):
    """檢測 be 動詞"""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    if word in ['is', 'was', 'were', 'are', 'be', 'been', "'s"]:
        return True, 1, False

    if word in ["isn't", "isnt", "wasn't", "wasnt", "weren't", "werent", "aren't", "arent"]:
        return True, 1, True

    if word in ['isn', 'wasn', 'weren', 'aren'] and start_idx + 2 < len(tokens):
        if tokens[start_idx + 1] == "'" and tokens[start_idx + 2].lower() in ['t', 'nt']:
            return True, 3, True

    return False, 0, False


def handle_that_with_negation(tokens, start_pos, end_pos):
    """處理 that + 否定"""
    result = []
    i = start_pos

    while i < end_pos:
        word = tokens[i]
        word_lower = word.lower()

        if word_lower == 'that' and i + 1 < end_pos:
            found, skip, is_negative = detect_be_verb(tokens, i + 1)

            if found and is_negative:
                i += 1 + skip
                result.append("not")
                continue

        result.append(word)
        i += 1

    return result


# ============================================
# 🆕 找到前一個屬性的結束位置
# ============================================

def find_previous_aspect_end(tokens, current_start_idx, all_aspect_positions):
    """
    找到前一個屬性的結束位置

    參數:
        tokens: 詞列表
        current_start_idx: 當前屬性的起始位置
        all_aspect_positions: 所有屬性的位置列表

    返回:
        前一個屬性的結束位置（如果沒有則返回 None）
    """
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            # 找到在當前屬性之前的最後一個屬性
            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(tokens, current_end_idx, all_aspect_positions):
    """找到下一個屬性的起始位置"""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# 🆕 核心改進：提取時檢測前後屬性邊界
# ============================================


def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """
    提取完整意見詞 - 帶前後邊界檢測 (V2 - 新增名詞補全邏輯)
    修正：針對 "interesting [dining] experience"，補回 "experience"
    """

    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)

    # ⭐ 找到前一個屬性的結束位置
    prev_aspect_end = find_previous_aspect_end(tokens, start_idx, all_aspect_positions)

    # ⭐ 找到下一個屬性的起始位置
    next_aspect_start = find_next_aspect_position(tokens, end_idx, all_aspect_positions)

    # ⭐ 如果有下一個屬性，限制向後搜索範圍
    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # ========================================
    # 模式 1: 前置修飾詞 (例如 interesting [dining])
    # ========================================

    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, modifier_type = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in [',', 'and', 'with', 'or', 'but']:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

        # ⭐⭐ 新增核心修正：名詞補全 (Noun Completion) ⭐⭐
        # 如果是 "interesting [dining] experience"，我們要把 experience 抓進來
        # 檢查屬性後面的那個字
        next_pos = end_idx + 1
        if next_pos < sentence_end:
            next_word = tokens[next_pos]

            # 定義禁用的後續詞 (動詞、介系詞、連接詞)
            # 我們只想要名詞，所以排除這些
            forbidden_next = {
                'is', 'was', 'are', 'were', 'be', 'been', # Be動詞
                'has', 'have', 'had', 'does', 'do',       # 助動詞
                'but', 'and', 'or', 'so', 'because',      # 連接詞
                'with', 'for', 'to', 'in', 'at', 'on', 'of', 'by', # 介系詞
                'that', 'which', 'who', 'this', 'it',     # 代詞
                ',', '.', '!', '?', ';', '-'              # 標點
            }

            # 如果下一個字是字母，且不是禁用詞，我們假設它是名詞並加入
            # (例如 experience, atmosphere, environment, options)
            if next_word.isalpha() and next_word.lower() not in forbidden_next:
                 opinion_words.append(next_word)

    # ========================================
    # 模式 2: be 動詞 + 形容詞 (如果前面沒有抓到 pre_modifiers 才做，或者依需求疊加)
    # ========================================

    # 只有當沒有抓到前置修飾詞，或者雖然抓到了但想要檢查後面有沒有 "is very good" 這種結構
    # 為了避免重複，通常如果有 pre_modifiers 且已經抓了名詞補全，這部分可以跳過
    # 但為了保險 (例如 "Good [food] was served")，我們保持原邏輯，但做一個判斷

    # 這裡維持您原本的邏輯，但如果您希望 Pre-modifier 優先，可以加一個 check
    should_check_post = True
    if pre_modifiers and pattern_type == "pre_modifier":
         # 如果已經是 "interesting experience"，通常不需要再找後面的 be 動詞
         pass

    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers: # 簡單起見，如果前面有修飾詞就不找 be 動詞了
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []

        if is_negative:
            post_modifiers.append("not")

        for i in range(opinion_start, opinion_end):
            word = tokens[i]

            if word in ['.', '!', '?', ';']:
                break

            if word.lower() == 'but' and next_aspect_start is not None:
                if i + 1 < len(tokens) and i + 1 >= next_aspect_start - 2:
                    break

            post_modifiers.append(word)

        if post_modifiers:
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # ========================================
    # 模式 3: 動詞短語
    # ========================================

    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in ['.', '!', '?', ';']:
                break
            temp_words.append(word)

        if temp_words:
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # ========================================
    # 模式 4: 上下文搜索
    # ========================================

    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, modifier_type = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                context_words = []
                for j in range(context_start, context_end):
                    if tokens[j] not in ['.', '!', '?', ';']:
                        context_words.append(tokens[j])

                opinion_words.extend(context_words)
                pattern_type = "context_search"
                break

            if word in ['.', '!', '?']:
                break

    # 處理 that + 否定
    opinion_words = handle_that_with_negation(opinion_words, 0, len(opinion_words))

    opinion_phrase = ' '.join(opinion_words).strip()

    return opinion_phrase, pattern_type



# ============================================
# 情感糾正
# ============================================

def correct_sentiment_enhanced(aspect, opinion, predicted_sentiment):
    """情感糾正"""

    opinion_lower = opinion.lower()

    if ' but ' in opinion_lower:
        parts = opinion_lower.split(' but ', 1)
        if len(parts) == 2:
            after_but = parts[1]

            negative_words = {
                'slow', 'expensive', 'small', 'limited', 'short',
                'terrible', 'bad', 'poor', 'disappointing', 'awful',
                'noisy', 'unstable', 'outdated', 'uncomfortable'
            }

            for neg_word in negative_words:
                if neg_word in after_but:
                    return 'Negative'

    for_the_patterns = ['for the price', 'for the cost', 'for the money', 'for price', 'for cost']
    for pattern in for_the_patterns:
        if pattern in opinion_lower:
            negative_indicators = ['small', 'limited', 'short', 'expensive', 'high', 'too']
            for indicator in negative_indicators:
                if indicator in opinion_lower:
                    return 'Negative'

    standalone_negative = {
        'limited', 'lacking', 'insufficient', 'inadequate',
        'disappointing', 'poor', 'bad', 'terrible', 'awful',
        'outdated', 'old', 'broken', 'slow', 'noisy',
        'uncomfortable', 'expensive', 'small', 'short'
    }

    opinion_words = opinion_lower.split()
    for word in opinion_words:
        clean_word = word.strip('.,!?;:')
        if clean_word in standalone_negative:
            return 'Negative'

    return predicted_sentiment


# ============================================
# 🆕 格式化顯示（移除結尾連接詞）
# ============================================


import re

def format_opinion_for_display(opinion):
    """
    格式化意見用於顯示 (V15 - 智慧保留 '比較級' 與 '性能指標')
    保留：'higher contrast, deeper blacks' (因含有 deeper)
    保留：'short, lasting only 4 hours' (因含有 only/數字)
    刪除：'colors, black frame' (普通列舉)
    """

    if not opinion:
        return ""

    # 重要名詞列表
    important_nouns = {
        'experience', 'quality', 'service', 'food', 'price',
        'performance', 'design', 'features', 'options', 'selection',
        'system', 'conditions', 'footage', 'screen', 'tasks',
        'gaming', 'back', 'frame', 'detail', 'mode', 'photos',
        'stabilization', 'colors', 'blacks', 'rate', 'lag',
        'connectivity', 'drawbacks', 'use', 'bugs', 'equipment',
        'slot', 'device', 'issues', 'people', 'day', 'sensor', 'lens',
        'center', 'location', 'access', 'contrast', 'brightness' # 新增 contrast
    }

    # 定義要移除的句首詞
    leading_words_to_remove = {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'also', 'too', 'either',
        'has', 'have', 'had',
        'contains', 'includes', 'consists',
        'offers', 'provides', 'features',
        'is', 'are', 'was', 'were', 'looks', 'feels',
        'it', 'itself', 'they', 'themselves', 'we', 'ourselves',
        'i', 'myself', 'you', 'yourself'
    }

    do_not_merge_words = {
        'a', 'an', 'the', 'it', 'is', 'we', 'i', 'you', 'he', 'she',
        'of', 'in', 'to', 'for', 'with', 'on', 'at', 'my', 'our'
    }

    general_cut_off = {
        'making', 'causing', 'forcing', 'leaving', 'rendering',
        'unless', 'except', 'besides', 'despite', 'although',
        'which', 'where', 'when', 'because', 'since'
    }

    post_dash_cut_off = {
        'with', 'and', 'but', 'including', 'plus', 'to', 'for', 'at'
    }

    # 1. 初步清理
    opinion = re.sub(r'([a-z])-(?=[A-Z])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'([a-zA-Z0-9])-(?=[a-zA-Z0-9])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    words = opinion.split()

    # 2. 移除句首冗餘詞
    while words and words[0].lower() in leading_words_to_remove:
        words.pop(0)

    # 3. 處理連字符合併
    cleaned_words = []
    i = 0
    while i < len(words):
        word = words[i]

        if i + 1 < len(words):
            next_word = words[i+1]
            if word.endswith('-') and len(word) > 1:
                if next_word.isdigit() or next_word.lower() in do_not_merge_words:
                    cleaned_words.append(word.rstrip('-'))
                else:
                    merged = word + next_word
                    cleaned_words.append(merged)
                    i += 2
                    continue
            elif next_word == '-' and i + 2 < len(words):
                 following_word = words[i+2]
                 if following_word.isdigit() or following_word.lower() in do_not_merge_words:
                     cleaned_words.append(word)
                     cleaned_words.append('-')
                     i += 2
                     continue
                 merged = word + "-" + following_word
                 cleaned_words.append(merged)
                 i += 3
                 continue
        cleaned_words.append(word)
        i += 1
    words = cleaned_words

    # 4. 核心截斷邏輯
    truncated_words = []
    valid_length = 0
    has_passed_dash = False

    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')
        is_truncator = False

        if word == '-' or word.startswith('-'):
            if i + 1 < len(words):
                next_w = words[i+1].lower()
                if next_w in {'i', 'we', 'it', 'he', 'she', 'they'}:
                    break
            has_passed_dash = True
            truncated_words.append(word)
            continue

        elif word.endswith('-') and '-' not in word[:-1] and valid_length >= 1:
            is_truncator = True

        if word_lower in general_cut_off and valid_length >= 2:
            break

        if has_passed_dash and word_lower in post_dash_cut_off:
            break

        if is_truncator and valid_length >= 1:
            break

        truncated_words.append(word)
        if word not in {',', 'and', 'but', '-'}:
            valid_length += 1

    words = truncated_words

    # 5. 修正逗號
    final_words = []
    for i, word in enumerate(words):
        if word == ',':
            if final_words and not final_words[-1].endswith(','):
                final_words[-1] = final_words[-1] + ','
        else:
            clean_word = word.lstrip(',')
            if clean_word:
                final_words.append(clean_word)
    words = final_words

    # 6. 移除結尾垃圾詞
    trailing_words_to_remove = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'too', 'either', 'also', 'but', 'however',
        'a', 'an', 'the', ',', '-', 'is', 'are', 'was', 'were',
        'need', 'justify', 'make', 'do', 'want', 'require', 'expect',
        'produces', 'provides', 'offers', 'features', 'includes', 'creates', 'shows'
    }

    dangling_adjectives = {
        'high', 'low', 'large', 'small', 'good', 'bad', 'great', 'poor',
        'long', 'short', 'new', 'old', 'many', 'few', 'heavy', 'light',
        'strong', 'weak', 'fast', 'slow'
    }

    prepositions_for_check = {'with', 'for', 'in', 'on', 'at', 'of', 'and', 'but', 'or'}

    while words:
        last_word = words[-1].lower().strip('.,!?;:')

        if last_word in important_nouns:
            break
        should_remove = False
        if last_word in trailing_words_to_remove or words[-1].endswith('-') or words[-1].isdigit():
            should_remove = True
        if not should_remove and len(words) >= 2:
            second_last = words[-2].lower().strip(',')
            if last_word in dangling_adjectives and second_last in prepositions_for_check:
                should_remove = True
        if words[-1].endswith(','):
            words[-1] = words[-1].rstrip(',')
            if not words[-1]:
                words.pop()
                continue
        if should_remove:
            words.pop()
        else:
            break

    # 7. 轉折詞處理
    result_text = ' '.join(words)
    connectors = [' but ', ' however ', ' though ', ' and ']

    for connector in connectors:
        if connector in result_text.lower():
            last_connector_idx = result_text.lower().rfind(connector)
            if last_connector_idx == -1: continue

            before_text = result_text[:last_connector_idx].strip()
            after_text_raw = result_text[last_connector_idx + len(connector):].strip()
            after_words = after_text_raw.split()
            should_cut = False

            if len(after_words) == 0:
                should_cut = True
            elif len(after_words) <= 2:
                bad_enders = {'i', 'it', 'we', 'he', 'she', 'they', 'a', 'an', 'the'}
                if any(w.lower() in bad_enders for w in after_words):
                    should_cut = True
            if connector.strip() == 'and' and after_words:
                first_after = after_words[0].lower()
                if first_after in {'the', 'a', 'an', 'this', 'my', 'our', 'it'}:
                    if len(after_words) <= 3:
                        should_cut = True

            # ⭐ 比較級例外：如果是 and deeper blacks，且包含比較級，不要切
            comparative_words = {
                'more', 'less', 'better', 'worse', 'higher', 'lower',
                'deeper', 'brighter', 'darker', 'faster', 'slower',
                'larger', 'smaller', 'stronger', 'weaker', 'stunning'
            }
            if any(w.lower() in comparative_words for w in after_words):
                should_cut = False

            if should_cut:
                result_text = before_text

    # 7b. ⭐ 智慧逗號過濾 (Intelligent Comma Filter) V15
    if ',' in result_text:
        last_comma_idx = result_text.rfind(',')
        fragment = result_text[last_comma_idx+1:].strip()
        fragment_words = fragment.split()

        should_keep_fragment = False

        if fragment_words:
            first_word = fragment_words[0].lower()

            # 特徵 1: 轉折詞
            if first_word in {'but', 'however', 'though', 'although', 'yet'}:
                should_keep_fragment = True

            # 特徵 2: 數字
            elif any(char.isdigit() for char in fragment):
                should_keep_fragment = True

            # 特徵 3: 強調詞與否定詞
            elif any(kw in fragment.lower() for kw in {'only', 'just', 'even', 'not', 'no', 'never'}):
                should_keep_fragment = True

            # 特徵 4: 特定動詞
            elif first_word == 'lasting':
                should_keep_fragment = True

            # ⭐ 特徵 5 (新增): 比較級與最高級形容詞
            # 如果包含 "deeper", "higher", "best", "most" 等詞，視為核心性能指標，保留
            comparative_words = {
                'more', 'less', 'better', 'worse', 'higher', 'lower',
                'deeper', 'brighter', 'darker', 'faster', 'slower',
                'larger', 'smaller', 'stronger', 'weaker', 'stunning',
                'best', 'worst', 'most', 'least', 'excellent', 'amazing'
            }
            if any(w.lower() in comparative_words for w in fragment_words):
                should_keep_fragment = True

        if not should_keep_fragment:
             result_text = result_text[:last_comma_idx].strip()

    # 8. 長度限制
    words = result_text.split()
    if len(words) > 12 and ' but ' not in result_text.lower():
         words = words[:12]
         result_text = ' '.join(words)

    result_text = re.sub(r',\s*,', ',', result_text)
    result_text = result_text.strip().rstrip('.,;-')

    return result_text



def is_valid_aspect(aspect):
    """檢查屬性是否有效"""
    if not aspect:
        return False

    cleaned = aspect.strip()

    if cleaned == "-" or not cleaned:
        return False

    if all(c in '.,!?;:-_' for c in cleaned):
        return False

    return True


# ============================================
# 分段函數
# ============================================

def split_long_review(text, max_words=80):
    """分割長評論"""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub_sentence = ' '.join(words[i:i+max_words])
                chunks.append(sub_sentence + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# 完整分析函數
# ============================================

def refine_aspect_term(aspect):
    """
    修正屬性詞 (Refine Aspect Term)
    目的：移除開頭無意義的單位、連字符或冠詞
    例如：
    - "inch AMOLED screen" -> "AMOLED screen"
    - "6.7-inch display"   -> "display"
    - "the battery"        -> "battery"
    - "- camera"           -> "camera"
    """
    if not aspect:
        return ""

    clean_aspect = aspect.strip()

    # 1. 移除開頭的計量單位 (及其前面的數字)
    # Regex 邏輯：
    # ^                     : 開頭
    # (?:[\d\.]+\s*-?\s*)?  : 非捕獲群組，匹配可選的數字 (如 "6.", "6.7", "100") 和可選的連字符
    # (?:inch|inches|cm|mm|kg|lbs|oz|hz|mah) : 常見單位
    # \b                    : 單詞邊界 (確保不會切斷單字，如 'in' 變成 'i')
    # \s+                   : 後面的空格

    units_pattern = r'^(?:[\d\.]+\s*-?\s*)?(?:inch|inches|cm|mm|kg|lbs|oz)\b\s*'
    clean_aspect = re.sub(units_pattern, '', clean_aspect, flags=re.IGNORECASE)

    # 2. 移除開頭的冠詞與代詞
    stopwords_pattern = r'^(the|a|an|this|that|my|our)\s+'
    clean_aspect = re.sub(stopwords_pattern, '', clean_aspect, flags=re.IGNORECASE)

    # 3. 移除開頭的連字符或特殊符號
    clean_aspect = re.sub(r'^[-:;]\s*', '', clean_aspect)

    return clean_aspect.strip()



def comprehensive_aspect_opinion_analysis_complete(reviews, aspect_extractor, max_words=80, verbose=True):
    """完整分析系統 - 最終修正版 V3"""

    if verbose:
        print("=" * 80)
        print("完整的屬性-意見分析系統 (最終修正版 V3)")
        print(f"分段閾值: {max_words} 字")
        print("=" * 80)

    all_results = []

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if verbose:
            print(f"\n{'='*80}")
            print(f"評論 #{idx} ({word_count} 字)")
            print(f"{'='*80}")

        if word_count <= max_words:
            if verbose:
                print(f"📝 直接分析")

            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]

            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if verbose:
                print(f"📄 需要分段處理")

            chunks = split_long_review(review, max_words=max_words)

            if verbose:
                print(f"   分成 {len(chunks)} 段")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = []
            for i, chunk_result in enumerate(chunk_results, 1):
                analysis_results.append({'chunk_id': i, 'result': chunk_result})

        aspect_opinion_pairs = []
        filtered_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, sentiment in zip(aspects, positions, sentiments):

                # ⭐⭐ 新增：在這裡清洗屬性詞 ⭐⭐
                aspect = refine_aspect_term(aspect)

                # 再次檢查清洗後是否為空
                if not is_valid_aspect(aspect):
                    filtered_count += 1
                    continue

                # ⭐ 傳入所有屬性位置
                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_count += 1
                    continue

                corrected_sentiment = correct_sentiment_enhanced(aspect, raw_opinion, sentiment)

                # ⭐ 格式化顯示（會移除結尾連接詞）
                display_opinion = format_opinion_for_display(raw_opinion)

                # ⭐ 再次檢查：如果格式化後為空，跳過
                if not display_opinion or display_opinion.strip() == "":
                    filtered_count += 1
                    continue

                aspect_opinion_pairs.append({
                    'aspect': aspect,
                    'opinion': display_opinion,
                    'raw_opinion': raw_opinion,
                    'sentiment': corrected_sentiment,
                    'original_sentiment': sentiment,
                    'formatted': f"{aspect}: {display_opinion}",
                    'pattern': pattern,
                    'chunk': chunk_data['chunk_id'] if len(analysis_results) > 1 else None
                })

        if verbose:
            print(f"\n🎯 屬性-意見提取結果:")
            print(f"{'序號':<6} {'屬性':<20} {'意見':<50} {'情感':<10} {'模式':<15}")
            print("-" * 95)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    emoji = "😊" if pair['sentiment'] == "Positive" else "😞"

                    sentiment_display = pair['sentiment']
                    if pair['sentiment'] != pair['original_sentiment']:
                        sentiment_display += "*"

                    print(f"{i:<6} {pair['aspect']:<20} {pair['opinion']:<50} "
                          f"{sentiment_display:<10} {pair['pattern']:<15}")

                if filtered_count > 0:
                    print(f"\n   ℹ️ 已自動過濾 {filtered_count} 個無效屬性")
            else:
                print(f"   ⚠️ 未提取到有效的屬性-意見對")

        if verbose and aspect_opinion_pairs:
            print(f"\n📋 完整的屬性-意見組合:")
            for i, pair in enumerate(aspect_opinion_pairs, 1):
                emoji = "😊" if pair['sentiment'] == "Positive" else "😞"
                chunk_info = f" [段{pair['chunk']}]" if pair['chunk'] else ""
                sentiment_note = " [糾正]" if pair['sentiment'] != pair['original_sentiment'] else ""
                print(f"   {i}. {pair['formatted']} {emoji}{chunk_info}{sentiment_note}")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
            'filtered_count': filtered_count
        })

    return all_results


# ============================================
# 管理報告
# ============================================

def generate_management_report_complete(results):
    """生成完整管理報告"""

    print("\n" + "=" * 80)
    print("管理分析報告")
    print("=" * 80)

    positive_pairs = []
    negative_pairs = []
    total_filtered = sum(r.get('filtered_count', 0) for r in results)
    total_corrected = 0

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

            if pair['sentiment'] != pair['original_sentiment']:
                total_corrected += 1

    total_pairs = len(positive_pairs) + len(negative_pairs)

    print(f"\n📊 總體統計:")
    print(f"   - 評論總數: {len(results)}")
    print(f"   - 識別有效屬性總數: {total_pairs}")
    if total_filtered > 0:
        print(f"   - 已過濾無效屬性: {total_filtered} 個")
    if total_corrected > 0:
        print(f"   - 情感判斷已糾正: {total_corrected} 個")
    if total_pairs > 0:
        print(f"   - 正面評價: {len(positive_pairs)} 個 ({len(positive_pairs)/total_pairs*100:.1f}%)")
        print(f"   - 負面評價: {len(negative_pairs)} 個 ({len(negative_pairs)/total_pairs*100:.1f}%)")

    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair['opinion'])

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair['opinion'])

    if positive_aspects:
        print(f"\n✅ 競爭優勢:")
        print(f"{'排名':<6} {'屬性':<20} {'客戶評價':<40}")
        print("-" * 70)

        sorted_positive = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)
        for i, (aspect, opinions) in enumerate(sorted_positive[:10], 1):
            opinion_text = opinions[0]
            count = f"({len(opinions)}次)" if len(opinions) > 1 else ""
            print(f"{i:<6} {aspect:<20} {opinion_text:<30} {count:<10}")

    if negative_aspects:
        print(f"\n⚠️ 需要改進:")
        print(f"{'排名':<6} {'屬性':<20} {'客戶抱怨':<35}")
        print("-" * 65)

        sorted_negative = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)
        for i, (aspect, opinions) in enumerate(sorted_negative[:10], 1):
            opinion = opinions[0]
            count = f"({len(opinions)}次)" if len(opinions) > 1 else ""
            print(f"{i:<6} {aspect:<20} {opinion:<30} {count:<10}")

    print(f"\n💡 行動建議:")
    if sorted_negative:
        print(f"   1. 優先改進: {sorted_negative[0][0]}")
    if sorted_positive:
        print(f"   2. 強化優勢: {sorted_positive[0][0]}")
    print(f"   3. 制定改善計畫並追蹤成效")

    print(f"\n" + "=" * 80)
    print("各評論提取統計")
    print("=" * 80)

    for result in results:
        status = "✓" if len(result['pairs']) > 0 else "✗"
        chunked = "是" if result['was_chunked'] else "否"

        print(f"\n評論 {result['review_id']} {status}")
        print(f"  字數: {result['word_count']}")
        if result['was_chunked']:
            print(f"  分段: {chunked} ({result['chunk_count']}段)")
        else:
            print(f"  分段: {chunked}")
        print(f"  提取: {len(result['pairs'])} 個屬性-意見對")
        if result['filtered_count'] > 0:
            print(f"  過濾: {result['filtered_count']} 個無效屬性")


# ============================================
# 執行測試
# ============================================

test_reviews = [
    "The food was delicious but service was slow",
    "Great camera quality, terrible battery life",
    "Beautiful design but very expensive price",

    """I bought this laptop last month and have mixed feelings about it.
    The screen quality is absolutely stunning with vivid colors and sharp resolution.
    The keyboard is comfortable for typing long documents. However, the battery life
    is disappointingly short, lasting only 4 hours. The trackpad is also very sensitive
    and causes accidental clicks. Overall, it's a decent laptop but overpriced for
    what you get.""",

    """This restaurant offers an interesting dining experience. The ambiance is cozy
    and romantic with dim lighting and soft music. The menu has creative dishes with
    unique flavor combinations. The steak was perfectly cooked and tender. Unfortunately,
    the portions are quite small for the price. The service was friendly but extremely
    slow - we waited 45 minutes for our main course. The wine selection is limited too.""",

    """I recently stayed at this hotel for a business trip and wanted to share my
    detailed experience. First, the location is absolutely perfect - right in the
    city center with easy access to public transportation, restaurants, and shopping
    areas. You can walk to most tourist attractions within 15 minutes. The check-in
    process was smooth and efficient, with friendly staff who spoke multiple languages.

    The room itself was clean and well-maintained with modern furnishings. The bed
    was extremely comfortable with high-quality linens and plenty of pillows. The
    bathroom was spacious with a large shower and premium toiletries. However, there
    were several issues that need improvement. The air conditioning was very noisy
    and disrupted my sleep throughout the night. The wifi connection was unstable
    and kept disconnecting during important video calls. The breakfast buffet was
    disappointing with limited options and food that wasn't very fresh.

    Additionally, the room service was slow and the food arrived cold. The gym
    facilities are outdated and lack modern equipment. Despite these problems,
    I would still recommend this hotel primarily because of its excellent location
    and comfortable beds. Just be prepared for some inconveniences.""",

    """I've been using this smartphone for three months now and have thoroughly
    tested all its features. Let me start with the positives. The camera system is
    truly impressive with exceptional photo quality in both daylight and low-light
    conditions. The 108MP main sensor captures stunning detail and the night mode
    produces clear, bright photos even in darkness. The video recording is smooth
    with excellent stabilization for 4K footage. The display is gorgeous - a 6.7-inch
    AMOLED screen with vibrant colors, deep blacks, and 120Hz refresh rate that makes
    scrolling incredibly smooth.

    The performance is excellent for daily tasks and gaming, with the processor
    handling everything I throw at it without lag or stuttering. The 5G connectivity
    is fast and reliable where available. The design is sleek and premium-feeling
    with a glass back and metal frame. However, there are significant drawbacks.
    The battery life is terrible - I barely make it through a full day with moderate
    use and heavy use requires charging twice daily. The phone gets uncomfortably hot
    during intensive tasks like gaming or video recording. The price is extremely high,
    making it hard to justify unless you need the absolute best camera. The software
    has occasional bugs and the bloatware is annoying. Storage options are limited
    with no microSD card slot. Overall, it's a powerful device but the battery issues
    and high price make it difficult to recommend to most people.""",
]

print("開始完整測試 (V3 - 修正所有問題)...")
print("="  * 80)

results = comprehensive_aspect_opinion_analysis_complete(
    test_reviews,
    aspect_extractor,
    max_words=80,
    verbose=True
)

generate_management_report_complete(results)

print("\n" + "=" * 80)
print("✅ 完整測試完成！")
print("=" * 80)





### 對真實評論數據進行全面方面級情感分析

由於 Aspect Sentiment Triplet Extraction (ASTE) 模型目前遇到技術問題，我們將使用之前成功運作的方面詞情感分類模型 (`aspect_extractor`) 對真實的店家評論數據 (`test_reviews`) 進行全面分析。這將幫助我們提取每個評論中的方面詞及其相關情感，並生成管理報告。

In [ ]:
aspect_extractor

In [ ]:
df_selected_data

In [ ]:
df_selected_data["評論內容"]

In [ ]:
test_reviews=df_selected_data["評論內容"]

# 執行全面的方面級意見分析
# 確保 test_reviews 和 aspect_extractor 已經存在
if 'test_reviews' in locals() and not test_reviews.empty and 'aspect_extractor' in locals():
    print("開始對實際店家評論進行全面方面級情感分析...")
    print("=" * 80)

    # 使用之前定義的 comprehensive_aspect_opinion_analysis_complete 函數
    # verbose=True 將顯示詳細的處理過程和結果
    ret_val = comprehensive_aspect_opinion_analysis_complete(
        test_reviews,
        aspect_extractor,
        max_words=80, # 每段評論的最大字數，可根據需求調整
        verbose=True
    )

    # 判斷回傳值類型以適配不同版本的函數
    if isinstance(ret_val, tuple) and len(ret_val) == 2:
        real_reviews_analysis_results, analysis_stats = ret_val
    else:
        print("\n⚠️ 檢測到舊版分析函數輸出 (僅回傳結果列表)，自動生成預設統計數據...")
        real_reviews_analysis_results = ret_val
        # 建立預設統計數據以避免後續錯誤
        analysis_stats = {
            'total_aspects': 0,
            'filtered_invalid_aspects': 0,
            'filtered_invalid_opinions': 0,
            'sentiment_corrections': 0
        }

    print("\n" + "=" * 80)
    print("✅ 實際評論分析完成！")
    print("=" * 80)

    # 生成管理報告，嘗試適配不同版本的報告函數
    try:
        generate_management_report_complete(real_reviews_analysis_results, analysis_stats)
    except TypeError:
        print("\n⚠️ 檢測到舊版報告函數，嘗試使用單參數調用...")
        generate_management_report_complete(real_reviews_analysis_results)
    except Exception as e:
        print(f"\n❌ 生成報告時發生錯誤: {e}")

else:
    print("錯誤: 'test_reviews' 或 'aspect_extractor' 變數未定義或為空。請確保之前的儲存格已執行。")

In [ ]:
test_reviews=df_selected_data["評論內容"]

# 執行全面的方面級意見分析
# 確保 test_reviews 和 aspect_extractor 已經存在
if 'test_reviews' in locals() and not test_reviews.empty and 'aspect_extractor' in locals():
    print("開始對實際店家評論進行全面方面級情感分析...")
    print("=" * 80)

    # 使用之前定義的 comprehensive_aspect_opinion_analysis_complete 函數
    # verbose=True 將顯示詳細的處理過程和結果
    ret_val = comprehensive_aspect_opinion_analysis_complete(
        test_reviews,
        aspect_extractor,
        max_words=80, # 每段評論的最大字數，可根據需求調整
        verbose=True
    )

    # 判斷回傳值類型以適配不同版本的函數
    if isinstance(ret_val, tuple) and len(ret_val) == 2:
        real_reviews_analysis_results, analysis_stats = ret_val
    else:
        print("\n⚠️ 檢測到舊版分析函數輸出 (僅回傳結果列表)，自動生成預設統計數據...")
        real_reviews_analysis_results = ret_val
        # 建立預設統計數據以避免後續錯誤
        analysis_stats = {
            'total_aspects': 0,
            'filtered_invalid_aspects': 0,
            'filtered_invalid_opinions': 0,
            'sentiment_corrections': 0
        }

    print("\n" + "=" * 80)
    print("✅ 實際評論分析完成！")
    print("=" * 80)

    # 生成管理報告，嘗試適配不同版本的報告函數
    try:
        generate_management_report_complete(real_reviews_analysis_results, analysis_stats)
    except TypeError:
        print("\n⚠️ 檢測到舊版報告函數，嘗試使用單參數調用...")
        generate_management_report_complete(real_reviews_analysis_results)
    except Exception as e:
        print(f"\n❌ 生成報告時發生錯誤: {e}")

else:
    print("錯誤: 'test_reviews' 或 'aspect_extractor' 變數未定義或為空。請確保之前的儲存格已執行。")

In [ ]:
# ABSA 改進版 V2 - 屬性-意見-情感提取系統
# 修正：無效意見詞、情感判斷錯誤、非屬性詞過濾

test_reviews=df_selected_data["評論內容"]


import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import re
from datetime import datetime

# ============================================
# 全域設定
# ============================================
OUTPUT_FILE = "absa_analysis_results.txt"

# ============================================
# is_adjective_or_adverb 函數（增強版）
# ============================================

def is_adjective_or_adverb(word):
    """判斷是否為形容詞或副詞 - 增強版"""

    positive_adjectives = {
        'beautiful', 'excellent', 'great', 'good', 'amazing', 'wonderful',
        'fantastic', 'outstanding', 'impressive', 'stunning', 'gorgeous',
        'perfect', 'superb', 'fabulous', 'marvelous', 'magnificent',
        'delicious', 'tasty', 'delightful', 'comfortable', 'cozy',
        'friendly', 'helpful', 'efficient', 'smooth', 'easy',
        'fast', 'quick', 'reliable', 'stable', 'secure',
        'spacious', 'large', 'big', 'bright', 'clean', 'modern',
        'innovative', 'creative', 'unique', 'special', 'exceptional',
        'high-quality', 'premium', 'luxurious', 'elegant', 'stylish',
        'soft', 'warm', 'cool', 'fresh', 'clear', 'sharp', 'vivid',
        'dim', 'bright', 'light', 'dark', 'crisp', 'smooth',
        'powerful', 'strong', 'solid', 'durable', 'sturdy',
        'romantic', 'peaceful', 'quiet', 'calm', 'relaxing',
        'tender', 'juicy', 'crispy', 'creamy', 'rich',
        'lightweight', 'portable', 'convenient', 'accessible',
        'nice', 'fine', 'lovely', 'charming', 'attractive',
        'pleasant', 'enjoyable', 'satisfying', 'adequate',
        'reasonable', 'decent', 'acceptable', 'maintained', 'tested',
        'interesting', 'awesome', 'incredible', 'terrific', 'cute',
        'attentive', 'professional', 'polite', 'welcoming', 'lively',
        'flavorful', 'seasoned', 'yummy', 'scrumptious'
    }

    negative_adjectives = {
        'terrible', 'bad', 'poor', 'awful', 'horrible', 'disappointing',
        'frustrating', 'annoying', 'uncomfortable', 'inconvenient',
        'slow', 'sluggish', 'laggy', 'unstable', 'unreliable',
        'expensive', 'overpriced', 'costly', 'pricey', 'high',
        'small', 'tiny', 'cramped', 'limited', 'insufficient',
        'short', 'brief', 'inadequate', 'lacking', 'missing',
        'noisy', 'loud', 'distracting', 'disturbing',
        'dirty', 'messy', 'cluttered', 'outdated', 'old', 'obsolete',
        'complicated', 'confusing', 'difficult', 'hard', 'complex',
        'broken', 'damaged', 'faulty', 'defective', 'flawed',
        'cold', 'hot', 'harsh', 'rough', 'stiff',
        'weak', 'fragile', 'unstable', 'insecure',
        'rude', 'unfriendly', 'unhelpful', 'careless',
        'sensitive', 'buggy', 'glitchy', 'sticky',
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross'
    }

    intensifiers = {
        'very', 'extremely', 'really', 'quite', 'too', 'so',
        'incredibly', 'amazingly', 'exceptionally', 'remarkably',
        'absolutely', 'totally', 'completely', 'utterly', 'entirely',
        'highly', 'fairly', 'rather', 'somewhat', 'pretty',
        'disappointingly', 'surprisingly', 'unusually', 'well', 'thoroughly',
        'super', 'especially', 'particularly'
    }

    word_lower = word.lower().strip('.,!?;:')

    if word_lower in positive_adjectives:
        return True, 'positive_adj'
    if word_lower in negative_adjectives:
        return True, 'negative_adj'
    if word_lower in intensifiers:
        return True, 'intensifier'

    adj_suffixes = ['ful', 'less', 'ous', 'ive', 'able', 'ible', 'al', 'ic', 'y', 'ing', 'ed']
    adv_suffixes = ['ly']

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    for suffix in adv_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adv'

    return False, 'none'


# ============================================
# 輔助函數
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """找到句子邊界"""
    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in ['.', '!', '?']:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in ['.', '!', '?']:
            sentence_end = i
            break

    return sentence_start, sentence_end


def detect_be_verb(tokens, start_idx):
    """檢測 be 動詞"""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    if word in ['is', 'was', 'were', 'are', 'be', 'been', "'s"]:
        return True, 1, False

    if word in ["isn't", "isnt", "wasn't", "wasnt", "weren't", "werent", "aren't", "arent"]:
        return True, 1, True

    if word in ['isn', 'wasn', 'weren', 'aren'] and start_idx + 2 < len(tokens):
        if tokens[start_idx + 1] == "'" and tokens[start_idx + 2].lower() in ['t', 'nt']:
            return True, 3, True

    return False, 0, False


def handle_that_with_negation(tokens, start_pos, end_pos):
    """處理 that + 否定"""
    result = []
    i = start_pos

    while i < end_pos:
        word = tokens[i]
        word_lower = word.lower()

        if word_lower == 'that' and i + 1 < end_pos:
            found, skip, is_negative = detect_be_verb(tokens, i + 1)

            if found and is_negative:
                i += 1 + skip
                result.append("not")
                continue

        result.append(word)
        i += 1

    return result


def find_previous_aspect_end(tokens, current_start_idx, all_aspect_positions):
    """找到前一個屬性的結束位置"""
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(tokens, current_end_idx, all_aspect_positions):
    """找到下一個屬性的起始位置"""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# ⭐ 新增：意見詞驗證函數
# ============================================

def is_valid_opinion(opinion):
    """驗證意見詞是否有效 - 新增"""
    if not opinion or len(opinion.strip()) < 2:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    # 1. 過濾純動作詞
    action_only_words = {
        'ordered', 'ordering', 'makes', 'making', 'arrived',
        'asked', 'asking', 'having', 'took', 'taking',
        'informed', 'told', 'said', 'called', 'comes', 'came',
        'showed', 'showing', 'walked', 'walking', 'served',
        'serving', 'brought', 'bringing', 'dropped', 'cooked',
        'cooking', 'fried', 'grilled', 'smoked', 'pulled'
    }

    # 2. 過濾無意義單詞
    meaningless_single = {
        'only', 'just', 'also', 'too', 'as', 'q', 'y', 'a', 'an', 'the',
        'outside', 'inside', 'here', 'there', 'next', 'back',
        'ordering', 'during', 'around', 'wherever', 'texanas'
    }

    # 3. 過濾人名
    common_names = {
        'joel', 'david', 'kaitlin', 'kaitlyn', 'christina', 'natalie',
        'melanie', 'sara', 'paul', 'morgan', 'fatima', 'veronica',
        'andy', 'nick', 'alyssa', 'janelle', 'kiara', 'braelyn',
        'gabby', 'adela', 'ray', 'kristina', 'tina', 'cameron',
        'zoe', 'greg', 'steven', 'jordan', 'sara'
    }

    # 4. 過濾位置描述
    location_keywords = {'close', 'near', 'hotel', 'magnolia', 'downtown', 'street'}

    # 單詞檢查
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')
        if clean_word in action_only_words | meaningless_single | common_names:
            return False
        # 單詞必須是形容詞
        is_adj, adj_type = is_adjective_or_adverb(clean_word)
        if not is_adj:
            return False

    # 雙詞檢查：如果都是動作詞或人名
    if len(words) == 2:
        if all(w.strip('.,') in action_only_words | common_names for w in words):
            return False

    # 檢查是否全是動作詞
    action_count = sum(1 for w in words if w.strip('.,') in action_only_words)
    if action_count == len(words):
        return False

    # 檢查人名（單獨出現）
    if len(words) <= 2 and any(w.strip('.,') in common_names for w in words):
        # 如果只有人名，無效
        non_name_words = [w for w in words if w.strip('.,') not in common_names]
        if len(non_name_words) == 0:
            return False

    # 檢查位置描述
    if len(words) >= 3:
        location_count = sum(1 for w in words if w.strip('.,') in location_keywords)
        if location_count >= 2:
            return False

    # 檢查無意義片段
    invalid_patterns = [
        r"^'s\s+name",  # 's name
        r"^and\s+waited$",
        r"^to\s+before\s+me",
        r"^de\s+camarón$",
        r"^son\s+gigantes\s+y$",
        r"^\d+\s*/",  # 5 / 等數字評分
        r"^or\s+maybe\s+both$",
        r"^for\s+dessert$",
        r"^going\s+forward$"
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    return True


# ============================================
# 核心：提取完整意見詞
# ============================================

def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """提取完整意見詞 - 帶前後邊界檢測"""

    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)

    prev_aspect_end = find_previous_aspect_end(tokens, start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(tokens, end_idx, all_aspect_positions)

    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # 模式 1: 前置修飾詞
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, modifier_type = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in [',', 'and', 'with', 'or', 'but']:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

        # 名詞補全
        next_pos = end_idx + 1
        if next_pos < sentence_end:
            next_word = tokens[next_pos]
            forbidden_next = {
                'is', 'was', 'are', 'were', 'be', 'been',
                'has', 'have', 'had', 'does', 'do',
                'but', 'and', 'or', 'so', 'because',
                'with', 'for', 'to', 'in', 'at', 'on', 'of', 'by',
                'that', 'which', 'who', 'this', 'it',
                ',', '.', '!', '?', ';', '-'
            }
            if next_word.isalpha() and next_word.lower() not in forbidden_next:
                opinion_words.append(next_word)

    # 模式 2: be 動詞 + 形容詞
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []

        if is_negative:
            post_modifiers.append("not")

        for i in range(opinion_start, opinion_end):
            word = tokens[i]

            if word in ['.', '!', '?', ';']:
                break

            if word.lower() == 'but' and next_aspect_start is not None:
                if i + 1 < len(tokens) and i + 1 >= next_aspect_start - 2:
                    break

            post_modifiers.append(word)

        if post_modifiers:
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # 模式 3: 動詞短語
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in ['.', '!', '?', ';']:
                break
            temp_words.append(word)

        if temp_words:
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # 模式 4: 上下文搜索
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, modifier_type = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                context_words = []
                for j in range(context_start, context_end):
                    if tokens[j] not in ['.', '!', '?', ';']:
                        context_words.append(tokens[j])

                opinion_words.extend(context_words)
                pattern_type = "context_search"
                break

            if word in ['.', '!', '?']:
                break

    opinion_words = handle_that_with_negation(opinion_words, 0, len(opinion_words))
    opinion_phrase = ' '.join(opinion_words).strip()

    return opinion_phrase, pattern_type


# ============================================
# ⭐ 情感糾正（修正版）
# ============================================

def correct_sentiment_enhanced(aspect, opinion, predicted_sentiment):
    """情感糾正 - 修正版：避免過度糾正"""

    opinion_lower = opinion.lower()

    # ⭐ 新增：強正面詞保護（不應被糾正為負面）
    strong_positive_words = {
        'excellent', 'amazing', 'fantastic', 'wonderful', 'perfect',
        'great', 'awesome', 'delicious', 'outstanding', 'best',
        'incredible', 'superb', 'terrific', 'fabulous', 'marvelous',
        'friendly', 'helpful', 'attentive', 'professional', 'cute',
        'nice', 'good', 'tasty', 'fresh', 'tender', 'juicy', 'crispy'
    }

    # 如果意見包含強正面詞，優先保持正面
    for pos_word in strong_positive_words:
        if pos_word in opinion_lower:
            # 檢查是否有否定詞
            negation_words = ['not', "n't", 'no', 'never', 'none', 'nothing']
            has_negation = any(neg in opinion_lower for neg in negation_words)
            if not has_negation:
                return 'Positive'

    # 處理 'but' 轉折
    if ' but ' in opinion_lower:
        parts = opinion_lower.split(' but ', 1)
        if len(parts) == 2:
            after_but = parts[1]

            negative_words = {
                'slow', 'expensive', 'small', 'limited', 'short',
                'terrible', 'bad', 'poor', 'disappointing', 'awful',
                'noisy', 'unstable', 'outdated', 'uncomfortable',
                'dry', 'bland', 'soggy', 'cold', 'burnt', 'overcooked'
            }

            for neg_word in negative_words:
                if neg_word in after_but:
                    return 'Negative'

    # 處理 "for the price" 模式
    for_the_patterns = ['for the price', 'for the cost', 'for the money', 'for price', 'for cost']
    for pattern in for_the_patterns:
        if pattern in opinion_lower:
            negative_indicators = ['small', 'limited', 'short', 'expensive', 'high', 'too']
            for indicator in negative_indicators:
                if indicator in opinion_lower:
                    return 'Negative'

    # 獨立負面詞檢查
    standalone_negative = {
        'limited', 'lacking', 'insufficient', 'inadequate',
        'disappointing', 'poor', 'bad', 'terrible', 'awful',
        'outdated', 'old', 'broken', 'slow', 'noisy',
        'uncomfortable', 'expensive', 'small', 'short',
        'bland', 'dry', 'soggy', 'cold', 'burnt', 'overcooked',
        'rude', 'unfriendly', 'unhelpful', 'horrible', 'pathetic',
        'lousy', 'mediocre', 'nasty', 'gross', 'fatty', 'greasy'
    }

    opinion_words = opinion_lower.split()
    for word in opinion_words:
        clean_word = word.strip('.,!?;:')
        if clean_word in standalone_negative:
            return 'Negative'

    return predicted_sentiment


# ============================================
# ⭐ 格式化顯示（增強版）
# ============================================

def format_opinion_for_display(opinion):
    """格式化意見用於顯示 - 增強版"""

    if not opinion:
        return ""

    # 重要名詞列表
    important_nouns = {
        'experience', 'quality', 'service', 'food', 'price',
        'performance', 'design', 'features', 'options', 'selection',
        'system', 'conditions', 'footage', 'screen', 'tasks',
        'gaming', 'back', 'frame', 'detail', 'mode', 'photos',
        'stabilization', 'colors', 'blacks', 'rate', 'lag',
        'connectivity', 'drawbacks', 'use', 'bugs', 'equipment',
        'slot', 'device', 'issues', 'people', 'day', 'sensor', 'lens',
        'center', 'location', 'access', 'contrast', 'brightness'
    }

    # 定義要移除的句首詞
    leading_words_to_remove = {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'also', 'too', 'either',
        'has', 'have', 'had',
        'contains', 'includes', 'consists',
        'offers', 'provides', 'features',
        'is', 'are', 'was', 'were', 'looks', 'feels',
        'it', 'itself', 'they', 'themselves', 'we', 'ourselves',
        'i', 'myself', 'you', 'yourself'
    }

    do_not_merge_words = {
        'a', 'an', 'the', 'it', 'is', 'we', 'i', 'you', 'he', 'she',
        'of', 'in', 'to', 'for', 'with', 'on', 'at', 'my', 'our'
    }

    general_cut_off = {
        'making', 'causing', 'forcing', 'leaving', 'rendering',
        'unless', 'except', 'besides', 'despite', 'although',
        'which', 'where', 'when', 'because', 'since'
    }

    post_dash_cut_off = {
        'with', 'and', 'but', 'including', 'plus', 'to', 'for', 'at'
    }

    # 1. 初步清理
    opinion = re.sub(r'([a-z])-(?=[A-Z])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'([a-zA-Z0-9])-(?=[a-zA-Z0-9])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    words = opinion.split()

    # 2. 移除句首冗餘詞
    while words and words[0].lower() in leading_words_to_remove:
        words.pop(0)

    # 3. 處理連字符合併
    cleaned_words = []
    i = 0
    while i < len(words):
        word = words[i]

        if i + 1 < len(words):
            next_word = words[i+1]
            if word.endswith('-') and len(word) > 1:
                if next_word.isdigit() or next_word.lower() in do_not_merge_words:
                    cleaned_words.append(word.rstrip('-'))
                else:
                    merged = word + next_word
                    cleaned_words.append(merged)
                    i += 2
                    continue
            elif next_word == '-' and i + 2 < len(words):
                following_word = words[i+2]
                if following_word.isdigit() or following_word.lower() in do_not_merge_words:
                    cleaned_words.append(word)
                    cleaned_words.append('-')
                    i += 2
                    continue
                merged = word + "-" + following_word
                cleaned_words.append(merged)
                i += 3
                continue
        cleaned_words.append(word)
        i += 1
    words = cleaned_words

    # 4. 核心截斷邏輯
    truncated_words = []
    valid_length = 0
    has_passed_dash = False

    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')
        is_truncator = False

        if word == '-' or word.startswith('-'):
            if i + 1 < len(words):
                next_w = words[i+1].lower()
                if next_w in {'i', 'we', 'it', 'he', 'she', 'they'}:
                    break
            has_passed_dash = True
            truncated_words.append(word)
            continue

        elif word.endswith('-') and '-' not in word[:-1] and valid_length >= 1:
            is_truncator = True

        if word_lower in general_cut_off and valid_length >= 2:
            break

        if has_passed_dash and word_lower in post_dash_cut_off:
            break

        if is_truncator and valid_length >= 1:
            break

        truncated_words.append(word)
        if word not in {',', 'and', 'but', '-'}:
            valid_length += 1

    words = truncated_words

    # 5. 修正逗號
    final_words = []
    for i, word in enumerate(words):
        if word == ',':
            if final_words and not final_words[-1].endswith(','):
                final_words[-1] = final_words[-1] + ','
        else:
            clean_word = word.lstrip(',')
            if clean_word:
                final_words.append(clean_word)
    words = final_words

    # 6. 移除結尾垃圾詞
    trailing_words_to_remove = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'too', 'either', 'also', 'but', 'however',
        'a', 'an', 'the', ',', '-', 'is', 'are', 'was', 'were',
        'need', 'justify', 'make', 'do', 'want', 'require', 'expect',
        'produces', 'provides', 'offers', 'features', 'includes', 'creates', 'shows',
        "'", '"', '(', 'it', 'i', 'we', 'they', 'he', 'she'
    }

    dangling_adjectives = {
        'high', 'low', 'large', 'small', 'good', 'bad', 'great', 'poor',
        'long', 'short', 'new', 'old', 'many', 'few', 'heavy', 'light',
        'strong', 'weak', 'fast', 'slow'
    }

    prepositions_for_check = {'with', 'for', 'in', 'on', 'at', 'of', 'and', 'but', 'or'}

    while words:
        last_word = words[-1].lower().strip('.,!?;:')

        if last_word in important_nouns:
            break
        should_remove = False
        if last_word in trailing_words_to_remove or words[-1].endswith('-') or words[-1].isdigit():
            should_remove = True
        if not should_remove and len(words) >= 2:
            second_last = words[-2].lower().strip(',')
            if last_word in dangling_adjectives and second_last in prepositions_for_check:
                should_remove = True
        if words[-1].endswith(','):
            words[-1] = words[-1].rstrip(',')
            if not words[-1]:
                words.pop()
                continue
        if should_remove:
            words.pop()
        else:
            break

    # 7. 轉折詞處理
    result_text = ' '.join(words)
    connectors = [' but ', ' however ', ' though ', ' and ']

    for connector in connectors:
        if connector in result_text.lower():
            last_connector_idx = result_text.lower().rfind(connector)
            if last_connector_idx == -1:
                continue

            before_text = result_text[:last_connector_idx].strip()
            after_text_raw = result_text[last_connector_idx + len(connector):].strip()
            after_words = after_text_raw.split()
            should_cut = False

            if len(after_words) == 0:
                should_cut = True
            elif len(after_words) <= 2:
                bad_enders = {'i', 'it', 'we', 'he', 'she', 'they', 'a', 'an', 'the'}
                if any(w.lower() in bad_enders for w in after_words):
                    should_cut = True
            if connector.strip() == 'and' and after_words:
                first_after = after_words[0].lower()
                if first_after in {'the', 'a', 'an', 'this', 'my', 'our', 'it'}:
                    if len(after_words) <= 3:
                        should_cut = True

            # 比較級例外
            comparative_words = {
                'more', 'less', 'better', 'worse', 'higher', 'lower',
                'deeper', 'brighter', 'darker', 'faster', 'slower',
                'larger', 'smaller', 'stronger', 'weaker', 'stunning'
            }
            if any(w.lower() in comparative_words for w in after_words):
                should_cut = False

            if should_cut:
                result_text = before_text

    # 8. 長度限制
    words = result_text.split()
    if len(words) > 12 and ' but ' not in result_text.lower():
        words = words[:12]
        result_text = ' '.join(words)

    result_text = re.sub(r',\s*,', ',', result_text)
    result_text = result_text.strip().rstrip('.,;-\'"(')

    return result_text


# ============================================
# ⭐ 屬性驗證（增強版）
# ============================================

def is_valid_aspect(aspect):
    """檢查屬性是否有效 - 增強版"""
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    if cleaned == "-" or not cleaned:
        return False

    if all(c in '.,!?;:-_' for c in cleaned):
        return False

    # ⭐ 新增：過濾非屬性詞
    invalid_aspects = {
        # 動詞/動名詞
        'waited', 'waiting', 'ordering', 'making', 'having', 'served',
        'serving', 'asked', 'asking', 'walked', 'walking',
        # 代詞
        'someone', 'she', 'he', 'they', 'it', 'we', 'i', 'you',
        # 冠詞
        'the', 'a', 'an',
        # 時間單位
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours',
        # 形容詞（不應作為屬性）
        'fast', 'hot', 'cold', 'new',
        # 其他
        'sub', 'drive', 'seat', 'wait', 'ordering'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """修正屬性詞"""
    if not aspect:
        return ""

    clean_aspect = aspect.strip()

    # 移除開頭的計量單位
    units_pattern = r'^(?:[\d\.]+\s*-?\s*)?(?:inch|inches|cm|mm|kg|lbs|oz)\b\s*'
    clean_aspect = re.sub(units_pattern, '', clean_aspect, flags=re.IGNORECASE)

    # 移除開頭的冠詞與代詞
    stopwords_pattern = r'^(the|a|an|this|that|my|our)\s+'
    clean_aspect = re.sub(stopwords_pattern, '', clean_aspect, flags=re.IGNORECASE)

    # 移除開頭的連字符或特殊符號
    clean_aspect = re.sub(r'^[-:;]\s*', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# 分段函數
# ============================================

def split_long_review(text, max_words=80):
    """分割長評論"""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub_sentence = ' '.join(words[i:i+max_words])
                chunks.append(sub_sentence + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# 文件輸出類
# ============================================

class FileLogger:
    """文件輸出記錄器"""
    def __init__(self, filename):
        self.filename = filename
        self.content = []

    def log(self, message=""):
        """記錄訊息"""
        self.content.append(message)
        print(message)

    def save(self):
        """保存到文件"""
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ 結果已保存到: {self.filename}")


# ============================================
# 完整分析函數（帶文件輸出）
# ============================================

def comprehensive_aspect_opinion_analysis_complete(reviews, aspect_extractor, max_words=80, verbose=True, logger=None):
    """完整分析系統 - 改進版 V2"""

    if logger:
        logger.log("=" * 80)
        logger.log("完整的屬性-意見分析系統 (改進版 V2)")
        logger.log(f"分段閾值: {max_words} 字")
        logger.log(f"分析時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 80)

    all_results = []

    # 統計計數
    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'sentiment_corrections': 0
    }

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'='*80}")
            logger.log(f"評論 #{idx} ({word_count} 字)")
            logger.log(f"{'='*80}")

        if word_count <= max_words:
            if logger and verbose:
                logger.log(f"📝 直接分析")

            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]

            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if logger and verbose:
                logger.log(f"📄 需要分段處理")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   分成 {len(chunks)} 段")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = []
            for i, chunk_result in enumerate(chunk_results, 1):
                analysis_results.append({'chunk_id': i, 'result': chunk_result})

        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, sentiment in zip(aspects, positions, sentiments):

                # 清洗屬性詞
                aspect = refine_aspect_term(aspect)

                # 屬性有效性檢查
                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # 提取意見詞
                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                # 意見詞有效性檢查
                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # 格式化意見詞
                display_opinion = format_opinion_for_display(raw_opinion)

                # 再次檢查格式化後的意見詞
                if not display_opinion or display_opinion.strip() == "":
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                # ⭐ 新增：意見詞有效性驗證
                if not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                # 情感糾正
                corrected_sentiment = correct_sentiment_enhanced(aspect, raw_opinion, sentiment)

                if corrected_sentiment != sentiment:
                    stats['sentiment_corrections'] += 1

                stats['total_aspects'] += 1

                aspect_opinion_pairs.append({
                    'aspect': aspect,
                    'opinion': display_opinion,
                    'raw_opinion': raw_opinion,
                    'sentiment': corrected_sentiment,
                    'original_sentiment': sentiment,
                    'formatted': f"{aspect}: {display_opinion}",
                    'pattern': pattern,
                    'chunk': chunk_data['chunk_id'] if len(analysis_results) > 1 else None
                })

        if logger and verbose:
            logger.log(f"\n🎯 屬性-意見提取結果:")
            logger.log(f"{'序號':<6} {'屬性':<20} {'意見':<50} {'情感':<10} {'模式':<15}")
            logger.log("-" * 95)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    sentiment_display = pair['sentiment']
                    if pair['sentiment'] != pair['original_sentiment']:
                        sentiment_display += "*"

                    # 截斷過長的意見
                    opinion_display = pair['opinion'][:47] + "..." if len(pair['opinion']) > 50 else pair['opinion']

                    logger.log(f"{i:<6} {pair['aspect']:<20} {opinion_display:<50} "
                              f"{sentiment_display:<10} {pair['pattern']:<15}")

                total_filtered = filtered_aspect_count + filtered_opinion_count
                if total_filtered > 0:
                    logger.log(f"\n   ℹ️ 已過濾: {filtered_aspect_count} 個無效屬性, {filtered_opinion_count} 個無效意見")
            else:
                logger.log(f"   ⚠️ 未提取到有效的屬性-意見對")

        if logger and verbose and aspect_opinion_pairs:
            logger.log(f"\n📋 完整的屬性-意見組合:")
            for i, pair in enumerate(aspect_opinion_pairs, 1):
                emoji = "😊" if pair['sentiment'] == "Positive" else "😞"
                chunk_info = f" [段{pair['chunk']}]" if pair['chunk'] else ""
                sentiment_note = " [糾正]" if pair['sentiment'] != pair['original_sentiment'] else ""
                logger.log(f"   {i}. {pair['formatted']} {emoji}{chunk_info}{sentiment_note}")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
            'filtered_aspect_count': filtered_aspect_count,
            'filtered_opinion_count': filtered_opinion_count
        })

    return all_results, stats


# ============================================
# 管理報告（增強版）
# ============================================

def generate_management_report_complete(results, stats, logger=None):
    """生成完整管理報告 - 增強版"""

    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 80)
    log("管理分析報告")
    log("=" * 80)

    positive_pairs = []
    negative_pairs = []
    total_corrected = 0

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

            if pair['sentiment'] != pair['original_sentiment']:
                total_corrected += 1

    total_pairs = len(positive_pairs) + len(negative_pairs)

    log(f"\n📊 總體統計:")
    log(f"   - 評論總數: {len(results)}")
    log(f"   - 識別有效屬性總數: {total_pairs}")
    log(f"   - 已過濾無效屬性: {stats['filtered_invalid_aspects']} 個")
    log(f"   - 已過濾無效意見: {stats['filtered_invalid_opinions']} 個")
    if total_corrected > 0:
        log(f"   - 情感判斷已糾正: {total_corrected} 個")
    if total_pairs > 0:
        log(f"   - 正面評價: {len(positive_pairs)} 個 ({len(positive_pairs)/total_pairs*100:.1f}%)")
        log(f"   - 負面評價: {len(negative_pairs)} 個 ({len(negative_pairs)/total_pairs*100:.1f}%)")

    # 彙整正面屬性
    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair['opinion'])

    # 彙整負面屬性
    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair['opinion'])

    # ⭐ 新增：過濾確認負面的屬性（避免誤判）
    def is_genuinely_negative(opinion):
        """確認意見真的是負面的"""
        positive_indicators = {
            'excellent', 'great', 'amazing', 'wonderful', 'perfect',
            'delicious', 'fantastic', 'awesome', 'best', 'cute', 'nice',
            'good', 'friendly', 'helpful', 'attentive', 'professional',
            'fresh', 'tasty', 'tender', 'juicy', 'crispy', 'terrific'
        }
        opinion_lower = opinion.lower()
        # 檢查是否有否定詞
        negation_words = ['not', "n't", 'no', 'never', 'none']
        has_negation = any(neg in opinion_lower for neg in negation_words)

        for pos in positive_indicators:
            if pos in opinion_lower and not has_negation:
                return False
        return True

    if positive_aspects:
        log(f"\n✅ 競爭優勢:")
        log(f"{'排名':<6} {'屬性':<20} {'客戶評價':<40} {'次數':<10}")
        log("-" * 80)

        sorted_positive = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)
        for i, (aspect, opinions) in enumerate(sorted_positive[:15], 1):
            opinion_text = opinions[0][:35] + "..." if len(opinions[0]) > 35 else opinions[0]
            count = f"({len(opinions)}次)"
            log(f"{i:<6} {aspect:<20} {opinion_text:<40} {count:<10}")

    if negative_aspects:
        log(f"\n⚠️ 需要改進:")
        log(f"{'排名':<6} {'屬性':<20} {'客戶抱怨':<40} {'次數':<10}")
        log("-" * 80)

        # 過濾真正負面的
        filtered_negative = {
            k: [op for op in v if is_genuinely_negative(op)]
            for k, v in negative_aspects.items()
        }
        filtered_negative = {k: v for k, v in filtered_negative.items() if v}

        sorted_negative = sorted(filtered_negative.items(), key=lambda x: len(x[1]), reverse=True)
        for i, (aspect, opinions) in enumerate(sorted_negative[:15], 1):
            opinion = opinions[0][:35] + "..." if len(opinions[0]) > 35 else opinions[0]
            count = f"({len(opinions)}次)"
            log(f"{i:<6} {aspect:<20} {opinion:<40} {count:<10}")

    log(f"\n💡 行動建議:")
    if 'sorted_negative' in dir() and sorted_negative:
        log(f"   1. 優先改進: {sorted_negative[0][0]}")
    if sorted_positive:
        log(f"   2. 強化優勢: {sorted_positive[0][0]}")
    log(f"   3. 制定改善計畫並追蹤成效")

    # 詳細統計
    log(f"\n" + "=" * 80)
    log("各評論提取統計摘要")
    log("=" * 80)

    # 只輸出有問題或特殊的評論
    problematic_reviews = []
    for result in results:
        if len(result['pairs']) == 0:
            problematic_reviews.append((result['review_id'], "無有效屬性"))
        elif result['filtered_aspect_count'] + result['filtered_opinion_count'] > 3:
            problematic_reviews.append((result['review_id'], f"過濾較多 ({result['filtered_aspect_count']}+{result['filtered_opinion_count']})"))

    if problematic_reviews:
        log(f"\n需注意的評論:")
        for review_id, issue in problematic_reviews[:20]:
            log(f"   評論 #{review_id}: {issue}")

    # 統計摘要
    log(f"\n📈 提取品質統計:")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   - 有效提取率: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   - 平均每則評論屬性數: {avg_pairs:.2f}")
    log(f"   - 總過濾數: {stats['filtered_invalid_aspects'] + stats['filtered_invalid_opinions']} 個")


# ============================================
# 主執行函數
# ============================================

def run_analysis(reviews, aspect_extractor, output_file="absa_analysis_results.txt"):
    """執行分析並輸出結果"""

    # 創建日誌記錄器
    logger = FileLogger(output_file)

    logger.log("=" * 80)
    logger.log("ABSA 屬性-意見-情感分析系統 (改進版 V2)")
    logger.log(f"執行時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"評論數量: {len(reviews)}")
    logger.log("=" * 80)

    # 執行分析
    results, stats = comprehensive_aspect_opinion_analysis_complete(
        reviews,
        aspect_extractor,
        max_words=80,
        verbose=True,
        logger=logger
    )

    logger.log("\n" + "=" * 80)
    logger.log("✅ 分析完成！")
    logger.log("=" * 80)

    # 生成管理報告
    generate_management_report_complete(results, stats, logger)

    # 保存結果
    logger.save()

    return results, stats


# ============================================
# 執行入口
# ============================================
if __name__ == "__main__":
    # 確保 test_reviews 和 aspect_extractor 已經存在
    if 'test_reviews' in dir() and len(test_reviews) > 0 and 'aspect_extractor' in dir():
        print("開始執行改進版 ABSA 分析...")
        results, stats = run_analysis(test_reviews, aspect_extractor, OUTPUT_FILE)
    else:
        print("請確保 test_reviews 和 aspect_extractor 變數已定義")
        print("\n使用方式:")
        print("1. 載入您的評論資料到 test_reviews 變數")
        print("2. 載入 PyABSA aspect_extractor")
        print("3. 執行: results, stats = run_analysis(test_reviews, aspect_extractor)")

In [ ]:
# AI 修改版 V3
# ABSA 改進版 V3 - 屬性-意見-情感提取系統
# 修正重點：
# 1. 否定句情感判斷（didn't, wasn't 等）
# 2. 更嚴格的意見詞驗證
# 3. 屬性詞格式清理（移除開頭的 and/or）
# 4. 管理報告品質提升

import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import re
from datetime import datetime

# ============================================
# 全域設定
# ============================================
OUTPUT_FILE = "absa_analysis_results_v3.txt"

# ============================================
# is_adjective_or_adverb 函數（增強版）
# ============================================

def is_adjective_or_adverb(word):
    """判斷是否為形容詞或副詞 - 增強版"""

    positive_adjectives = {
        'beautiful', 'excellent', 'great', 'good', 'amazing', 'wonderful',
        'fantastic', 'outstanding', 'impressive', 'stunning', 'gorgeous',
        'perfect', 'superb', 'fabulous', 'marvelous', 'magnificent',
        'delicious', 'tasty', 'delightful', 'comfortable', 'cozy',
        'friendly', 'helpful', 'efficient', 'smooth', 'easy',
        'fast', 'quick', 'reliable', 'stable', 'secure',
        'spacious', 'large', 'big', 'bright', 'clean', 'modern',
        'innovative', 'creative', 'unique', 'special', 'exceptional',
        'high-quality', 'premium', 'luxurious', 'elegant', 'stylish',
        'soft', 'warm', 'cool', 'fresh', 'clear', 'sharp', 'vivid',
        'dim', 'bright', 'light', 'dark', 'crisp', 'smooth',
        'powerful', 'strong', 'solid', 'durable', 'sturdy',
        'romantic', 'peaceful', 'quiet', 'calm', 'relaxing',
        'tender', 'juicy', 'crispy', 'creamy', 'rich',
        'lightweight', 'portable', 'convenient', 'accessible',
        'nice', 'fine', 'lovely', 'charming', 'attractive',
        'pleasant', 'enjoyable', 'satisfying', 'adequate',
        'reasonable', 'decent', 'acceptable', 'maintained', 'tested',
        'interesting', 'awesome', 'incredible', 'terrific', 'cute',
        'attentive', 'professional', 'polite', 'welcoming', 'lively',
        'flavorful', 'seasoned', 'yummy', 'scrumptious'
    }

    negative_adjectives = {
        'terrible', 'bad', 'poor', 'awful', 'horrible', 'disappointing',
        'frustrating', 'annoying', 'uncomfortable', 'inconvenient',
        'slow', 'sluggish', 'laggy', 'unstable', 'unreliable',
        'expensive', 'overpriced', 'costly', 'pricey', 'high',
        'small', 'tiny', 'cramped', 'limited', 'insufficient',
        'short', 'brief', 'inadequate', 'lacking', 'missing',
        'noisy', 'loud', 'distracting', 'disturbing',
        'dirty', 'messy', 'cluttered', 'outdated', 'old', 'obsolete',
        'complicated', 'confusing', 'difficult', 'hard', 'complex',
        'broken', 'damaged', 'faulty', 'defective', 'flawed',
        'cold', 'hot', 'harsh', 'rough', 'stiff',
        'weak', 'fragile', 'unstable', 'insecure',
        'rude', 'unfriendly', 'unhelpful', 'careless',
        'sensitive', 'buggy', 'glitchy', 'sticky',
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross'
    }

    intensifiers = {
        'very', 'extremely', 'really', 'quite', 'too', 'so',
        'incredibly', 'amazingly', 'exceptionally', 'remarkably',
        'absolutely', 'totally', 'completely', 'utterly', 'entirely',
        'highly', 'fairly', 'rather', 'somewhat', 'pretty',
        'disappointingly', 'surprisingly', 'unusually', 'well', 'thoroughly',
        'super', 'especially', 'particularly'
    }

    word_lower = word.lower().strip('.,!?;:')

    if word_lower in positive_adjectives:
        return True, 'positive_adj'
    if word_lower in negative_adjectives:
        return True, 'negative_adj'
    if word_lower in intensifiers:
        return True, 'intensifier'

    adj_suffixes = ['ful', 'less', 'ous', 'ive', 'able', 'ible', 'al', 'ic', 'y', 'ing', 'ed']
    adv_suffixes = ['ly']

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    for suffix in adv_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adv'

    return False, 'none'


# ============================================
# 輔助函數
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """找到句子邊界"""
    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in ['.', '!', '?']:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in ['.', '!', '?']:
            sentence_end = i
            break

    return sentence_start, sentence_end


def detect_be_verb(tokens, start_idx):
    """檢測 be 動詞"""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    if word in ['is', 'was', 'were', 'are', 'be', 'been', "'s"]:
        return True, 1, False

    if word in ["isn't", "isnt", "wasn't", "wasnt", "weren't", "werent", "aren't", "arent"]:
        return True, 1, True

    if word in ['isn', 'wasn', 'weren', 'aren'] and start_idx + 2 < len(tokens):
        if tokens[start_idx + 1] == "'" and tokens[start_idx + 2].lower() in ['t', 'nt']:
            return True, 3, True

    return False, 0, False


def handle_that_with_negation(tokens, start_pos, end_pos):
    """處理 that + 否定"""
    result = []
    i = start_pos

    while i < end_pos:
        word = tokens[i]
        word_lower = word.lower()

        if word_lower == 'that' and i + 1 < end_pos:
            found, skip, is_negative = detect_be_verb(tokens, i + 1)

            if found and is_negative:
                i += 1 + skip
                result.append("not")
                continue

        result.append(word)
        i += 1

    return result


def find_previous_aspect_end(tokens, current_start_idx, all_aspect_positions):
    """找到前一個屬性的結束位置"""
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(tokens, current_end_idx, all_aspect_positions):
    """找到下一個屬性的起始位置"""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# ⭐ V3 新增：檢測否定結構
# ============================================

def detect_negation_in_opinion(opinion):
    """
    檢測意見中是否包含否定結構
    返回: (has_negation, negation_type)
    """
    opinion_lower = opinion.lower()

    # 1. 直接否定詞
    direct_negations = [
        "not ", "n't ", "no ", "never ", "none ", "nothing ",
        "didn't", "didn't", "wasn't", "wasn't", "weren't", "weren't",
        "isn't", "isn't", "aren't", "aren't", "don't", "don't",
        "doesn't", "doesn't", "won't", "won't", "couldn't", "couldn't",
        "shouldn't", "shouldn't", "wouldn't", "wouldn't",
        "can't", "can't", "cannot"
    ]

    for neg in direct_negations:
        if neg in opinion_lower:
            return True, 'direct'

    # 2. 分詞形式的否定 (didn ' t, wasn ' t 等)
    tokenized_negations = [
        ("didn", "'", "t"),
        ("wasn", "'", "t"),
        ("weren", "'", "t"),
        ("isn", "'", "t"),
        ("aren", "'", "t"),
        ("don", "'", "t"),
        ("doesn", "'", "t"),
        ("won", "'", "t"),
        ("couldn", "'", "t"),
        ("shouldn", "'", "t"),
        ("wouldn", "'", "t"),
        ("can", "'", "t"),
    ]

    for neg_tuple in tokenized_negations:
        pattern = r'\b' + neg_tuple[0] + r'\s*' + neg_tuple[1] + r'\s*' + neg_tuple[2] + r'\b'
        if re.search(pattern, opinion_lower):
            return True, 'tokenized'

    # 3. 缺乏/沒有 類型
    lacking_phrases = [
        "lack of", "lacking", "without", "missing", "absent",
        "no sign of", "devoid of"
    ]

    for phrase in lacking_phrases:
        if phrase in opinion_lower:
            return True, 'lacking'

    return False, None


# ============================================
# ⭐ V3 改進：意見詞驗證函數
# ============================================

def is_valid_opinion(opinion):
    """驗證意見詞是否有效 - V3 增強版"""
    if not opinion or len(opinion.strip()) < 2:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    # 1. 過濾純動作詞
    action_only_words = {
        'ordered', 'ordering', 'makes', 'making', 'arrived',
        'asked', 'asking', 'having', 'took', 'taking',
        'informed', 'told', 'said', 'called', 'comes', 'came',
        'showed', 'showing', 'walked', 'walking', 'served',
        'serving', 'brought', 'bringing', 'dropped', 'cooked',
        'cooking', 'fried', 'grilled', 'smoked', 'pulled',
        'hoped', 'enjoy', 'member', 'available'  # V3 新增
    }

    # 2. 過濾無意義單詞
    meaningless_single = {
        'only', 'just', 'also', 'too', 'as', 'q', 'y', 'a', 'an', 'the',
        'outside', 'inside', 'here', 'there', 'next', 'back',
        'ordering', 'during', 'around', 'wherever', 'texanas',
        'specialty', 'available', 'if', 'when', 'would'  # V3 新增
    }

    # 3. 過濾人名
    common_names = {
        'joel', 'david', 'kaitlin', 'kaitlyn', 'christina', 'natalie',
        'melanie', 'sara', 'paul', 'morgan', 'fatima', 'veronica',
        'andy', 'nick', 'alyssa', 'janelle', 'kiara', 'braelyn',
        'gabby', 'adela', 'ray', 'kristina', 'tina', 'cameron',
        'zoe', 'greg', 'steven', 'jordan', 'sara'
    }

    # 4. 過濾位置描述
    location_keywords = {'close', 'near', 'hotel', 'magnolia', 'downtown', 'street'}

    # 單詞檢查
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')
        if clean_word in action_only_words | meaningless_single | common_names:
            return False
        # 單詞必須是形容詞
        is_adj, adj_type = is_adjective_or_adverb(clean_word)
        if not is_adj:
            return False

    # 雙詞檢查
    if len(words) == 2:
        clean_words = [w.strip('.,') for w in words]
        if all(w in action_only_words | common_names | meaningless_single for w in clean_words):
            return False
        # V3: 檢查 "動詞 + if/when/would" 模式
        if clean_words[1] in {'if', 'when', 'would', 'could', 'should'}:
            return False
        if clean_words[0] in action_only_words and clean_words[1] in {'if', 'when', 'would', 'over'}:
            return False

    # 檢查是否全是動作詞
    action_count = sum(1 for w in words if w.strip('.,') in action_only_words)
    if action_count == len(words):
        return False

    # 檢查人名（單獨出現）
    if len(words) <= 2 and any(w.strip('.,') in common_names for w in words):
        non_name_words = [w for w in words if w.strip('.,') not in common_names]
        if len(non_name_words) == 0:
            return False

    # 檢查位置描述
    if len(words) >= 3:
        location_count = sum(1 for w in words if w.strip('.,') in location_keywords)
        if location_count >= 2:
            return False

    # V3: 更多無意義模式
    invalid_patterns = [
        r"^'s\s+name",
        r"^and\s+waited$",
        r"^to\s+before\s+me",
        r"^de\s+camarón$",
        r"^son\s+gigantes\s+y$",
        r"^\d+\s*/",
        r"^or\s+maybe\s+both$",
        r"^for\s+dessert$",
        r"^going\s+forward$",
        r"^hoped\s+would$",           # V3 新增
        r"^enjoy\s+when$",            # V3 新增
        r"^asked\s+if$",              # V3 新增
        r"^took\s+over$",             # V3 新增
        r"^member\s+where",           # V3 新增
        r"^came\s+out\s+and\s+it",    # V3 新增
        r"^n\s+cheese\s+was",         # V3 新增
        r"^the\s+grilled\s+cheese",   # V3 新增
        r"^can\s+make\s+anything",    # V3 新增
        r"^rustic\s+eatery",          # V3 新增
        r"before\s+he\s+ever\s+made", # V3 新增
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    # V3: 檢查意見是否包含至少一個情感詞
    has_sentiment_word = False
    sentiment_indicators = {
        # 正面
        'good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic',
        'delicious', 'tasty', 'fresh', 'crispy', 'tender', 'perfect',
        'friendly', 'helpful', 'attentive', 'professional', 'nice',
        'lovely', 'beautiful', 'solid', 'rich', 'flavorful',
        # 負面
        'bad', 'terrible', 'awful', 'horrible', 'poor', 'slow',
        'rude', 'cold', 'dry', 'bland', 'soggy', 'greasy', 'fatty',
        'disappointing', 'mediocre', 'overcooked', 'undercooked',
        'expensive', 'overpriced', 'crowded', 'noisy', 'dirty',
        'sticky', 'burnt', 'stale'
    }

    for word in words:
        clean_w = word.strip('.,!?;:\'"').lower()
        if clean_w in sentiment_indicators:
            has_sentiment_word = True
            break
        # 檢查形容詞後綴
        is_adj, _ = is_adjective_or_adverb(clean_w)
        if is_adj:
            has_sentiment_word = True
            break

    # 如果意見詞太短（<=3詞）且沒有情感詞，可能無效
    if len(words) <= 3 and not has_sentiment_word:
        # 但如果包含否定結構，可能仍有意義
        has_neg, _ = detect_negation_in_opinion(opinion_lower)
        if not has_neg:
            return False

    return True


# ============================================
# 核心：提取完整意見詞
# ============================================

def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """提取完整意見詞 - 帶前後邊界檢測"""

    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)

    prev_aspect_end = find_previous_aspect_end(tokens, start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(tokens, end_idx, all_aspect_positions)

    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # 模式 1: 前置修飾詞
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, modifier_type = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in [',', 'and', 'with', 'or', 'but']:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

        next_pos = end_idx + 1
        if next_pos < sentence_end:
            next_word = tokens[next_pos]
            forbidden_next = {
                'is', 'was', 'are', 'were', 'be', 'been',
                'has', 'have', 'had', 'does', 'do',
                'but', 'and', 'or', 'so', 'because',
                'with', 'for', 'to', 'in', 'at', 'on', 'of', 'by',
                'that', 'which', 'who', 'this', 'it',
                ',', '.', '!', '?', ';', '-'
            }
            if next_word.isalpha() and next_word.lower() not in forbidden_next:
                opinion_words.append(next_word)

    # 模式 2: be 動詞 + 形容詞
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []

        if is_negative:
            post_modifiers.append("not")

        for i in range(opinion_start, opinion_end):
            word = tokens[i]

            if word in ['.', '!', '?', ';']:
                break

            if word.lower() == 'but' and next_aspect_start is not None:
                if i + 1 < len(tokens) and i + 1 >= next_aspect_start - 2:
                    break

            post_modifiers.append(word)

        if post_modifiers:
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # 模式 3: 動詞短語
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in ['.', '!', '?', ';']:
                break
            temp_words.append(word)

        if temp_words:
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # 模式 4: 上下文搜索
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, modifier_type = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                context_words = []
                for j in range(context_start, context_end):
                    if tokens[j] not in ['.', '!', '?', ';']:
                        context_words.append(tokens[j])

                opinion_words.extend(context_words)
                pattern_type = "context_search"
                break

            if word in ['.', '!', '?']:
                break

    opinion_words = handle_that_with_negation(opinion_words, 0, len(opinion_words))
    opinion_phrase = ' '.join(opinion_words).strip()

    return opinion_phrase, pattern_type


# ============================================
# ⭐ V3 改進：情感糾正（修正否定句處理）
# ============================================

def correct_sentiment_enhanced(aspect, opinion, predicted_sentiment):
    """情感糾正 - V3 修正版：正確處理否定句"""

    opinion_lower = opinion.lower()

    # ⭐ V3 核心修正：首先檢測否定結構
    has_negation, neg_type = detect_negation_in_opinion(opinion_lower)

    # 強正面詞
    strong_positive_words = {
        'excellent', 'amazing', 'fantastic', 'wonderful', 'perfect',
        'great', 'awesome', 'delicious', 'outstanding', 'best',
        'incredible', 'superb', 'terrific', 'fabulous', 'marvelous',
        'friendly', 'helpful', 'attentive', 'professional', 'cute',
        'nice', 'good', 'tasty', 'fresh', 'tender', 'juicy', 'crispy'
    }

    # 強負面詞
    strong_negative_words = {
        'terrible', 'awful', 'horrible', 'bad', 'poor', 'disappointing',
        'rude', 'slow', 'cold', 'dry', 'bland', 'soggy', 'greasy',
        'fatty', 'overcooked', 'undercooked', 'burnt', 'stale',
        'dirty', 'messy', 'noisy', 'crowded', 'expensive', 'overpriced',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross'
    }

    # ⭐ V3 關鍵邏輯：如果有否定結構
    if has_negation:
        # 檢查是否否定了正面詞 -> 應該是負面
        for pos_word in strong_positive_words:
            if pos_word in opinion_lower:
                # "wasn't good", "didn't have fresh" -> Negative
                return 'Negative'

        # 檢查是否否定了負面詞 -> 應該是正面（雙重否定）
        for neg_word in strong_negative_words:
            if neg_word in opinion_lower:
                # "not bad", "wasn't terrible" -> Positive
                return 'Positive'

        # 其他有否定結構的情況，傾向負面
        return 'Negative'

    # 如果沒有否定結構，檢查是否包含正面詞
    for pos_word in strong_positive_words:
        if pos_word in opinion_lower:
            return 'Positive'

    # 處理 'but' 轉折
    if ' but ' in opinion_lower:
        parts = opinion_lower.split(' but ', 1)
        if len(parts) == 2:
            after_but = parts[1]

            for neg_word in strong_negative_words:
                if neg_word in after_but:
                    return 'Negative'

    # 處理 "for the price" 模式
    for_the_patterns = ['for the price', 'for the cost', 'for the money', 'for price', 'for cost']
    for pattern in for_the_patterns:
        if pattern in opinion_lower:
            negative_indicators = ['small', 'limited', 'short', 'expensive', 'high', 'too']
            for indicator in negative_indicators:
                if indicator in opinion_lower:
                    return 'Negative'

    # 獨立負面詞檢查
    opinion_words = opinion_lower.split()
    for word in opinion_words:
        clean_word = word.strip('.,!?;:')
        if clean_word in strong_negative_words:
            return 'Negative'

    return predicted_sentiment


# ============================================
# 格式化顯示（增強版）
# ============================================

def format_opinion_for_display(opinion):
    """格式化意見用於顯示 - 增強版"""

    if not opinion:
        return ""

    important_nouns = {
        'experience', 'quality', 'service', 'food', 'price',
        'performance', 'design', 'features', 'options', 'selection',
        'system', 'conditions', 'footage', 'screen', 'tasks',
        'gaming', 'back', 'frame', 'detail', 'mode', 'photos',
        'stabilization', 'colors', 'blacks', 'rate', 'lag',
        'connectivity', 'drawbacks', 'use', 'bugs', 'equipment',
        'slot', 'device', 'issues', 'people', 'day', 'sensor', 'lens',
        'center', 'location', 'access', 'contrast', 'brightness'
    }

    leading_words_to_remove = {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'also', 'too', 'either',
        'has', 'have', 'had',
        'contains', 'includes', 'consists',
        'offers', 'provides', 'features',
        'is', 'are', 'was', 'were', 'looks', 'feels',
        'it', 'itself', 'they', 'themselves', 'we', 'ourselves',
        'i', 'myself', 'you', 'yourself'
    }

    do_not_merge_words = {
        'a', 'an', 'the', 'it', 'is', 'we', 'i', 'you', 'he', 'she',
        'of', 'in', 'to', 'for', 'with', 'on', 'at', 'my', 'our'
    }

    general_cut_off = {
        'making', 'causing', 'forcing', 'leaving', 'rendering',
        'unless', 'except', 'besides', 'despite', 'although',
        'which', 'where', 'when', 'because', 'since'
    }

    post_dash_cut_off = {
        'with', 'and', 'but', 'including', 'plus', 'to', 'for', 'at'
    }

    opinion = re.sub(r'([a-z])-(?=[A-Z])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'([a-zA-Z0-9])-(?=[a-zA-Z0-9])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    words = opinion.split()

    while words and words[0].lower() in leading_words_to_remove:
        words.pop(0)

    cleaned_words = []
    i = 0
    while i < len(words):
        word = words[i]

        if i + 1 < len(words):
            next_word = words[i+1]
            if word.endswith('-') and len(word) > 1:
                if next_word.isdigit() or next_word.lower() in do_not_merge_words:
                    cleaned_words.append(word.rstrip('-'))
                else:
                    merged = word + next_word
                    cleaned_words.append(merged)
                    i += 2
                    continue
            elif next_word == '-' and i + 2 < len(words):
                following_word = words[i+2]
                if following_word.isdigit() or following_word.lower() in do_not_merge_words:
                    cleaned_words.append(word)
                    cleaned_words.append('-')
                    i += 2
                    continue
                merged = word + "-" + following_word
                cleaned_words.append(merged)
                i += 3
                continue
        cleaned_words.append(word)
        i += 1
    words = cleaned_words

    truncated_words = []
    valid_length = 0
    has_passed_dash = False

    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')
        is_truncator = False

        if word == '-' or word.startswith('-'):
            if i + 1 < len(words):
                next_w = words[i+1].lower()
                if next_w in {'i', 'we', 'it', 'he', 'she', 'they'}:
                    break
            has_passed_dash = True
            truncated_words.append(word)
            continue

        elif word.endswith('-') and '-' not in word[:-1] and valid_length >= 1:
            is_truncator = True

        if word_lower in general_cut_off and valid_length >= 2:
            break

        if has_passed_dash and word_lower in post_dash_cut_off:
            break

        if is_truncator and valid_length >= 1:
            break

        truncated_words.append(word)
        if word not in {',', 'and', 'but', '-'}:
            valid_length += 1

    words = truncated_words

    final_words = []
    for i, word in enumerate(words):
        if word == ',':
            if final_words and not final_words[-1].endswith(','):
                final_words[-1] = final_words[-1] + ','
        else:
            clean_word = word.lstrip(',')
            if clean_word:
                final_words.append(clean_word)
    words = final_words

    trailing_words_to_remove = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'too', 'either', 'also', 'but', 'however',
        'a', 'an', 'the', ',', '-', 'is', 'are', 'was', 'were',
        'need', 'justify', 'make', 'do', 'want', 'require', 'expect',
        'produces', 'provides', 'offers', 'features', 'includes', 'creates', 'shows',
        "'", '"', '(', 'it', 'i', 'we', 'they', 'he', 'she'
    }

    dangling_adjectives = {
        'high', 'low', 'large', 'small', 'good', 'bad', 'great', 'poor',
        'long', 'short', 'new', 'old', 'many', 'few', 'heavy', 'light',
        'strong', 'weak', 'fast', 'slow'
    }

    prepositions_for_check = {'with', 'for', 'in', 'on', 'at', 'of', 'and', 'but', 'or'}

    while words:
        last_word = words[-1].lower().strip('.,!?;:')

        if last_word in important_nouns:
            break
        should_remove = False
        if last_word in trailing_words_to_remove or words[-1].endswith('-') or words[-1].isdigit():
            should_remove = True
        if not should_remove and len(words) >= 2:
            second_last = words[-2].lower().strip(',')
            if last_word in dangling_adjectives and second_last in prepositions_for_check:
                should_remove = True
        if words[-1].endswith(','):
            words[-1] = words[-1].rstrip(',')
            if not words[-1]:
                words.pop()
                continue
        if should_remove:
            words.pop()
        else:
            break

    result_text = ' '.join(words)
    connectors = [' but ', ' however ', ' though ', ' and ']

    for connector in connectors:
        if connector in result_text.lower():
            last_connector_idx = result_text.lower().rfind(connector)
            if last_connector_idx == -1:
                continue

            before_text = result_text[:last_connector_idx].strip()
            after_text_raw = result_text[last_connector_idx + len(connector):].strip()
            after_words = after_text_raw.split()
            should_cut = False

            if len(after_words) == 0:
                should_cut = True
            elif len(after_words) <= 2:
                bad_enders = {'i', 'it', 'we', 'he', 'she', 'they', 'a', 'an', 'the'}
                if any(w.lower() in bad_enders for w in after_words):
                    should_cut = True
            if connector.strip() == 'and' and after_words:
                first_after = after_words[0].lower()
                if first_after in {'the', 'a', 'an', 'this', 'my', 'our', 'it'}:
                    if len(after_words) <= 3:
                        should_cut = True

            comparative_words = {
                'more', 'less', 'better', 'worse', 'higher', 'lower',
                'deeper', 'brighter', 'darker', 'faster', 'slower',
                'larger', 'smaller', 'stronger', 'weaker', 'stunning'
            }
            if any(w.lower() in comparative_words for w in after_words):
                should_cut = False

            if should_cut:
                result_text = before_text

    words = result_text.split()
    if len(words) > 12 and ' but ' not in result_text.lower():
        words = words[:12]
        result_text = ' '.join(words)

    result_text = re.sub(r',\s*,', ',', result_text)
    result_text = result_text.strip().rstrip('.,;-\'"(')

    return result_text


# ============================================
# ⭐ V3 改進：屬性驗證（增強版）
# ============================================

def is_valid_aspect(aspect):
    """檢查屬性是否有效 - V3 增強版"""
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    if cleaned == "-" or not cleaned:
        return False

    if all(c in '.,!?;:-_' for c in cleaned):
        return False

    # V3: 過濾以 and/or 開頭的屬性
    if cleaned.startswith('and ') or cleaned.startswith('or '):
        return False

    invalid_aspects = {
        'waited', 'waiting', 'ordering', 'making', 'having', 'served',
        'serving', 'asked', 'asking', 'walked', 'walking',
        'someone', 'she', 'he', 'they', 'it', 'we', 'i', 'you',
        'the', 'a', 'an',
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours',
        'fast', 'hot', 'cold', 'new',
        'sub', 'drive', 'seat', 'wait', 'ordering'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """修正屬性詞 - V3 增強版"""
    if not aspect:
        return ""

    clean_aspect = aspect.strip()

    # V3: 移除開頭的 and/or
    clean_aspect = re.sub(r'^(and|or)\s+', '', clean_aspect, flags=re.IGNORECASE)

    units_pattern = r'^(?:[\d\.]+\s*-?\s*)?(?:inch|inches|cm|mm|kg|lbs|oz)\b\s*'
    clean_aspect = re.sub(units_pattern, '', clean_aspect, flags=re.IGNORECASE)

    stopwords_pattern = r'^(the|a|an|this|that|my|our)\s+'
    clean_aspect = re.sub(stopwords_pattern, '', clean_aspect, flags=re.IGNORECASE)

    clean_aspect = re.sub(r'^[-:;]\s*', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# 分段函數
# ============================================

def split_long_review(text, max_words=80):
    """分割長評論"""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub_sentence = ' '.join(words[i:i+max_words])
                chunks.append(sub_sentence + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# 文件輸出類
# ============================================

class FileLogger:
    """文件輸出記錄器"""
    def __init__(self, filename):
        self.filename = filename
        self.content = []

    def log(self, message=""):
        self.content.append(message)
        print(message)

    def save(self):
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ 結果已保存到: {self.filename}")


# ============================================
# 完整分析函數
# ============================================

def comprehensive_aspect_opinion_analysis_complete(reviews, aspect_extractor, max_words=80, verbose=True, logger=None):
    """完整分析系統 - V3 改進版"""

    if logger:
        logger.log("=" * 80)
        logger.log("完整的屬性-意見分析系統 (V3 改進版)")
        logger.log(f"分段閾值: {max_words} 字")
        logger.log(f"分析時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 80)

    all_results = []

    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'sentiment_corrections': 0
    }

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'='*80}")
            logger.log(f"評論 #{idx} ({word_count} 字)")
            logger.log(f"{'='*80}")

        if word_count <= max_words:
            if logger and verbose:
                logger.log(f"📝 直接分析")

            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]

            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if logger and verbose:
                logger.log(f"📄 需要分段處理")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   分成 {len(chunks)} 段")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = []
            for i, chunk_result in enumerate(chunk_results, 1):
                analysis_results.append({'chunk_id': i, 'result': chunk_result})

        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, sentiment in zip(aspects, positions, sentiments):

                aspect = refine_aspect_term(aspect)

                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                display_opinion = format_opinion_for_display(raw_opinion)

                if not display_opinion or display_opinion.strip() == "":
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                if not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                corrected_sentiment = correct_sentiment_enhanced(aspect, raw_opinion, sentiment)

                if corrected_sentiment != sentiment:
                    stats['sentiment_corrections'] += 1

                stats['total_aspects'] += 1

                aspect_opinion_pairs.append({
                    'aspect': aspect,
                    'opinion': display_opinion,
                    'raw_opinion': raw_opinion,
                    'sentiment': corrected_sentiment,
                    'original_sentiment': sentiment,
                    'formatted': f"{aspect}: {display_opinion}",
                    'pattern': pattern,
                    'chunk': chunk_data['chunk_id'] if len(analysis_results) > 1 else None
                })

        if logger and verbose:
            logger.log(f"\n🎯 屬性-意見提取結果:")
            logger.log(f"{'序號':<6} {'屬性':<20} {'意見':<50} {'情感':<10} {'模式':<15}")
            logger.log("-" * 95)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    sentiment_display = pair['sentiment']
                    if pair['sentiment'] != pair['original_sentiment']:
                        sentiment_display += "*"

                    opinion_display = pair['opinion'][:47] + "..." if len(pair['opinion']) > 50 else pair['opinion']

                    logger.log(f"{i:<6} {pair['aspect']:<20} {opinion_display:<50} "
                              f"{sentiment_display:<10} {pair['pattern']:<15}")

                total_filtered = filtered_aspect_count + filtered_opinion_count
                if total_filtered > 0:
                    logger.log(f"\n   ℹ️ 已過濾: {filtered_aspect_count} 個無效屬性, {filtered_opinion_count} 個無效意見")
            else:
                logger.log(f"   ⚠️ 未提取到有效的屬性-意見對")

        if logger and verbose and aspect_opinion_pairs:
            logger.log(f"\n📋 完整的屬性-意見組合:")
            for i, pair in enumerate(aspect_opinion_pairs, 1):
                emoji = "😊" if pair['sentiment'] == "Positive" else "😞"
                chunk_info = f" [段{pair['chunk']}]" if pair['chunk'] else ""
                sentiment_note = " [糾正]" if pair['sentiment'] != pair['original_sentiment'] else ""
                logger.log(f"   {i}. {pair['formatted']} {emoji}{chunk_info}{sentiment_note}")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
            'filtered_aspect_count': filtered_aspect_count,
            'filtered_opinion_count': filtered_opinion_count
        })

    return all_results, stats


# ============================================
# ⭐ V3 改進：管理報告（增強版）
# ============================================

def generate_management_report_complete(results, stats, logger=None):
    """生成完整管理報告 - V3 增強版"""

    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 80)
    log("管理分析報告 (V3)")
    log("=" * 80)

    positive_pairs = []
    negative_pairs = []
    total_corrected = 0

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

            if pair['sentiment'] != pair['original_sentiment']:
                total_corrected += 1

    total_pairs = len(positive_pairs) + len(negative_pairs)

    log(f"\n📊 總體統計:")
    log(f"   - 評論總數: {len(results)}")
    log(f"   - 識別有效屬性總數: {total_pairs}")
    log(f"   - 已過濾無效屬性: {stats['filtered_invalid_aspects']} 個")
    log(f"   - 已過濾無效意見: {stats['filtered_invalid_opinions']} 個")
    if total_corrected > 0:
        log(f"   - 情感判斷已糾正: {total_corrected} 個")
    if total_pairs > 0:
        log(f"   - 正面評價: {len(positive_pairs)} 個 ({len(positive_pairs)/total_pairs*100:.1f}%)")
        log(f"   - 負面評價: {len(negative_pairs)} 個 ({len(negative_pairs)/total_pairs*100:.1f}%)")

    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair['opinion'])

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair['opinion'])

    # V3: 更嚴格的負面意見驗證
    def is_genuinely_negative(opinion):
        """確認意見真的是負面的 - V3 增強版"""
        positive_indicators = {
            'excellent', 'great', 'amazing', 'wonderful', 'perfect',
            'delicious', 'fantastic', 'awesome', 'best', 'cute', 'nice',
            'good', 'friendly', 'helpful', 'attentive', 'professional',
            'fresh', 'tasty', 'tender', 'juicy', 'crispy', 'terrific',
            'solid', 'rich', 'flavorful', 'lovely', 'beautiful'
        }

        # V3: 非情感詞列表
        non_sentiment_words = {
            'specialty', 'rustic', 'eatery', 'grilled', 'cheese', 'brisket',
            'anything', 'make', 'can', 'went', 'came', 'took', 'had'
        }

        opinion_lower = opinion.lower()
        opinion_words = set(opinion_lower.split())

        # 檢查是否主要是非情感詞
        non_sentiment_count = len(opinion_words & non_sentiment_words)
        if non_sentiment_count >= len(opinion_words) * 0.5:
            return False

        # 檢查是否有否定詞
        has_negation, _ = detect_negation_in_opinion(opinion_lower)

        # 如果有否定詞，且包含正面詞，則是負面的
        if has_negation:
            for pos in positive_indicators:
                if pos in opinion_lower:
                    return True

        # 如果沒有否定詞，且包含正面詞，則不是負面的
        for pos in positive_indicators:
            if pos in opinion_lower:
                return False

        return True

    # V3: 更嚴格的正面意見驗證
    def is_genuinely_positive(opinion):
        """確認意見真的是正面的 - V3 新增"""
        negative_indicators = {
            'terrible', 'awful', 'horrible', 'bad', 'poor', 'disappointing',
            'rude', 'slow', 'cold', 'dry', 'bland', 'soggy', 'greasy',
            'fatty', 'overcooked', 'burnt', 'stale', 'dirty', 'messy',
            'mediocre', 'pathetic', 'lousy', 'nasty', 'gross'
        }

        opinion_lower = opinion.lower()

        # 檢查是否有否定詞
        has_negation, _ = detect_negation_in_opinion(opinion_lower)

        # 如果有否定詞，且包含負面詞，則是正面的（雙重否定）
        if has_negation:
            for neg in negative_indicators:
                if neg in opinion_lower:
                    return True
            # 有否定詞但沒有負面詞，可能是否定正面詞，不是正面
            return False

        # 如果沒有否定詞，檢查是否包含負面詞
        for neg in negative_indicators:
            if neg in opinion_lower:
                return False

        return True

    if positive_aspects:
        log(f"\n✅ 競爭優勢:")
        log(f"{'排名':<6} {'屬性':<20} {'客戶評價':<40} {'次數':<10}")
        log("-" * 80)

        # V3: 過濾真正正面的
        filtered_positive = {
            k: [op for op in v if is_genuinely_positive(op)]
            for k, v in positive_aspects.items()
        }
        filtered_positive = {k: v for k, v in filtered_positive.items() if v}

        sorted_positive = sorted(filtered_positive.items(), key=lambda x: len(x[1]), reverse=True)
        for i, (aspect, opinions) in enumerate(sorted_positive[:15], 1):
            opinion_text = opinions[0][:35] + "..." if len(opinions[0]) > 35 else opinions[0]
            count = f"({len(opinions)}次)"
            log(f"{i:<6} {aspect:<20} {opinion_text:<40} {count:<10}")

    if negative_aspects:
        log(f"\n⚠️ 需要改進:")
        log(f"{'排名':<6} {'屬性':<20} {'客戶抱怨':<40} {'次數':<10}")
        log("-" * 80)

        filtered_negative = {
            k: [op for op in v if is_genuinely_negative(op)]
            for k, v in negative_aspects.items()
        }
        filtered_negative = {k: v for k, v in filtered_negative.items() if v}

        sorted_negative = sorted(filtered_negative.items(), key=lambda x: len(x[1]), reverse=True)
        for i, (aspect, opinions) in enumerate(sorted_negative[:15], 1):
            opinion = opinions[0][:35] + "..." if len(opinions[0]) > 35 else opinions[0]
            count = f"({len(opinions)}次)"
            log(f"{i:<6} {aspect:<20} {opinion:<40} {count:<10}")

    log(f"\n💡 行動建議:")
    if 'sorted_negative' in dir() and sorted_negative:
        log(f"   1. 優先改進: {sorted_negative[0][0]}")
    if 'sorted_positive' in dir() and sorted_positive:
        log(f"   2. 強化優勢: {sorted_positive[0][0]}")
    log(f"   3. 制定改善計畫並追蹤成效")

    log(f"\n" + "=" * 80)
    log("各評論提取統計摘要")
    log("=" * 80)

    problematic_reviews = []
    for result in results:
        if len(result['pairs']) == 0:
            problematic_reviews.append((result['review_id'], "無有效屬性"))
        elif result['filtered_aspect_count'] + result['filtered_opinion_count'] > 3:
            problematic_reviews.append((result['review_id'], f"過濾較多 ({result['filtered_aspect_count']}+{result['filtered_opinion_count']})"))

    if problematic_reviews:
        log(f"\n需注意的評論:")
        for review_id, issue in problematic_reviews[:20]:
            log(f"   評論 #{review_id}: {issue}")

    log(f"\n📈 提取品質統計:")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   - 有效提取率: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   - 平均每則評論屬性數: {avg_pairs:.2f}")
    log(f"   - 總過濾數: {stats['filtered_invalid_aspects'] + stats['filtered_invalid_opinions']} 個")


# ============================================
# 主執行函數
# ============================================

def run_analysis(reviews, aspect_extractor, output_file="absa_analysis_results_v3.txt"):
    """執行分析並輸出結果"""

    logger = FileLogger(output_file)

    logger.log("=" * 80)
    logger.log("ABSA 屬性-意見-情感分析系統 (V3 改進版)")
    logger.log(f"執行時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"評論數量: {len(reviews)}")
    logger.log("=" * 80)

    results, stats = comprehensive_aspect_opinion_analysis_complete(
        reviews,
        aspect_extractor,
        max_words=80,
        verbose=True,
        logger=logger
    )

    logger.log("\n" + "=" * 80)
    logger.log("✅ 分析完成！")
    logger.log("=" * 80)

    generate_management_report_complete(results, stats, logger)

    logger.save()

    return results, stats


# ============================================
# 執行入口
# ============================================
if __name__ == "__main__":
    if 'test_reviews' in dir() and len(test_reviews) > 0 and 'aspect_extractor' in dir():
        print("開始執行 V3 改進版 ABSA 分析...")
        results, stats = run_analysis(test_reviews, aspect_extractor, OUTPUT_FILE)
    else:
        print("請確保 test_reviews 和 aspect_extractor 變數已定義")
        print("\n使用方式:")
        print("1. 載入您的評論資料到 test_reviews 變數")
        print("2. 載入 PyABSA aspect_extractor")
        print("3. 執行: results, stats = run_analysis(test_reviews, aspect_extractor)")


In [ ]:
# =============================================================================
# ABSA 深度洞察分析系統 V4
# =============================================================================
# 功能：
# 1. 解析 V3 結果並去重
# 2. 屬性語義聚類（將相似屬性歸類到顧客旅程階段）
# 3. 深度洞察報告生成
# 4. 輸出 Excel 報告
# =============================================================================

import re
import pandas as pd
from collections import defaultdict, Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 第一部分：解析 V3 結果文件
# =============================================================================

def parse_v3_results(file_path):
    """解析 V3 結果文件，提取所有屬性-意見-情感三元組"""

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 提取所有評論區塊
    review_pattern = r'評論 #(\d+) \((\d+) 字\)'
    pair_pattern = r'(\d+)\.\s+([^:]+):\s+(.+?)\s+(😊|😞)(?:\s+\[段\d+\])?(?:\s+\[糾正\])?'

    all_pairs = []
    current_review_id = None

    lines = content.split('\n')

    for i, line in enumerate(lines):
        # 檢查是否是評論標題
        review_match = re.search(review_pattern, line)
        if review_match:
            current_review_id = int(review_match.group(1))
            continue

        # 檢查是否是屬性-意見對
        pair_match = re.search(pair_pattern, line.strip())
        if pair_match and current_review_id:
            idx = pair_match.group(1)
            aspect = pair_match.group(2).strip()
            opinion = pair_match.group(3).strip()
            emoji = pair_match.group(4)
            sentiment = 'Positive' if emoji == '😊' else 'Negative'

            all_pairs.append({
                'review_id': current_review_id,
                'aspect': aspect,
                'opinion': opinion,
                'sentiment': sentiment,
                'aspect_lower': aspect.lower().strip(),
                'opinion_lower': opinion.lower().strip()
            })

    return pd.DataFrame(all_pairs)


# =============================================================================
# 第二部分：資料清理與去重
# =============================================================================

def clean_and_deduplicate(df):
    """清理資料並去除重複"""

    print("=" * 60)
    print("資料清理與去重")
    print("=" * 60)

    original_count = len(df)
    print(f"原始資料筆數: {original_count}")

    # 1. 移除完全重複的記錄（同評論、同屬性、同意見）
    df_dedup = df.drop_duplicates(subset=['review_id', 'aspect_lower', 'opinion_lower'])
    after_exact_dedup = len(df_dedup)
    print(f"移除完全重複後: {after_exact_dedup} (移除 {original_count - after_exact_dedup} 筆)")

    # 2. 移除同評論中相同屬性的重複（保留第一個）
    df_dedup = df_dedup.drop_duplicates(subset=['review_id', 'aspect_lower'], keep='first')
    after_aspect_dedup = len(df_dedup)
    print(f"移除同評論重複屬性後: {after_aspect_dedup} (移除 {after_exact_dedup - after_aspect_dedup} 筆)")

    # 3. 清理屬性名稱
    def clean_aspect(aspect):
        # 移除開頭的 and/or/the/a
        cleaned = re.sub(r'^(and|or|the|a|an)\s+', '', aspect.strip(), flags=re.IGNORECASE)
        # 移除特殊字符
        cleaned = re.sub(r'^[-:;]\s*', '', cleaned)
        return cleaned.strip()

    df_dedup['aspect_cleaned'] = df_dedup['aspect'].apply(clean_aspect)

    # 4. 移除無效屬性
    invalid_aspects = {
        'waited', 'waiting', 'ordering', 'making', 'having',
        'she', 'he', 'they', 'it', 'we', 'i', 'you', 'someone',
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours'
    }

    df_dedup = df_dedup[~df_dedup['aspect_cleaned'].str.lower().isin(invalid_aspects)]
    df_dedup = df_dedup[df_dedup['aspect_cleaned'].str.len() > 1]

    final_count = len(df_dedup)
    print(f"移除無效屬性後: {final_count} (移除 {after_aspect_dedup - final_count} 筆)")
    print(f"\n最終有效資料: {final_count} 筆")

    return df_dedup


# =============================================================================
# 第三部分：屬性語義聚類（顧客旅程框架）
# =============================================================================

# 定義顧客旅程階段與屬性映射
CUSTOMER_JOURNEY_MAPPING = {
    # 階段 1: 來店動機 (Why they come)
    'visit_motivation': {
        'keywords': ['location', 'downtown', 'hotel', 'game', 'astros', 'event',
                     'recommendation', 'reviews', 'rating', 'yelp', 'google'],
        'aspects': []
    },

    # 階段 2: 第一印象 (First Impression)
    'first_impression': {
        'keywords': ['atmosphere', 'vibe', 'ambiance', 'decor', 'decoration',
                     'interior', 'exterior', 'entrance', 'space', 'place',
                     'restaurant', 'bar', 'patio', 'seating', 'music', 'noise',
                     'crowd', 'busy', 'wait', 'line', 'parking'],
        'aspects': []
    },

    # 階段 3: 服務體驗 (Service Experience)
    'service_experience': {
        'keywords': ['service', 'server', 'waiter', 'waitress', 'staff',
                     'manager', 'host', 'hostess', 'bartender', 'busser',
                     'attentive', 'friendly', 'rude', 'slow', 'fast', 'quick'],
        'person_names': ['joel', 'paul', 'kaitlyn', 'kaitlin', 'christina',
                        'natalie', 'melanie', 'sara', 'morgan', 'fatima',
                        'andy', 'nick', 'alyssa', 'janelle', 'kiara'],
        'aspects': []
    },

    # 階段 4: 食物品質 (Food Quality)
    'food_quality': {
        'keywords': ['food', 'meal', 'dish', 'taste', 'flavor', 'texture',
                     'portion', 'size', 'presentation', 'quality', 'fresh',
                     'cooked', 'seasoning', 'sauce', 'spicy', 'bland'],
        'aspects': []
    },

    # 階段 5: 具體菜品 (Specific Menu Items)
    'menu_items': {
        'keywords': ['chicken', 'waffles', 'brisket', 'nachos', 'burger',
                     'wings', 'fries', 'tacos', 'sandwich', 'salad', 'soup',
                     'steak', 'ribs', 'pork', 'bacon', 'eggs', 'pancakes',
                     'mac', 'cheese', 'grits', 'coleslaw', 'beans', 'corn',
                     'bread', 'biscuit', 'dessert', 'pie', 'cake'],
        'aspects': []
    },

    # 階段 6: 飲品 (Beverages)
    'beverages': {
        'keywords': ['drink', 'drinks', 'cocktail', 'cocktails', 'beer', 'beers',
                     'wine', 'moonshine', 'margarita', 'mimosa', 'bloody mary',
                     'coffee', 'tea', 'juice', 'water', 'soda', 'lemonade',
                     'bar', 'happy hour'],
        'aspects': []
    },

    # 階段 7: 價值感知 (Value Perception)
    'value_perception': {
        'keywords': ['price', 'prices', 'cost', 'value', 'money', 'worth',
                     'expensive', 'cheap', 'affordable', 'reasonable',
                     'overpriced', 'deal', 'discount', 'happy hour'],
        'aspects': []
    },

    # 階段 8: 環境設施 (Facilities)
    'facilities': {
        'keywords': ['table', 'tables', 'chair', 'booth', 'restroom', 'bathroom',
                     'clean', 'dirty', 'sticky', 'menu', 'utensils', 'napkin',
                     'plate', 'glass', 'air conditioning', 'temperature'],
        'aspects': []
    },

    # 階段 9: 整體體驗 (Overall Experience)
    'overall_experience': {
        'keywords': ['experience', 'overall', 'visit', 'time', 'night',
                     'lunch', 'dinner', 'brunch', 'breakfast', 'return',
                     'recommend', 'back', 'again'],
        'aspects': []
    }
}

# 情感細分類別
SENTIMENT_SUBCATEGORIES = {
    'highly_positive': ['excellent', 'amazing', 'fantastic', 'perfect', 'outstanding',
                        'incredible', 'best', 'wonderful', 'awesome', 'superb'],
    'positive': ['good', 'great', 'nice', 'lovely', 'friendly', 'helpful',
                 'tasty', 'delicious', 'fresh', 'solid'],
    'neutral_positive': ['ok', 'okay', 'decent', 'fine', 'average', 'standard'],
    'neutral_negative': ['mediocre', 'disappointing', 'underwhelming'],
    'negative': ['bad', 'poor', 'slow', 'cold', 'bland', 'dry', 'soggy',
                 'rude', 'unfriendly', 'dirty'],
    'highly_negative': ['terrible', 'awful', 'horrible', 'worst', 'disgusting',
                        'pathetic', 'nasty', 'gross']
}


def classify_aspect_to_journey(aspect, opinion):
    """將屬性分類到顧客旅程階段"""

    aspect_lower = aspect.lower()
    opinion_lower = opinion.lower()
    combined = f"{aspect_lower} {opinion_lower}"

    # 優先檢查具體菜品
    menu_keywords = CUSTOMER_JOURNEY_MAPPING['menu_items']['keywords']
    for keyword in menu_keywords:
        if keyword in aspect_lower:
            return 'menu_items'

    # 檢查飲品
    beverage_keywords = CUSTOMER_JOURNEY_MAPPING['beverages']['keywords']
    for keyword in beverage_keywords:
        if keyword in aspect_lower:
            return 'beverages'

    # 檢查服務相關（包含人名）
    service_keywords = CUSTOMER_JOURNEY_MAPPING['service_experience']['keywords']
    person_names = CUSTOMER_JOURNEY_MAPPING['service_experience'].get('person_names', [])
    for keyword in service_keywords:
        if keyword in aspect_lower:
            return 'service_experience'
    for name in person_names:
        if name in combined:
            return 'service_experience'

    # 檢查其他階段
    for stage, config in CUSTOMER_JOURNEY_MAPPING.items():
        if stage in ['menu_items', 'beverages', 'service_experience']:
            continue
        for keyword in config['keywords']:
            if keyword in aspect_lower or keyword in opinion_lower:
                return stage

    # 默認歸類到食物品質
    return 'food_quality'


def get_sentiment_intensity(opinion):
    """獲取情感強度"""

    opinion_lower = opinion.lower()

    for intensity, keywords in SENTIMENT_SUBCATEGORIES.items():
        for keyword in keywords:
            if keyword in opinion_lower:
                return intensity

    return 'neutral'


def add_journey_classification(df):
    """為資料添加顧客旅程分類"""

    df['journey_stage'] = df.apply(
        lambda row: classify_aspect_to_journey(row['aspect_cleaned'], row['opinion']),
        axis=1
    )

    df['sentiment_intensity'] = df['opinion'].apply(get_sentiment_intensity)

    return df


# =============================================================================
# 第四部分：屬性標準化與聚合
# =============================================================================

# 屬性同義詞映射
ASPECT_SYNONYMS = {
    # 服務人員
    'server': ['server', 'waiter', 'waitress', 'waitstaff'],
    'staff': ['staff', 'employee', 'team', 'crew'],
    'manager': ['manager', 'gm', 'general manager'],
    'bartender': ['bartender', 'barman', 'barmaid'],
    'host': ['host', 'hostess', 'greeter'],

    # 食物總類
    'food': ['food', 'meal', 'dish', 'cuisine'],

    # 具體菜品
    'chicken_and_waffles': ['chicken and waffles', 'waffles', 'chicken waffles',
                            'waffle', 'hot chicken'],
    'brisket': ['brisket', 'beef brisket', 'smoked brisket'],
    'nachos': ['nachos', 'brisket nachos', 'nacho'],
    'wings': ['wings', 'chicken wings', 'wing'],
    'burger': ['burger', 'hamburger', 'cheeseburger'],
    'fries': ['fries', 'french fries', 'fry'],
    'tacos': ['tacos', 'taco', 'pulled pork tacos'],
    'mac_and_cheese': ['mac', 'mac and cheese', 'macaroni'],

    # 飲品
    'drinks': ['drinks', 'drink', 'beverages', 'beverage'],
    'cocktails': ['cocktail', 'cocktails', 'mixed drink'],
    'beer': ['beer', 'beers', 'draft', 'ale'],
    'moonshine': ['moonshine', 'shine'],
    'mimosa': ['mimosa', 'mimosas'],

    # 環境
    'atmosphere': ['atmosphere', 'vibe', 'ambiance', 'ambience', 'environment'],
    'decor': ['decor', 'decoration', 'interior', 'design'],
    'seating': ['seating', 'seat', 'seats', 'booth', 'table', 'tables'],

    # 服務屬性
    'service': ['service'],
    'wait_time': ['wait', 'waiting', 'wait time', 'waited'],

    # 價格
    'price': ['price', 'prices', 'cost', 'pricing'],
    'value': ['value', 'worth', 'deal'],

    # 其他
    'location': ['location', 'place', 'spot'],
    'parking': ['parking', 'park'],
    'cleanliness': ['clean', 'cleanliness', 'dirty', 'sticky']
}


def standardize_aspect(aspect):
    """標準化屬性名稱"""

    aspect_lower = aspect.lower().strip()

    for standard_name, synonyms in ASPECT_SYNONYMS.items():
        for synonym in synonyms:
            if synonym in aspect_lower or aspect_lower in synonym:
                return standard_name

    return aspect_lower


def aggregate_aspects(df):
    """聚合相似屬性"""

    df['aspect_standardized'] = df['aspect_cleaned'].apply(standardize_aspect)

    return df


# =============================================================================
# 第五部分：深度洞察分析
# =============================================================================

def generate_deep_insights(df):
    """生成深度洞察報告"""

    insights = {}

    # 1. 整體統計
    insights['overall'] = {
        'total_reviews': df['review_id'].nunique(),
        'total_mentions': len(df),
        'positive_count': len(df[df['sentiment'] == 'Positive']),
        'negative_count': len(df[df['sentiment'] == 'Negative']),
        'positive_rate': len(df[df['sentiment'] == 'Positive']) / len(df) * 100
    }

    # 2. 顧客旅程階段分析
    journey_analysis = {}
    for stage in CUSTOMER_JOURNEY_MAPPING.keys():
        stage_df = df[df['journey_stage'] == stage]
        if len(stage_df) > 0:
            journey_analysis[stage] = {
                'total_mentions': len(stage_df),
                'positive_count': len(stage_df[stage_df['sentiment'] == 'Positive']),
                'negative_count': len(stage_df[stage_df['sentiment'] == 'Negative']),
                'satisfaction_rate': len(stage_df[stage_df['sentiment'] == 'Positive']) / len(stage_df) * 100,
                'top_positive_aspects': stage_df[stage_df['sentiment'] == 'Positive']['aspect_standardized'].value_counts().head(5).to_dict(),
                'top_negative_aspects': stage_df[stage_df['sentiment'] == 'Negative']['aspect_standardized'].value_counts().head(5).to_dict(),
                'sample_positive_opinions': stage_df[stage_df['sentiment'] == 'Positive'][['aspect_cleaned', 'opinion']].head(3).values.tolist(),
                'sample_negative_opinions': stage_df[stage_df['sentiment'] == 'Negative'][['aspect_cleaned', 'opinion']].head(3).values.tolist()
            }
    insights['journey_analysis'] = journey_analysis

    # 3. 熱門菜品分析
    menu_df = df[df['journey_stage'] == 'menu_items']
    menu_analysis = {}

    for aspect in menu_df['aspect_standardized'].unique():
        aspect_df = menu_df[menu_df['aspect_standardized'] == aspect]
        if len(aspect_df) >= 3:  # 至少3次提及
            pos_count = len(aspect_df[aspect_df['sentiment'] == 'Positive'])
            neg_count = len(aspect_df[aspect_df['sentiment'] == 'Negative'])

            menu_analysis[aspect] = {
                'total_mentions': len(aspect_df),
                'positive_count': pos_count,
                'negative_count': neg_count,
                'satisfaction_rate': pos_count / len(aspect_df) * 100 if len(aspect_df) > 0 else 0,
                'positive_opinions': aspect_df[aspect_df['sentiment'] == 'Positive']['opinion'].tolist()[:5],
                'negative_opinions': aspect_df[aspect_df['sentiment'] == 'Negative']['opinion'].tolist()[:5]
            }

    insights['menu_analysis'] = dict(sorted(menu_analysis.items(),
                                            key=lambda x: x[1]['total_mentions'],
                                            reverse=True))

    # 4. 服務人員分析
    service_df = df[df['journey_stage'] == 'service_experience']

    # 提取提到的服務人員名字
    person_names = CUSTOMER_JOURNEY_MAPPING['service_experience'].get('person_names', [])
    staff_mentions = defaultdict(lambda: {'positive': 0, 'negative': 0, 'opinions': []})

    for _, row in service_df.iterrows():
        combined = f"{row['aspect_cleaned']} {row['opinion']}".lower()
        for name in person_names:
            if name in combined:
                if row['sentiment'] == 'Positive':
                    staff_mentions[name]['positive'] += 1
                else:
                    staff_mentions[name]['negative'] += 1
                staff_mentions[name]['opinions'].append(row['opinion'][:50])

    insights['staff_mentions'] = dict(staff_mentions)

    # 5. 問題熱點分析
    negative_df = df[df['sentiment'] == 'Negative']

    # 按旅程階段統計負面評價
    problem_hotspots = {}
    for stage in CUSTOMER_JOURNEY_MAPPING.keys():
        stage_neg = negative_df[negative_df['journey_stage'] == stage]
        if len(stage_neg) > 0:
            problem_hotspots[stage] = {
                'count': len(stage_neg),
                'percentage': len(stage_neg) / len(negative_df) * 100,
                'top_issues': stage_neg['aspect_standardized'].value_counts().head(5).to_dict(),
                'sample_complaints': stage_neg[['aspect_cleaned', 'opinion']].head(5).values.tolist()
            }

    insights['problem_hotspots'] = dict(sorted(problem_hotspots.items(),
                                               key=lambda x: x[1]['count'],
                                               reverse=True))

    # 6. 情感強度分析
    intensity_analysis = df.groupby(['sentiment_intensity', 'sentiment']).size().unstack(fill_value=0)
    insights['sentiment_intensity'] = intensity_analysis.to_dict()

    return insights


# =============================================================================
# 第六部分：報告生成
# =============================================================================

def generate_text_report(df, insights, output_file):
    """生成詳細文字報告"""

    report = []

    def add_line(text=""):
        report.append(text)

    # 標題
    add_line("=" * 80)
    add_line("餐廳評論深度洞察分析報告")
    add_line(f"分析時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    add_line("=" * 80)

    # 1. 執行摘要
    add_line("\n" + "=" * 80)
    add_line("📊 執行摘要 (Executive Summary)")
    add_line("=" * 80)

    overall = insights['overall']
    add_line(f"\n總評論數: {overall['total_reviews']} 則")
    add_line(f"有效屬性-意見對: {overall['total_mentions']} 組")
    add_line(f"正面評價: {overall['positive_count']} ({overall['positive_rate']:.1f}%)")
    add_line(f"負面評價: {overall['negative_count']} ({100 - overall['positive_rate']:.1f}%)")

    # 關鍵發現
    add_line("\n🔑 關鍵發現:")

    # 找出最佳和最差的旅程階段
    journey = insights['journey_analysis']
    best_stage = max(journey.items(), key=lambda x: x[1]['satisfaction_rate'])
    worst_stage = min(journey.items(), key=lambda x: x[1]['satisfaction_rate'])

    stage_names_zh = {
        'visit_motivation': '來店動機',
        'first_impression': '第一印象',
        'service_experience': '服務體驗',
        'food_quality': '食物品質',
        'menu_items': '具體菜品',
        'beverages': '飲品',
        'value_perception': '價值感知',
        'facilities': '環境設施',
        'overall_experience': '整體體驗'
    }

    add_line(f"   ✅ 最佳體驗環節: {stage_names_zh.get(best_stage[0], best_stage[0])} "
             f"(滿意度 {best_stage[1]['satisfaction_rate']:.1f}%)")
    add_line(f"   ⚠️ 最需改進環節: {stage_names_zh.get(worst_stage[0], worst_stage[0])} "
             f"(滿意度 {worst_stage[1]['satisfaction_rate']:.1f}%)")

    # 2. 顧客旅程分析
    add_line("\n" + "=" * 80)
    add_line("🚶 顧客旅程分析 (Customer Journey Analysis)")
    add_line("=" * 80)

    for stage, data in sorted(journey.items(), key=lambda x: x[1]['total_mentions'], reverse=True):
        stage_name = stage_names_zh.get(stage, stage)
        add_line(f"\n{'─' * 60}")
        add_line(f"📍 {stage_name}")
        add_line(f"{'─' * 60}")
        add_line(f"   提及次數: {data['total_mentions']} | "
                 f"正面: {data['positive_count']} | "
                 f"負面: {data['negative_count']} | "
                 f"滿意度: {data['satisfaction_rate']:.1f}%")

        if data['top_positive_aspects']:
            add_line(f"\n   ✅ 正面評價焦點:")
            for aspect, count in list(data['top_positive_aspects'].items())[:3]:
                add_line(f"      • {aspect}: {count}次")

        if data['sample_positive_opinions']:
            add_line(f"   📝 正面評價範例:")
            for aspect, opinion in data['sample_positive_opinions'][:2]:
                add_line(f"      「{aspect}: {opinion[:60]}...」" if len(opinion) > 60 else f"      「{aspect}: {opinion}」")

        if data['top_negative_aspects']:
            add_line(f"\n   ⚠️ 負面評價焦點:")
            for aspect, count in list(data['top_negative_aspects'].items())[:3]:
                add_line(f"      • {aspect}: {count}次")

        if data['sample_negative_opinions']:
            add_line(f"   📝 負面評價範例:")
            for aspect, opinion in data['sample_negative_opinions'][:2]:
                add_line(f"      「{aspect}: {opinion[:60]}...」" if len(opinion) > 60 else f"      「{aspect}: {opinion}」")

    # 3. 菜品詳細分析
    add_line("\n" + "=" * 80)
    add_line("🍽️ 菜品詳細分析 (Menu Items Analysis)")
    add_line("=" * 80)

    menu = insights['menu_analysis']

    add_line(f"\n{'菜品':<25} {'提及':<8} {'正面':<8} {'負面':<8} {'滿意度':<10}")
    add_line("-" * 60)

    for item, data in list(menu.items())[:15]:
        sat_rate = f"{data['satisfaction_rate']:.1f}%"
        add_line(f"{item:<25} {data['total_mentions']:<8} {data['positive_count']:<8} "
                 f"{data['negative_count']:<8} {sat_rate:<10}")

    # 菜品深度洞察
    add_line("\n📋 菜品深度洞察:")

    for item, data in list(menu.items())[:8]:
        if data['total_mentions'] >= 5:
            add_line(f"\n   🍴 {item.upper().replace('_', ' ')}")
            add_line(f"      滿意度: {data['satisfaction_rate']:.1f}% ({data['total_mentions']}次提及)")

            if data['positive_opinions']:
                add_line(f"      ✅ 顧客讚賞:")
                for op in data['positive_opinions'][:3]:
                    add_line(f"         • {op[:70]}..." if len(op) > 70 else f"         • {op}")

            if data['negative_opinions']:
                add_line(f"      ⚠️ 顧客抱怨:")
                for op in data['negative_opinions'][:3]:
                    add_line(f"         • {op[:70]}..." if len(op) > 70 else f"         • {op}")

    # 4. 服務人員分析
    add_line("\n" + "=" * 80)
    add_line("👥 服務人員分析 (Staff Analysis)")
    add_line("=" * 80)

    staff = insights['staff_mentions']
    if staff:
        add_line(f"\n{'服務人員':<15} {'正面':<10} {'負面':<10} {'總計':<10}")
        add_line("-" * 50)

        for name, data in sorted(staff.items(), key=lambda x: x[1]['positive'] + x[1]['negative'], reverse=True):
            total = data['positive'] + data['negative']
            if total >= 2:
                add_line(f"{name.title():<15} {data['positive']:<10} {data['negative']:<10} {total:<10}")

        # 表現優異的員工
        top_performers = [(name, data) for name, data in staff.items()
                          if data['positive'] >= 3 and data['negative'] == 0]
        if top_performers:
            add_line("\n   🌟 表現優異員工:")
            for name, data in top_performers:
                add_line(f"      • {name.title()}: {data['positive']} 次正面提及")

    # 5. 問題熱點分析
    add_line("\n" + "=" * 80)
    add_line("🚨 問題熱點分析 (Problem Hotspots)")
    add_line("=" * 80)

    hotspots = insights['problem_hotspots']

    add_line(f"\n{'環節':<20} {'負面數':<10} {'佔比':<10} {'主要問題':<30}")
    add_line("-" * 70)

    for stage, data in list(hotspots.items())[:5]:
        stage_name = stage_names_zh.get(stage, stage)
        top_issue = list(data['top_issues'].keys())[0] if data['top_issues'] else 'N/A'
        add_line(f"{stage_name:<20} {data['count']:<10} {data['percentage']:.1f}%{'':<5} {top_issue:<30}")

    # 詳細問題分析
    add_line("\n📋 詳細問題分析:")

    for stage, data in list(hotspots.items())[:3]:
        stage_name = stage_names_zh.get(stage, stage)
        add_line(f"\n   🔴 {stage_name} ({data['count']}個負面評價)")

        add_line(f"      主要問題:")
        for issue, count in list(data['top_issues'].items())[:5]:
            add_line(f"         • {issue}: {count}次")

        add_line(f"      典型抱怨:")
        for aspect, opinion in data['sample_complaints'][:3]:
            add_line(f"         「{aspect}: {opinion[:60]}...」" if len(opinion) > 60 else f"         「{aspect}: {opinion}」")

    # 6. 行動建議
    add_line("\n" + "=" * 80)
    add_line("💡 行動建議 (Action Recommendations)")
    add_line("=" * 80)

    add_line("\n🔴 緊急改進 (需立即處理):")

    # 基於分析結果生成建議
    if 'service_experience' in hotspots and hotspots['service_experience']['count'] > 20:
        add_line("   1. 【服務速度】")
        add_line("      • 問題: 服務速度慢是最常見抱怨")
        add_line("      • 建議: 增加尖峰時段人手、優化點餐流程、設定服務時間標準")
        add_line("      • KPI: 將平均等待時間降低 30%")

    if 'food_quality' in hotspots:
        add_line("   2. 【食物品質一致性】")
        add_line("      • 問題: 食物品質不穩定（過乾/過淡/過冷）")
        add_line("      • 建議: 加強廚房品管、建立出餐前檢查機制")
        add_line("      • KPI: 將食物相關負評降低 40%")

    add_line("\n🟡 中期優化 (1-3個月):")
    add_line("   3. 【員工培訓】")
    add_line("      • 針對負面提及的員工進行服務培訓")
    add_line("      • 表揚並學習表現優異員工的服務方式")

    add_line("   4. 【菜單優化】")
    add_line("      • 強化高滿意度菜品的行銷")
    add_line("      • 檢討低滿意度菜品的食譜或下架")

    add_line("\n🟢 長期策略 (3-6個月):")
    add_line("   5. 【顧客體驗整體提升】")
    add_line("      • 建立顧客回饋追蹤系統")
    add_line("      • 定期進行滿意度調查")
    add_line("      • 設計忠誠顧客獎勵計畫")

    # 保存報告
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report))

    print(f"\n✅ 報告已保存到: {output_file}")

    return '\n'.join(report)


def generate_excel_report(df, insights, output_file):
    """生成 Excel 報告"""

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

        # Sheet 1: 原始資料（去重後）
        df_export = df[['review_id', 'aspect_cleaned', 'opinion', 'sentiment',
                        'journey_stage', 'aspect_standardized', 'sentiment_intensity']].copy()
        df_export.columns = ['評論ID', '屬性', '意見', '情感', '旅程階段', '標準化屬性', '情感強度']
        df_export.to_excel(writer, sheet_name='原始資料', index=False)

        # Sheet 2: 旅程階段統計
        journey_stats = []
        stage_names_zh = {
            'visit_motivation': '來店動機',
            'first_impression': '第一印象',
            'service_experience': '服務體驗',
            'food_quality': '食物品質',
            'menu_items': '具體菜品',
            'beverages': '飲品',
            'value_perception': '價值感知',
            'facilities': '環境設施',
            'overall_experience': '整體體驗'
        }

        for stage, data in insights['journey_analysis'].items():
            journey_stats.append({
                '旅程階段': stage_names_zh.get(stage, stage),
                '階段代碼': stage,
                '提及次數': data['total_mentions'],
                '正面評價': data['positive_count'],
                '負面評價': data['negative_count'],
                '滿意度(%)': round(data['satisfaction_rate'], 1)
            })

        df_journey = pd.DataFrame(journey_stats)
        df_journey = df_journey.sort_values('提及次數', ascending=False)
        df_journey.to_excel(writer, sheet_name='旅程階段統計', index=False)

        # Sheet 3: 菜品分析
        menu_stats = []
        for item, data in insights['menu_analysis'].items():
            menu_stats.append({
                '菜品': item.replace('_', ' ').title(),
                '提及次數': data['total_mentions'],
                '正面評價': data['positive_count'],
                '負面評價': data['negative_count'],
                '滿意度(%)': round(data['satisfaction_rate'], 1),
                '正面評價範例': ' | '.join(data['positive_opinions'][:3]),
                '負面評價範例': ' | '.join(data['negative_opinions'][:3])
            })

        df_menu = pd.DataFrame(menu_stats)
        df_menu.to_excel(writer, sheet_name='菜品分析', index=False)

        # Sheet 4: 屬性統計
        aspect_stats = df.groupby(['aspect_standardized', 'sentiment']).size().unstack(fill_value=0)
        aspect_stats['總計'] = aspect_stats.sum(axis=1)
        aspect_stats['滿意度(%)'] = (aspect_stats.get('Positive', 0) / aspect_stats['總計'] * 100).round(1)
        aspect_stats = aspect_stats.sort_values('總計', ascending=False)
        aspect_stats.to_excel(writer, sheet_name='屬性統計')

        # Sheet 5: 負面評價詳情
        df_negative = df[df['sentiment'] == 'Negative'][['review_id', 'aspect_cleaned', 'opinion', 'journey_stage']].copy()
        df_negative.columns = ['評論ID', '屬性', '意見', '旅程階段']
        df_negative.to_excel(writer, sheet_name='負面評價詳情', index=False)

        # Sheet 6: 正面評價詳情
        df_positive = df[df['sentiment'] == 'Positive'][['review_id', 'aspect_cleaned', 'opinion', 'journey_stage']].copy()
        df_positive.columns = ['評論ID', '屬性', '意見', '旅程階段']
        df_positive.to_excel(writer, sheet_name='正面評價詳情', index=False)

        # Sheet 7: 摘要統計
        summary_data = {
            '指標': ['總評論數', '有效屬性-意見對', '正面評價數', '負面評價數',
                    '正面評價比例(%)', '負面評價比例(%)', '平均每則評論屬性數'],
            '數值': [
                insights['overall']['total_reviews'],
                insights['overall']['total_mentions'],
                insights['overall']['positive_count'],
                insights['overall']['negative_count'],
                round(insights['overall']['positive_rate'], 1),
                round(100 - insights['overall']['positive_rate'], 1),
                round(insights['overall']['total_mentions'] / insights['overall']['total_reviews'], 2)
            ]
        }
        df_summary = pd.DataFrame(summary_data)
        df_summary.to_excel(writer, sheet_name='摘要統計', index=False)

    print(f"✅ Excel 報告已保存到: {output_file}")


# =============================================================================
# 主程式
# =============================================================================

def main(v3_result_file, output_prefix="absa_deep_analysis"):
    """主程式"""

    print("=" * 60)
    print("ABSA 深度洞察分析系統 V4")
    print("=" * 60)

    # 1. 解析 V3 結果
    print("\n[1/5] 解析 V3 結果文件...")
    df = parse_v3_results(v3_result_file)
    print(f"   解析完成，共 {len(df)} 筆資料")

    # 2. 資料清理與去重
    print("\n[2/5] 資料清理與去重...")
    df = clean_and_deduplicate(df)

    # 3. 顧客旅程分類
    print("\n[3/5] 顧客旅程分類...")
    df = add_journey_classification(df)
    df = aggregate_aspects(df)
    print(f"   分類完成")

    # 4. 深度洞察分析
    print("\n[4/5] 深度洞察分析...")
    insights = generate_deep_insights(df)
    print(f"   分析完成")

    # 5. 生成報告
    print("\n[5/5] 生成報告...")

    text_report_file = f"{output_prefix}_report.txt"
    excel_report_file = f"{output_prefix}_report.xlsx"

    generate_text_report(df, insights, text_report_file)
    generate_excel_report(df, insights, excel_report_file)

    print("\n" + "=" * 60)
    print("✅ 分析完成！")
    print("=" * 60)
    print(f"\n輸出文件:")
    print(f"   📄 文字報告: {text_report_file}")
    print(f"   📊 Excel報告: {excel_report_file}")

    return df, insights


# =============================================================================
# 執行
# =============================================================================

if __name__ == "__main__":
    # 請修改為您的 V3 結果文件路徑
    V3_RESULT_FILE = "absa_analysis_results_v3.txt"
    OUTPUT_PREFIX = "absa_deep_analysis"

    df, insights = main(V3_RESULT_FILE, OUTPUT_PREFIX)

    # 顯示快速摘要
    print("\n" + "=" * 60)
    print("快速摘要")
    print("=" * 60)
    print(f"總評論數: {insights['overall']['total_reviews']}")
    print(f"有效屬性-意見對: {insights['overall']['total_mentions']}")
    print(f"正面評價: {insights['overall']['positive_count']} ({insights['overall']['positive_rate']:.1f}%)")
    print(f"負面評價: {insights['overall']['negative_count']} ({100-insights['overall']['positive_rate']:.1f}%)")

In [ ]:
"""
================================================================================
ABSA (Aspect-Based Sentiment Analysis) System - V4 Universal Edition
================================================================================

Purpose:
    This system extracts aspect-opinion-sentiment triplets from customer reviews.
    It is designed to be domain-agnostic and can be applied to restaurants, hotels,
    products, services, or any other review domain.

Key Features:
    1. Aspect extraction and validation
    2. Opinion extraction with boundary detection
    3. Multi-source sentiment analysis (VADER, TextBlob, Transformer)
    4. Sentiment correction using rules and sentiment scores
    5. Comprehensive reporting and DataFrame export

Version: 4.0 Universal
Author: AI Assistant
Date: 2025
================================================================================
"""

import warnings
warnings.filterwarnings('ignore')

import re
from collections import Counter
from datetime import datetime

# ============================================
# SECTION 1: GLOBAL CONFIGURATION
# ============================================
"""
Why this section exists:
    - Centralizes all configuration parameters in one place
    - Makes it easy to adjust settings without modifying code logic
    - Allows users to quickly customize the system for their needs
"""

OUTPUT_FILE = "absa_analysis_results_v4_universal.txt"

# Maximum words per chunk when splitting long reviews
# Why 80: Balance between context preservation and model performance
MAX_WORDS_PER_CHUNK = 80

# Minimum confidence threshold for sentiment analysis correction
# Why 0.6: Prevents over-correction when sentiment analysis is uncertain
SENTIMENT_CONFIDENCE_THRESHOLD = 0.6

# Sentiment score thresholds for override decisions
# Why 0.4/-0.4: Strong enough to indicate clear sentiment mismatch
SENTIMENT_OVERRIDE_THRESHOLD = 0.4


# ============================================
# SECTION 2: SENTIMENT ANALYZER INITIALIZATION
# ============================================
"""
Why this section exists:
    - Initializes multiple sentiment analysis tools for ensemble approach
    - Ensemble methods are more robust than single-source analysis
    - Gracefully handles missing dependencies (not all tools are required)

Why multiple analyzers:
    - VADER: Fast, rule-based, good for social media/reviews
    - TextBlob: Simple, pattern-based, provides subjectivity scores
    - Transformer: Deep learning, most accurate but slower
    - Using all three provides more reliable results through consensus
"""

# Global variable to store initialized analyzers (singleton pattern)
# Why singleton: Avoid re-initializing expensive models multiple times
SENTIMENT_ANALYZERS = None


def initialize_sentiment_analyzers():
    """
    Initialize all available sentiment analysis tools.

    Why this function exists:
        - Centralizes all sentiment tool initialization
        - Handles import errors gracefully
        - Provides feedback on which tools are available

    Returns:
        dict: Dictionary containing initialized analyzers (or None if unavailable)
    """
    analyzers = {}

    # ----- VADER Initialization -----
    # Why VADER: Specifically designed for social media and reviews
    # Handles emoticons, slang, and intensity modifiers well
    try:
        import nltk
        # Download lexicon quietly (won't re-download if exists)
        nltk.download('vader_lexicon', quiet=True)
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        analyzers['vader'] = SentimentIntensityAnalyzer()
        print("✅ VADER sentiment analyzer loaded successfully")
    except Exception as e:
        print(f"⚠️ VADER not available: {e}")
        analyzers['vader'] = None

    # ----- TextBlob Initialization -----
    # Why TextBlob: Provides both polarity AND subjectivity scores
    # Subjectivity helps identify factual vs. opinion statements
    try:
        from textblob import TextBlob
        analyzers['textblob'] = TextBlob
        print("✅ TextBlob sentiment analyzer loaded successfully")
    except Exception as e:
        print(f"⚠️ TextBlob not available: {e}")
        analyzers['textblob'] = None

    # ----- Transformer Initialization (Optional) -----
    # Why Transformer: Most accurate for complex sentiment patterns
    # Why optional: Requires significant memory and is slower
    try:
        from transformers import pipeline
        # Use DistilBERT for balance between accuracy and speed
        # device=-1 forces CPU usage (more compatible)
        analyzers['transformer'] = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=-1
        )
        print("✅ Transformer sentiment analyzer loaded successfully")
    except Exception as e:
        print(f"ℹ️ Transformer not available (optional): {e}")
        analyzers['transformer'] = None

    return analyzers


def get_sentiment_analyzers():
    """
    Get or initialize sentiment analyzers (singleton pattern).

    Why singleton pattern:
        - Analyzers are expensive to initialize (especially Transformer)
        - Reusing the same instances improves performance
        - Ensures consistent behavior across the application

    Returns:
        dict: Dictionary of initialized sentiment analyzers
    """
    global SENTIMENT_ANALYZERS
    if SENTIMENT_ANALYZERS is None:
        SENTIMENT_ANALYZERS = initialize_sentiment_analyzers()
    return SENTIMENT_ANALYZERS


# ============================================
# SECTION 3: UNIVERSAL WORD LISTS
# ============================================
"""
Why this section exists:
    - Provides domain-agnostic word lists for validation and analysis
    - Separates data from logic for easier maintenance
    - Uses functions instead of global variables for lazy loading

Design principle:
    - All word lists are UNIVERSAL and not specific to any business
    - Can be easily extended for specific domains if needed
"""


def get_common_english_names():
    """
    Get a comprehensive list of common English first names.

    Why this function exists:
        - Person names in reviews usually refer to staff members
        - Names alone don't provide sentiment information
        - Filtering names improves opinion extraction quality

    Why not use NER (Named Entity Recognition):
        - NER is slower and requires additional dependencies
        - This list covers 95%+ of common cases
        - Can be combined with NER for higher accuracy if needed

    Returns:
        set: Set of common English first names (lowercase)
    """
    # Male names (top 100 US names)
    male_names = {
        'james', 'john', 'robert', 'michael', 'william', 'david', 'richard',
        'joseph', 'thomas', 'charles', 'christopher', 'daniel', 'matthew',
        'anthony', 'mark', 'donald', 'steven', 'paul', 'andrew', 'joshua',
        'kenneth', 'kevin', 'brian', 'george', 'timothy', 'ronald', 'edward',
        'jason', 'jeffrey', 'ryan', 'jacob', 'gary', 'nicholas', 'eric',
        'jonathan', 'stephen', 'larry', 'justin', 'scott', 'brandon', 'benjamin',
        'samuel', 'raymond', 'gregory', 'frank', 'alexander', 'patrick', 'jack',
        'dennis', 'jerry', 'tyler', 'aaron', 'jose', 'adam', 'nathan', 'henry',
        'douglas', 'zachary', 'peter', 'kyle', 'noah', 'ethan', 'jeremy',
        'walter', 'christian', 'keith', 'roger', 'terry', 'austin', 'sean',
        'gerald', 'carl', 'harold', 'dylan', 'arthur', 'lawrence', 'jordan',
        'jesse', 'bryan', 'billy', 'bruce', 'gabriel', 'joe', 'logan', 'albert',
        'willie', 'alan', 'eugene', 'russell', 'vincent', 'philip', 'bobby',
        'johnny', 'bradley', 'roy', 'ralph', 'randy', 'wayne', 'elijah',
        'ray', 'nick', 'andy', 'greg', 'mike', 'steve', 'tom', 'bob', 'jim', 'dan'
    }

    # Female names (top 100 US names)
    female_names = {
        'mary', 'patricia', 'jennifer', 'linda', 'barbara', 'elizabeth', 'susan',
        'jessica', 'sarah', 'karen', 'lisa', 'nancy', 'betty', 'margaret', 'sandra',
        'ashley', 'kimberly', 'emily', 'donna', 'michelle', 'dorothy', 'carol',
        'amanda', 'melissa', 'deborah', 'stephanie', 'rebecca', 'sharon', 'laura',
        'cynthia', 'kathleen', 'amy', 'angela', 'shirley', 'anna', 'brenda',
        'pamela', 'emma', 'nicole', 'helen', 'samantha', 'katherine', 'christine',
        'debra', 'rachel', 'carolyn', 'janet', 'catherine', 'maria', 'heather',
        'diane', 'ruth', 'julie', 'olivia', 'joyce', 'virginia', 'victoria',
        'kelly', 'lauren', 'christina', 'joan', 'evelyn', 'judith', 'megan',
        'andrea', 'cheryl', 'hannah', 'jacqueline', 'martha', 'gloria', 'teresa',
        'ann', 'sara', 'madison', 'frances', 'kathryn', 'janice', 'jean', 'abigail',
        'alice', 'judy', 'sophia', 'grace', 'denise', 'amber', 'doris', 'marilyn',
        'danielle', 'beverly', 'isabella', 'theresa', 'diana', 'natalie', 'brittany',
        'charlotte', 'marie', 'kayla', 'alexis', 'lori', 'tina', 'zoe',
        'melanie', 'morgan', 'veronica', 'alyssa', 'kristina', 'cameron', 'gabby'
    }

    return male_names | female_names


def get_positive_sentiment_words():
    """
    Get universal positive sentiment indicator words.

    Why this function exists:
        - Provides a reliable set of words that indicate positive sentiment
        - Used for sentiment correction when model predictions seem wrong
        - Organized by category for easier maintenance

    Design principle:
        - Only includes words that are UNIVERSALLY positive
        - Avoids domain-specific terms (no "delicious" for non-food domains)
        - Wait, for reviews, food terms are still universal across restaurants

    Returns:
        set: Set of positive sentiment words (lowercase)
    """
    return {
        # ----- General Quality -----
        # Why: These words universally indicate high quality
        'good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic',
        'outstanding', 'superb', 'perfect', 'incredible', 'awesome',
        'terrific', 'fabulous', 'marvelous', 'exceptional', 'best',
        'impressive', 'remarkable', 'magnificent', 'splendid', 'brilliant',

        # ----- Food-Related (Universal for F&B) -----
        # Why: Common across all restaurant reviews regardless of cuisine
        'delicious', 'tasty', 'yummy', 'flavorful', 'savory', 'fresh',
        'tender', 'juicy', 'crispy', 'creamy', 'rich', 'light',
        'authentic', 'homemade', 'seasoned', 'aromatic', 'scrumptious',

        # ----- Service-Related -----
        # Why: Universal for any service industry (restaurants, hotels, retail)
        'friendly', 'helpful', 'attentive', 'professional', 'courteous',
        'prompt', 'efficient', 'welcoming', 'accommodating', 'polite',
        'responsive', 'knowledgeable', 'patient', 'thorough', 'dedicated',

        # ----- Environment/Atmosphere -----
        # Why: Applies to any physical location
        'clean', 'comfortable', 'cozy', 'spacious', 'nice', 'lovely',
        'beautiful', 'charming', 'pleasant', 'relaxing', 'quiet',
        'modern', 'elegant', 'stylish', 'inviting', 'warm',

        # ----- Value-Related -----
        # Why: Universal concern for all consumers
        'reasonable', 'affordable', 'worth', 'value', 'cheap', 'bargain',
        'fair', 'inexpensive', 'economical'
    }


def get_negative_sentiment_words():
    """
    Get universal negative sentiment indicator words.

    Why this function exists:
        - Provides a reliable set of words that indicate negative sentiment
        - Used for sentiment correction when model predictions seem wrong
        - Mirrors the structure of positive words for consistency

    Returns:
        set: Set of negative sentiment words (lowercase)
    """
    return {
        # ----- General Quality -----
        # Why: These words universally indicate poor quality
        'bad', 'terrible', 'awful', 'horrible', 'poor', 'disappointing',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross', 'worst',
        'inferior', 'subpar', 'unacceptable', 'dreadful', 'appalling',

        # ----- Food-Related (Universal for F&B) -----
        # Why: Common food complaints across all cuisines
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty', 'stale',
        'cold', 'lukewarm', 'frozen', 'rubbery', 'tough', 'chewy',
        'flavorless', 'mushy', 'bitter', 'sour', 'spoiled',

        # ----- Service-Related -----
        # Why: Universal service complaints
        'rude', 'slow', 'unfriendly', 'unhelpful', 'inattentive',
        'unprofessional', 'careless', 'dismissive', 'indifferent',
        'incompetent', 'negligent', 'impolite', 'disrespectful',

        # ----- Environment/Atmosphere -----
        # Why: Universal environmental complaints
        'dirty', 'messy', 'noisy', 'crowded', 'cramped', 'uncomfortable',
        'dark', 'smelly', 'hot', 'stuffy', 'outdated', 'dingy',
        'filthy', 'cluttered', 'rundown', 'shabby',

        # ----- Value-Related -----
        # Why: Universal pricing complaints
        'expensive', 'overpriced', 'costly', 'pricey', 'ripoff',
        'unreasonable', 'exorbitant'
    }


def get_action_verbs():
    """
    Get a list of action verbs that should not be treated as opinions.

    Why this function exists:
        - Action verbs describe what happened, not how someone felt
        - "I ordered the pizza" is factual, not an opinion
        - Filtering these improves opinion extraction accuracy

    Why include all tenses:
        - Reviews use various tenses ("ordered", "ordering", "orders")
        - Comprehensive coverage prevents false positives

    Returns:
        set: Set of action verbs in various forms (lowercase)
    """
    # Base verbs with their conjugations
    return {
        # Ordering/Requesting
        'order', 'ordered', 'ordering', 'orders',
        'ask', 'asked', 'asking', 'asks',
        'request', 'requested', 'requesting', 'requests',

        # Coming/Going
        'come', 'came', 'coming', 'comes',
        'go', 'went', 'going', 'goes', 'gone',
        'arrive', 'arrived', 'arriving', 'arrives',
        'leave', 'left', 'leaving', 'leaves',
        'walk', 'walked', 'walking', 'walks',
        'enter', 'entered', 'entering', 'enters',
        'visit', 'visited', 'visiting', 'visits',

        # Getting/Taking
        'get', 'got', 'getting', 'gets',
        'take', 'took', 'taking', 'takes', 'taken',
        'bring', 'brought', 'bringing', 'brings',
        'receive', 'received', 'receiving', 'receives',

        # Making/Doing
        'make', 'made', 'making', 'makes',
        'do', 'did', 'doing', 'does', 'done',
        'prepare', 'prepared', 'preparing', 'prepares',
        'cook', 'cooked', 'cooking', 'cooks',

        # Communication
        'say', 'said', 'saying', 'says',
        'tell', 'told', 'telling', 'tells',
        'call', 'called', 'calling', 'calls',
        'inform', 'informed', 'informing', 'informs',
        'mention', 'mentioned', 'mentioning', 'mentions',

        # Service actions
        'serve', 'served', 'serving', 'serves',
        'show', 'showed', 'showing', 'shows', 'shown',
        'give', 'gave', 'giving', 'gives', 'given',
        'seat', 'seated', 'seating', 'seats',

        # Customer actions
        'try', 'tried', 'trying', 'tries',
        'want', 'wanted', 'wanting', 'wants',
        'need', 'needed', 'needing', 'needs',
        'hope', 'hoped', 'hoping', 'hopes',
        'expect', 'expected', 'expecting', 'expects',
        'enjoy', 'enjoyed', 'enjoying', 'enjoys',
        'pay', 'paid', 'paying', 'pays',
        'wait', 'waited', 'waiting', 'waits',
        'sit', 'sat', 'sitting', 'sits',
        'eat', 'ate', 'eating', 'eats', 'eaten',
        'drink', 'drank', 'drinking', 'drinks', 'drunk'
    }


def get_function_words():
    """
    Get a list of function words that carry no sentiment meaning.

    Why this function exists:
        - Function words (articles, prepositions, etc.) have grammatical roles
        - They don't express opinions or sentiment
        - Filtering these improves signal-to-noise ratio

    Returns:
        set: Set of function words (lowercase)
    """
    return {
        # Articles
        'a', 'an', 'the',

        # Demonstratives
        'this', 'that', 'these', 'those',

        # Pronouns
        'i', 'me', 'my', 'mine', 'myself',
        'you', 'your', 'yours', 'yourself',
        'he', 'him', 'his', 'himself',
        'she', 'her', 'hers', 'herself',
        'it', 'its', 'itself',
        'we', 'us', 'our', 'ours', 'ourselves',
        'they', 'them', 'their', 'theirs', 'themselves',

        # Prepositions
        'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
        'from', 'about', 'into', 'through', 'during', 'before',
        'after', 'above', 'below', 'between', 'under', 'over',

        # Conjunctions
        'and', 'or', 'but', 'so', 'yet', 'nor',
        'if', 'when', 'where', 'while', 'as', 'because', 'since',
        'although', 'though', 'unless', 'until',

        # Adverbs of degree (without sentiment)
        'only', 'just', 'also', 'too', 'even', 'still',
        'already', 'always', 'never', 'ever', 'often',

        # Auxiliary verbs
        'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'has', 'have', 'had', 'having',
        'do', 'does', 'did',
        'will', 'would', 'shall', 'should',
        'can', 'could', 'may', 'might', 'must',

        # Other function words
        'here', 'there', 'some', 'any', 'no', 'every',
        'each', 'both', 'all', 'most', 'many', 'much',
        'few', 'little', 'other', 'another', 'such'
    }


def get_intensifiers():
    """
    Get a list of intensifier words that modify sentiment strength.

    Why this function exists:
        - Intensifiers amplify or reduce sentiment (e.g., "very good" vs "good")
        - They are important for sentiment analysis but aren't sentiments themselves
        - Recognizing them helps in proper opinion extraction

    Returns:
        set: Set of intensifier words (lowercase)
    """
    return {
        # Strong intensifiers (amplify)
        'very', 'extremely', 'incredibly', 'amazingly', 'exceptionally',
        'remarkably', 'absolutely', 'totally', 'completely', 'utterly',
        'entirely', 'thoroughly', 'perfectly', 'highly', 'deeply',
        'truly', 'really', 'genuinely', 'seriously',

        # Moderate intensifiers
        'quite', 'rather', 'fairly', 'pretty', 'somewhat',
        'reasonably', 'moderately', 'relatively',

        # Weak intensifiers (reduce)
        'slightly', 'a bit', 'a little', 'mildly', 'barely',
        'hardly', 'scarcely',

        # Excess intensifiers (often negative context)
        'too', 'overly', 'excessively',

        # Emphatic
        'so', 'such', 'super', 'especially', 'particularly'
    }


# ============================================
# SECTION 4: ADJECTIVE/ADVERB DETECTION
# ============================================
"""
Why this section exists:
    - Adjectives and adverbs are the primary carriers of sentiment
    - "The food was DELICIOUS" - "delicious" is the opinion
    - Accurate detection enables better opinion extraction
"""


def is_adjective_or_adverb(word):
    """
    Determine if a word is an adjective or adverb.

    Why this function exists:
        - Adjectives and adverbs typically express opinions
        - This helps identify sentiment-bearing words
        - Uses both dictionary lookup and morphological rules

    Why two approaches (dictionary + suffix):
        - Dictionary: High precision for known words
        - Suffix rules: Catches new/uncommon words (e.g., "instagrammable")

    Args:
        word (str): The word to check

    Returns:
        tuple: (is_adj_or_adv: bool, word_type: str)
               word_type: 'positive_adj', 'negative_adj', 'intensifier',
                         'suffix_adj', 'suffix_adv', 'none'
    """
    # Clean the word
    word_lower = word.lower().strip('.,!?;:\'"')

    if not word_lower:
        return False, 'none'

    # Check against known positive adjectives
    positive_words = get_positive_sentiment_words()
    if word_lower in positive_words:
        return True, 'positive_adj'

    # Check against known negative adjectives
    negative_words = get_negative_sentiment_words()
    if word_lower in negative_words:
        return True, 'negative_adj'

    # Check against intensifiers
    intensifiers = get_intensifiers()
    if word_lower in intensifiers:
        return True, 'intensifier'

    # ----- Morphological Rules -----
    # Why: Catches adjectives/adverbs not in our dictionaries

    # Common adjective suffixes
    # Why these: Highly reliable indicators of adjective status
    adj_suffixes = [
        'ful',   # beautiful, wonderful
        'less',  # careless, tasteless
        'ous',   # delicious, gorgeous
        'ive',   # impressive, attentive
        'able',  # comfortable, reasonable
        'ible',  # incredible, accessible
        'al',    # exceptional, professional
        'ic',    # fantastic, authentic
        'ish',   # smallish, reddish
        'ent',   # excellent, different
        'ant',   # pleasant, elegant
        'ory',   # satisfactory
        'ary',   # ordinary, customary
    ]

    # Check adjective suffixes
    for suffix in adj_suffixes:
        # Why len check: Avoid false positives like "of" matching "ful"
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    # Common adverb suffix
    # Why just 'ly': This is the primary English adverb marker
    if word_lower.endswith('ly') and len(word_lower) > 4:
        # Exclude words that end in 'ly' but aren't adverbs
        non_adverb_ly = {'family', 'only', 'early', 'likely', 'friendly', 'lonely'}
        if word_lower not in non_adverb_ly:
            return True, 'suffix_adv'

    # Past participles used as adjectives (e.g., "disappointed", "satisfied")
    # Why: These are common in reviews ("I was disappointed")
    if word_lower.endswith('ed') and len(word_lower) > 4:
        # Check if it's a sentiment-bearing participle
        sentiment_participles = {
            'disappointed', 'satisfied', 'pleased', 'impressed', 'amazed',
            'surprised', 'disgusted', 'frustrated', 'annoyed', 'delighted',
            'thrilled', 'excited', 'bored', 'tired', 'exhausted',
            'overwhelmed', 'underwhelmed', 'overpriced'
        }
        if word_lower in sentiment_participles:
            return True, 'suffix_adj'

    # Present participles used as adjectives (e.g., "amazing", "disappointing")
    if word_lower.endswith('ing') and len(word_lower) > 5:
        sentiment_ing = {
            'amazing', 'disappointing', 'disgusting', 'interesting', 'boring',
            'exciting', 'frustrating', 'annoying', 'satisfying', 'refreshing',
            'relaxing', 'welcoming', 'inviting', 'appealing', 'appalling'
        }
        if word_lower in sentiment_ing:
            return True, 'suffix_adj'

    return False, 'none'


# ============================================
# SECTION 5: NEGATION DETECTION
# ============================================
"""
Why this section exists:
    - Negation completely reverses sentiment meaning
    - "The food was NOT good" = negative (despite "good" being positive)
    - Accurate negation detection is critical for sentiment accuracy

Common negation challenges:
    - Contractions: "wasn't", "didn't", "isn't"
    - Tokenized contractions: "was", "n't" (separate tokens)
    - Implicit negation: "lack of", "without", "fail to"
"""


def detect_negation_in_opinion(opinion):
    """
    Detect if an opinion contains negation.

    Why this function exists:
        - Negation flips sentiment polarity
        - "good" is positive, but "not good" is negative
        - Must handle various negation forms

    Why multiple detection methods:
        - Direct: "not", "no", "never"
        - Contractions: "didn't", "wasn't"
        - Tokenized: "did n't" (some tokenizers split these)
        - Implicit: "lack of", "without"

    Args:
        opinion (str): The opinion text to check

    Returns:
        tuple: (has_negation: bool, negation_type: str or None)
    """
    opinion_lower = opinion.lower()

    # ----- Method 1: Direct negation words -----
    # Why list form: Need to match as complete words or with punctuation
    direct_negations = [
        'not ', "n't ", 'no ', 'never ', 'none ', 'nothing ',
        'neither ', 'nobody ', 'nowhere ', 'cannot '
    ]

    for neg in direct_negations:
        if neg in opinion_lower or opinion_lower.startswith(neg.strip()):
            return True, 'direct'

    # ----- Method 2: Contracted negations -----
    # Why: Very common in informal reviews ("wasn't", "didn't")
    contracted_negations = [
        "didn't", "didn't", "did not",
        "wasn't", "wasn't", "was not",
        "weren't", "weren't", "were not",
        "isn't", "isn't", "is not",
        "aren't", "aren't", "are not",
        "don't", "don't", "do not",
        "doesn't", "doesn't", "does not",
        "won't", "won't", "will not",
        "wouldn't", "wouldn't", "would not",
        "couldn't", "couldn't", "could not",
        "shouldn't", "shouldn't", "should not",
        "can't", "can't", "cannot",
        "haven't", "haven't", "have not",
        "hasn't", "hasn't", "has not",
        "hadn't", "hadn't", "had not"
    ]

    for neg in contracted_negations:
        if neg in opinion_lower:
            return True, 'contraction'

    # ----- Method 3: Tokenized contractions -----
    # Why: Some NLP tools split "didn't" into ["did", "n't"]
    tokenized_patterns = [
        (r"\bdidn?\s*['']\s*t\b", "didn't"),
        (r"\bwasn?\s*['']\s*t\b", "wasn't"),
        (r"\bweren?\s*['']\s*t\b", "weren't"),
        (r"\bisn?\s*['']\s*t\b", "isn't"),
        (r"\baren?\s*['']\s*t\b", "aren't"),
        (r"\bdon?\s*['']\s*t\b", "don't"),
        (r"\bdoesn?\s*['']\s*t\b", "doesn't"),
        (r"\bwon?\s*['']\s*t\b", "won't"),
        (r"\bwouldn?\s*['']\s*t\b", "wouldn't"),
        (r"\bcouldn?\s*['']\s*t\b", "couldn't"),
        (r"\bshouldn?\s*['']\s*t\b", "shouldn't"),
        (r"\bcan?\s*['']\s*t\b", "can't"),
    ]

    for pattern, _ in tokenized_patterns:
        if re.search(pattern, opinion_lower):
            return True, 'tokenized'

    # ----- Method 4: Implicit negation -----
    # Why: Some phrases imply negation without using "not"
    implicit_negations = [
        'lack of', 'lacking', 'lacks',
        'without', 'absent', 'absence of',
        'missing', 'miss', 'missed',
        'fail to', 'failed to', 'fails to',
        'unable to', 'incapable of',
        'devoid of', 'free of',  # context-dependent
        'no sign of', 'no trace of'
    ]

    for phrase in implicit_negations:
        if phrase in opinion_lower:
            return True, 'implicit'

    return False, None


# ============================================
# SECTION 6: SENTENCE BOUNDARY DETECTION
# ============================================
"""
Why this section exists:
    - Aspects and their opinions are typically within the same sentence
    - Crossing sentence boundaries often leads to incorrect pairings
    - Example: "The food was good. The service was slow."
      - "food" should pair with "good", not "slow"
"""


def find_sentence_boundaries(tokens, aspect_end_idx):
    """
    Find the sentence boundaries containing a given aspect.

    Why this function exists:
        - Limits opinion search to the same sentence as the aspect
        - Prevents cross-sentence opinion misattribution
        - Improves extraction precision

    Args:
        tokens (list): List of tokens from the review
        aspect_end_idx (int): Index of the last token of the aspect

    Returns:
        tuple: (sentence_start_idx, sentence_end_idx)
    """
    # Sentence-ending punctuation
    # Why these: Standard English sentence terminators
    sentence_enders = {'.', '!', '?'}

    # Find sentence start (search backwards)
    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in sentence_enders:
            sentence_start = i + 1  # Start after the punctuation
            break

    # Find sentence end (search forwards)
    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in sentence_enders:
            sentence_end = i  # End at the punctuation (exclusive)
            break

    return sentence_start, sentence_end


def find_previous_aspect_end(current_start_idx, all_aspect_positions):
    """
    Find the end position of the previous aspect in the sentence.

    Why this function exists:
        - Prevents opinion words from being attributed to wrong aspects
        - Example: "great food and terrible service"
          - "great" belongs to "food", not to "service"
        - Sets a left boundary for opinion extraction

    Args:
        current_start_idx (int): Start index of current aspect
        all_aspect_positions (list): Positions of all aspects

    Returns:
        int or None: End index of previous aspect, or None if no previous aspect
    """
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            # Handle nested position lists
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            # Check if this aspect comes before current and is the closest
            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(current_end_idx, all_aspect_positions):
    """
    Find the start position of the next aspect in the sentence.

    Why this function exists:
        - Prevents opinion extraction from bleeding into next aspect's territory
        - Sets a right boundary for opinion extraction
        - Example: "The food was good and the service was slow"
          - Opinion extraction for "food" should stop before "service"

    Args:
        current_end_idx (int): End index of current aspect
        all_aspect_positions (list): Positions of all aspects

    Returns:
        int or None: Start index of next aspect, or None if no next aspect
    """
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            # Handle nested position lists
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            # Check if this aspect comes after current and is the closest
            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# SECTION 7: BE-VERB DETECTION
# ============================================
"""
Why this section exists:
    - "Be" verbs connect aspects to their descriptors
    - "The food IS delicious" - "is" connects "food" to "delicious"
    - Detecting be-verbs helps identify the "aspect IS opinion" pattern
    - Negated be-verbs are especially important ("isn't", "wasn't")
"""


def detect_be_verb(tokens, start_idx):
    """
    Detect if a be-verb appears at the given position.

    Why this function exists:
        - Be-verbs are key connectors in "X is Y" patterns
        - Negated be-verbs ("isn't", "wasn't") flip sentiment
        - This is one of the most common opinion expression patterns

    Args:
        tokens (list): List of tokens
        start_idx (int): Index to check for be-verb

    Returns:
        tuple: (found: bool, skip_count: int, is_negative: bool)
               skip_count: Number of tokens to skip (for tokenized forms)
    """
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    # Standard be-verbs (non-negated)
    be_verbs = {'is', 'was', 'were', 'are', 'be', 'been', "'s", "s"}
    if word in be_verbs:
        return True, 1, False

    # Contracted negated be-verbs
    negated_be_verbs = {
        "isn't", "isnt", "isn't",
        "wasn't", "wasnt", "wasn't",
        "weren't", "werent", "weren't",
        "aren't", "arent", "aren't"
    }
    if word in negated_be_verbs:
        return True, 1, True

    # Tokenized negated forms: "isn", "'", "t"
    # Why: Some tokenizers split contractions
    partial_negated = {'isn', 'wasn', 'weren', 'aren'}
    if word in partial_negated and start_idx + 2 < len(tokens):
        if tokens[start_idx + 1] in {"'", "'"} and tokens[start_idx + 2].lower() in {'t', 'nt'}:
            return True, 3, True  # Skip 3 tokens

    return False, 0, False


# ============================================
# SECTION 8: OPINION VALIDATION
# ============================================
"""
Why this section exists:
    - Not all extracted text is a valid opinion
    - "I ordered the pizza" - "ordered the pizza" is not an opinion
    - "The pizza was delicious" - "delicious" IS an opinion
    - Validation filters out non-opinion extractions
"""


def is_valid_opinion(opinion):
    """
    Validate whether extracted text is a genuine opinion.

    Why this function exists:
        - Filters out non-opinion extractions
        - Improves precision of the extraction system
        - Removes meaningless fragments

    Validation criteria:
        1. Not empty or too short
        2. Not just action verbs
        3. Not just function words
        4. Not just person names
        5. Contains at least one sentiment-bearing word
        6. Doesn't match invalid patterns

    Args:
        opinion (str): The extracted opinion text

    Returns:
        bool: True if the opinion is valid, False otherwise
    """
    # Basic checks
    if not opinion or len(opinion.strip()) < 2:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    if not words:
        return False

    # Get word lists
    action_verbs = get_action_verbs()
    function_words = get_function_words()
    common_names = get_common_english_names()
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()
    all_sentiment_words = positive_words | negative_words

    # ----- Check 1: Single word validation -----
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')

        # Reject if it's just an action verb
        if clean_word in action_verbs:
            return False

        # Reject if it's just a function word
        if clean_word in function_words:
            return False

        # Reject if it's just a name
        if clean_word in common_names:
            return False

        # Single word should be an adjective/adverb
        is_adj, _ = is_adjective_or_adverb(clean_word)
        if not is_adj:
            return False

    # ----- Check 2: Two-word validation -----
    if len(words) == 2:
        clean_words = [w.strip('.,!?;:\'"') for w in words]

        # Reject if both are action verbs or function words
        non_content_words = action_verbs | function_words
        if all(w in non_content_words for w in clean_words):
            return False

        # Reject if both are names
        if all(w in common_names for w in clean_words):
            return False

        # Reject "verb + conditional" patterns
        conditionals = {'if', 'when', 'would', 'could', 'should', 'might', 'may'}
        if clean_words[0] in action_verbs and clean_words[1] in conditionals:
            return False

    # ----- Check 3: All action verbs -----
    clean_words = [w.strip('.,!?;:\'"') for w in words]
    action_count = sum(1 for w in clean_words if w in action_verbs)
    if action_count == len(words):
        return False

    # ----- Check 4: Only person names -----
    if len(words) <= 3:
        non_name_words = [w for w in clean_words if w not in common_names]
        if len(non_name_words) == 0:
            return False

    # ----- Check 5: Invalid patterns -----
    # Why regex: Catches specific grammatical fragments that aren't opinions
    invalid_patterns = [
        r"^'s\s+",                    # 's name, 's order (possessive fragment)
        r"^\d+\s*/",                  # 5 /10 (rating fragment)
        r"^or\s+maybe\s+both$",       # Indecisive fragment
        r"^going\s+forward$",         # Time reference
        r"^hoped\s+would$",           # Incomplete thought
        r"^asked\s+if$",              # Incomplete question
        r"^came\s+out\s+and\s+it$",   # Incomplete description
        r"^for\s+dessert$",           # Category, not opinion
        r"^to\s+before\s+me$",        # Grammatical fragment
        r"^and\s+waited$",            # Action fragment
        r"^member\s+where",           # Incomplete clause
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    # ----- Check 6: Contains sentiment word -----
    # Why: Opinions should express some sentiment
    has_sentiment = False

    for word in clean_words:
        # Check against sentiment dictionaries
        if word in all_sentiment_words:
            has_sentiment = True
            break

        # Check if it's an adjective/adverb
        is_adj, _ = is_adjective_or_adverb(word)
        if is_adj:
            has_sentiment = True
            break

    # Short opinions without sentiment words are likely invalid
    # Unless they contain negation (which modifies meaning)
    if len(words) <= 3 and not has_sentiment:
        has_negation, _ = detect_negation_in_opinion(opinion_lower)
        if not has_negation:
            return False

    return True


# ============================================
# SECTION 9: ASPECT VALIDATION AND REFINEMENT
# ============================================
"""
Why this section exists:
    - Not all extracted aspects are valid product/service attributes
    - "She" is not an aspect; "service" is an aspect
    - Validation ensures extracted aspects are meaningful
"""


def is_valid_aspect(aspect):
    """
    Validate whether extracted text is a genuine aspect.

    Why this function exists:
        - Filters out non-aspect extractions
        - Aspects should be nouns representing product/service attributes
        - Pronouns, verbs, and function words are not aspects

    Args:
        aspect (str): The extracted aspect text

    Returns:
        bool: True if the aspect is valid, False otherwise
    """
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    # Empty or punctuation-only
    if not cleaned or cleaned == "-":
        return False

    if all(c in '.,!?;:-_\'"' for c in cleaned):
        return False

    # Starts with conjunction (incomplete phrase)
    if cleaned.startswith('and ') or cleaned.startswith('or '):
        return False

    # Invalid aspect words
    invalid_aspects = {
        # ----- Verbs (actions, not attributes) -----
        'wait', 'waited', 'waiting',
        'order', 'ordered', 'ordering',
        'serve', 'served', 'serving',
        'ask', 'asked', 'asking',
        'walk', 'walked', 'walking',
        'make', 'made', 'making',
        'have', 'having', 'had',
        'come', 'came', 'coming',
        'go', 'went', 'going',

        # ----- Pronouns (references, not attributes) -----
        'i', 'me', 'my', 'mine',
        'you', 'your', 'yours',
        'he', 'him', 'his',
        'she', 'her', 'hers',
        'it', 'its',
        'we', 'us', 'our', 'ours',
        'they', 'them', 'their', 'theirs',
        'someone', 'anyone', 'everyone', 'nobody',
        'something', 'anything', 'everything', 'nothing',

        # ----- Articles (function words) -----
        'the', 'a', 'an',

        # ----- Time units (measurements, not attributes) -----
        'min', 'mins', 'minute', 'minutes',
        'hour', 'hours',
        'second', 'seconds',
        'day', 'days',
        'week', 'weeks',
        'month', 'months',
        'year', 'years',

        # ----- Pure adjectives (opinions, not aspects) -----
        # Why: These describe aspects, they are not aspects themselves
        'good', 'bad', 'great', 'nice', 'poor',
        'fast', 'slow', 'hot', 'cold', 'new', 'old',
        'big', 'small', 'large', 'little',

        # ----- Generic words (too vague) -----
        'thing', 'things', 'stuff',
        'way', 'ways',
        'time', 'times',
        'place', 'places',
        'part', 'parts'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """
    Clean and standardize an aspect term.

    Why this function exists:
        - Removes noise from extracted aspects
        - Standardizes format for better aggregation
        - Example: "the Food" -> "food", "and service" -> "service"

    Args:
        aspect (str): The raw extracted aspect

    Returns:
        str: Cleaned aspect term
    """
    if not aspect:
        return ""

    clean_aspect = aspect.strip()

    # Remove leading conjunctions
    # Why: Incomplete phrases like "and service" should become "service"
    clean_aspect = re.sub(r'^(and|or|but)\s+', '', clean_aspect, flags=re.IGNORECASE)

    # Remove measurement units
    # Why: "10-inch pizza" should focus on "pizza", not the size
    units_pattern = r'^(?:[\d\.]+\s*-?\s*)?(?:inch|inches|cm|mm|kg|lbs|lb|oz|g|ml)\b\s*'
    clean_aspect = re.sub(units_pattern, '', clean_aspect, flags=re.IGNORECASE)

    # Remove leading articles and determiners
    # Why: "the food" should be normalized to "food"
    stopwords_pattern = r'^(the|a|an|this|that|my|our|their|his|her|your|some|any)\s+'
    clean_aspect = re.sub(stopwords_pattern, '', clean_aspect, flags=re.IGNORECASE)

    # Remove leading punctuation
    clean_aspect = re.sub(r'^[-:;,.\'"]+\s*', '', clean_aspect)

    # Remove trailing punctuation
    clean_aspect = re.sub(r'[-:;,.\'"]+$', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# SECTION 10: OPINION EXTRACTION
# ============================================
"""
Why this section exists:
    - This is the core algorithm for extracting opinions associated with aspects
    - Handles multiple patterns:
      1. Pre-modifier: "delicious food" - opinion before aspect
      2. Be-adjective: "food is delicious" - opinion after be-verb
      3. Verb phrase: "food tastes amazing" - opinion in verb phrase
      4. Context search: fallback for complex sentences
"""


def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """
    Extract the opinion associated with an aspect, respecting boundaries.

    Why this function exists:
        - Core function for pairing aspects with their opinions
        - Uses multiple extraction patterns for comprehensive coverage
        - Respects sentence and aspect boundaries to avoid misattribution

    Why multiple patterns:
        - Different grammatical structures require different approaches
        - No single pattern catches all opinion expressions
        - Prioritized order: pre-modifier > be-adjective > verb phrase > context

    Args:
        tokens (list): Tokenized review text
        aspect_positions (list): Token positions of the aspect
        aspect_text (str): The aspect text (for reference)
        all_aspect_positions (list): Positions of all aspects (for boundary detection)

    Returns:
        tuple: (opinion_phrase: str, pattern_type: str)
    """
    # Parse aspect positions
    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1] if len(aspect_positions) > 1 else aspect_positions[0]
    else:
        return "", "none"

    # Get boundaries
    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)
    prev_aspect_end = find_previous_aspect_end(start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(end_idx, all_aspect_positions)

    # Adjust sentence end based on next aspect
    # Why: Don't extract beyond the next aspect's territory
    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # ===== PATTERN 1: Pre-modifier =====
    # Example: "delicious food", "very friendly staff"
    # Why first: Most direct opinion-aspect relationship
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)  # Look up to 4 words back

    # Respect previous aspect boundary
    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, modifier_type = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in {',', 'and', 'with', 'or', 'but'}:
            # Reset on separators - these might start a new phrase
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

        # Check for trailing noun (e.g., "great dining experience")
        # Why: Sometimes the full opinion includes a noun after the aspect
        next_pos = end_idx + 1
        if next_pos < sentence_end:
            next_word = tokens[next_pos]
            # Words that shouldn't follow
            forbidden_next = {
                'is', 'was', 'are', 'were', 'be', 'been',
                'has', 'have', 'had', 'does', 'do',
                'but', 'and', 'or', 'so', 'because',
                'with', 'for', 'to', 'in', 'at', 'on', 'of', 'by',
                'that', 'which', 'who', 'this', 'it',
                ',', '.', '!', '?', ';', '-'
            }
            if next_word.isalpha() and next_word.lower() not in forbidden_next:
                opinion_words.append(next_word)

    # ===== PATTERN 2: Be-verb + Adjective =====
    # Example: "food is delicious", "service was excellent"
    # Why: Very common pattern in reviews
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:  # Only if no pre-modifiers found
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)  # Max 15 words

        post_modifiers = []

        # Add negation if detected
        if is_negative:
            post_modifiers.append("not")

        for i in range(opinion_start, opinion_end):
            word = tokens[i]

            # Stop at sentence enders
            if word in {'.', '!', '?', ';'}:
                break

            # Stop at 'but' if it leads to next aspect
            if word.lower() == 'but' and next_aspect_start is not None:
                if i + 1 < len(tokens) and i + 1 >= next_aspect_start - 2:
                    break

            post_modifiers.append(word)

        if post_modifiers:
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # ===== PATTERN 3: Verb Phrase =====
    # Example: "food tastes great", "service took forever"
    # Why: Catches opinions expressed through action verbs
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)  # Max 8 words

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            temp_words.append(word)

        if temp_words:
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # ===== PATTERN 4: Context Search (Fallback) =====
    # Why: Last resort for complex sentences
    # Searches for any adjective/adverb near the aspect
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, _ = is_adjective_or_adverb(word)

            if is_modifier:
                # Found an adjective - get context around it
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                context_words = []
                for j in range(context_start, context_end):
                    if tokens[j] not in {'.', '!', '?', ';'}:
                        context_words.append(tokens[j])

                opinion_words.extend(context_words)
                pattern_type = "context_search"
                break

            # Stop at sentence boundaries
            if word in {'.', '!', '?'}:
                break

    # Handle "that + negation" patterns
    # Example: "quality that isn't great" -> "not great"
    opinion_words = handle_that_with_negation(opinion_words)

    opinion_phrase = ' '.join(opinion_words).strip()

    return opinion_phrase, pattern_type


def handle_that_with_negation(words):
    """
    Process "that + negated be-verb" patterns.

    Why this function exists:
        - "quality that isn't great" should capture negation
        - Transforms complex negation patterns to simple form
        - Example: "that isn't" -> removes "that", keeps negation

    Args:
        words (list): List of words to process

    Returns:
        list: Processed word list
    """
    if not words:
        return words

    result = []
    i = 0

    while i < len(words):
        word = words[i]
        word_lower = word.lower()

        if word_lower == 'that' and i + 1 < len(words):
            # Check if followed by negated be-verb
            next_word = words[i + 1].lower()
            negated_be = {"isn't", "isn't", "wasn't", "wasn't", "aren't", "aren't", "weren't", "weren't"}

            if next_word in negated_be:
                # Skip "that", add "not" to capture negation
                result.append("not")
                i += 2  # Skip both "that" and the negated verb
                continue

        result.append(word)
        i += 1

    return result


# ============================================
# SECTION 11: OPINION FORMATTING
# ============================================
"""
Why this section exists:
    - Raw extracted opinions often need cleaning
    - "the food was , extremely - delicious" -> "extremely delicious"
    - Formatting improves readability and analysis quality
"""


def format_opinion_for_display(opinion):
    """
    Format and clean an extracted opinion for display.

    Why this function exists:
        - Raw extractions contain noise (extra punctuation, fragments)
        - Consistent formatting improves report quality
        - Removes trailing garbage words

    Processing steps:
        1. Fix hyphenation and punctuation
        2. Remove leading filler words
        3. Handle word merging for hyphenated terms
        4. Truncate at appropriate points
        5. Remove trailing garbage words
        6. Handle connectors (but, and, however)
        7. Enforce length limits

    Args:
        opinion (str): Raw extracted opinion

    Returns:
        str: Cleaned and formatted opinion
    """
    if not opinion:
        return ""

    # ----- Step 1: Fix basic punctuation issues -----
    # Fix camelCase hyphenation
    opinion = re.sub(r'([a-z])-(?=[A-Z])', r'\1 - ', opinion)
    # Fix double commas
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    # Standardize hyphens
    opinion = re.sub(r'([a-zA-Z0-9])-(?=[a-zA-Z0-9])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    words = opinion.split()
    if not words:
        return ""

    # ----- Step 2: Remove leading filler words -----
    # Why: These don't contribute to the opinion meaning
    leading_fillers = {
        # Articles and demonstratives
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        # Adverbs
        'also', 'too', 'either',
        # Verbs
        'has', 'have', 'had', 'is', 'are', 'was', 'were',
        'looks', 'feels', 'seems',
        'contains', 'includes', 'consists',
        'offers', 'provides', 'features',
        # Pronouns
        'it', 'itself', 'they', 'themselves', 'we', 'ourselves',
        'i', 'myself', 'you', 'yourself'
    }

    while words and words[0].lower() in leading_fillers:
        words.pop(0)

    if not words:
        return ""

    # ----- Step 3: Handle hyphenated word merging -----
    # Why: "well - known" should become "well-known"
    do_not_merge = {
        'a', 'an', 'the', 'it', 'is', 'we', 'i', 'you', 'he', 'she',
        'of', 'in', 'to', 'for', 'with', 'on', 'at', 'my', 'our'
    }

    cleaned_words = []
    i = 0
    while i < len(words):
        word = words[i]

        # Check for "word-" pattern
        if i + 1 < len(words):
            next_word = words[i + 1]
            if word.endswith('-') and len(word) > 1:
                if next_word.isdigit() or next_word.lower() in do_not_merge:
                    cleaned_words.append(word.rstrip('-'))
                else:
                    # Merge: "well-" + "known" = "well-known"
                    merged = word + next_word
                    cleaned_words.append(merged)
                    i += 2
                    continue
            elif next_word == '-' and i + 2 < len(words):
                following = words[i + 2]
                if following.isdigit() or following.lower() in do_not_merge:
                    cleaned_words.append(word)
                    cleaned_words.append('-')
                    i += 2
                    continue
                # Merge: "well" + "-" + "known" = "well-known"
                merged = word + "-" + following
                cleaned_words.append(merged)
                i += 3
                continue

        cleaned_words.append(word)
        i += 1

    words = cleaned_words

    # ----- Step 4: Truncate at cutoff points -----
    # Why: Certain words indicate the opinion is complete
    cutoff_words = {
        'making', 'causing', 'forcing', 'leaving', 'rendering',
        'unless', 'except', 'besides', 'despite', 'although',
        'which', 'where', 'when', 'because', 'since'
    }

    truncated = []
    valid_length = 0

    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')

        # Handle dash
        if word == '-' or word.startswith('-'):
            if i + 1 < len(words):
                next_w = words[i + 1].lower()
                # Stop if dash is followed by pronoun (new clause)
                if next_w in {'i', 'we', 'it', 'he', 'she', 'they'}:
                    break
            truncated.append(word)
            continue

        # Stop at cutoff words (if we have enough content)
        if word_lower in cutoff_words and valid_length >= 2:
            break

        truncated.append(word)
        if word not in {',', 'and', 'but', '-'}:
            valid_length += 1

    words = truncated

    # ----- Step 5: Fix comma placement -----
    final_words = []
    for word in words:
        if word == ',':
            if final_words and not final_words[-1].endswith(','):
                final_words[-1] = final_words[-1] + ','
        else:
            clean_word = word.lstrip(',')
            if clean_word:
                final_words.append(clean_word)

    words = final_words

    # ----- Step 6: Remove trailing garbage words -----
    # Why: Incomplete phrases at the end should be removed
    trailing_garbage = {
        # Conjunctions and prepositions
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'too', 'either', 'also', 'but', 'however',
        # Articles
        'a', 'an', 'the',
        # Punctuation
        ',', '-',
        # Verbs
        'is', 'are', 'was', 'were',
        'need', 'justify', 'make', 'do', 'want', 'require', 'expect',
        'produces', 'provides', 'offers', 'features', 'includes',
        # Pronouns
        "'", '"', '(', 'it', 'i', 'we', 'they', 'he', 'she'
    }

    # Important nouns that should not be removed
    # Why: These complete the opinion meaning
    important_nouns = {
        'experience', 'quality', 'service', 'food', 'price',
        'taste', 'flavor', 'portion', 'value', 'atmosphere',
        'staff', 'server', 'waiter', 'waitress', 'manager',
        'location', 'ambiance', 'decor', 'menu', 'selection',
        'meal', 'dish', 'order', 'drink', 'dessert'
    }

    while words:
        last_word = words[-1].lower().strip('.,!?;:')

        # Don't remove important nouns
        if last_word in important_nouns:
            break

        # Check if it's trailing garbage
        should_remove = False
        if last_word in trailing_garbage:
            should_remove = True
        elif words[-1].endswith('-'):
            should_remove = True
        elif words[-1].isdigit():
            should_remove = True

        # Remove trailing comma
        if words[-1].endswith(','):
            words[-1] = words[-1].rstrip(',')
            if not words[-1]:
                words.pop()
                continue

        if should_remove:
            words.pop()
        else:
            break

    # ----- Step 7: Handle connectors -----
    # Why: "good but" -> "good" (incomplete comparison)
    result_text = ' '.join(words)
    connectors = [' but ', ' however ', ' though ', ' and ']

    for connector in connectors:
        if connector in result_text.lower():
            last_idx = result_text.lower().rfind(connector)
            before = result_text[:last_idx].strip()
            after = result_text[last_idx + len(connector):].strip().split()

            # Remove if nothing meaningful follows
            should_cut = False
            if not after:
                should_cut = True
            elif len(after) <= 2:
                bad_enders = {'i', 'it', 'we', 'he', 'she', 'they', 'a', 'an', 'the'}
                if any(w.lower() in bad_enders for w in after):
                    should_cut = True

            # Exception: keep comparatives
            comparatives = {
                'more', 'less', 'better', 'worse', 'higher', 'lower',
                'deeper', 'brighter', 'darker', 'faster', 'slower',
                'larger', 'smaller', 'stronger', 'weaker'
            }
            if any(w.lower() in comparatives for w in after):
                should_cut = False

            if should_cut:
                result_text = before

    # ----- Step 8: Length limit -----
    # Why: Very long opinions are usually extraction errors
    words = result_text.split()
    if len(words) > 12 and ' but ' not in result_text.lower():
        words = words[:12]
        result_text = ' '.join(words)

    # Final cleanup
    result_text = re.sub(r',\s*,', ',', result_text)
    result_text = result_text.strip().rstrip('.,;-\'"(')

    return result_text


# ============================================
# SECTION 12: SENTIMENT ANALYSIS
# ============================================
"""
Why this section exists:
    - Multiple sentiment sources provide more robust analysis
    - Ensemble methods reduce individual analyzer errors
    - Provides confidence scores for decision making
"""


def analyze_sentiment_scores(text):
    """
    Analyze text sentiment using multiple tools.

    Why this function exists:
        - Single sentiment analyzer can be wrong
        - Ensemble of multiple analyzers is more reliable
        - Provides detailed scores for nuanced analysis

    Why these specific analyzers:
        - VADER: Fast, handles informal text well
        - TextBlob: Provides subjectivity alongside polarity
        - Transformer: Most accurate for complex patterns

    Args:
        text (str): Text to analyze

    Returns:
        dict: Sentiment scores from all analyzers and ensemble
    """
    analyzers = get_sentiment_analyzers()
    results = {}

    # Handle empty text
    if not text or len(text.strip()) < 2:
        return {
            'vader': None,
            'textblob': None,
            'transformer': None,
            'ensemble': {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}
        }

    # ----- VADER Analysis -----
    if analyzers.get('vader'):
        try:
            vader_scores = analyzers['vader'].polarity_scores(text)
            results['vader'] = {
                'compound': vader_scores['compound'],  # -1 to 1
                'positive': vader_scores['pos'],       # 0 to 1
                'negative': vader_scores['neg'],       # 0 to 1
                'neutral': vader_scores['neu']         # 0 to 1
            }
        except Exception:
            results['vader'] = None
    else:
        results['vader'] = None

    # ----- TextBlob Analysis -----
    if analyzers.get('textblob'):
        try:
            blob = analyzers['textblob'](text)
            results['textblob'] = {
                'polarity': blob.sentiment.polarity,         # -1 to 1
                'subjectivity': blob.sentiment.subjectivity  # 0 to 1
            }
        except Exception:
            results['textblob'] = None
    else:
        results['textblob'] = None

    # ----- Transformer Analysis -----
    if analyzers.get('transformer'):
        try:
            # Truncate for transformer (max 512 tokens typically)
            truncated = text[:500] if len(text) > 500 else text
            trans_result = analyzers['transformer'](truncated)[0]

            # Convert to -1 to 1 scale
            if trans_result['label'] == 'POSITIVE':
                normalized = trans_result['score']
            else:
                normalized = -trans_result['score']

            results['transformer'] = {
                'label': trans_result['label'],
                'score': trans_result['score'],
                'normalized_score': normalized
            }
        except Exception:
            results['transformer'] = None
    else:
        results['transformer'] = None

    # ----- Calculate Ensemble Score -----
    results['ensemble'] = calculate_ensemble_score(results)

    return results


def calculate_ensemble_score(sentiment_results):
    """
    Calculate weighted ensemble sentiment score.

    Why this function exists:
        - Combines multiple analyzer outputs into single score
        - Uses weighted average for better reliability
        - Provides confidence based on analyzer agreement

    Weights rationale:
        - VADER (0.4): Best for social media/reviews, most reliable for this domain
        - TextBlob (0.3): Good general performance
        - Transformer (0.3): Most accurate but can overfit

    Args:
        sentiment_results (dict): Results from individual analyzers

    Returns:
        dict: Ensemble score, label, and confidence
    """
    scores = []
    weights = []

    # Collect available scores with their weights
    if sentiment_results.get('vader') and sentiment_results['vader'].get('compound') is not None:
        scores.append(sentiment_results['vader']['compound'])
        weights.append(0.4)  # VADER weight

    if sentiment_results.get('textblob') and sentiment_results['textblob'].get('polarity') is not None:
        scores.append(sentiment_results['textblob']['polarity'])
        weights.append(0.3)  # TextBlob weight

    if sentiment_results.get('transformer') and sentiment_results['transformer'].get('normalized_score') is not None:
        scores.append(sentiment_results['transformer']['normalized_score'])
        weights.append(0.3)  # Transformer weight

    # Handle no available scores
    if not scores:
        return {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}

    # Normalize weights (in case not all analyzers available)
    total_weight = sum(weights)
    normalized_weights = [w / total_weight for w in weights]

    # Calculate weighted average
    ensemble_score = sum(s * w for s, w in zip(scores, normalized_weights))

    # Calculate confidence based on agreement
    # Why: High variance = low confidence
    if len(scores) > 1:
        variance = sum((s - ensemble_score) ** 2 for s in scores) / len(scores)
        confidence = max(0, 1 - variance)
    else:
        confidence = 0.7  # Single source default confidence

    # Determine label
    # Why thresholds: 0.05 buffer to avoid noise-based classification
    if ensemble_score >= 0.05:
        label = 'Positive'
    elif ensemble_score <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return {
        'score': round(ensemble_score, 4),
        'label': label,
        'confidence': round(confidence, 4)
    }


# ============================================
# SECTION 13: SENTIMENT CORRECTION
# ============================================
"""
Why this section exists:
    - Model predictions aren't always accurate
    - Rules can catch obvious errors (negation handling)
    - Sentiment scores can override when confident
    - Combining approaches gives best results
"""


def correct_sentiment_with_analysis(aspect, opinion, predicted_sentiment, sentiment_scores=None):
    """
    Correct sentiment prediction using rules and sentiment analysis.

    Why this function exists:
        - PyABSA model can make errors, especially with negation
        - Rules catch clear cases (negation + positive word = negative)
        - Sentiment scores provide additional signal
        - Multi-layer correction improves accuracy

    Correction priority:
        1. Negation detection (highest priority - most reliable)
        2. Sentiment score override (when highly confident)
        3. Keyword detection (when no negation)
        4. 'But' clause analysis

    Args:
        aspect (str): The aspect being evaluated
        opinion (str): The opinion text
        predicted_sentiment (str): Model's prediction ('Positive' or 'Negative')
        sentiment_scores (dict): Scores from sentiment analysis (optional)

    Returns:
        tuple: (corrected_sentiment, correction_reason)
    """
    opinion_lower = opinion.lower()

    # Get sentiment word lists
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()

    # Detect negation
    has_negation, neg_type = detect_negation_in_opinion(opinion_lower)

    # ===== RULE 1: Negation Handling (Highest Priority) =====
    # Why first: Negation is the most common cause of sentiment errors
    if has_negation:
        # Check if negating a positive word -> Negative
        for pos_word in positive_words:
            if pos_word in opinion_lower:
                return 'Negative', f'negation_positive ({pos_word})'

        # Check if negating a negative word -> Positive (double negative)
        for neg_word in negative_words:
            if neg_word in opinion_lower:
                return 'Positive', f'double_negation ({neg_word})'

        # Has negation but no clear sentiment word - likely negative
        # Why: "didn't work", "wasn't available" are typically complaints
        return 'Negative', f'negation_general ({neg_type})'

    # ===== RULE 2: Sentiment Score Override =====
    # Why: When sentiment analysis strongly disagrees with prediction
    if sentiment_scores:
        ensemble = sentiment_scores.get('ensemble', {})
        ensemble_score = ensemble.get('score', 0)
        confidence = ensemble.get('confidence', 0)

        # Only override with high confidence
        if confidence >= SENTIMENT_CONFIDENCE_THRESHOLD:
            # Predicted positive but scores strongly negative
            if predicted_sentiment == 'Positive' and ensemble_score < -SENTIMENT_OVERRIDE_THRESHOLD:
                # Verify no strong positive words present
                has_strong_positive = any(pw in opinion_lower for pw in positive_words)
                if not has_strong_positive:
                    return 'Negative', f'sentiment_override ({ensemble_score:.2f})'

            # Predicted negative but scores strongly positive
            if predicted_sentiment == 'Negative' and ensemble_score > SENTIMENT_OVERRIDE_THRESHOLD:
                # Verify no strong negative words present
                has_strong_negative = any(nw in opinion_lower for nw in negative_words)
                if not has_strong_negative:
                    return 'Positive', f'sentiment_override ({ensemble_score:.2f})'

    # ===== RULE 3: Keyword Detection =====
    # Why: If clear sentiment words present and no negation, trust them
    for pos_word in positive_words:
        if pos_word in opinion_lower:
            if predicted_sentiment != 'Positive':
                return 'Positive', f'positive_keyword ({pos_word})'

    for neg_word in negative_words:
        if neg_word in opinion_lower:
            if predicted_sentiment != 'Negative':
                return 'Negative', f'negative_keyword ({neg_word})'

    # ===== RULE 4: 'But' Clause Analysis =====
    # Why: "The food was good but the service was terrible"
    # The sentiment after 'but' often dominates
    if ' but ' in opinion_lower:
        parts = opinion_lower.split(' but ', 1)
        if len(parts) == 2:
            after_but = parts[1]

            # Check what's after 'but'
            for neg_word in negative_words:
                if neg_word in after_but:
                    return 'Negative', f'but_negative ({neg_word})'

            for pos_word in positive_words:
                if pos_word in after_but:
                    return 'Positive', f'but_positive ({pos_word})'

    # No correction needed
    return predicted_sentiment, None


# ============================================
# SECTION 14: REVIEW CHUNKING
# ============================================
"""
Why this section exists:
    - Long reviews can exceed model context limits
    - Shorter chunks are processed more accurately
    - Preserves sentence boundaries for context
"""


def split_long_review(text, max_words=MAX_WORDS_PER_CHUNK):
    """
    Split long reviews into manageable chunks.

    Why this function exists:
        - NLP models have input length limits
        - Very long text can reduce extraction accuracy
        - Chunks should preserve sentence integrity

    Why sentence-based splitting:
        - Breaking mid-sentence loses context
        - Sentence boundaries are natural breakpoints
        - Aspects and opinions usually within same sentence

    Args:
        text (str): The review text to split
        max_words (int): Maximum words per chunk

    Returns:
        list: List of text chunks
    """
    # Split by sentence endings
    # Why replace: Treat ! and ? as sentence enders like .
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        # Handle very long sentences (rare but possible)
        if sentence_length > max_words:
            # Flush current chunk
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            # Split long sentence by word count
            for i in range(0, sentence_length, max_words):
                sub_sentence = ' '.join(words[i:i + max_words])
                chunks.append(sub_sentence + '.')
            continue

        # Check if adding this sentence exceeds limit
        if current_length + sentence_length > max_words and current_chunk:
            # Flush current chunk and start new one
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            # Add to current chunk
            current_chunk.append(sentence)
            current_length += sentence_length

    # Don't forget the last chunk
    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# SECTION 15: RESULT DATA STRUCTURES
# ============================================
"""
Why this section exists:
    - Standardizes output format
    - Includes all relevant information
    - Makes results easy to analyze and export
"""


def create_aspect_opinion_pair(aspect, opinion, raw_opinion, sentiment,
                                original_sentiment, pattern, chunk_id,
                                sentiment_scores, correction_reason):
    """
    Create a standardized result dictionary for an aspect-opinion pair.

    Why this function exists:
        - Ensures consistent output structure
        - Captures all analysis information
        - Easy to convert to DataFrame or other formats

    Args:
        aspect (str): The aspect term
        opinion (str): Formatted opinion
        raw_opinion (str): Original extracted opinion
        sentiment (str): Final sentiment label
        original_sentiment (str): Model's original prediction
        pattern (str): Extraction pattern used
        chunk_id (int): Which chunk this came from (if split)
        sentiment_scores (dict): Sentiment analysis scores
        correction_reason (str): Why sentiment was corrected (if any)

    Returns:
        dict: Comprehensive result dictionary
    """
    # Extract scores for easy access
    vader_compound = None
    textblob_polarity = None
    transformer_score = None
    ensemble_score = None
    ensemble_confidence = None

    if sentiment_scores:
        if sentiment_scores.get('vader'):
            vader_compound = sentiment_scores['vader'].get('compound')
        if sentiment_scores.get('textblob'):
            textblob_polarity = sentiment_scores['textblob'].get('polarity')
        if sentiment_scores.get('transformer'):
            transformer_score = sentiment_scores['transformer'].get('normalized_score')
        if sentiment_scores.get('ensemble'):
            ensemble_score = sentiment_scores['ensemble'].get('score')
            ensemble_confidence = sentiment_scores['ensemble'].get('confidence')

    return {
        # Core extraction results
        'aspect': aspect,
        'opinion': opinion,
        'raw_opinion': raw_opinion,
        'formatted': f"{aspect}: {opinion}",
        'pattern': pattern,
        'chunk': chunk_id,

        # Sentiment results (three layers)
        'original_sentiment': original_sentiment,  # Model prediction
        'sentiment': sentiment,                     # Final result
        'correction_reason': correction_reason,    # Why corrected (if any)

        # Detailed sentiment scores
        'sentiment_scores': {
            'vader_compound': vader_compound,
            'textblob_polarity': textblob_polarity,
            'transformer_score': transformer_score,
            'ensemble_score': ensemble_score,
            'ensemble_confidence': ensemble_confidence
        }
    }


# ============================================
# SECTION 16: FILE LOGGING
# ============================================
"""
Why this section exists:
    - Records analysis process and results
    - Enables both console output and file saving
    - Useful for debugging and reporting
"""


class FileLogger:
    """
    Simple logger that writes to both console and file.

    Why this class exists:
        - Captures all output for later review
        - Avoids print statement duplication
        - Easy to save complete analysis log
    """

    def __init__(self, filename):
        """
        Initialize logger.

        Args:
            filename (str): Output file path
        """
        self.filename = filename
        self.content = []

    def log(self, message=""):
        """
        Log a message.

        Args:
            message (str): Message to log
        """
        self.content.append(message)
        print(message)

    def save(self):
        """
        Save all logged content to file.
        """
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ Results saved to: {self.filename}")


# ============================================
# SECTION 17: MAIN ANALYSIS FUNCTION
# ============================================
"""
Why this section exists:
    - Orchestrates the complete analysis pipeline
    - Processes all reviews and collects results
    - Provides detailed logging and statistics
"""


def comprehensive_aspect_opinion_analysis(reviews, aspect_extractor,
                                          max_words=MAX_WORDS_PER_CHUNK,
                                          verbose=True, logger=None,
                                          use_sentiment_analysis=True):
    """
    Run complete ABSA analysis on a collection of reviews.

    Why this function exists:
        - Main entry point for analysis
        - Handles chunking, extraction, validation, and correction
        - Collects comprehensive statistics

    Pipeline steps:
        1. For each review:
           a. Split into chunks if needed
           b. Extract aspects using PyABSA
           c. Validate and clean aspects
           d. Extract opinions with boundary detection
           e. Validate and format opinions
           f. Analyze sentiment scores
           g. Apply sentiment correction
           h. Create result records
        2. Collect statistics
        3. Return all results

    Args:
        reviews: Iterable of review texts
        aspect_extractor: PyABSA aspect extractor model
        max_words (int): Max words per chunk
        verbose (bool): Whether to log detailed output
        logger (FileLogger): Logger instance
        use_sentiment_analysis (bool): Whether to use sentiment scoring

    Returns:
        tuple: (results_list, statistics_dict)
    """
    # Log header
    if logger:
        logger.log("=" * 80)
        logger.log("ABSA Analysis System - V4 Universal Edition")
        logger.log(f"Chunk size: {max_words} words")
        logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
        logger.log(f"Analysis time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 80)

    # Initialize sentiment analyzers if needed
    if use_sentiment_analysis:
        _ = get_sentiment_analyzers()

    # Results and statistics
    all_results = []
    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'sentiment_corrections': 0,
        'correction_reasons': Counter()
    }

    # Process each review
    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'=' * 80}")
            logger.log(f"Review #{idx} ({word_count} words)")
            logger.log("=" * 80)

        # ----- Step 1: Chunking -----
        if word_count <= max_words:
            # Process directly
            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]
            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            # Split into chunks
            if logger and verbose:
                logger.log("📄 Splitting long review into chunks...")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   Split into {len(chunks)} chunks")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = [
                {'chunk_id': i + 1, 'result': r}
                for i, r in enumerate(chunk_results)
            ]

        # ----- Step 2: Process extracted aspects -----
        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, original_sentiment in zip(aspects, positions, sentiments):

                # ----- Validate and clean aspect -----
                aspect = refine_aspect_term(aspect)

                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # ----- Extract opinion -----
                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # ----- Format and validate opinion -----
                display_opinion = format_opinion_for_display(raw_opinion)

                if not display_opinion or not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                # ----- Sentiment analysis -----
                sentiment_scores = None
                if use_sentiment_analysis:
                    sentiment_scores = analyze_sentiment_scores(display_opinion)

                # ----- Sentiment correction -----
                corrected_sentiment, correction_reason = correct_sentiment_with_analysis(
                    aspect, raw_opinion, original_sentiment, sentiment_scores
                )

                # Track corrections
                if corrected_sentiment != original_sentiment:
                    stats['sentiment_corrections'] += 1
                    if correction_reason:
                        stats['correction_reasons'][correction_reason] += 1

                stats['total_aspects'] += 1

                # ----- Create result record -----
                pair = create_aspect_opinion_pair(
                    aspect=aspect,
                    opinion=display_opinion,
                    raw_opinion=raw_opinion,
                    sentiment=corrected_sentiment,
                    original_sentiment=original_sentiment,
                    pattern=pattern,
                    chunk_id=chunk_data['chunk_id'] if len(analysis_results) > 1 else None,
                    sentiment_scores=sentiment_scores,
                    correction_reason=correction_reason
                )
                aspect_opinion_pairs.append(pair)

        # ----- Log results for this review -----
        if logger and verbose:
            logger.log(f"\n🎯 Extraction Results:")
            header = f"{'#':<3} {'Aspect':<18} {'Opinion':<35} {'Orig':<8} {'Final':<8} {'Score':<8}"
            logger.log(header)
            logger.log("-" * 90)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    opinion_disp = pair['opinion'][:32] + "..." if len(pair['opinion']) > 35 else pair['opinion']

                    scores = pair['sentiment_scores']
                    if scores and scores.get('ensemble_score') is not None:
                        score_str = f"{scores['ensemble_score']:+.2f}"
                    else:
                        score_str = "N/A"

                    final = pair['sentiment']
                    if pair['correction_reason']:
                        final += "*"

                    logger.log(f"{i:<3} {pair['aspect']:<18} {opinion_disp:<35} "
                              f"{pair['original_sentiment']:<8} {final:<8} {score_str:<8}")

                if filtered_aspect_count + filtered_opinion_count > 0:
                    logger.log(f"\n   ℹ️ Filtered: {filtered_aspect_count} aspects, {filtered_opinion_count} opinions")
            else:
                logger.log("   ⚠️ No valid aspect-opinion pairs found")

        # Store review results
        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
            'filtered_aspect_count': filtered_aspect_count,
            'filtered_opinion_count': filtered_opinion_count
        })

    return all_results, stats


# ============================================
# SECTION 18: REPORT GENERATION
# ============================================
"""
Why this section exists:
    - Summarizes analysis results for decision-making
    - Identifies strengths and areas for improvement
    - Provides actionable insights
"""


def generate_management_report(results, stats, logger=None):
    """
    Generate a management-focused analysis report.

    Why this function exists:
        - Transforms raw data into actionable insights
        - Highlights competitive strengths and improvement areas
        - Provides executive summary of customer sentiment

    Report sections:
        1. Overall statistics
        2. Correction statistics
        3. Competitive advantages (top positive aspects)
        4. Areas for improvement (top negative aspects)
        5. Recommendations

    Args:
        results (list): Analysis results
        stats (dict): Analysis statistics
        logger (FileLogger): Logger instance
    """
    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 80)
    log("MANAGEMENT ANALYSIS REPORT")
    log("=" * 80)

    # Collect all pairs by sentiment
    positive_pairs = []
    negative_pairs = []

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

    total_pairs = len(positive_pairs) + len(negative_pairs)

    # ----- Overall Statistics -----
    log(f"\n📊 OVERALL STATISTICS")
    log(f"   • Total reviews analyzed: {len(results)}")
    log(f"   • Total aspects identified: {total_pairs}")
    log(f"   • Invalid aspects filtered: {stats['filtered_invalid_aspects']}")
    log(f"   • Invalid opinions filtered: {stats['filtered_invalid_opinions']}")

    if stats['sentiment_corrections'] > 0:
        log(f"   • Sentiments corrected: {stats['sentiment_corrections']}")

    if total_pairs > 0:
        pos_pct = len(positive_pairs) / total_pairs * 100
        neg_pct = len(negative_pairs) / total_pairs * 100
        log(f"   • Positive mentions: {len(positive_pairs)} ({pos_pct:.1f}%)")
        log(f"   • Negative mentions: {len(negative_pairs)} ({neg_pct:.1f}%)")

    # ----- Correction Statistics -----
    if stats['correction_reasons']:
        log(f"\n📈 CORRECTION BREAKDOWN")
        for reason, count in stats['correction_reasons'].most_common(10):
            log(f"   • {reason}: {count}")

    # ----- Aggregate by aspect -----
    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair)

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair)

    # ----- Competitive Advantages -----
    if positive_aspects:
        log(f"\n✅ COMPETITIVE ADVANTAGES (Top Positive Aspects)")
        log(f"{'Rank':<5} {'Aspect':<20} {'Count':<8} {'Avg Score':<10} {'Sample Opinion':<35}")
        log("-" * 80)

        sorted_pos = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_pos[:15], 1):
            count = len(pairs)

            # Calculate average sentiment score
            scores = [
                p['sentiment_scores']['ensemble_score']
                for p in pairs
                if p['sentiment_scores'].get('ensemble_score') is not None
            ]
            avg_score = sum(scores) / len(scores) if scores else 0

            sample = pairs[0]['opinion'][:32] + "..." if len(pairs[0]['opinion']) > 35 else pairs[0]['opinion']

            log(f"{i:<5} {aspect:<20} {count:<8} {avg_score:+.2f}      {sample:<35}")

    # ----- Areas for Improvement -----
    if negative_aspects:
        log(f"\n⚠️ AREAS FOR IMPROVEMENT (Top Negative Aspects)")
        log(f"{'Rank':<5} {'Aspect':<20} {'Count':<8} {'Avg Score':<10} {'Sample Complaint':<35}")
        log("-" * 80)

        sorted_neg = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_neg[:15], 1):
            count = len(pairs)

            scores = [
                p['sentiment_scores']['ensemble_score']
                for p in pairs
                if p['sentiment_scores'].get('ensemble_score') is not None
            ]
            avg_score = sum(scores) / len(scores) if scores else 0

            sample = pairs[0]['opinion'][:32] + "..." if len(pairs[0]['opinion']) > 35 else pairs[0]['opinion']

            log(f"{i:<5} {aspect:<20} {count:<8} {avg_score:+.2f}      {sample:<35}")

    # ----- Recommendations -----
    log(f"\n💡 RECOMMENDATIONS")

    if negative_aspects:
        worst = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   1. PRIORITY: Address '{worst[0]}' ({len(worst[1])} negative mentions)")

    if positive_aspects:
        best = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   2. LEVERAGE: Promote '{best[0]}' ({len(best[1])} positive mentions)")

    log(f"   3. MONITOR: Track sentiment score trends over time")
    log(f"   4. INVESTIGATE: Review corrected sentiments for accuracy")

    # ----- Quality Statistics -----
    log(f"\n📉 EXTRACTION QUALITY METRICS")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   • Extraction success rate: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   • Average aspects per review: {avg_pairs:.2f}")


# ============================================
# SECTION 19: DATA EXPORT
# ============================================
"""
Why this section exists:
    - Enables further analysis in other tools
    - DataFrame format is widely compatible
    - Preserves all analysis details
"""


def export_results_to_dataframe(results):
    """
    Export analysis results to a pandas DataFrame.

    Why this function exists:
        - DataFrames enable easy filtering and analysis
        - Compatible with Excel, CSV export
        - Can be used with visualization tools

    Args:
        results (list): Analysis results

    Returns:
        pd.DataFrame: Results in tabular format
    """
    import pandas as pd

    rows = []

    for result in results:
        review_id = result['review_id']

        for pair in result['pairs']:
            scores = pair.get('sentiment_scores', {})

            row = {
                'review_id': review_id,
                'aspect': pair['aspect'],
                'opinion': pair['opinion'],
                'raw_opinion': pair['raw_opinion'],
                'original_sentiment': pair['original_sentiment'],
                'corrected_sentiment': pair['sentiment'],
                'correction_reason': pair.get('correction_reason'),
                'extraction_pattern': pair['pattern'],
                'chunk_id': pair.get('chunk'),
                'vader_compound': scores.get('vader_compound'),
                'textblob_polarity': scores.get('textblob_polarity'),
                'transformer_score': scores.get('transformer_score'),
                'ensemble_score': scores.get('ensemble_score'),
                'ensemble_confidence': scores.get('ensemble_confidence')
            }
            rows.append(row)

    return pd.DataFrame(rows)


# ============================================
# SECTION 20: MAIN EXECUTION
# ============================================
"""
Why this section exists:
    - Provides easy-to-use entry point
    - Combines all analysis steps
    - Handles output and export
"""


def run_analysis(reviews, aspect_extractor,
                 output_file="absa_analysis_results_v4.txt",
                 use_sentiment_analysis=True):
    """
    Run complete ABSA analysis and generate outputs.

    Why this function exists:
        - Single function to run entire analysis
        - Generates both log file and DataFrame
        - Easy to use for end users

    Args:
        reviews: Iterable of review texts (list, Series, etc.)
        aspect_extractor: PyABSA aspect extractor model
        output_file (str): Path for output log file
        use_sentiment_analysis (bool): Enable sentiment scoring

    Returns:
        tuple: (results, stats, dataframe)
    """
    # Create logger
    logger = FileLogger(output_file)

    # Log header
    logger.log("=" * 80)
    logger.log("ABSA ANALYSIS SYSTEM - V4 UNIVERSAL EDITION")
    logger.log(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"Reviews to analyze: {len(reviews)}")
    logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
    logger.log("=" * 80)

    # Run analysis
    results, stats = comprehensive_aspect_opinion_analysis(
        reviews,
        aspect_extractor,
        max_words=MAX_WORDS_PER_CHUNK,
        verbose=True,
        logger=logger,
        use_sentiment_analysis=use_sentiment_analysis
    )

    # Log completion
    logger.log("\n" + "=" * 80)
    logger.log("✅ ANALYSIS COMPLETE")
    logger.log("=" * 80)

    # Generate report
    generate_management_report(results, stats, logger)

    # Save log
    logger.save()

    # Export to DataFrame
    df_results = export_results_to_dataframe(results)

    return results, stats, df_results


# ============================================
# SECTION 21: ENTRY POINT
# ============================================
"""
Why this section exists:
    - Allows running as standalone script
    - Provides usage instructions
    - Handles missing dependencies gracefully
"""

if __name__ == "__main__":
    print("=" * 60)
    print("ABSA Analysis System - V4 Universal Edition")
    print("=" * 60)

    # Check if required variables exist
    if 'test_reviews' in dir() and len(test_reviews) > 0 and 'aspect_extractor' in dir():
        print("\n📊 Starting analysis...")
        print("Installing required packages...")

        # Install dependencies
        import subprocess
        subprocess.run(['pip', 'install', '-q', 'nltk', 'textblob'], check=True)

        # Run analysis
        results, stats, df_results = run_analysis(
            test_reviews,
            aspect_extractor,
            OUTPUT_FILE,
            use_sentiment_analysis=True
        )

        print(f"\n📋 Results DataFrame shape: {df_results.shape}")
        print(df_results.head())

    else:
        print("\n⚠️ Required variables not found.")
        print("\nUsage:")
        print("  1. Load your reviews into 'test_reviews' variable")
        print("  2. Load PyABSA extractor into 'aspect_extractor' variable")
        print("  3. Run: results, stats, df = run_analysis(test_reviews, aspect_extractor)")
        print("\nExample:")
        print("  from pyabsa import AspectTermExtraction as ATEPC")
        print("  aspect_extractor = ATEPC.AspectExtractor('multilingual')")
        print("  test_reviews = df['review_column']")
        print("  results, stats, df = run_analysis(test_reviews, aspect_extractor)")

In [ ]:
"""
================================================================================
ABSA (Aspect-Based Sentiment Analysis) System - V5 Enhanced Edition
================================================================================

改進項目：
1. 新增慣用語/俚語辭典處理
2. 新增信心度過濾機制
3. 改進Opinion邊界檢測
4. 增強管理報告（含多個代表性意見）
5. 修正情緒修正邏輯

Version: 5.0 Enhanced
Date: 2025
================================================================================
"""

import warnings
warnings.filterwarnings('ignore')

import re
from collections import Counter
from datetime import datetime

# ============================================
# SECTION 1: ENHANCED CONFIGURATION
# ============================================

OUTPUT_FILE = "absa_analysis_results_v5_enhanced.txt"
MAX_WORDS_PER_CHUNK = 80

# 新增：信心度過濾閾值
SENTIMENT_CONFIDENCE_THRESHOLD = 0.6
SENTIMENT_OVERRIDE_THRESHOLD = 0.4

# 🆕 新增：最低情緒分數閾值（分數太接近0表示情緒不明確）
MINIMUM_SENTIMENT_SCORE_THRESHOLD = 0.25

# 🆕 新增：分析器一致性閾值
MINIMUM_AGREEMENT_THRESHOLD = 0.40

# ============================================
# SECTION 2: IDIOM DICTIONARIES (NEW)
# ============================================
"""
🆕 新增章節：慣用語辭典
解決問題：正面慣用語被否定邏輯誤判為負面
例如："did not disappoint" 被誤判為負面
"""

def get_positive_idioms():
    """
    獲取正面慣用語辭典
    這些表達方式雖含有否定詞，但整體語意為正面
    """
    return {
        # === 雙重否定 = 肯定 ===
        "did not disappoint": "Positive",
        "didn't disappoint": "Positive",
        "does not disappoint": "Positive",
        "doesn't disappoint": "Positive",
        "never disappoints": "Positive",
        "never disappointed": "Positive",
        "not disappointed": "Positive",
        "wasn't disappointed": "Positive",
        "weren't disappointed": "Positive",

        # === 最高級否定句型 ===
        "could not have been better": "Positive",
        "couldn't have been better": "Positive",
        "could not have been nicer": "Positive",
        "couldn't have been nicer": "Positive",
        "could not have been friendlier": "Positive",
        "couldn't have been friendlier": "Positive",
        "could not have been more helpful": "Positive",
        "couldn't have been more helpful": "Positive",
        "could not have been more attentive": "Positive",
        "couldn't have been more attentive": "Positive",
        "could not ask for more": "Positive",
        "couldn't ask for more": "Positive",
        "could not be happier": "Positive",
        "couldn't be happier": "Positive",

        # === 無可挑剔型 ===
        "can't complain": "Positive",
        "cannot complain": "Positive",
        "nothing to complain about": "Positive",
        "no complaints": "Positive",
        "left nothing to be desired": "Positive",
        "leaves nothing to be desired": "Positive",
        "nothing short of": "Positive",  # "nothing short of amazing"

        # === 俚語（正面） ===
        "killed it": "Positive",
        "nailed it": "Positive",
        "hit the spot": "Positive",
        "hits the spot": "Positive",
        "on point": "Positive",
        "on fire": "Positive",
        "off the hook": "Positive",
        "off the chain": "Positive",
        "off the charts": "Positive",
        "out of this world": "Positive",
        "to die for": "Positive",
        "die for": "Positive",
        "the bomb": "Positive",
        "da bomb": "Positive",
        "was bomb": "Positive",
        "is fire": "Positive",
        "was fire": "Positive",
        "chef's kiss": "Positive",
        "slaps": "Positive",
        "bussin": "Positive",
        "bussin'": "Positive",
        "hits different": "Positive",
        "top notch": "Positive",
        "top-notch": "Positive",
        "a must try": "Positive",
        "must try": "Positive",
        "a must": "Positive",
        "a gem": "Positive",
        "hidden gem": "Positive",
        "saved the day": "Positive",

        # === 餐飲特定 ===
        "finger licking good": "Positive",
        "finger-licking good": "Positive",
        "melts in your mouth": "Positive",
        "melt in your mouth": "Positive",
        "fall off the bone": "Positive",
        "falls off the bone": "Positive",
        "cooked to perfection": "Positive",
        "done to perfection": "Positive",
    }


def get_negative_idioms():
    """
    獲取負面慣用語辭典
    """
    return {
        # === 負面俚語 ===
        "left a lot to be desired": "Negative",
        "leaves a lot to be desired": "Negative",
        "left much to be desired": "Negative",
        "leaves much to be desired": "Negative",
        "not my cup of tea": "Negative",
        "wasn't my cup of tea": "Negative",
        "nothing to write home about": "Negative",
        "nothing special": "Negative",
        "seen better days": "Negative",
        "has seen better days": "Negative",
        "hit or miss": "Negative",
        "meh": "Negative",
        "just ok": "Negative",
        "just okay": "Negative",
        "waste of money": "Negative",
        "waste of time": "Negative",
        "rip off": "Negative",
        "rip-off": "Negative",
        "not worth it": "Negative",
        "not worth the price": "Negative",
        "not worth the hype": "Negative",
        "overhyped": "Negative",
        "over-hyped": "Negative",
        "overrated": "Negative",
        "over-rated": "Negative",

        # === 餐飲特定負面 ===
        "tasted like cardboard": "Negative",
        "like eating cardboard": "Negative",
        "rubber chicken": "Negative",
        "hockey puck": "Negative",
        "sat under a heat lamp": "Negative",
    }


# ============================================
# SECTION 3: SENTIMENT ANALYZER INITIALIZATION
# ============================================

SENTIMENT_ANALYZERS = None


def initialize_sentiment_analyzers():
    """Initialize all available sentiment analysis tools."""
    analyzers = {}

    try:
        import nltk
        nltk.download('vader_lexicon', quiet=True)
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        analyzers['vader'] = SentimentIntensityAnalyzer()
        print("✅ VADER sentiment analyzer loaded successfully")
    except Exception as e:
        print(f"⚠️ VADER not available: {e}")
        analyzers['vader'] = None

    try:
        from textblob import TextBlob
        analyzers['textblob'] = TextBlob
        print("✅ TextBlob sentiment analyzer loaded successfully")
    except Exception as e:
        print(f"⚠️ TextBlob not available: {e}")
        analyzers['textblob'] = None

    try:
        from transformers import pipeline
        analyzers['transformer'] = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=-1
        )
        print("✅ Transformer sentiment analyzer loaded successfully")
    except Exception as e:
        print(f"ℹ️ Transformer not available (optional): {e}")
        analyzers['transformer'] = None

    return analyzers


def get_sentiment_analyzers():
    """Get or initialize sentiment analyzers (singleton pattern)."""
    global SENTIMENT_ANALYZERS
    if SENTIMENT_ANALYZERS is None:
        SENTIMENT_ANALYZERS = initialize_sentiment_analyzers()
    return SENTIMENT_ANALYZERS


# ============================================
# SECTION 4: WORD LISTS
# ============================================

def get_common_english_names():
    """Get a comprehensive list of common English first names."""
    male_names = {
        'james', 'john', 'robert', 'michael', 'william', 'david', 'richard',
        'joseph', 'thomas', 'charles', 'christopher', 'daniel', 'matthew',
        'anthony', 'mark', 'donald', 'steven', 'paul', 'andrew', 'joshua',
        'kenneth', 'kevin', 'brian', 'george', 'timothy', 'ronald', 'edward',
        'jason', 'jeffrey', 'ryan', 'jacob', 'gary', 'nicholas', 'eric',
        'jonathan', 'stephen', 'larry', 'justin', 'scott', 'brandon', 'benjamin',
        'samuel', 'raymond', 'gregory', 'frank', 'alexander', 'patrick', 'jack',
        'dennis', 'jerry', 'tyler', 'aaron', 'jose', 'adam', 'nathan', 'henry',
        'douglas', 'zachary', 'peter', 'kyle', 'noah', 'ethan', 'jeremy',
        'walter', 'christian', 'keith', 'roger', 'terry', 'austin', 'sean',
        'gerald', 'carl', 'harold', 'dylan', 'arthur', 'lawrence', 'jordan',
        'jesse', 'bryan', 'billy', 'bruce', 'gabriel', 'joe', 'logan', 'albert',
        'willie', 'alan', 'eugene', 'russell', 'vincent', 'philip', 'bobby',
        'johnny', 'bradley', 'roy', 'ralph', 'randy', 'wayne', 'elijah',
        'ray', 'nick', 'andy', 'greg', 'mike', 'steve', 'tom', 'bob', 'jim', 'dan',
        # 常見服務生名字
        'joel', 'paul', 'morgan', 'david', 'jordan', 'chris', 'matt'
    }

    female_names = {
        'mary', 'patricia', 'jennifer', 'linda', 'barbara', 'elizabeth', 'susan',
        'jessica', 'sarah', 'karen', 'lisa', 'nancy', 'betty', 'margaret', 'sandra',
        'ashley', 'kimberly', 'emily', 'donna', 'michelle', 'dorothy', 'carol',
        'amanda', 'melissa', 'deborah', 'stephanie', 'rebecca', 'sharon', 'laura',
        'cynthia', 'kathleen', 'amy', 'angela', 'shirley', 'anna', 'brenda',
        'pamela', 'emma', 'nicole', 'helen', 'samantha', 'katherine', 'christine',
        'debra', 'rachel', 'carolyn', 'janet', 'catherine', 'maria', 'heather',
        'diane', 'ruth', 'julie', 'olivia', 'joyce', 'virginia', 'victoria',
        'kelly', 'lauren', 'christina', 'joan', 'evelyn', 'judith', 'megan',
        'andrea', 'cheryl', 'hannah', 'jacqueline', 'martha', 'gloria', 'teresa',
        'ann', 'sara', 'madison', 'frances', 'kathryn', 'janice', 'jean', 'abigail',
        'alice', 'judy', 'sophia', 'grace', 'denise', 'amber', 'doris', 'marilyn',
        'danielle', 'beverly', 'isabella', 'theresa', 'diana', 'natalie', 'brittany',
        'charlotte', 'marie', 'kayla', 'alexis', 'lori', 'tina', 'zoe',
        'melanie', 'morgan', 'veronica', 'alyssa', 'kristina', 'cameron', 'gabby',
        # 常見服務生名字
        'sara', 'fatima', 'kaitlin', 'kaitlyn', 'natalie', 'melanie', 'christina'
    }

    return male_names | female_names


def get_positive_sentiment_words():
    """Get universal positive sentiment indicator words."""
    return {
        # General Quality
        'good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic',
        'outstanding', 'superb', 'perfect', 'incredible', 'awesome',
        'terrific', 'fabulous', 'marvelous', 'exceptional', 'best',
        'impressive', 'remarkable', 'magnificent', 'splendid', 'brilliant',
        'stellar', 'phenomenal', 'spectacular', 'extraordinary',

        # Food-Related
        'delicious', 'tasty', 'yummy', 'flavorful', 'savory', 'fresh',
        'tender', 'juicy', 'crispy', 'creamy', 'rich', 'light',
        'authentic', 'homemade', 'seasoned', 'aromatic', 'scrumptious',
        'divine', 'heavenly', 'mouthwatering',

        # Service-Related
        'friendly', 'helpful', 'attentive', 'professional', 'courteous',
        'prompt', 'efficient', 'welcoming', 'accommodating', 'polite',
        'responsive', 'knowledgeable', 'patient', 'thorough', 'dedicated',
        'personable', 'warm', 'gracious', 'sweet',

        # Environment/Atmosphere
        'clean', 'comfortable', 'cozy', 'spacious', 'nice', 'lovely',
        'beautiful', 'charming', 'pleasant', 'relaxing', 'quiet',
        'modern', 'elegant', 'stylish', 'inviting', 'cute',
        'trendy', 'vibrant', 'lively',

        # Value-Related
        'reasonable', 'affordable', 'worth', 'value', 'cheap', 'bargain',
        'fair', 'inexpensive', 'economical'
    }


def get_negative_sentiment_words():
    """Get universal negative sentiment indicator words."""
    return {
        # General Quality
        'bad', 'terrible', 'awful', 'horrible', 'poor', 'disappointing',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross', 'worst',
        'inferior', 'subpar', 'unacceptable', 'dreadful', 'appalling',
        'atrocious', 'abysmal', 'horrendous',

        # Food-Related
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty', 'stale',
        'cold', 'lukewarm', 'frozen', 'rubbery', 'tough', 'chewy',
        'flavorless', 'mushy', 'bitter', 'sour', 'spoiled',
        'unseasoned', 'unappetizing',

        # Service-Related
        'rude', 'slow', 'unfriendly', 'unhelpful', 'inattentive',
        'unprofessional', 'careless', 'dismissive', 'indifferent',
        'incompetent', 'negligent', 'impolite', 'disrespectful',
        'horrible', 'terrible', 'awful', 'ignored',

        # Environment/Atmosphere
        'dirty', 'messy', 'noisy', 'crowded', 'cramped', 'uncomfortable',
        'dark', 'smelly', 'hot', 'stuffy', 'outdated', 'dingy',
        'filthy', 'cluttered', 'rundown', 'shabby', 'sticky',

        # Value-Related
        'expensive', 'overpriced', 'costly', 'pricey', 'ripoff',
        'unreasonable', 'exorbitant'
    }


def get_action_verbs():
    """Get a list of action verbs that should not be treated as opinions."""
    return {
        'order', 'ordered', 'ordering', 'orders',
        'ask', 'asked', 'asking', 'asks',
        'request', 'requested', 'requesting', 'requests',
        'come', 'came', 'coming', 'comes',
        'go', 'went', 'going', 'goes', 'gone',
        'arrive', 'arrived', 'arriving', 'arrives',
        'leave', 'left', 'leaving', 'leaves',
        'walk', 'walked', 'walking', 'walks',
        'enter', 'entered', 'entering', 'enters',
        'visit', 'visited', 'visiting', 'visits',
        'get', 'got', 'getting', 'gets',
        'take', 'took', 'taking', 'takes', 'taken',
        'bring', 'brought', 'bringing', 'brings',
        'receive', 'received', 'receiving', 'receives',
        'make', 'made', 'making', 'makes',
        'do', 'did', 'doing', 'does', 'done',
        'prepare', 'prepared', 'preparing', 'prepares',
        'cook', 'cooked', 'cooking', 'cooks',
        'say', 'said', 'saying', 'says',
        'tell', 'told', 'telling', 'tells',
        'call', 'called', 'calling', 'calls',
        'inform', 'informed', 'informing', 'informs',
        'mention', 'mentioned', 'mentioning', 'mentions',
        'serve', 'served', 'serving', 'serves',
        'show', 'showed', 'showing', 'shows', 'shown',
        'give', 'gave', 'giving', 'gives', 'given',
        'seat', 'seated', 'seating', 'seats',
        'try', 'tried', 'trying', 'tries',
        'want', 'wanted', 'wanting', 'wants',
        'need', 'needed', 'needing', 'needs',
        'hope', 'hoped', 'hoping', 'hopes',
        'expect', 'expected', 'expecting', 'expects',
        'enjoy', 'enjoyed', 'enjoying', 'enjoys',
        'pay', 'paid', 'paying', 'pays',
        'wait', 'waited', 'waiting', 'waits',
        'sit', 'sat', 'sitting', 'sits',
        'eat', 'ate', 'eating', 'eats', 'eaten',
        'drink', 'drank', 'drinking', 'drinks', 'drunk'
    }


def get_function_words():
    """Get a list of function words that carry no sentiment meaning."""
    return {
        'a', 'an', 'the',
        'this', 'that', 'these', 'those',
        'i', 'me', 'my', 'mine', 'myself',
        'you', 'your', 'yours', 'yourself',
        'he', 'him', 'his', 'himself',
        'she', 'her', 'hers', 'herself',
        'it', 'its', 'itself',
        'we', 'us', 'our', 'ours', 'ourselves',
        'they', 'them', 'their', 'theirs', 'themselves',
        'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
        'from', 'about', 'into', 'through', 'during', 'before',
        'after', 'above', 'below', 'between', 'under', 'over',
        'and', 'or', 'but', 'so', 'yet', 'nor',
        'if', 'when', 'where', 'while', 'as', 'because', 'since',
        'although', 'though', 'unless', 'until',
        'only', 'just', 'also', 'too', 'even', 'still',
        'already', 'always', 'never', 'ever', 'often',
        'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'has', 'have', 'had', 'having',
        'do', 'does', 'did',
        'will', 'would', 'shall', 'should',
        'can', 'could', 'may', 'might', 'must',
        'here', 'there', 'some', 'any', 'no', 'every',
        'each', 'both', 'all', 'most', 'many', 'much',
        'few', 'little', 'other', 'another', 'such'
    }


def get_intensifiers():
    """Get a list of intensifier words that modify sentiment strength."""
    return {
        'very', 'extremely', 'incredibly', 'amazingly', 'exceptionally',
        'remarkably', 'absolutely', 'totally', 'completely', 'utterly',
        'entirely', 'thoroughly', 'perfectly', 'highly', 'deeply',
        'truly', 'really', 'genuinely', 'seriously',
        'quite', 'rather', 'fairly', 'pretty', 'somewhat',
        'reasonably', 'moderately', 'relatively',
        'slightly', 'a bit', 'a little', 'mildly', 'barely',
        'hardly', 'scarcely',
        'too', 'overly', 'excessively',
        'so', 'such', 'super', 'especially', 'particularly'
    }


# ============================================
# SECTION 5: ADJECTIVE/ADVERB DETECTION
# ============================================

def is_adjective_or_adverb(word):
    """Determine if a word is an adjective or adverb."""
    word_lower = word.lower().strip('.,!?;:\'"')

    if not word_lower:
        return False, 'none'

    positive_words = get_positive_sentiment_words()
    if word_lower in positive_words:
        return True, 'positive_adj'

    negative_words = get_negative_sentiment_words()
    if word_lower in negative_words:
        return True, 'negative_adj'

    intensifiers = get_intensifiers()
    if word_lower in intensifiers:
        return True, 'intensifier'

    adj_suffixes = [
        'ful', 'less', 'ous', 'ive', 'able', 'ible',
        'al', 'ic', 'ish', 'ent', 'ant', 'ory', 'ary',
    ]

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    if word_lower.endswith('ly') and len(word_lower) > 4:
        non_adverb_ly = {'family', 'only', 'early', 'likely', 'friendly', 'lonely'}
        if word_lower not in non_adverb_ly:
            return True, 'suffix_adv'

    sentiment_participles = {
        'disappointed', 'satisfied', 'pleased', 'impressed', 'amazed',
        'surprised', 'disgusted', 'frustrated', 'annoyed', 'delighted',
        'thrilled', 'excited', 'bored', 'tired', 'exhausted',
        'overwhelmed', 'underwhelmed', 'overpriced'
    }
    if word_lower in sentiment_participles:
        return True, 'suffix_adj'

    sentiment_ing = {
        'amazing', 'disappointing', 'disgusting', 'interesting', 'boring',
        'exciting', 'frustrating', 'annoying', 'satisfying', 'refreshing',
        'relaxing', 'welcoming', 'inviting', 'appealing', 'appalling'
    }
    if word_lower in sentiment_ing:
        return True, 'suffix_adj'

    return False, 'none'


# ============================================
# SECTION 6: ENHANCED NEGATION DETECTION
# ============================================

def check_idiom_first(text):
    """
    🆕 優先檢查是否為慣用語
    返回 (is_idiom, sentiment, idiom_matched)
    """
    text_lower = text.lower()

    # 檢查正面慣用語
    positive_idioms = get_positive_idioms()
    for idiom, sentiment in positive_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    # 檢查負面慣用語
    negative_idioms = get_negative_idioms()
    for idiom, sentiment in negative_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    return False, None, None


def detect_negation_in_opinion(opinion):
    """
    Detect if an opinion contains negation.
    🆕 改進：先檢查慣用語，避免誤判
    """
    opinion_lower = opinion.lower()

    # 🆕 優先檢查慣用語
    is_idiom, idiom_sentiment, idiom_matched = check_idiom_first(opinion_lower)
    if is_idiom:
        # 如果是慣用語，不視為普通否定
        return False, f'idiom:{idiom_matched}'

    # 直接否定詞
    direct_negations = [
        'not ', "n't ", 'no ', 'never ', 'none ', 'nothing ',
        'neither ', 'nobody ', 'nowhere ', 'cannot '
    ]

    for neg in direct_negations:
        if neg in opinion_lower or opinion_lower.startswith(neg.strip()):
            return True, 'direct'

    # 縮寫否定
    contracted_negations = [
        "didn't", "didn't", "did not",
        "wasn't", "wasn't", "was not",
        "weren't", "weren't", "were not",
        "isn't", "isn't", "is not",
        "aren't", "aren't", "are not",
        "don't", "don't", "do not",
        "doesn't", "doesn't", "does not",
        "won't", "won't", "will not",
        "wouldn't", "wouldn't", "would not",
        "couldn't", "couldn't", "could not",
        "shouldn't", "shouldn't", "should not",
        "can't", "can't", "cannot",
        "haven't", "haven't", "have not",
        "hasn't", "hasn't", "has not",
        "hadn't", "hadn't", "had not"
    ]

    for neg in contracted_negations:
        if neg in opinion_lower:
            return True, 'contraction'

    return False, None


# ============================================
# SECTION 7: BOUNDARY DETECTION
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """Find the sentence boundaries containing a given aspect."""
    sentence_enders = {'.', '!', '?'}

    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in sentence_enders:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in sentence_enders:
            sentence_end = i
            break

    return sentence_start, sentence_end


def find_previous_aspect_end(current_start_idx, all_aspect_positions):
    """Find the end position of the previous aspect in the sentence."""
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(current_end_idx, all_aspect_positions):
    """Find the start position of the next aspect in the sentence."""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# SECTION 8: BE-VERB DETECTION
# ============================================

def detect_be_verb(tokens, start_idx):
    """Detect if a be-verb appears at the given position."""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    be_verbs = {'is', 'was', 'were', 'are', 'be', 'been', "'s", "s"}
    if word in be_verbs:
        return True, 1, False

    negated_be_verbs = {
        "isn't", "isnt", "isn't",
        "wasn't", "wasnt", "wasn't",
        "weren't", "werent", "weren't",
        "aren't", "arent", "aren't"
    }
    if word in negated_be_verbs:
        return True, 1, True

    partial_negated = {'isn', 'wasn', 'weren', 'aren'}
    if word in partial_negated and start_idx + 2 < len(tokens):
        if tokens[start_idx + 1] in {"'", "'"} and tokens[start_idx + 2].lower() in {'t', 'nt'}:
            return True, 3, True

    return False, 0, False


# ============================================
# SECTION 9: ENHANCED OPINION VALIDATION
# ============================================

def validate_opinion_completeness(opinion):
    """
    🆕 新增：驗證 opinion 是否完整且有意義
    """
    if not opinion or len(opinion.strip()) < 2:
        return False, 'empty'

    words = opinion.lower().strip().split()
    if not words:
        return False, 'empty'

    # 規則 1: 不能以連接詞開頭
    conjunctions = {'and', 'or', 'but', 'so', 'yet', 'nor', 'for', 'if'}
    if words[0] in conjunctions:
        return False, 'starts_with_conjunction'

    # 規則 2: 如果只有一個詞，必須是情緒詞
    if len(words) == 1:
        intensifiers = get_intensifiers()
        if words[0] in intensifiers:
            return False, 'only_intensifier'

        function_words = get_function_words()
        if words[0] in function_words:
            return False, 'only_function_word'

    # 規則 3: 檢查是否為純敘述句（非意見）
    factual_patterns = [
        r'^(he|she|they|it|we|i)\s+(said|told|asked|mentioned|replied)',
        r'^(was|were|is|are)\s+(out of|available|unavailable|closed)',
        r'^if\s+(it|they|she|he|we)\s+(was|were|is|are)',
        r'^that\s+(it|they|she|he|we)',
        r'^\d+\s*/',  # Rating like "10/"
        r'^with\s+(a\s+)?side',  # "with a side of"
    ]

    opinion_text = opinion.lower()
    for pattern in factual_patterns:
        if re.search(pattern, opinion_text):
            return False, 'factual_statement'

    # 規則 4: 必須包含至少一個形容詞/副詞或情緒詞
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()
    all_sentiment = positive_words | negative_words

    has_sentiment_word = False
    for word in words:
        clean_word = word.strip('.,!?;:\'"')
        if clean_word in all_sentiment:
            has_sentiment_word = True
            break
        is_adj, _ = is_adjective_or_adverb(clean_word)
        if is_adj:
            has_sentiment_word = True
            break

    # 短句如果沒有情緒詞，可能無效
    if len(words) <= 4 and not has_sentiment_word:
        # 但檢查是否為慣用語
        is_idiom, _, _ = check_idiom_first(opinion)
        if not is_idiom:
            return False, 'no_sentiment_signal'

    return True, None


def is_valid_opinion(opinion):
    """Validate whether extracted text is a genuine opinion."""
    if not opinion or len(opinion.strip()) < 2:
        return False

    # 🆕 首先進行完整性驗證
    is_complete, _ = validate_opinion_completeness(opinion)
    if not is_complete:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    if not words:
        return False

    action_verbs = get_action_verbs()
    function_words = get_function_words()
    common_names = get_common_english_names()
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()
    all_sentiment_words = positive_words | negative_words

    # 單詞驗證
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')
        if clean_word in action_verbs:
            return False
        if clean_word in function_words:
            return False
        if clean_word in common_names:
            return False
        is_adj, _ = is_adjective_or_adverb(clean_word)
        if not is_adj and clean_word not in all_sentiment_words:
            return False

    # 雙詞驗證
    if len(words) == 2:
        clean_words = [w.strip('.,!?;:\'"') for w in words]
        non_content_words = action_verbs | function_words
        if all(w in non_content_words for w in clean_words):
            return False
        if all(w in common_names for w in clean_words):
            return False

    # 全為動詞
    clean_words = [w.strip('.,!?;:\'"') for w in words]
    action_count = sum(1 for w in clean_words if w in action_verbs)
    if action_count == len(words):
        return False

    # 無效模式
    invalid_patterns = [
        r"^'s\s+",
        r"^\d+\s*/",
        r"^or\s+maybe\s+both$",
        r"^going\s+forward$",
        r"^hoped\s+would$",
        r"^asked\s+if$",
        r"^came\s+out\s+and\s+it$",
        r"^for\s+dessert$",
        r"^to\s+before\s+me$",
        r"^and\s+waited$",
        r"^member\s+where",
        r"^n\s+cheese",  # 🆕 修正 "n cheese" 這種截斷
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    return True


# ============================================
# SECTION 10: ASPECT VALIDATION
# ============================================

def is_valid_aspect(aspect):
    """Validate whether extracted text is a genuine aspect."""
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    if not cleaned or cleaned == "-":
        return False

    if all(c in '.,!?;:-_\'"' for c in cleaned):
        return False

    if cleaned.startswith('and ') or cleaned.startswith('or '):
        return False

    # 🆕 新增：排除介係詞和連接詞作為 aspect
    invalid_single_words = {'of', 'to', 'for', 'with', 'at', 'in', 'on', 'by', 'and', 'or', 'but', 'n'}
    if cleaned in invalid_single_words:
        return False

    invalid_aspects = {
        'wait', 'waited', 'waiting',
        'order', 'ordered', 'ordering',
        'serve', 'served', 'serving',
        'ask', 'asked', 'asking',
        'walk', 'walked', 'walking',
        'make', 'made', 'making',
        'have', 'having', 'had',
        'come', 'came', 'coming',
        'go', 'went', 'going',
        'i', 'me', 'my', 'mine',
        'you', 'your', 'yours',
        'he', 'him', 'his',
        'she', 'her', 'hers',
        'it', 'its',
        'we', 'us', 'our', 'ours',
        'they', 'them', 'their', 'theirs',
        'someone', 'anyone', 'everyone', 'nobody',
        'something', 'anything', 'everything', 'nothing',
        'the', 'a', 'an',
        'min', 'mins', 'minute', 'minutes',
        'hour', 'hours',
        'second', 'seconds',
        'day', 'days',
        'week', 'weeks',
        'month', 'months',
        'year', 'years',
        'good', 'bad', 'great', 'nice', 'poor',
        'fast', 'slow', 'hot', 'cold', 'new', 'old',
        'big', 'small', 'large', 'little',
        'thing', 'things', 'stuff',
        'way', 'ways',
        'time', 'times',
        'place', 'places',
        'part', 'parts'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """Clean and standardize an aspect term."""
    if not aspect:
        return ""

    clean_aspect = aspect.strip()
    clean_aspect = re.sub(r'^(and|or|but)\s+', '', clean_aspect, flags=re.IGNORECASE)
    units_pattern = r'^(?:[\d\.]+\s*-?\s*)?(?:inch|inches|cm|mm|kg|lbs|lb|oz|g|ml)\b\s*'
    clean_aspect = re.sub(units_pattern, '', clean_aspect, flags=re.IGNORECASE)
    stopwords_pattern = r'^(the|a|an|this|that|my|our|their|his|her|your|some|any)\s+'
    clean_aspect = re.sub(stopwords_pattern, '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^[-:;,.\'"]+\s*', '', clean_aspect)
    clean_aspect = re.sub(r'[-:;,.\'"]+$', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# SECTION 11: OPINION EXTRACTION
# ============================================

def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """Extract the opinion associated with an aspect, respecting boundaries."""
    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1] if len(aspect_positions) > 1 else aspect_positions[0]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)
    prev_aspect_end = find_previous_aspect_end(start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(end_idx, all_aspect_positions)

    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # PATTERN 1: Pre-modifier
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, modifier_type = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in {',', 'and', 'with', 'or', 'but'}:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

        next_pos = end_idx + 1
        if next_pos < sentence_end:
            next_word = tokens[next_pos]
            forbidden_next = {
                'is', 'was', 'are', 'were', 'be', 'been',
                'has', 'have', 'had', 'does', 'do',
                'but', 'and', 'or', 'so', 'because',
                'with', 'for', 'to', 'in', 'at', 'on', 'of', 'by',
                'that', 'which', 'who', 'this', 'it',
                ',', '.', '!', '?', ';', '-'
            }
            if next_word.isalpha() and next_word.lower() not in forbidden_next:
                opinion_words.append(next_word)

    # PATTERN 2: Be-verb + Adjective
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []

        if is_negative:
            post_modifiers.append("not")

        for i in range(opinion_start, opinion_end):
            word = tokens[i]

            if word in {'.', '!', '?', ';'}:
                break

            if word.lower() == 'but' and next_aspect_start is not None:
                if i + 1 < len(tokens) and i + 1 >= next_aspect_start - 2:
                    break

            post_modifiers.append(word)

        if post_modifiers:
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # PATTERN 3: Verb Phrase
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            temp_words.append(word)

        if temp_words:
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # PATTERN 4: Context Search
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, _ = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                context_words = []
                for j in range(context_start, context_end):
                    if tokens[j] not in {'.', '!', '?', ';'}:
                        context_words.append(tokens[j])

                opinion_words.extend(context_words)
                pattern_type = "context_search"
                break

            if word in {'.', '!', '?'}:
                break

    opinion_words = handle_that_with_negation(opinion_words)
    opinion_phrase = ' '.join(opinion_words).strip()

    return opinion_phrase, pattern_type


def handle_that_with_negation(words):
    """Process 'that + negated be-verb' patterns."""
    if not words:
        return words

    result = []
    i = 0

    while i < len(words):
        word = words[i]
        word_lower = word.lower()

        if word_lower == 'that' and i + 1 < len(words):
            next_word = words[i + 1].lower()
            negated_be = {"isn't", "isn't", "wasn't", "wasn't", "aren't", "aren't", "weren't", "weren't"}

            if next_word in negated_be:
                result.append("not")
                i += 2
                continue

        result.append(word)
        i += 1

    return result


# ============================================
# SECTION 12: ENHANCED OPINION FORMATTING
# ============================================

def format_opinion_for_display(opinion):
    """Format and clean an extracted opinion for display."""
    if not opinion:
        return ""

    # 🆕 移除開頭的連接詞
    opinion = re.sub(r'^(and|or|but|so)\s+', '', opinion, flags=re.IGNORECASE)

    opinion = re.sub(r'([a-z])-(?=[A-Z])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'([a-zA-Z0-9])-(?=[a-zA-Z0-9])', r'\1 - ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    words = opinion.split()
    if not words:
        return ""

    leading_fillers = {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'also', 'too', 'either',
        'has', 'have', 'had', 'is', 'are', 'was', 'were',
        'looks', 'feels', 'seems',
        'contains', 'includes', 'consists',
        'offers', 'provides', 'features',
        'it', 'itself', 'they', 'themselves', 'we', 'ourselves',
        'i', 'myself', 'you', 'yourself',
        # 🆕 新增
        'and', 'or', 'but', 'so'
    }

    while words and words[0].lower() in leading_fillers:
        words.pop(0)

    if not words:
        return ""

    do_not_merge = {
        'a', 'an', 'the', 'it', 'is', 'we', 'i', 'you', 'he', 'she',
        'of', 'in', 'to', 'for', 'with', 'on', 'at', 'my', 'our'
    }

    cleaned_words = []
    i = 0
    while i < len(words):
        word = words[i]

        if i + 1 < len(words):
            next_word = words[i + 1]
            if word.endswith('-') and len(word) > 1:
                if next_word.isdigit() or next_word.lower() in do_not_merge:
                    cleaned_words.append(word.rstrip('-'))
                else:
                    merged = word + next_word
                    cleaned_words.append(merged)
                    i += 2
                    continue
            elif next_word == '-' and i + 2 < len(words):
                following = words[i + 2]
                if following.isdigit() or following.lower() in do_not_merge:
                    cleaned_words.append(word)
                    cleaned_words.append('-')
                    i += 2
                    continue
                merged = word + "-" + following
                cleaned_words.append(merged)
                i += 3
                continue

        cleaned_words.append(word)
        i += 1

    words = cleaned_words

    cutoff_words = {
        'making', 'causing', 'forcing', 'leaving', 'rendering',
        'unless', 'except', 'besides', 'despite', 'although',
        'which', 'where', 'when', 'because', 'since'
    }

    truncated = []
    valid_length = 0

    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')

        if word == '-' or word.startswith('-'):
            if i + 1 < len(words):
                next_w = words[i + 1].lower()
                if next_w in {'i', 'we', 'it', 'he', 'she', 'they'}:
                    break
            truncated.append(word)
            continue

        if word_lower in cutoff_words and valid_length >= 2:
            break

        truncated.append(word)
        if word not in {',', 'and', 'but', '-'}:
            valid_length += 1

    words = truncated

    final_words = []
    for word in words:
        if word == ',':
            if final_words and not final_words[-1].endswith(','):
                final_words[-1] = final_words[-1] + ','
        else:
            clean_word = word.lstrip(',')
            if clean_word:
                final_words.append(clean_word)

    words = final_words

    trailing_garbage = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'too', 'either', 'also', 'but', 'however',
        'a', 'an', 'the',
        ',', '-',
        'is', 'are', 'was', 'were',
        'need', 'justify', 'make', 'do', 'want', 'require', 'expect',
        'produces', 'provides', 'offers', 'features', 'includes',
        "'", '"', '(', 'it', 'i', 'we', 'they', 'he', 'she'
    }

    important_nouns = {
        'experience', 'quality', 'service', 'food', 'price',
        'taste', 'flavor', 'portion', 'value', 'atmosphere',
        'staff', 'server', 'waiter', 'waitress', 'manager',
        'location', 'ambiance', 'decor', 'menu', 'selection',
        'meal', 'dish', 'order', 'drink', 'dessert'
    }

    while words:
        last_word = words[-1].lower().strip('.,!?;:')

        if last_word in important_nouns:
            break

        should_remove = False
        if last_word in trailing_garbage:
            should_remove = True
        elif words[-1].endswith('-'):
            should_remove = True
        elif words[-1].isdigit():
            should_remove = True

        if words[-1].endswith(','):
            words[-1] = words[-1].rstrip(',')
            if not words[-1]:
                words.pop()
                continue

        if should_remove:
            words.pop()
        else:
            break

    result_text = ' '.join(words)
    connectors = [' but ', ' however ', ' though ', ' and ']

    for connector in connectors:
        if connector in result_text.lower():
            last_idx = result_text.lower().rfind(connector)
            before = result_text[:last_idx].strip()
            after = result_text[last_idx + len(connector):].strip().split()

            should_cut = False
            if not after:
                should_cut = True
            elif len(after) <= 2:
                bad_enders = {'i', 'it', 'we', 'he', 'she', 'they', 'a', 'an', 'the'}
                if any(w.lower() in bad_enders for w in after):
                    should_cut = True

            comparatives = {
                'more', 'less', 'better', 'worse', 'higher', 'lower',
                'deeper', 'brighter', 'darker', 'faster', 'slower',
                'larger', 'smaller', 'stronger', 'weaker'
            }
            if any(w.lower() in comparatives for w in after):
                should_cut = False

            if should_cut:
                result_text = before

    words = result_text.split()
    if len(words) > 12 and ' but ' not in result_text.lower():
        words = words[:12]
        result_text = ' '.join(words)

    result_text = re.sub(r',\s*,', ',', result_text)
    result_text = result_text.strip().rstrip('.,;-\'"(')

    # 🆕 最後再次確認不以連接詞開頭
    result_text = re.sub(r'^(and|or|but|so)\s+', '', result_text, flags=re.IGNORECASE)

    return result_text


# ============================================
# SECTION 13: SENTIMENT ANALYSIS
# ============================================

def analyze_sentiment_scores(text):
    """Analyze text sentiment using multiple tools."""
    analyzers = get_sentiment_analyzers()
    results = {}

    if not text or len(text.strip()) < 2:
        return {
            'vader': None,
            'textblob': None,
            'transformer': None,
            'ensemble': {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}
        }

    if analyzers.get('vader'):
        try:
            vader_scores = analyzers['vader'].polarity_scores(text)
            results['vader'] = {
                'compound': vader_scores['compound'],
                'positive': vader_scores['pos'],
                'negative': vader_scores['neg'],
                'neutral': vader_scores['neu']
            }
        except Exception:
            results['vader'] = None
    else:
        results['vader'] = None

    if analyzers.get('textblob'):
        try:
            blob = analyzers['textblob'](text)
            results['textblob'] = {
                'polarity': blob.sentiment.polarity,
                'subjectivity': blob.sentiment.subjectivity
            }
        except Exception:
            results['textblob'] = None
    else:
        results['textblob'] = None

    if analyzers.get('transformer'):
        try:
            truncated = text[:500] if len(text) > 500 else text
            trans_result = analyzers['transformer'](truncated)[0]

            if trans_result['label'] == 'POSITIVE':
                normalized = trans_result['score']
            else:
                normalized = -trans_result['score']

            results['transformer'] = {
                'label': trans_result['label'],
                'score': trans_result['score'],
                'normalized_score': normalized
            }
        except Exception:
            results['transformer'] = None
    else:
        results['transformer'] = None

    results['ensemble'] = calculate_ensemble_score(results)

    return results


def calculate_ensemble_score(sentiment_results):
    """Calculate weighted ensemble sentiment score."""
    scores = []
    weights = []

    if sentiment_results.get('vader') and sentiment_results['vader'].get('compound') is not None:
        scores.append(sentiment_results['vader']['compound'])
        weights.append(0.4)

    if sentiment_results.get('textblob') and sentiment_results['textblob'].get('polarity') is not None:
        scores.append(sentiment_results['textblob']['polarity'])
        weights.append(0.3)

    if sentiment_results.get('transformer') and sentiment_results['transformer'].get('normalized_score') is not None:
        scores.append(sentiment_results['transformer']['normalized_score'])
        weights.append(0.3)

    if not scores:
        return {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}

    total_weight = sum(weights)
    normalized_weights = [w / total_weight for w in weights]

    ensemble_score = sum(s * w for s, w in zip(scores, normalized_weights))

    if len(scores) > 1:
        variance = sum((s - ensemble_score) ** 2 for s in scores) / len(scores)
        confidence = max(0, 1 - variance)
    else:
        confidence = 0.7

    if ensemble_score >= 0.05:
        label = 'Positive'
    elif ensemble_score <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return {
        'score': round(ensemble_score, 4),
        'label': label,
        'confidence': round(confidence, 4)
    }


# ============================================
# SECTION 14: ENHANCED SENTIMENT CORRECTION
# ============================================

def correct_sentiment_with_analysis(aspect, opinion, predicted_sentiment, sentiment_scores=None):
    """
    🆕 改進版情緒修正函數
    優先順序：慣用語 > 特殊句型 > 否定邏輯 > 關鍵字 > 分數覆蓋
    """
    opinion_lower = opinion.lower()

    # ===== 最高優先：慣用語檢查 =====
    is_idiom, idiom_sentiment, idiom_matched = check_idiom_first(opinion_lower)
    if is_idiom:
        if predicted_sentiment != idiom_sentiment:
            return idiom_sentiment, f'idiom_override ({idiom_matched})'
        return predicted_sentiment, None

    # ===== 特殊句型檢查 =====
    # "could not have been [positive_word]" = 極度正面
    superlative_patterns = [
        r"could(n't| not) have been (more )?(friendly|helpful|nicer|better|attentive|professional)",
        r"could(n't| not) ask for (more|better)",
        r"could(n't| not) be (happier|better)",
    ]
    for pattern in superlative_patterns:
        if re.search(pattern, opinion_lower):
            if predicted_sentiment != 'Positive':
                return 'Positive', f'superlative_positive'
            return predicted_sentiment, None

    # Get sentiment word lists
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()

    # ===== 否定邏輯（更謹慎應用） =====
    has_negation, neg_type = detect_negation_in_opinion(opinion_lower)

    if has_negation and not neg_type.startswith('idiom:'):
        # 檢查是否否定正面詞 -> 負面
        for pos_word in positive_words:
            if pos_word in opinion_lower:
                # 但要排除 "not only ... but also" 這種句型
                if "not only" in opinion_lower:
                    break
                return 'Negative', f'negation_positive ({pos_word})'

        # 檢查是否否定負面詞 -> 正面（雙重否定）
        for neg_word in negative_words:
            if neg_word in opinion_lower:
                return 'Positive', f'double_negation ({neg_word})'

        # 有否定但無明確情緒詞 - 通常是負面
        # 但要更謹慎，檢查分數
        if sentiment_scores:
            score = sentiment_scores.get('ensemble', {}).get('score', 0)
            if score > 0.3:
                # 分數偏正，可能是誤判
                return predicted_sentiment, None
        return 'Negative', f'negation_general ({neg_type})'

    # ===== 分數覆蓋 =====
    if sentiment_scores:
        ensemble = sentiment_scores.get('ensemble', {})
        ensemble_score = ensemble.get('score', 0)
        confidence = ensemble.get('confidence', 0)

        if confidence >= SENTIMENT_CONFIDENCE_THRESHOLD:
            if predicted_sentiment == 'Positive' and ensemble_score < -SENTIMENT_OVERRIDE_THRESHOLD:
                has_strong_positive = any(pw in opinion_lower for pw in positive_words)
                if not has_strong_positive:
                    return 'Negative', f'sentiment_override ({ensemble_score:.2f})'

            if predicted_sentiment == 'Negative' and ensemble_score > SENTIMENT_OVERRIDE_THRESHOLD:
                has_strong_negative = any(nw in opinion_lower for nw in negative_words)
                if not has_strong_negative:
                    return 'Positive', f'sentiment_override ({ensemble_score:.2f})'

    # ===== 關鍵字檢測 =====
    for pos_word in positive_words:
        if pos_word in opinion_lower:
            if predicted_sentiment != 'Positive':
                return 'Positive', f'positive_keyword ({pos_word})'

    for neg_word in negative_words:
        if neg_word in opinion_lower:
            if predicted_sentiment != 'Negative':
                return 'Negative', f'negative_keyword ({neg_word})'

    # ===== 'But' 子句分析 =====
    if ' but ' in opinion_lower:
        parts = opinion_lower.split(' but ', 1)
        if len(parts) == 2:
            after_but = parts[1]

            for neg_word in negative_words:
                if neg_word in after_but:
                    return 'Negative', f'but_negative ({neg_word})'

            for pos_word in positive_words:
                if pos_word in after_but:
                    return 'Positive', f'but_positive ({pos_word})'

    return predicted_sentiment, None


# ============================================
# SECTION 15: CONFIDENCE-BASED FILTERING (NEW)
# ============================================

def should_include_extraction(sentiment_scores, opinion, predicted_sentiment):
    """
    🆕 新增：基於信心度決定是否保留萃取結果
    """
    if not sentiment_scores:
        return True, None

    ensemble = sentiment_scores.get('ensemble', {})
    score = ensemble.get('score', 0)
    confidence = ensemble.get('confidence', 0)

    # 規則 1: 分數太接近 0，情緒不明確
    if abs(score) < MINIMUM_SENTIMENT_SCORE_THRESHOLD:
        # 但如果是慣用語，則保留
        is_idiom, _, _ = check_idiom_first(opinion.lower())
        if not is_idiom:
            return False, f'low_confidence_score (|{score:.2f}| < {MINIMUM_SENTIMENT_SCORE_THRESHOLD})'

    # 規則 2: 分析器之間意見分歧太大
    if confidence < MINIMUM_AGREEMENT_THRESHOLD:
        # 但如果有明確的情緒詞，則保留
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        all_sentiment = positive_words | negative_words

        has_clear_word = any(w in opinion.lower() for w in all_sentiment)
        if not has_clear_word:
            return False, f'low_agreement ({confidence:.2f} < {MINIMUM_AGREEMENT_THRESHOLD})'

    return True, None


# ============================================
# SECTION 16: RESULT DATA STRUCTURES
# ============================================

def create_aspect_opinion_pair(aspect, opinion, raw_opinion, sentiment,
                                original_sentiment, pattern, chunk_id,
                                sentiment_scores, correction_reason,
                                filtered_reason=None):
    """Create a standardized result dictionary for an aspect-opinion pair."""
    vader_compound = None
    textblob_polarity = None
    transformer_score = None
    ensemble_score = None
    ensemble_confidence = None

    if sentiment_scores:
        if sentiment_scores.get('vader'):
            vader_compound = sentiment_scores['vader'].get('compound')
        if sentiment_scores.get('textblob'):
            textblob_polarity = sentiment_scores['textblob'].get('polarity')
        if sentiment_scores.get('transformer'):
            transformer_score = sentiment_scores['transformer'].get('normalized_score')
        if sentiment_scores.get('ensemble'):
            ensemble_score = sentiment_scores['ensemble'].get('score')
            ensemble_confidence = sentiment_scores['ensemble'].get('confidence')

    return {
        'aspect': aspect,
        'opinion': opinion,
        'raw_opinion': raw_opinion,
        'formatted': f"{aspect}: {opinion}",
        'pattern': pattern,
        'chunk': chunk_id,
        'original_sentiment': original_sentiment,
        'sentiment': sentiment,
        'correction_reason': correction_reason,
        'filtered_reason': filtered_reason,  # 🆕 新增
        'sentiment_scores': {
            'vader_compound': vader_compound,
            'textblob_polarity': textblob_polarity,
            'transformer_score': transformer_score,
            'ensemble_score': ensemble_score,
            'ensemble_confidence': ensemble_confidence
        }
    }


# ============================================
# SECTION 17: FILE LOGGING
# ============================================

class FileLogger:
    """Simple logger that writes to both console and file."""

    def __init__(self, filename):
        self.filename = filename
        self.content = []

    def log(self, message=""):
        self.content.append(message)
        print(message)

    def save(self):
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ Results saved to: {self.filename}")


# ============================================
# SECTION 18: REVIEW CHUNKING
# ============================================

def split_long_review(text, max_words=MAX_WORDS_PER_CHUNK):
    """Split long reviews into manageable chunks."""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub_sentence = ' '.join(words[i:i + max_words])
                chunks.append(sub_sentence + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# SECTION 19: MAIN ANALYSIS FUNCTION
# ============================================

def comprehensive_aspect_opinion_analysis(reviews, aspect_extractor,
                                          max_words=MAX_WORDS_PER_CHUNK,
                                          verbose=True, logger=None,
                                          use_sentiment_analysis=True,
                                          apply_confidence_filter=True):  # 🆕 新增參數
    """Run complete ABSA analysis on a collection of reviews."""

    if logger:
        logger.log("=" * 80)
        logger.log("ABSA Analysis System - V5 Enhanced Edition")
        logger.log(f"Chunk size: {max_words} words")
        logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
        logger.log(f"Confidence filter: {'Enabled' if apply_confidence_filter else 'Disabled'}")  # 🆕
        logger.log(f"Min score threshold: {MINIMUM_SENTIMENT_SCORE_THRESHOLD}")  # 🆕
        logger.log(f"Analysis time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 80)

    if use_sentiment_analysis:
        _ = get_sentiment_analyzers()

    all_results = []
    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'filtered_low_confidence': 0,  # 🆕 新增
        'sentiment_corrections': 0,
        'correction_reasons': Counter(),
        'idiom_detections': 0,  # 🆕 新增
    }

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'=' * 80}")
            logger.log(f"Review #{idx} ({word_count} words)")
            logger.log("=" * 80)

        if word_count <= max_words:
            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]
            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if logger and verbose:
                logger.log("📄 Splitting long review into chunks...")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   Split into {len(chunks)} chunks")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = [
                {'chunk_id': i + 1, 'result': r}
                for i, r in enumerate(chunk_results)
            ]

        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0
        filtered_confidence_count = 0  # 🆕

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, original_sentiment in zip(aspects, positions, sentiments):

                aspect = refine_aspect_term(aspect)

                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                display_opinion = format_opinion_for_display(raw_opinion)

                if not display_opinion or not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                sentiment_scores = None
                if use_sentiment_analysis:
                    sentiment_scores = analyze_sentiment_scores(display_opinion)

                # 🆕 信心度過濾
                if apply_confidence_filter and sentiment_scores:
                    should_include, filter_reason = should_include_extraction(
                        sentiment_scores, display_opinion, original_sentiment
                    )
                    if not should_include:
                        filtered_confidence_count += 1
                        stats['filtered_low_confidence'] += 1
                        continue

                corrected_sentiment, correction_reason = correct_sentiment_with_analysis(
                    aspect, raw_opinion, original_sentiment, sentiment_scores
                )

                if corrected_sentiment != original_sentiment:
                    stats['sentiment_corrections'] += 1
                    if correction_reason:
                        stats['correction_reasons'][correction_reason] += 1
                        if 'idiom' in correction_reason:
                            stats['idiom_detections'] += 1

                stats['total_aspects'] += 1

                pair = create_aspect_opinion_pair(
                    aspect=aspect,
                    opinion=display_opinion,
                    raw_opinion=raw_opinion,
                    sentiment=corrected_sentiment,
                    original_sentiment=original_sentiment,
                    pattern=pattern,
                    chunk_id=chunk_data['chunk_id'] if len(analysis_results) > 1 else None,
                    sentiment_scores=sentiment_scores,
                    correction_reason=correction_reason
                )
                aspect_opinion_pairs.append(pair)

        if logger and verbose:
            logger.log(f"\n🎯 Extraction Results:")
            header = f"{'#':<3} {'Aspect':<18} {'Opinion':<35} {'Orig':<8} {'Final':<8} {'Score':<8}"
            logger.log(header)
            logger.log("-" * 90)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    opinion_disp = pair['opinion'][:32] + "..." if len(pair['opinion']) > 35 else pair['opinion']

                    scores = pair['sentiment_scores']
                    if scores and scores.get('ensemble_score') is not None:
                        score_str = f"{scores['ensemble_score']:+.2f}"
                    else:
                        score_str = "N/A"

                    final = pair['sentiment']
                    if pair['correction_reason']:
                        final += "*"

                    logger.log(f"{i:<3} {pair['aspect']:<18} {opinion_disp:<35} "
                              f"{pair['original_sentiment']:<8} {final:<8} {score_str:<8}")

                total_filtered = filtered_aspect_count + filtered_opinion_count + filtered_confidence_count
                if total_filtered > 0:
                    logger.log(f"\n   ℹ️ Filtered: {filtered_aspect_count} aspects, "
                              f"{filtered_opinion_count} opinions, "
                              f"{filtered_confidence_count} low confidence")
            else:
                logger.log("   ⚠️ No valid aspect-opinion pairs found")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
            'filtered_aspect_count': filtered_aspect_count,
            'filtered_opinion_count': filtered_opinion_count,
            'filtered_confidence_count': filtered_confidence_count  # 🆕
        })

    return all_results, stats


# ============================================
# SECTION 20: ENHANCED REPORT GENERATION
# ============================================

def select_representative_opinions(pairs, n=3):
    """
    🆕 新增：選擇 n 個最具代表性的意見
    """
    # 按分數絕對值排序
    sorted_pairs = sorted(
        pairs,
        key=lambda p: abs(p['sentiment_scores'].get('ensemble_score', 0) if p['sentiment_scores'] else 0),
        reverse=True
    )

    quality_pairs = []
    for p in sorted_pairs:
        opinion = p['opinion']

        # 排除過短
        if len(opinion.split()) < 2:
            continue

        # 排除以連接詞開頭
        if opinion.lower().strip().startswith(('and ', 'or ', 'but ', 'so ')):
            continue

        # 排除過度截斷
        if opinion.count('...') > 0 and len(opinion) < 20:
            continue

        # 排除非意見性文字
        is_complete, _ = validate_opinion_completeness(opinion)
        if not is_complete:
            continue

        quality_pairs.append(p)

    # 去重
    unique_opinions = []
    seen_stems = set()
    for p in quality_pairs:
        stem = ' '.join(p['opinion'].lower().split()[:3])
        if stem not in seen_stems:
            seen_stems.add(stem)
            unique_opinions.append(p)

    return unique_opinions[:n]


def generate_management_report(results, stats, logger=None):
    """Generate enhanced management-focused analysis report."""
    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 80)
    log("MANAGEMENT ANALYSIS REPORT (Enhanced)")
    log("=" * 80)

    positive_pairs = []
    negative_pairs = []

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

    total_pairs = len(positive_pairs) + len(negative_pairs)

    log(f"\n📊 OVERALL STATISTICS")
    log(f"   • Total reviews analyzed: {len(results)}")
    log(f"   • Total aspects identified: {total_pairs}")
    log(f"   • Invalid aspects filtered: {stats['filtered_invalid_aspects']}")
    log(f"   • Invalid opinions filtered: {stats['filtered_invalid_opinions']}")
    log(f"   • Low confidence filtered: {stats.get('filtered_low_confidence', 0)}")  # 🆕

    if stats['sentiment_corrections'] > 0:
        log(f"   • Sentiments corrected: {stats['sentiment_corrections']}")
        log(f"   • Idiom detections: {stats.get('idiom_detections', 0)}")  # 🆕

    if total_pairs > 0:
        pos_pct = len(positive_pairs) / total_pairs * 100
        neg_pct = len(negative_pairs) / total_pairs * 100
        log(f"   • Positive mentions: {len(positive_pairs)} ({pos_pct:.1f}%)")
        log(f"   • Negative mentions: {len(negative_pairs)} ({neg_pct:.1f}%)")

    if stats['correction_reasons']:
        log(f"\n📈 CORRECTION BREAKDOWN")
        for reason, count in stats['correction_reasons'].most_common(10):
            log(f"   • {reason}: {count}")

    # Aggregate by aspect
    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair)

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair)

    # 🆕 改進：顯示多個代表性意見
    if positive_aspects:
        log(f"\n✅ COMPETITIVE ADVANTAGES (Top Positive Aspects)")
        log("-" * 80)

        sorted_pos = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_pos[:10], 1):
            count = len(pairs)

            scores = [
                p['sentiment_scores']['ensemble_score']
                for p in pairs
                if p['sentiment_scores'].get('ensemble_score') is not None
            ]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg score: {avg_score:+.2f})")

            # 選擇代表性意見
            rep_opinions = select_representative_opinions(pairs, n=3)
            for rep in rep_opinions:
                opinion_text = rep['opinion'][:60]
                if len(rep['opinion']) > 60:
                    opinion_text += "..."
                log(f"   • \"{opinion_text}\"")

    if negative_aspects:
        log(f"\n⚠️ AREAS FOR IMPROVEMENT (Top Negative Aspects)")
        log("-" * 80)

        sorted_neg = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_neg[:10], 1):
            count = len(pairs)

            scores = [
                p['sentiment_scores']['ensemble_score']
                for p in pairs
                if p['sentiment_scores'].get('ensemble_score') is not None
            ]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg score: {avg_score:+.2f})")

            rep_opinions = select_representative_opinions(pairs, n=3)
            for rep in rep_opinions:
                opinion_text = rep['opinion'][:60]
                if len(rep['opinion']) > 60:
                    opinion_text += "..."
                log(f"   • \"{opinion_text}\"")

    log(f"\n💡 RECOMMENDATIONS")

    if negative_aspects:
        worst = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   1. PRIORITY: Address '{worst[0]}' ({len(worst[1])} negative mentions)")

    if positive_aspects:
        best = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   2. LEVERAGE: Promote '{best[0]}' ({len(best[1])} positive mentions)")

    log(f"   3. MONITOR: Track sentiment score trends over time")
    log(f"   4. INVESTIGATE: Review corrected sentiments for accuracy")

    log(f"\n📉 EXTRACTION QUALITY METRICS")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   • Extraction success rate: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   • Average aspects per review: {avg_pairs:.2f}")

    # 🆕 新增品質指標
    total_filtered = (stats['filtered_invalid_aspects'] +
                      stats['filtered_invalid_opinions'] +
                      stats.get('filtered_low_confidence', 0))
    total_extracted = total_pairs + total_filtered
    if total_extracted > 0:
        quality_rate = total_pairs / total_extracted * 100
        log(f"   • Quality pass rate: {total_pairs}/{total_extracted} ({quality_rate:.1f}%)")


# ============================================
# SECTION 21: DATA EXPORT
# ============================================

def export_results_to_dataframe(results):
    """Export analysis results to a pandas DataFrame."""
    import pandas as pd

    rows = []

    for result in results:
        review_id = result['review_id']

        for pair in result['pairs']:
            scores = pair.get('sentiment_scores', {})

            row = {
                'review_id': review_id,
                'aspect': pair['aspect'],
                'opinion': pair['opinion'],
                'raw_opinion': pair['raw_opinion'],
                'original_sentiment': pair['original_sentiment'],
                'corrected_sentiment': pair['sentiment'],
                'correction_reason': pair.get('correction_reason'),
                'extraction_pattern': pair['pattern'],
                'chunk_id': pair.get('chunk'),
                'vader_compound': scores.get('vader_compound'),
                'textblob_polarity': scores.get('textblob_polarity'),
                'transformer_score': scores.get('transformer_score'),
                'ensemble_score': scores.get('ensemble_score'),
                'ensemble_confidence': scores.get('ensemble_confidence')
            }
            rows.append(row)

    return pd.DataFrame(rows)


# ============================================
# SECTION 22: MAIN EXECUTION
# ============================================

def run_analysis(reviews, aspect_extractor,
                 output_file="absa_analysis_results_v5.txt",
                 use_sentiment_analysis=True,
                 apply_confidence_filter=True):
    """Run complete ABSA analysis and generate outputs."""

    logger = FileLogger(output_file)

    logger.log("=" * 80)
    logger.log("ABSA ANALYSIS SYSTEM - V5 ENHANCED EDITION")
    logger.log(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"Reviews to analyze: {len(reviews)}")
    logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
    logger.log(f"Confidence filter: {'Enabled' if apply_confidence_filter else 'Disabled'}")
    logger.log("=" * 80)

    results, stats = comprehensive_aspect_opinion_analysis(
        reviews,
        aspect_extractor,
        max_words=MAX_WORDS_PER_CHUNK,
        verbose=True,
        logger=logger,
        use_sentiment_analysis=use_sentiment_analysis,
        apply_confidence_filter=apply_confidence_filter
    )

    logger.log("\n" + "=" * 80)
    logger.log("✅ ANALYSIS COMPLETE")
    logger.log("=" * 80)

    generate_management_report(results, stats, logger)

    logger.save()

    df_results = export_results_to_dataframe(results)

    return results, stats, df_results


if __name__ == "__main__":
    print("=" * 60)
    print("ABSA Analysis System - V5 Enhanced Edition")
    print("=" * 60)
    print("\n改進項目：")
    print("  1. 新增慣用語/俚語辭典處理")
    print("  2. 新增信心度過濾機制")
    print("  3. 改進Opinion邊界檢測")
    print("  4. 增強管理報告（含多個代表性意見）")
    print("  5. 修正情緒修正邏輯")
    print("\n使用方式：")
    print("  results, stats, df = run_analysis(test_reviews, aspect_extractor)")

In [ ]:
results, stats, df = run_analysis(test_reviews, aspect_extractor)

In [ ]:
"""
================================================================================
ABSA (Aspect-Based Sentiment Analysis) System - V5 Enhanced Universal Edition
================================================================================

Purpose:
    Extract aspect-opinion-sentiment triplets from restaurant reviews.
    Designed for UNIVERSAL application across ALL restaurant types:
    - Fast food, casual dining, fine dining
    - All cuisines (American, Asian, Mexican, Italian, etc.)
    - All service models (dine-in, takeout, delivery, food trucks)

Key Enhancements in V5:
    1. Idiom/Slang Dictionary - Handles "did not disappoint", "killed it", etc.
    2. Confidence-Based Filtering - Excludes low-confidence extractions
    3. Improved Opinion Boundary Detection - Avoids fragments and factual statements
    4. Enhanced Management Report - Multiple representative opinions per aspect
    5. Universal Restaurant Vocabulary - Multi-cuisine support

Version: 5.0 Enhanced Universal
Author: HAOS Framework Research Team
Date: 2025
================================================================================
"""

import warnings
warnings.filterwarnings('ignore')

import re
from collections import Counter
from datetime import datetime

# ============================================
# SECTION 1: GLOBAL CONFIGURATION
# ============================================

OUTPUT_FILE = "absa_analysis_results_v5_universal.txt"
MAX_WORDS_PER_CHUNK = 80

# Sentiment correction thresholds
SENTIMENT_CONFIDENCE_THRESHOLD = 0.6
SENTIMENT_OVERRIDE_THRESHOLD = 0.4

# NEW: Confidence-based filtering thresholds
# Extractions with |score| < this threshold will be excluded
MINIMUM_SENTIMENT_SCORE_THRESHOLD = 0.25

# Minimum agreement level between sentiment analyzers
MINIMUM_AGREEMENT_THRESHOLD = 0.40


# ============================================
# SECTION 2: IDIOM/SLANG DICTIONARIES
# ============================================
"""
CRITICAL SECTION: Idiom Detection

Problem Solved:
    Many English expressions use negation words but convey POSITIVE meaning.
    Example: "The service did not disappoint" = POSITIVE (not negative!)

These idioms are universal across ALL restaurant types:
    - Fast food: "The fries hit the spot"
    - Fine dining: "The tasting menu was to die for"
    - BBQ: "Ribs fall off the bone"
    - Any cuisine: "Our server killed it tonight"
"""


def get_positive_idioms():
    """
    Dictionary of positive idioms/slang expressions.

    Categories:
    1. Double Negative = Positive ("did not disappoint")
    2. Superlative Negation = Maximum Positive ("could not have been better")
    3. Slang = Positive ("killed it", "fire", "bussin")
    4. Food-Specific = Positive ("melt in your mouth")

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DOUBLE NEGATIVE = POSITIVE ==========
        # Pattern: "did not [negative_word]" → Positive
        # Logic: NOT disappointing = satisfying
        "did not disappoint": "Positive",
        "didn't disappoint": "Positive",
        "does not disappoint": "Positive",
        "doesn't disappoint": "Positive",
        "never disappoints": "Positive",
        "never disappointed": "Positive",
        "not disappointed": "Positive",
        "wasn't disappointed": "Positive",
        "weren't disappointed": "Positive",
        "not let down": "Positive",
        "didn't let us down": "Positive",

        # ========== SUPERLATIVE NEGATION = MAXIMUM POSITIVE ==========
        # Pattern: "could not have been more [positive_adj]"
        # Logic: Impossible to be MORE positive = maximum level achieved
        "could not have been better": "Positive",
        "couldn't have been better": "Positive",
        "could not have been nicer": "Positive",
        "couldn't have been nicer": "Positive",
        "could not have been friendlier": "Positive",
        "couldn't have been friendlier": "Positive",
        "could not have been more helpful": "Positive",
        "couldn't have been more helpful": "Positive",
        "could not have been more attentive": "Positive",
        "couldn't have been more attentive": "Positive",
        "could not have been more professional": "Positive",
        "couldn't have been more professional": "Positive",
        "could not ask for more": "Positive",
        "couldn't ask for more": "Positive",
        "could not ask for better": "Positive",
        "couldn't ask for better": "Positive",
        "could not be happier": "Positive",
        "couldn't be happier": "Positive",
        "could not be more satisfied": "Positive",
        "couldn't be more satisfied": "Positive",

        # ========== NO COMPLAINTS = POSITIVE ==========
        "can't complain": "Positive",
        "cannot complain": "Positive",
        "nothing to complain about": "Positive",
        "no complaints": "Positive",
        "zero complaints": "Positive",
        "left nothing to be desired": "Positive",
        "leaves nothing to be desired": "Positive",

        # ========== SLANG/INFORMAL = POSITIVE ==========
        # Common in casual reviews, especially younger demographics
        "killed it": "Positive",           # Performed excellently
        "nailed it": "Positive",           # Got it exactly right
        "crushed it": "Positive",          # Exceeded expectations
        "smashed it": "Positive",          # Did amazingly well
        "hit the spot": "Positive",        # Perfectly satisfying
        "hits the spot": "Positive",
        "hit different": "Positive",       # Especially good
        "hits different": "Positive",
        "on point": "Positive",            # Exactly right
        "on fire": "Positive",             # Performing excellently
        "off the hook": "Positive",        # Extremely good
        "off the chain": "Positive",       # Amazing
        "off the charts": "Positive",      # Exceptionally high quality
        "out of this world": "Positive",   # Extraordinary
        "to die for": "Positive",          # Extremely desirable
        "die for": "Positive",
        "the bomb": "Positive",            # Excellent
        "da bomb": "Positive",
        "was bomb": "Positive",            # "The pizza was bomb"
        "is bomb": "Positive",
        "is fire": "Positive",             # "These wings are fire"
        "was fire": "Positive",
        "straight fire": "Positive",
        "chef's kiss": "Positive",         # Perfect
        "chefs kiss": "Positive",
        "slaps": "Positive",               # "This mac and cheese slaps"
        "bussin": "Positive",              # "Food was bussin"
        "bussin'": "Positive",
        "top notch": "Positive",
        "top-notch": "Positive",
        "first rate": "Positive",
        "first-rate": "Positive",
        "A1": "Positive",                  # Top quality
        "a-1": "Positive",
        "10/10": "Positive",
        "10 out of 10": "Positive",
        "five stars": "Positive",
        "5 stars": "Positive",

        # ========== FOOD-SPECIFIC POSITIVE IDIOMS ==========
        # Universal across cuisines
        "melt in your mouth": "Positive",
        "melts in your mouth": "Positive",
        "melted in my mouth": "Positive",
        "fall off the bone": "Positive",   # Tender meat
        "falls off the bone": "Positive",
        "fell off the bone": "Positive",
        "cooked to perfection": "Positive",
        "done to perfection": "Positive",
        "seasoned to perfection": "Positive",
        "finger licking good": "Positive",
        "finger-licking good": "Positive",
        "fresh out of the oven": "Positive",
        "fresh off the grill": "Positive",
        "made with love": "Positive",
        "like grandma used to make": "Positive",
        "like mama used to make": "Positive",
        "home cooked": "Positive",
        "homemade taste": "Positive",

        # ========== RECOMMENDATION POSITIVE ==========
        "a must try": "Positive",
        "must try": "Positive",
        "a must": "Positive",
        "a must visit": "Positive",
        "must visit": "Positive",
        "a gem": "Positive",
        "hidden gem": "Positive",
        "best kept secret": "Positive",
        "saved the day": "Positive",
        "made my day": "Positive",
        "worth the wait": "Positive",
        "worth the drive": "Positive",
        "worth every penny": "Positive",
        "worth the price": "Positive",
        "bang for your buck": "Positive",
        "will be back": "Positive",
        "coming back": "Positive",
        "definitely returning": "Positive",
        "can't wait to come back": "Positive",
    }


def get_negative_idioms():
    """
    Dictionary of negative idioms/slang expressions.

    Categories:
    1. Disappointment expressions
    2. Mediocrity expressions
    3. Poor value expressions
    4. Food-specific negative idioms

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DISAPPOINTMENT ==========
        "left a lot to be desired": "Negative",
        "leaves a lot to be desired": "Negative",
        "left much to be desired": "Negative",
        "leaves much to be desired": "Negative",
        "nothing to write home about": "Negative",
        "not my cup of tea": "Negative",
        "wasn't my cup of tea": "Negative",
        "seen better days": "Negative",
        "has seen better days": "Negative",
        "let down": "Negative",
        "let us down": "Negative",
        "fell short": "Negative",
        "falls short": "Negative",
        "missed the mark": "Negative",
        "misses the mark": "Negative",

        # ========== MEDIOCRITY ==========
        "hit or miss": "Negative",
        "meh": "Negative",
        "just ok": "Negative",
        "just okay": "Negative",
        "nothing special": "Negative",
        "middle of the road": "Negative",
        "run of the mill": "Negative",
        "average at best": "Negative",
        "could be better": "Negative",
        "room for improvement": "Negative",
        "needs work": "Negative",

        # ========== POOR VALUE ==========
        "waste of money": "Negative",
        "waste of time": "Negative",
        "rip off": "Negative",
        "rip-off": "Negative",
        "ripoff": "Negative",
        "not worth it": "Negative",
        "not worth the price": "Negative",
        "not worth the money": "Negative",
        "not worth the wait": "Negative",
        "not worth the hype": "Negative",
        "overpriced": "Negative",
        "over priced": "Negative",
        "overhyped": "Negative",
        "over-hyped": "Negative",
        "overrated": "Negative",
        "over-rated": "Negative",

        # ========== FOOD-SPECIFIC NEGATIVE ==========
        "tasted like cardboard": "Negative",
        "tastes like cardboard": "Negative",
        "like eating cardboard": "Negative",
        "rubber chicken": "Negative",
        "hockey puck": "Negative",         # Overcooked burger
        "sat under a heat lamp": "Negative",
        "sitting under a heat lamp": "Negative",
        "straight from the freezer": "Negative",
        "microwaved": "Negative",
        "reheated": "Negative",
        "day old": "Negative",
        "stale": "Negative",

        # ========== SERVICE NEGATIVE ==========
        "worst service ever": "Negative",
        "worst experience ever": "Negative",
        "never coming back": "Negative",
        "will not return": "Negative",
        "won't be back": "Negative",
        "avoid this place": "Negative",
        "stay away": "Negative",
        "save your money": "Negative",
    }


# ============================================
# SECTION 3: SENTIMENT ANALYZER INITIALIZATION
# ============================================

SENTIMENT_ANALYZERS = None


def initialize_sentiment_analyzers():
    """
    Initialize multiple sentiment analysis tools for ensemble approach.

    Why Multiple Analyzers:
        - VADER: Fast, rule-based, optimized for social media/reviews
        - TextBlob: Pattern-based, provides subjectivity scores
        - Transformer: Deep learning, most accurate for complex patterns
        - Ensemble of all three provides robust results

    Returns:
        dict: Initialized analyzers (None if unavailable)
    """
    analyzers = {}

    # VADER Initialization
    try:
        import nltk
        nltk.download('vader_lexicon', quiet=True)
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        analyzers['vader'] = SentimentIntensityAnalyzer()
        print("✅ VADER sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ VADER not available: {e}")
        analyzers['vader'] = None

    # TextBlob Initialization
    try:
        from textblob import TextBlob
        analyzers['textblob'] = TextBlob
        print("✅ TextBlob sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ TextBlob not available: {e}")
        analyzers['textblob'] = None

    # Transformer Initialization (Optional)
    try:
        from transformers import pipeline
        analyzers['transformer'] = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=-1
        )
        print("✅ Transformer sentiment analyzer loaded")
    except Exception as e:
        print(f"ℹ️ Transformer not available (optional): {e}")
        analyzers['transformer'] = None

    return analyzers


def get_sentiment_analyzers():
    """Get or initialize sentiment analyzers (singleton pattern)."""
    global SENTIMENT_ANALYZERS
    if SENTIMENT_ANALYZERS is None:
        SENTIMENT_ANALYZERS = initialize_sentiment_analyzers()
    return SENTIMENT_ANALYZERS


# ============================================
# SECTION 4: UNIVERSAL RESTAURANT VOCABULARY
# ============================================
"""
Universal Vocabulary Design:
    These word lists are designed to work across ALL restaurant types.
    They cover common aspects, opinions, and patterns found in reviews
    regardless of cuisine or service model.
"""


def get_common_english_names():
    """
    Common first names to filter from opinions.

    Rationale:
        Reviews often mention server names: "Sarah was great"
        Names alone don't provide sentiment - the opinion does.

    Returns:
        set: Common English first names (lowercase)
    """
    male_names = {
        'james', 'john', 'robert', 'michael', 'william', 'david', 'richard',
        'joseph', 'thomas', 'charles', 'christopher', 'daniel', 'matthew',
        'anthony', 'mark', 'donald', 'steven', 'paul', 'andrew', 'joshua',
        'kenneth', 'kevin', 'brian', 'george', 'timothy', 'ronald', 'edward',
        'jason', 'jeffrey', 'ryan', 'jacob', 'gary', 'nicholas', 'eric',
        'jonathan', 'stephen', 'larry', 'justin', 'scott', 'brandon', 'benjamin',
        'samuel', 'raymond', 'gregory', 'frank', 'alexander', 'patrick', 'jack',
        'joel', 'paul', 'morgan', 'jordan', 'chris', 'matt', 'andy', 'ray',
        'nick', 'greg', 'mike', 'steve', 'tom', 'bob', 'jim', 'dan'
    }

    female_names = {
        'mary', 'patricia', 'jennifer', 'linda', 'barbara', 'elizabeth', 'susan',
        'jessica', 'sarah', 'karen', 'lisa', 'nancy', 'betty', 'margaret', 'sandra',
        'ashley', 'kimberly', 'emily', 'donna', 'michelle', 'dorothy', 'carol',
        'amanda', 'melissa', 'deborah', 'stephanie', 'rebecca', 'sharon', 'laura',
        'sara', 'fatima', 'kaitlin', 'kaitlyn', 'natalie', 'melanie', 'christina',
        'veronica', 'morgan', 'jordan', 'gabby', 'tina', 'cameron'
    }

    return male_names | female_names


def get_positive_sentiment_words():
    """
    Universal positive sentiment words for restaurant reviews.

    Categories:
        - General quality descriptors
        - Food-specific positive terms
        - Service-related positive terms
        - Atmosphere/environment terms
        - Value-related terms

    Returns:
        set: Positive sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic',
        'outstanding', 'superb', 'perfect', 'incredible', 'awesome',
        'terrific', 'fabulous', 'marvelous', 'exceptional', 'best',
        'impressive', 'remarkable', 'magnificent', 'splendid', 'brilliant',
        'stellar', 'phenomenal', 'spectacular', 'extraordinary', 'superb',

        # ===== FOOD-SPECIFIC (Universal across cuisines) =====
        'delicious', 'tasty', 'yummy', 'flavorful', 'savory', 'fresh',
        'tender', 'juicy', 'crispy', 'creamy', 'rich', 'light',
        'authentic', 'homemade', 'seasoned', 'aromatic', 'scrumptious',
        'divine', 'heavenly', 'mouthwatering', 'succulent', 'zesty',
        'perfectly cooked', 'well-seasoned', 'flavorsome',

        # ===== SERVICE-RELATED =====
        'friendly', 'helpful', 'attentive', 'professional', 'courteous',
        'prompt', 'efficient', 'welcoming', 'accommodating', 'polite',
        'responsive', 'knowledgeable', 'patient', 'thorough', 'dedicated',
        'personable', 'warm', 'gracious', 'sweet', 'caring',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'clean', 'comfortable', 'cozy', 'spacious', 'nice', 'lovely',
        'beautiful', 'charming', 'pleasant', 'relaxing', 'quiet',
        'modern', 'elegant', 'stylish', 'inviting', 'cute',
        'trendy', 'vibrant', 'lively', 'romantic', 'intimate',

        # ===== VALUE-RELATED =====
        'reasonable', 'affordable', 'worth', 'value', 'cheap', 'bargain',
        'fair', 'inexpensive', 'economical', 'generous'
    }


def get_negative_sentiment_words():
    """
    Universal negative sentiment words for restaurant reviews.

    Returns:
        set: Negative sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'bad', 'terrible', 'awful', 'horrible', 'poor', 'disappointing',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross', 'worst',
        'inferior', 'subpar', 'unacceptable', 'dreadful', 'appalling',
        'atrocious', 'abysmal', 'horrendous', 'disgusting',

        # ===== FOOD-SPECIFIC =====
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty', 'stale',
        'cold', 'lukewarm', 'frozen', 'rubbery', 'tough', 'chewy',
        'flavorless', 'mushy', 'bitter', 'sour', 'spoiled',
        'unseasoned', 'unappetizing', 'oily', 'oversalted',

        # ===== SERVICE-RELATED =====
        'rude', 'slow', 'unfriendly', 'unhelpful', 'inattentive',
        'unprofessional', 'careless', 'dismissive', 'indifferent',
        'incompetent', 'negligent', 'impolite', 'disrespectful',
        'ignored', 'forgotten', 'invisible',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'dirty', 'messy', 'noisy', 'crowded', 'cramped', 'uncomfortable',
        'dark', 'smelly', 'hot', 'stuffy', 'outdated', 'dingy',
        'filthy', 'cluttered', 'rundown', 'shabby', 'sticky',

        # ===== VALUE-RELATED =====
        'expensive', 'overpriced', 'costly', 'pricey', 'ripoff',
        'unreasonable', 'exorbitant', 'small portions'
    }


def get_action_verbs():
    """
    Action verbs that should NOT be treated as opinions.

    Rationale:
        "I ordered the pasta" - "ordered" is action, not opinion
        "The waiter served us" - "served" is action, not opinion

    Returns:
        set: Action verbs in various forms
    """
    return {
        'order', 'ordered', 'ordering', 'orders',
        'ask', 'asked', 'asking', 'asks',
        'request', 'requested', 'requesting', 'requests',
        'come', 'came', 'coming', 'comes',
        'go', 'went', 'going', 'goes', 'gone',
        'arrive', 'arrived', 'arriving', 'arrives',
        'leave', 'left', 'leaving', 'leaves',
        'walk', 'walked', 'walking', 'walks',
        'enter', 'entered', 'entering', 'enters',
        'visit', 'visited', 'visiting', 'visits',
        'get', 'got', 'getting', 'gets',
        'take', 'took', 'taking', 'takes', 'taken',
        'bring', 'brought', 'bringing', 'brings',
        'receive', 'received', 'receiving', 'receives',
        'make', 'made', 'making', 'makes',
        'do', 'did', 'doing', 'does', 'done',
        'prepare', 'prepared', 'preparing', 'prepares',
        'cook', 'cooked', 'cooking', 'cooks',
        'say', 'said', 'saying', 'says',
        'tell', 'told', 'telling', 'tells',
        'call', 'called', 'calling', 'calls',
        'serve', 'served', 'serving', 'serves',
        'show', 'showed', 'showing', 'shows', 'shown',
        'give', 'gave', 'giving', 'gives', 'given',
        'seat', 'seated', 'seating', 'seats',
        'try', 'tried', 'trying', 'tries',
        'want', 'wanted', 'wanting', 'wants',
        'need', 'needed', 'needing', 'needs',
        'hope', 'hoped', 'hoping', 'hopes',
        'expect', 'expected', 'expecting', 'expects',
        'enjoy', 'enjoyed', 'enjoying', 'enjoys',
        'pay', 'paid', 'paying', 'pays',
        'wait', 'waited', 'waiting', 'waits',
        'sit', 'sat', 'sitting', 'sits',
        'eat', 'ate', 'eating', 'eats', 'eaten',
        'drink', 'drank', 'drinking', 'drinks', 'drunk'
    }


def get_function_words():
    """
    Function words that carry no sentiment meaning.

    Returns:
        set: Articles, prepositions, pronouns, etc.
    """
    return {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'i', 'me', 'my', 'mine', 'myself',
        'you', 'your', 'yours', 'yourself',
        'he', 'him', 'his', 'himself',
        'she', 'her', 'hers', 'herself',
        'it', 'its', 'itself',
        'we', 'us', 'our', 'ours', 'ourselves',
        'they', 'them', 'their', 'theirs', 'themselves',
        'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
        'from', 'about', 'into', 'through', 'during', 'before',
        'after', 'above', 'below', 'between', 'under', 'over',
        'and', 'or', 'but', 'so', 'yet', 'nor',
        'if', 'when', 'where', 'while', 'as', 'because', 'since',
        'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'has', 'have', 'had', 'having',
        'do', 'does', 'did',
        'will', 'would', 'shall', 'should',
        'can', 'could', 'may', 'might', 'must',
        'here', 'there', 'some', 'any', 'no', 'every'
    }


def get_intensifiers():
    """
    Intensifier words that modify sentiment strength.

    Returns:
        set: Intensifier words
    """
    return {
        'very', 'extremely', 'incredibly', 'amazingly', 'exceptionally',
        'remarkably', 'absolutely', 'totally', 'completely', 'utterly',
        'entirely', 'thoroughly', 'perfectly', 'highly', 'deeply',
        'truly', 'really', 'genuinely', 'seriously',
        'quite', 'rather', 'fairly', 'pretty', 'somewhat',
        'reasonably', 'moderately', 'relatively',
        'slightly', 'a bit', 'a little', 'mildly', 'barely',
        'too', 'overly', 'excessively',
        'so', 'such', 'super', 'especially', 'particularly'
    }


# ============================================
# SECTION 5: IDIOM DETECTION FUNCTIONS
# ============================================

def check_for_idiom(text):
    """
    Check if text contains any known idiom.

    PRIORITY: This check runs BEFORE negation detection.

    Args:
        text (str): Text to check

    Returns:
        tuple: (is_idiom: bool, sentiment: str or None, matched_idiom: str or None)

    Example:
        >>> check_for_idiom("The food did not disappoint")
        (True, "Positive", "did not disappoint")

        >>> check_for_idiom("The food was cold")
        (False, None, None)
    """
    text_lower = text.lower()

    # Check positive idioms
    positive_idioms = get_positive_idioms()
    for idiom, sentiment in positive_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    # Check negative idioms
    negative_idioms = get_negative_idioms()
    for idiom, sentiment in negative_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    return False, None, None


# ============================================
# SECTION 6: ADJECTIVE/ADVERB DETECTION
# ============================================

def is_adjective_or_adverb(word):
    """
    Determine if a word is an adjective or adverb.

    Uses two approaches:
        1. Dictionary lookup (known sentiment words)
        2. Morphological rules (suffix patterns)

    Args:
        word (str): Word to check

    Returns:
        tuple: (is_adj_or_adv: bool, word_type: str)
    """
    word_lower = word.lower().strip('.,!?;:\'"')

    if not word_lower:
        return False, 'none'

    # Check sentiment dictionaries
    if word_lower in get_positive_sentiment_words():
        return True, 'positive_adj'

    if word_lower in get_negative_sentiment_words():
        return True, 'negative_adj'

    if word_lower in get_intensifiers():
        return True, 'intensifier'

    # Morphological rules
    adj_suffixes = ['ful', 'less', 'ous', 'ive', 'able', 'ible',
                    'al', 'ic', 'ish', 'ent', 'ant', 'ory', 'ary']

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    # Adverb suffix
    if word_lower.endswith('ly') and len(word_lower) > 4:
        non_adverb_ly = {'family', 'only', 'early', 'likely', 'friendly', 'lonely'}
        if word_lower not in non_adverb_ly:
            return True, 'suffix_adv'

    # Past participles as adjectives
    sentiment_participles = {
        'disappointed', 'satisfied', 'pleased', 'impressed', 'amazed',
        'surprised', 'disgusted', 'frustrated', 'annoyed', 'delighted',
        'thrilled', 'excited', 'bored', 'tired', 'exhausted',
        'overwhelmed', 'underwhelmed', 'overpriced'
    }
    if word_lower in sentiment_participles:
        return True, 'suffix_adj'

    # Present participles as adjectives
    sentiment_ing = {
        'amazing', 'disappointing', 'disgusting', 'interesting', 'boring',
        'exciting', 'frustrating', 'annoying', 'satisfying', 'refreshing',
        'relaxing', 'welcoming', 'inviting', 'appealing', 'appalling'
    }
    if word_lower in sentiment_ing:
        return True, 'suffix_adj'

    return False, 'none'


# ============================================
# SECTION 7: NEGATION DETECTION
# ============================================

def detect_negation_in_opinion(opinion):
    """
    Detect negation in opinion text.

    IMPORTANT: Check idioms FIRST before calling this function.

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (has_negation: bool, negation_type: str or None)

    Example:
        >>> detect_negation_in_opinion("not good")
        (True, 'direct')

        >>> detect_negation_in_opinion("wasn't fresh")
        (True, 'contraction')
    """
    opinion_lower = opinion.lower()

    # SAFETY CHECK: If this is an idiom, don't treat as simple negation
    is_idiom, _, _ = check_for_idiom(opinion_lower)
    if is_idiom:
        return False, 'idiom_detected'

    # Direct negation words
    direct_negations = [
        'not ', "n't ", 'no ', 'never ', 'none ', 'nothing ',
        'neither ', 'nobody ', 'nowhere ', 'cannot '
    ]

    for neg in direct_negations:
        if neg in opinion_lower or opinion_lower.startswith(neg.strip()):
            return True, 'direct'

    # Contracted negations
    contracted_negations = [
        "didn't", "wasn't", "weren't", "isn't", "aren't",
        "don't", "doesn't", "won't", "wouldn't", "couldn't",
        "shouldn't", "can't", "haven't", "hasn't", "hadn't"
    ]

    for neg in contracted_negations:
        if neg in opinion_lower:
            return True, 'contraction'

    return False, None


# ============================================
# SECTION 8: BOUNDARY DETECTION
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """Find sentence boundaries containing the aspect."""
    sentence_enders = {'.', '!', '?'}

    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in sentence_enders:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in sentence_enders:
            sentence_end = i
            break

    return sentence_start, sentence_end


def find_previous_aspect_end(current_start_idx, all_aspect_positions):
    """Find end position of previous aspect."""
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(current_end_idx, all_aspect_positions):
    """Find start position of next aspect."""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# SECTION 9: BE-VERB DETECTION
# ============================================

def detect_be_verb(tokens, start_idx):
    """Detect be-verb at given position."""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    be_verbs = {'is', 'was', 'were', 'are', 'be', 'been', "'s", "s"}
    if word in be_verbs:
        return True, 1, False

    negated_be_verbs = {
        "isn't", "isnt", "wasn't", "wasnt",
        "weren't", "werent", "aren't", "arent"
    }
    if word in negated_be_verbs:
        return True, 1, True

    return False, 0, False


# ============================================
# SECTION 10: OPINION VALIDATION
# ============================================

def validate_opinion_completeness(opinion):
    """
    Validate that opinion is complete and meaningful.

    Rules:
        1. Cannot start with conjunction (incomplete phrase)
        2. Single intensifier alone is invalid
        3. Cannot be factual statement (reporting, not opinion)
        4. Must contain sentiment signal

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (is_valid: bool, reason: str or None)

    Examples:
        >>> validate_opinion_completeness("and was delicious")
        (False, "starts_with_conjunction")

        >>> validate_opinion_completeness("she said yes")
        (False, "factual_statement")

        >>> validate_opinion_completeness("absolutely delicious")
        (True, None)
    """
    if not opinion or len(opinion.strip()) < 2:
        return False, 'empty'

    words = opinion.lower().strip().split()
    if not words:
        return False, 'empty'

    # Rule 1: Cannot start with conjunction
    conjunctions = {'and', 'or', 'but', 'so', 'yet', 'nor', 'for'}
    if words[0] in conjunctions:
        return False, 'starts_with_conjunction'

    # Rule 2: Single intensifier is invalid
    if len(words) == 1:
        intensifiers = get_intensifiers()
        if words[0] in intensifiers:
            return False, 'only_intensifier'

        function_words = get_function_words()
        if words[0] in function_words:
            return False, 'only_function_word'

    # Rule 3: Cannot be factual statement
    factual_patterns = [
        r'^(he|she|they|it|we|i)\s+(said|told|asked|mentioned|replied)',
        r'^(was|were|is|are)\s+(out of|available|unavailable)',
        r'^if\s+(it|they|she|he|we)\s+(was|were|is|are)',
        r'^that\s+(it|they)',
        r'^\d+\s*/',
        r'^with\s+(a\s+)?side',
    ]

    opinion_text = opinion.lower()
    for pattern in factual_patterns:
        if re.search(pattern, opinion_text):
            return False, 'factual_statement'

    # Rule 4: Must contain sentiment signal (for short opinions)
    if len(words) <= 4:
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        all_sentiment = positive_words | negative_words

        has_sentiment = False
        for word in words:
            clean_word = word.strip('.,!?;:\'"')
            if clean_word in all_sentiment:
                has_sentiment = True
                break
            is_adj, _ = is_adjective_or_adverb(clean_word)
            if is_adj:
                has_sentiment = True
                break

        if not has_sentiment:
            # Exception: Idioms should pass
            is_idiom, _, _ = check_for_idiom(opinion)
            if not is_idiom:
                return False, 'no_sentiment_signal'

    return True, None


def is_valid_opinion(opinion):
    """
    Check if extracted text is a valid opinion.

    Args:
        opinion (str): Extracted opinion text

    Returns:
        bool: True if valid opinion
    """
    if not opinion or len(opinion.strip()) < 2:
        return False

    # Use completeness validation
    is_complete, _ = validate_opinion_completeness(opinion)
    if not is_complete:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    if not words:
        return False

    action_verbs = get_action_verbs()
    function_words = get_function_words()
    common_names = get_common_english_names()

    # Single word validation
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')
        if clean_word in action_verbs:
            return False
        if clean_word in function_words:
            return False
        if clean_word in common_names:
            return False

    # Invalid patterns
    invalid_patterns = [
        r"^'s\s+",
        r"^\d+\s*/",
        r"^or\s+maybe\s+both$",
        r"^going\s+forward$",
        r"^n\s+cheese",  # Truncated "mac n cheese"
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    return True


# ============================================
# SECTION 11: ASPECT VALIDATION
# ============================================

def is_valid_aspect(aspect):
    """Check if extracted text is a valid aspect."""
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    if not cleaned or cleaned == "-":
        return False

    if all(c in '.,!?;:-_\'"' for c in cleaned):
        return False

    # Invalid single words (prepositions, conjunctions, etc.)
    invalid_single = {'of', 'to', 'for', 'with', 'at', 'in', 'on', 'by',
                      'and', 'or', 'but', 'n', 'the', 'a', 'an'}
    if cleaned in invalid_single:
        return False

    invalid_aspects = {
        # Verbs
        'wait', 'waited', 'waiting', 'order', 'ordered', 'ordering',
        'serve', 'served', 'serving', 'ask', 'asked', 'asking',
        # Pronouns
        'i', 'me', 'my', 'you', 'your', 'he', 'him', 'his',
        'she', 'her', 'it', 'its', 'we', 'us', 'our', 'they', 'them',
        # Time units
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours',
        # Adjectives (opinions, not aspects)
        'good', 'bad', 'great', 'nice', 'poor',
        # Generic
        'thing', 'things', 'stuff', 'way', 'time', 'place'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """Clean and standardize aspect term."""
    if not aspect:
        return ""

    clean_aspect = aspect.strip()
    clean_aspect = re.sub(r'^(and|or|but)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^(the|a|an|this|that|my|our)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^[-:;,.\'"]+\s*', '', clean_aspect)
    clean_aspect = re.sub(r'[-:;,.\'"]+$', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# SECTION 12: OPINION EXTRACTION
# ============================================

def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """
    Extract opinion associated with aspect, respecting boundaries.

    Extraction Patterns (Priority Order):
        1. Pre-modifier: "delicious food" → opinion before aspect
        2. Be-adjective: "food is delicious" → opinion after be-verb
        3. Verb phrase: "food tastes great" → opinion in verb phrase
        4. Context search: Fallback for complex sentences

    Args:
        tokens: Tokenized review
        aspect_positions: Position indices of aspect
        aspect_text: Aspect text (for reference)
        all_aspect_positions: All aspect positions (for boundaries)

    Returns:
        tuple: (opinion_phrase: str, pattern_type: str)
    """
    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1] if len(aspect_positions) > 1 else aspect_positions[0]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)
    prev_aspect_end = find_previous_aspect_end(start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(end_idx, all_aspect_positions)

    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # PATTERN 1: Pre-modifier
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, _ = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in {',', 'and', 'with', 'or', 'but'}:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

    # PATTERN 2: Be-verb + Adjective
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []
        if is_negative:
            post_modifiers.append("not")

        for i in range(opinion_start, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            post_modifiers.append(word)

        if post_modifiers:
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # PATTERN 3: Verb Phrase
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            temp_words.append(word)

        if temp_words:
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # PATTERN 4: Context Search
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, _ = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                for j in range(context_start, context_end):
                    if tokens[j] not in {'.', '!', '?', ';'}:
                        opinion_words.append(tokens[j])

                pattern_type = "context_search"
                break

    opinion_phrase = ' '.join(opinion_words).strip()
    return opinion_phrase, pattern_type


# ============================================
# SECTION 13: OPINION FORMATTING
# ============================================

def format_opinion_for_display(opinion):
    """
    Format and clean extracted opinion for display.

    Processing Steps:
        1. Remove leading conjunctions ("and", "or", "but")
        2. Remove leading function words
        3. Handle hyphenation
        4. Truncate at cutoff words
        5. Remove trailing garbage
        6. Enforce length limit

    Args:
        opinion (str): Raw extracted opinion

    Returns:
        str: Cleaned and formatted opinion
    """
    if not opinion:
        return ""

    # Remove leading conjunctions
    opinion = re.sub(r'^(and|or|but|so)\s+', '', opinion, flags=re.IGNORECASE)

    # Fix punctuation
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    words = opinion.split()
    if not words:
        return ""

    # Remove leading fillers
    leading_fillers = {
        'a', 'an', 'the', 'this', 'that', 'also', 'too',
        'has', 'have', 'had', 'is', 'are', 'was', 'were',
        'it', 'itself', 'they', 'themselves', 'we', 'i',
        'and', 'or', 'but', 'so'
    }

    while words and words[0].lower() in leading_fillers:
        words.pop(0)

    if not words:
        return ""

    # Truncate at cutoff words
    cutoff_words = {
        'making', 'causing', 'forcing', 'leaving',
        'unless', 'except', 'despite', 'although',
        'which', 'where', 'when', 'because', 'since'
    }

    truncated = []
    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')
        if word_lower in cutoff_words and i >= 2:
            break
        truncated.append(word)

    words = truncated

    # Remove trailing garbage
    trailing_garbage = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'a', 'an', 'the', ',', '-',
        'is', 'are', 'was', 'were',
        'it', 'i', 'we', 'they'
    }

    while words:
        last_word = words[-1].lower().strip('.,!?;:')
        if last_word in trailing_garbage or words[-1].endswith('-'):
            words.pop()
        else:
            break

    result = ' '.join(words)

    # Length limit
    words = result.split()
    if len(words) > 12:
        words = words[:12]
        result = ' '.join(words)

    # Final cleanup
    result = result.strip().rstrip('.,;-\'"(')
    result = re.sub(r'^(and|or|but|so)\s+', '', result, flags=re.IGNORECASE)

    return result


# ============================================
# SECTION 14: SENTIMENT ANALYSIS
# ============================================

def analyze_sentiment_scores(text):
    """
    Analyze text sentiment using multiple tools.

    Returns ensemble score combining:
        - VADER (weight: 0.4)
        - TextBlob (weight: 0.3)
        - Transformer (weight: 0.3)

    Args:
        text (str): Text to analyze

    Returns:
        dict: Sentiment scores from all analyzers and ensemble
    """
    analyzers = get_sentiment_analyzers()
    results = {}

    if not text or len(text.strip()) < 2:
        return {
            'vader': None,
            'textblob': None,
            'transformer': None,
            'ensemble': {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}
        }

    # VADER
    if analyzers.get('vader'):
        try:
            vader_scores = analyzers['vader'].polarity_scores(text)
            results['vader'] = {'compound': vader_scores['compound']}
        except:
            results['vader'] = None
    else:
        results['vader'] = None

    # TextBlob
    if analyzers.get('textblob'):
        try:
            blob = analyzers['textblob'](text)
            results['textblob'] = {'polarity': blob.sentiment.polarity}
        except:
            results['textblob'] = None
    else:
        results['textblob'] = None

    # Transformer
    if analyzers.get('transformer'):
        try:
            truncated = text[:500] if len(text) > 500 else text
            trans_result = analyzers['transformer'](truncated)[0]
            normalized = trans_result['score'] if trans_result['label'] == 'POSITIVE' else -trans_result['score']
            results['transformer'] = {'normalized_score': normalized}
        except:
            results['transformer'] = None
    else:
        results['transformer'] = None

    # Calculate ensemble
    results['ensemble'] = calculate_ensemble_score(results)

    return results


def calculate_ensemble_score(sentiment_results):
    """Calculate weighted ensemble sentiment score."""
    scores = []
    weights = []

    if sentiment_results.get('vader') and sentiment_results['vader'].get('compound') is not None:
        scores.append(sentiment_results['vader']['compound'])
        weights.append(0.4)

    if sentiment_results.get('textblob') and sentiment_results['textblob'].get('polarity') is not None:
        scores.append(sentiment_results['textblob']['polarity'])
        weights.append(0.3)

    if sentiment_results.get('transformer') and sentiment_results['transformer'].get('normalized_score') is not None:
        scores.append(sentiment_results['transformer']['normalized_score'])
        weights.append(0.3)

    if not scores:
        return {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}

    total_weight = sum(weights)
    normalized_weights = [w / total_weight for w in weights]
    ensemble_score = sum(s * w for s, w in zip(scores, normalized_weights))

    # Calculate confidence (agreement)
    if len(scores) > 1:
        variance = sum((s - ensemble_score) ** 2 for s in scores) / len(scores)
        confidence = max(0, 1 - variance)
    else:
        confidence = 0.7

    # Determine label
    if ensemble_score >= 0.05:
        label = 'Positive'
    elif ensemble_score <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return {
        'score': round(ensemble_score, 4),
        'label': label,
        'confidence': round(confidence, 4)
    }


# ============================================
# SECTION 15: ENHANCED SENTIMENT CORRECTION
# ============================================

# Strong sentiment indicators for sanity checking
OBVIOUS_NEGATIVE_INDICATORS = {
    'terrible', 'horrible', 'awful', 'worst', 'disgusting', 'nasty',
    'gross', 'pathetic', 'abysmal', 'dreadful', 'atrocious', 'appalling',
    'incompetent', 'useless', 'unacceptable', 'inexcusable', 'nightmare'
}

OBVIOUS_POSITIVE_INDICATORS = {
    'excellent', 'amazing', 'wonderful', 'fantastic', 'outstanding',
    'superb', 'incredible', 'phenomenal', 'magnificent', 'exceptional',
    'perfect', 'best', 'love', 'loved', 'brilliant', 'marvelous'
}


def sanity_check_sentiment(opinion, predicted_sentiment):
    """
    Sanity check to catch obvious sentiment misclassifications.

    Example:
        "TERRIBLE service" should NEVER be classified as Positive
        "EXCELLENT food" should NEVER be classified as Negative

    This is a safety net for when other correction logic fails.

    Args:
        opinion (str): Opinion text
        predicted_sentiment (str): Current sentiment prediction

    Returns:
        tuple: (corrected_sentiment, correction_reason) or (None, None) if no change
    """
    opinion_lower = opinion.lower()
    words = opinion_lower.split()

    # Check for obvious negative indicators
    for neg_word in OBVIOUS_NEGATIVE_INDICATORS:
        if neg_word in words or neg_word in opinion_lower:
            # Check if negated (e.g., "not terrible")
            neg_index = opinion_lower.find(neg_word)
            prefix = opinion_lower[max(0, neg_index-10):neg_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Positive':
                    return 'Negative', f'sanity_check_negative ({neg_word})'

    # Check for obvious positive indicators
    for pos_word in OBVIOUS_POSITIVE_INDICATORS:
        if pos_word in words or pos_word in opinion_lower:
            # Check if negated
            pos_index = opinion_lower.find(pos_word)
            prefix = opinion_lower[max(0, pos_index-10):pos_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Negative':
                    return 'Positive', f'sanity_check_positive ({pos_word})'

    return None, None


def correct_sentiment_with_analysis(aspect, opinion, predicted_sentiment, sentiment_scores=None):
    """
    Correct sentiment prediction using rules and analysis.

    PRIORITY ORDER (Critical for accuracy):
        0. SANITY CHECK (NEW) - Catch obvious misclassifications like "TERRIBLE" → Positive
        1. Idiom detection - "did not disappoint" → Positive
        2. Superlative patterns - "could not have been better" → Positive
        3. Negation + sentiment word
        4. Score override (when highly confident)
        5. Keyword detection

    Args:
        aspect (str): Aspect term
        opinion (str): Opinion text
        predicted_sentiment (str): Model's prediction
        sentiment_scores (dict): Sentiment analysis scores

    Returns:
        tuple: (corrected_sentiment, correction_reason)
    """
    opinion_lower = opinion.lower()

    # ========== PRIORITY 0: SANITY CHECK (NEW - HIGHEST) ==========
    # Catch obvious cases like "TERRIBLE" classified as Positive
    sanity_result, sanity_reason = sanity_check_sentiment(opinion, predicted_sentiment)
    if sanity_result:
        return sanity_result, sanity_reason

    # ========== PRIORITY 1: IDIOM DETECTION ==========
    is_idiom, idiom_sentiment, idiom_matched = check_for_idiom(opinion_lower)
    if is_idiom:
        if predicted_sentiment != idiom_sentiment:
            return idiom_sentiment, f'idiom_override ({idiom_matched})'
        return predicted_sentiment, None

    # ========== PRIORITY 2: SUPERLATIVE PATTERNS ==========
    superlative_patterns = [
        r"could(n't| not) have been (more )?(friendly|helpful|better|nicer|attentive|professional)",
        r"could(n't| not) ask for (more|better)",
        r"could(n't| not) be (happier|better)",
    ]
    for pattern in superlative_patterns:
        if re.search(pattern, opinion_lower):
            if predicted_sentiment != 'Positive':
                return 'Positive', 'superlative_positive'
            return predicted_sentiment, None

    # Get sentiment word lists
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()

    # ========== PRIORITY 3: NEGATION LOGIC ==========
    has_negation, neg_type = detect_negation_in_opinion(opinion_lower)

    if has_negation and neg_type != 'idiom_detected':
        # Negating positive word → Negative
        for pos_word in positive_words:
            if pos_word in opinion_lower:
                if "not only" not in opinion_lower:  # Exception
                    return 'Negative', f'negation_positive ({pos_word})'

        # Negating negative word → Positive (double negative)
        for neg_word in negative_words:
            if neg_word in opinion_lower:
                return 'Positive', f'double_negation ({neg_word})'

        # General negation with no clear word
        if sentiment_scores:
            score = sentiment_scores.get('ensemble', {}).get('score', 0)
            if score > 0.3:
                return predicted_sentiment, None
        return 'Negative', f'negation_general ({neg_type})'

    # ========== PRIORITY 4: SCORE OVERRIDE ==========
    if sentiment_scores:
        ensemble = sentiment_scores.get('ensemble', {})
        ensemble_score = ensemble.get('score', 0)
        confidence = ensemble.get('confidence', 0)

        if confidence >= SENTIMENT_CONFIDENCE_THRESHOLD:
            if predicted_sentiment == 'Positive' and ensemble_score < -SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(pw in opinion_lower for pw in positive_words):
                    return 'Negative', f'sentiment_override ({ensemble_score:.2f})'

            if predicted_sentiment == 'Negative' and ensemble_score > SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(nw in opinion_lower for nw in negative_words):
                    return 'Positive', f'sentiment_override ({ensemble_score:.2f})'

    # ========== PRIORITY 5: KEYWORD DETECTION ==========
    for pos_word in positive_words:
        if pos_word in opinion_lower and predicted_sentiment != 'Positive':
            return 'Positive', f'positive_keyword ({pos_word})'

    for neg_word in negative_words:
        if neg_word in opinion_lower and predicted_sentiment != 'Negative':
            return 'Negative', f'negative_keyword ({neg_word})'

    return predicted_sentiment, None


# ============================================
# SECTION 16: CONFIDENCE-BASED FILTERING
# ============================================

def should_include_extraction(sentiment_scores, opinion, predicted_sentiment):
    """
    Determine if extraction should be included based on confidence.

    Exclusion Rules:
        1. |score| < MINIMUM_SENTIMENT_SCORE_THRESHOLD (unclear sentiment)
        2. confidence < MINIMUM_AGREEMENT_THRESHOLD (analyzers disagree)

    Exceptions:
        - Known idioms are always included
        - Clear sentiment words override low scores

    Args:
        sentiment_scores (dict): Sentiment analysis results
        opinion (str): Opinion text
        predicted_sentiment (str): Model prediction

    Returns:
        tuple: (include: bool, reason: str or None)
    """
    if not sentiment_scores:
        return True, None

    ensemble = sentiment_scores.get('ensemble', {})
    score = ensemble.get('score', 0)
    confidence = ensemble.get('confidence', 0)

    # Rule 1: Score too close to zero
    if abs(score) < MINIMUM_SENTIMENT_SCORE_THRESHOLD:
        # Exception: Idioms should be included
        is_idiom, _, _ = check_for_idiom(opinion.lower())
        if not is_idiom:
            return False, f'low_score (|{score:.2f}| < {MINIMUM_SENTIMENT_SCORE_THRESHOLD})'

    # Rule 2: Low agreement between analyzers
    if confidence < MINIMUM_AGREEMENT_THRESHOLD:
        # Exception: Clear sentiment words
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        has_clear_word = any(w in opinion.lower() for w in positive_words | negative_words)
        if not has_clear_word:
            return False, f'low_agreement ({confidence:.2f})'

    return True, None


# ============================================
# SECTION 17: RESULT STRUCTURES
# ============================================

def create_aspect_opinion_pair(aspect, opinion, raw_opinion, sentiment,
                                original_sentiment, pattern, chunk_id,
                                sentiment_scores, correction_reason):
    """Create standardized result dictionary."""

    scores_dict = {
        'vader_compound': None,
        'textblob_polarity': None,
        'transformer_score': None,
        'ensemble_score': None,
        'ensemble_confidence': None
    }

    if sentiment_scores:
        if sentiment_scores.get('vader'):
            scores_dict['vader_compound'] = sentiment_scores['vader'].get('compound')
        if sentiment_scores.get('textblob'):
            scores_dict['textblob_polarity'] = sentiment_scores['textblob'].get('polarity')
        if sentiment_scores.get('transformer'):
            scores_dict['transformer_score'] = sentiment_scores['transformer'].get('normalized_score')
        if sentiment_scores.get('ensemble'):
            scores_dict['ensemble_score'] = sentiment_scores['ensemble'].get('score')
            scores_dict['ensemble_confidence'] = sentiment_scores['ensemble'].get('confidence')

    return {
        'aspect': aspect,
        'opinion': opinion,
        'raw_opinion': raw_opinion,
        'formatted': f"{aspect}: {opinion}",
        'pattern': pattern,
        'chunk': chunk_id,
        'original_sentiment': original_sentiment,
        'sentiment': sentiment,
        'correction_reason': correction_reason,
        'sentiment_scores': scores_dict
    }


# ============================================
# SECTION 18: FILE LOGGING
# ============================================

class FileLogger:
    """Logger for both console and file output."""

    def __init__(self, filename):
        self.filename = filename
        self.content = []

    def log(self, message=""):
        self.content.append(message)
        print(message)

    def save(self):
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ Results saved to: {self.filename}")


# ============================================
# SECTION 19: REVIEW CHUNKING
# ============================================

def split_long_review(text, max_words=MAX_WORDS_PER_CHUNK):
    """Split long reviews into sentence-preserving chunks."""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub = ' '.join(words[i:i + max_words])
                chunks.append(sub + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# SECTION 20: MAIN ANALYSIS FUNCTION
# ============================================

def comprehensive_aspect_opinion_analysis(reviews, aspect_extractor,
                                          max_words=MAX_WORDS_PER_CHUNK,
                                          verbose=True, logger=None,
                                          use_sentiment_analysis=True,
                                          apply_confidence_filter=True):
    """
    Run complete ABSA analysis on restaurant reviews.

    Args:
        reviews: Iterable of review texts
        aspect_extractor: PyABSA aspect extractor
        max_words: Max words per chunk
        verbose: Enable detailed logging
        logger: FileLogger instance
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence-based filtering

    Returns:
        tuple: (results_list, statistics_dict)
    """
    if logger:
        logger.log("=" * 80)
        logger.log("ABSA Analysis System - V5 Enhanced Universal Edition")
        logger.log(f"Chunk size: {max_words} words")
        logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
        logger.log(f"Confidence filter: {'Enabled' if apply_confidence_filter else 'Disabled'}")
        logger.log(f"Min score threshold: {MINIMUM_SENTIMENT_SCORE_THRESHOLD}")
        logger.log(f"Analysis time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 80)

    if use_sentiment_analysis:
        _ = get_sentiment_analyzers()

    all_results = []
    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'filtered_low_confidence': 0,
        'sentiment_corrections': 0,
        'correction_reasons': Counter(),
        'idiom_detections': 0,
        'sanity_check_corrections': 0,  # NEW: Track sanity check fixes
    }

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'=' * 80}")
            logger.log(f"Review #{idx} ({word_count} words)")
            logger.log("=" * 80)

        # Chunking
        if word_count <= max_words:
            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]
            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if logger and verbose:
                logger.log("📄 Splitting long review into chunks...")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   Split into {len(chunks)} chunks")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = [
                {'chunk_id': i + 1, 'result': r}
                for i, r in enumerate(chunk_results)
            ]

        # Process extractions
        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0
        filtered_confidence_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, original_sentiment in zip(aspects, positions, sentiments):

                # Validate aspect
                aspect = refine_aspect_term(aspect)
                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Extract opinion
                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Format and validate opinion
                display_opinion = format_opinion_for_display(raw_opinion)

                if not display_opinion or not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                # Sentiment analysis
                sentiment_scores = None
                if use_sentiment_analysis:
                    sentiment_scores = analyze_sentiment_scores(display_opinion)

                # Confidence filtering
                if apply_confidence_filter and sentiment_scores:
                    should_include, _ = should_include_extraction(
                        sentiment_scores, display_opinion, original_sentiment
                    )
                    if not should_include:
                        filtered_confidence_count += 1
                        stats['filtered_low_confidence'] += 1
                        continue

                # Sentiment correction
                corrected_sentiment, correction_reason = correct_sentiment_with_analysis(
                    aspect, raw_opinion, original_sentiment, sentiment_scores
                )

                if corrected_sentiment != original_sentiment:
                    stats['sentiment_corrections'] += 1
                    if correction_reason:
                        stats['correction_reasons'][correction_reason] += 1
                        if 'idiom' in correction_reason:
                            stats['idiom_detections'] += 1
                        if 'sanity_check' in correction_reason:
                            stats['sanity_check_corrections'] += 1

                stats['total_aspects'] += 1

                # Create result
                pair = create_aspect_opinion_pair(
                    aspect=aspect,
                    opinion=display_opinion,
                    raw_opinion=raw_opinion,
                    sentiment=corrected_sentiment,
                    original_sentiment=original_sentiment,
                    pattern=pattern,
                    chunk_id=chunk_data['chunk_id'] if len(analysis_results) > 1 else None,
                    sentiment_scores=sentiment_scores,
                    correction_reason=correction_reason
                )
                aspect_opinion_pairs.append(pair)

        # Log results
        if logger and verbose:
            logger.log(f"\n🎯 Extraction Results:")
            header = f"{'#':<3} {'Aspect':<18} {'Opinion':<35} {'Orig':<8} {'Final':<8} {'Score':<8}"
            logger.log(header)
            logger.log("-" * 90)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    opinion_disp = pair['opinion'][:32] + "..." if len(pair['opinion']) > 35 else pair['opinion']

                    scores = pair['sentiment_scores']
                    score_str = f"{scores['ensemble_score']:+.2f}" if scores.get('ensemble_score') else "N/A"

                    final = pair['sentiment']
                    if pair['correction_reason']:
                        final += "*"

                    logger.log(f"{i:<3} {pair['aspect']:<18} {opinion_disp:<35} "
                              f"{pair['original_sentiment']:<8} {final:<8} {score_str:<8}")

                total_filtered = filtered_aspect_count + filtered_opinion_count + filtered_confidence_count
                if total_filtered > 0:
                    logger.log(f"\n   ℹ️ Filtered: {filtered_aspect_count} aspects, "
                              f"{filtered_opinion_count} opinions, "
                              f"{filtered_confidence_count} low confidence")
            else:
                logger.log("   ⚠️ No valid aspect-opinion pairs found")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
        })

    return all_results, stats


# ============================================
# SECTION 21: ENHANCED REPORT GENERATION
# ============================================

def select_representative_opinions(pairs, n=3, target_sentiment=None):
    """
    Select high-quality representative opinions.

    Selection Criteria:
        1. Sentiment matches target (CRITICAL: positive section shows only positive opinions)
        2. High absolute sentiment score (clear sentiment)
        3. Sufficient length (not fragments)
        4. Passes completeness validation
        5. Diverse (not similar to already selected)
        6. No negative words in positive section (and vice versa)

    Args:
        pairs: List of aspect-opinion pairs
        n: Number of opinions to select
        target_sentiment: 'Positive' or 'Negative' - MUST match this sentiment

    Returns:
        list: Selected representative opinions
    """
    # CRITICAL FIX: Filter by target sentiment FIRST
    if target_sentiment:
        pairs = [p for p in pairs if p.get('sentiment') == target_sentiment]

    # Sort by score magnitude (clearest sentiment first)
    sorted_pairs = sorted(
        pairs,
        key=lambda p: abs(p['sentiment_scores'].get('ensemble_score', 0) if p['sentiment_scores'] else 0),
        reverse=True
    )

    # Define words that indicate opposite sentiment
    strong_negative_words = {
        'terrible', 'horrible', 'awful', 'worst', 'bad', 'poor', 'disgusting',
        'rude', 'slow', 'cold', 'bland', 'disappointing', 'mediocre', 'nasty',
        'gross', 'dirty', 'stale', 'overpriced', 'incompetent', 'never'
    }
    strong_positive_words = {
        'great', 'excellent', 'amazing', 'wonderful', 'fantastic', 'perfect',
        'delicious', 'friendly', 'awesome', 'best', 'love', 'outstanding',
        'superb', 'incredible', 'fresh', 'tasty', 'beautiful', 'attentive'
    }

    selected = []
    seen_stems = set()

    for pair in sorted_pairs:
        opinion = pair['opinion']
        opinion_lower = opinion.lower()

        # CRITICAL: Skip if opinion contains strong opposite-sentiment words
        if target_sentiment == 'Positive':
            # For positive section, skip opinions with strong negative words
            if any(neg_word in opinion_lower for neg_word in strong_negative_words):
                continue
        elif target_sentiment == 'Negative':
            # For negative section, verify it actually sounds negative
            # (optional: could skip if too many positive words)
            pass

        # Quality checks
        if len(opinion.split()) < 2:
            continue

        if opinion_lower.startswith(('and ', 'or ', 'but ')):
            continue

        is_valid, _ = validate_opinion_completeness(opinion)
        if not is_valid:
            continue

        # Diversity check (avoid similar opinions)
        stem = ' '.join(opinion_lower.split()[:3])
        if stem in seen_stems:
            continue

        seen_stems.add(stem)
        selected.append(pair)

        if len(selected) >= n:
            break

    return selected


def generate_management_report(results, stats, logger=None):
    """
    Generate enhanced management report with multiple representative opinions.

    Report Sections:
        1. Overall Statistics
        2. Correction Breakdown
        3. Competitive Advantages (with 2-3 example opinions each)
        4. Areas for Improvement (with 2-3 example complaints each)
        5. Recommendations
        6. Quality Metrics
    """
    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 80)
    log("MANAGEMENT ANALYSIS REPORT")
    log("Universal Restaurant Review Analysis")
    log("=" * 80)

    # Aggregate by sentiment
    positive_pairs = []
    negative_pairs = []

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

    total_pairs = len(positive_pairs) + len(negative_pairs)

    # ===== STATISTICS =====
    log(f"\n📊 OVERALL STATISTICS")
    log(f"   • Total reviews analyzed: {len(results)}")
    log(f"   • Total aspects extracted: {total_pairs}")
    log(f"   • Invalid aspects filtered: {stats['filtered_invalid_aspects']}")
    log(f"   • Invalid opinions filtered: {stats['filtered_invalid_opinions']}")
    log(f"   • Low confidence filtered: {stats.get('filtered_low_confidence', 0)}")

    if stats['sentiment_corrections'] > 0:
        log(f"   • Sentiments corrected: {stats['sentiment_corrections']}")
        log(f"   • Idiom detections: {stats.get('idiom_detections', 0)}")
        log(f"   • Sanity check fixes: {stats.get('sanity_check_corrections', 0)}")

    if total_pairs > 0:
        pos_pct = len(positive_pairs) / total_pairs * 100
        neg_pct = len(negative_pairs) / total_pairs * 100
        log(f"   • Positive mentions: {len(positive_pairs)} ({pos_pct:.1f}%)")
        log(f"   • Negative mentions: {len(negative_pairs)} ({neg_pct:.1f}%)")

    # ===== CORRECTION BREAKDOWN =====
    if stats['correction_reasons']:
        log(f"\n📈 CORRECTION BREAKDOWN")
        for reason, count in stats['correction_reasons'].most_common(10):
            log(f"   • {reason}: {count}")

    # Aggregate by aspect
    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair)

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair)

    # ===== COMPETITIVE ADVANTAGES =====
    if positive_aspects:
        log(f"\n✅ COMPETITIVE ADVANTAGES (Top Positive Aspects)")
        log("-" * 80)

        sorted_pos = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_pos[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # Multiple representative opinions - MUST be positive sentiment
            examples = select_representative_opinions(pairs, n=3, target_sentiment='Positive')
            for ex in examples:
                opinion_text = ex['opinion'][:55]
                if len(ex['opinion']) > 55:
                    opinion_text += "..."
                # Show score for each example
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0
                log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid positive examples found, note it
            if not examples:
                log(f'   • [No clear positive examples available]')

    # ===== AREAS FOR IMPROVEMENT =====
    if negative_aspects:
        log(f"\n⚠️ AREAS FOR IMPROVEMENT (Top Negative Aspects)")
        log("-" * 80)

        sorted_neg = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_neg[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # Multiple representative opinions - MUST be negative sentiment
            examples = select_representative_opinions(pairs, n=3, target_sentiment='Negative')
            for ex in examples:
                opinion_text = ex['opinion'][:55]
                if len(ex['opinion']) > 55:
                    opinion_text += "..."
                # Show score for each example
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0
                log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid negative examples found, note it
            if not examples:
                log(f'   • [No clear negative examples available]')

    # ===== RECOMMENDATIONS =====
    log(f"\n💡 RECOMMENDATIONS")

    if negative_aspects:
        worst = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   1. PRIORITY: Address '{worst[0]}' ({len(worst[1])} negative mentions)")

    if positive_aspects:
        best = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   2. LEVERAGE: Promote '{best[0]}' ({len(best[1])} positive mentions)")

    log(f"   3. MONITOR: Track sentiment trends over time")
    log(f"   4. INVESTIGATE: Review corrected sentiments for accuracy")

    # ===== QUALITY METRICS =====
    log(f"\n📉 EXTRACTION QUALITY METRICS")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   • Extraction success rate: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   • Average aspects per review: {avg_pairs:.2f}")

    total_filtered = (stats['filtered_invalid_aspects'] +
                      stats['filtered_invalid_opinions'] +
                      stats.get('filtered_low_confidence', 0))
    total_extracted = total_pairs + total_filtered
    if total_extracted > 0:
        quality_rate = total_pairs / total_extracted * 100
        log(f"   • Quality pass rate: {total_pairs}/{total_extracted} ({quality_rate:.1f}%)")


# ============================================
# SECTION 22: DATA EXPORT
# ============================================

def export_results_to_dataframe(results):
    """Export results to pandas DataFrame."""
    import pandas as pd

    rows = []
    for result in results:
        review_id = result['review_id']
        for pair in result['pairs']:
            scores = pair.get('sentiment_scores', {})
            rows.append({
                'review_id': review_id,
                'aspect': pair['aspect'],
                'opinion': pair['opinion'],
                'original_sentiment': pair['original_sentiment'],
                'corrected_sentiment': pair['sentiment'],
                'correction_reason': pair.get('correction_reason'),
                'ensemble_score': scores.get('ensemble_score'),
                'ensemble_confidence': scores.get('ensemble_confidence')
            })

    return pd.DataFrame(rows)


# ============================================
# SECTION 23: MAIN EXECUTION
# ============================================

def run_analysis(reviews, aspect_extractor,
                 output_file="absa_results_v5_universal.txt",
                 use_sentiment_analysis=True,
                 apply_confidence_filter=True):
    """
    Run complete ABSA analysis pipeline.

    Args:
        reviews: Review texts (list or Series)
        aspect_extractor: PyABSA extractor
        output_file: Output log file path
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence filtering

    Returns:
        tuple: (results, stats, dataframe)
    """
    logger = FileLogger(output_file)

    logger.log("=" * 80)
    logger.log("ABSA ANALYSIS SYSTEM - V5 ENHANCED UNIVERSAL EDITION")
    logger.log(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"Reviews to analyze: {len(reviews)}")
    logger.log("=" * 80)

    results, stats = comprehensive_aspect_opinion_analysis(
        reviews,
        aspect_extractor,
        max_words=MAX_WORDS_PER_CHUNK,
        verbose=True,
        logger=logger,
        use_sentiment_analysis=use_sentiment_analysis,
        apply_confidence_filter=apply_confidence_filter
    )

    logger.log("\n" + "=" * 80)
    logger.log("✅ ANALYSIS COMPLETE")
    logger.log("=" * 80)

    generate_management_report(results, stats, logger)
    logger.save()

    df_results = export_results_to_dataframe(results)

    return results, stats, df_results


if __name__ == "__main__":
    print("=" * 60)
    print("ABSA Analysis System - V5 Enhanced Universal Edition")
    print("=" * 60)
    print("\nDesigned for UNIVERSAL restaurant review analysis:")
    print("  • All restaurant types (fast food to fine dining)")
    print("  • All cuisines (American, Asian, Mexican, etc.)")
    print("  • All service models (dine-in, takeout, delivery)")
    print("\nKey Enhancements:")
    print("  1. Idiom/Slang Dictionary (~80 expressions)")
    print("  2. Confidence-Based Filtering")
    print("  3. Improved Opinion Boundary Detection")
    print("  4. Enhanced Management Report")
    print("\nUsage:")
    print("  results, stats, df = run_analysis(reviews, aspect_extractor)")




In [ ]:
results, stats, df = run_analysis(test_reviews, aspect_extractor)

In [ ]:
"""
================================================================================
ABSA (Aspect-Based Sentiment Analysis) System - V5 Enhanced Universal Edition
================================================================================

Purpose:
    Extract aspect-opinion-sentiment triplets from restaurant reviews.
    Designed for UNIVERSAL application across ALL restaurant types:
    - Fast food, casual dining, fine dining
    - All cuisines (American, Asian, Mexican, Italian, etc.)
    - All service models (dine-in, takeout, delivery, food trucks)

Key Enhancements in V5:
    1. Idiom/Slang Dictionary - Handles "did not disappoint", "killed it", etc.
    2. Confidence-Based Filtering - Excludes low-confidence extractions
    3. Improved Opinion Boundary Detection - Avoids fragments and factual statements
    4. Enhanced Management Report - Multiple representative opinions per aspect
    5. Universal Restaurant Vocabulary - Multi-cuisine support

Version: 5.0 Enhanced Universal
Author: HAOS Framework Research Team
Date: 2025
================================================================================
"""

import warnings
warnings.filterwarnings('ignore')

import re
from collections import Counter
from datetime import datetime

# ============================================
# SECTION 1: GLOBAL CONFIGURATION
# ============================================

OUTPUT_FILE = "absa_analysis_results_v5_universal.txt"
MAX_WORDS_PER_CHUNK = 80

# Sentiment correction thresholds
SENTIMENT_CONFIDENCE_THRESHOLD = 0.6
SENTIMENT_OVERRIDE_THRESHOLD = 0.4

# NEW: Confidence-based filtering thresholds
# Extractions with |score| < this threshold will be excluded
MINIMUM_SENTIMENT_SCORE_THRESHOLD = 0.25

# Minimum agreement level between sentiment analyzers
MINIMUM_AGREEMENT_THRESHOLD = 0.40


# ============================================
# SECTION 2: IDIOM/SLANG DICTIONARIES
# ============================================
"""
CRITICAL SECTION: Idiom Detection

Problem Solved:
    Many English expressions use negation words but convey POSITIVE meaning.
    Example: "The service did not disappoint" = POSITIVE (not negative!)

These idioms are universal across ALL restaurant types:
    - Fast food: "The fries hit the spot"
    - Fine dining: "The tasting menu was to die for"
    - BBQ: "Ribs fall off the bone"
    - Any cuisine: "Our server killed it tonight"
"""


def get_positive_idioms():
    """
    Dictionary of positive idioms/slang expressions.

    Categories:
    1. Double Negative = Positive ("did not disappoint")
    2. Superlative Negation = Maximum Positive ("could not have been better")
    3. Slang = Positive ("killed it", "fire", "bussin")
    4. Food-Specific = Positive ("melt in your mouth")

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DOUBLE NEGATIVE = POSITIVE ==========
        # Pattern: "did not [negative_word]" → Positive
        # Logic: NOT disappointing = satisfying
        "did not disappoint": "Positive",
        "didn't disappoint": "Positive",
        "does not disappoint": "Positive",
        "doesn't disappoint": "Positive",
        "never disappoints": "Positive",
        "never disappointed": "Positive",
        "not disappointed": "Positive",
        "wasn't disappointed": "Positive",
        "weren't disappointed": "Positive",
        "not let down": "Positive",
        "didn't let us down": "Positive",

        # ========== SUPERLATIVE NEGATION = MAXIMUM POSITIVE ==========
        # Pattern: "could not have been more [positive_adj]"
        # Logic: Impossible to be MORE positive = maximum level achieved
        "could not have been better": "Positive",
        "couldn't have been better": "Positive",
        "could not have been nicer": "Positive",
        "couldn't have been nicer": "Positive",
        "could not have been friendlier": "Positive",
        "couldn't have been friendlier": "Positive",
        "could not have been more helpful": "Positive",
        "couldn't have been more helpful": "Positive",
        "could not have been more attentive": "Positive",
        "couldn't have been more attentive": "Positive",
        "could not have been more professional": "Positive",
        "couldn't have been more professional": "Positive",
        "could not ask for more": "Positive",
        "couldn't ask for more": "Positive",
        "could not ask for better": "Positive",
        "couldn't ask for better": "Positive",
        "could not be happier": "Positive",
        "couldn't be happier": "Positive",
        "could not be more satisfied": "Positive",
        "couldn't be more satisfied": "Positive",

        # ========== NO COMPLAINTS = POSITIVE ==========
        "can't complain": "Positive",
        "cannot complain": "Positive",
        "nothing to complain about": "Positive",
        "no complaints": "Positive",
        "zero complaints": "Positive",
        "left nothing to be desired": "Positive",
        "leaves nothing to be desired": "Positive",

        # ========== SLANG/INFORMAL = POSITIVE ==========
        # Common in casual reviews, especially younger demographics
        "killed it": "Positive",           # Performed excellently
        "nailed it": "Positive",           # Got it exactly right
        "crushed it": "Positive",          # Exceeded expectations
        "smashed it": "Positive",          # Did amazingly well
        "hit the spot": "Positive",        # Perfectly satisfying
        "hits the spot": "Positive",
        "hit different": "Positive",       # Especially good
        "hits different": "Positive",
        "on point": "Positive",            # Exactly right
        "on fire": "Positive",             # Performing excellently
        "off the hook": "Positive",        # Extremely good
        "off the chain": "Positive",       # Amazing
        "off the charts": "Positive",      # Exceptionally high quality
        "out of this world": "Positive",   # Extraordinary
        "to die for": "Positive",          # Extremely desirable
        "die for": "Positive",
        "the bomb": "Positive",            # Excellent
        "da bomb": "Positive",
        "was bomb": "Positive",            # "The pizza was bomb"
        "is bomb": "Positive",
        "is fire": "Positive",             # "These wings are fire"
        "was fire": "Positive",
        "straight fire": "Positive",
        "chef's kiss": "Positive",         # Perfect
        "chefs kiss": "Positive",
        "slaps": "Positive",               # "This mac and cheese slaps"
        "bussin": "Positive",              # "Food was bussin"
        "bussin'": "Positive",
        "top notch": "Positive",
        "top-notch": "Positive",
        "first rate": "Positive",
        "first-rate": "Positive",
        "A1": "Positive",                  # Top quality
        "a-1": "Positive",
        "10/10": "Positive",
        "10 out of 10": "Positive",
        "five stars": "Positive",
        "5 stars": "Positive",

        # ========== FOOD-SPECIFIC POSITIVE IDIOMS ==========
        # Universal across cuisines
        "melt in your mouth": "Positive",
        "melts in your mouth": "Positive",
        "melted in my mouth": "Positive",
        "fall off the bone": "Positive",   # Tender meat
        "falls off the bone": "Positive",
        "fell off the bone": "Positive",
        "cooked to perfection": "Positive",
        "done to perfection": "Positive",
        "seasoned to perfection": "Positive",
        "finger licking good": "Positive",
        "finger-licking good": "Positive",
        "fresh out of the oven": "Positive",
        "fresh off the grill": "Positive",
        "made with love": "Positive",
        "like grandma used to make": "Positive",
        "like mama used to make": "Positive",
        "home cooked": "Positive",
        "homemade taste": "Positive",

        # ========== RECOMMENDATION POSITIVE ==========
        "a must try": "Positive",
        "must try": "Positive",
        "a must": "Positive",
        "a must visit": "Positive",
        "must visit": "Positive",
        "a gem": "Positive",
        "hidden gem": "Positive",
        "best kept secret": "Positive",
        "saved the day": "Positive",
        "made my day": "Positive",
        "worth the wait": "Positive",
        "worth the drive": "Positive",
        "worth every penny": "Positive",
        "worth the price": "Positive",
        "bang for your buck": "Positive",
        "will be back": "Positive",
        "coming back": "Positive",
        "definitely returning": "Positive",
        "can't wait to come back": "Positive",
    }


def get_negative_idioms():
    """
    Dictionary of negative idioms/slang expressions.

    Categories:
    1. Disappointment expressions
    2. Mediocrity expressions
    3. Poor value expressions
    4. Food-specific negative idioms

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DISAPPOINTMENT ==========
        "left a lot to be desired": "Negative",
        "leaves a lot to be desired": "Negative",
        "left much to be desired": "Negative",
        "leaves much to be desired": "Negative",
        "nothing to write home about": "Negative",
        "not my cup of tea": "Negative",
        "wasn't my cup of tea": "Negative",
        "seen better days": "Negative",
        "has seen better days": "Negative",
        "let down": "Negative",
        "let us down": "Negative",
        "fell short": "Negative",
        "falls short": "Negative",
        "missed the mark": "Negative",
        "misses the mark": "Negative",

        # ========== MEDIOCRITY ==========
        "hit or miss": "Negative",
        "meh": "Negative",
        "just ok": "Negative",
        "just okay": "Negative",
        "nothing special": "Negative",
        "middle of the road": "Negative",
        "run of the mill": "Negative",
        "average at best": "Negative",
        "could be better": "Negative",
        "room for improvement": "Negative",
        "needs work": "Negative",

        # ========== POOR VALUE ==========
        "waste of money": "Negative",
        "waste of time": "Negative",
        "rip off": "Negative",
        "rip-off": "Negative",
        "ripoff": "Negative",
        "not worth it": "Negative",
        "not worth the price": "Negative",
        "not worth the money": "Negative",
        "not worth the wait": "Negative",
        "not worth the hype": "Negative",
        "overpriced": "Negative",
        "over priced": "Negative",
        "overhyped": "Negative",
        "over-hyped": "Negative",
        "overrated": "Negative",
        "over-rated": "Negative",

        # ========== FOOD-SPECIFIC NEGATIVE ==========
        "tasted like cardboard": "Negative",
        "tastes like cardboard": "Negative",
        "like eating cardboard": "Negative",
        "rubber chicken": "Negative",
        "hockey puck": "Negative",         # Overcooked burger
        "sat under a heat lamp": "Negative",
        "sitting under a heat lamp": "Negative",
        "straight from the freezer": "Negative",
        "microwaved": "Negative",
        "reheated": "Negative",
        "day old": "Negative",
        "stale": "Negative",

        # ========== SERVICE NEGATIVE ==========
        "worst service ever": "Negative",
        "worst experience ever": "Negative",
        "never coming back": "Negative",
        "will not return": "Negative",
        "won't be back": "Negative",
        "avoid this place": "Negative",
        "stay away": "Negative",
        "save your money": "Negative",
    }


# ============================================
# SECTION 3: SENTIMENT ANALYZER INITIALIZATION
# ============================================

SENTIMENT_ANALYZERS = None


def initialize_sentiment_analyzers():
    """
    Initialize multiple sentiment analysis tools for ensemble approach.

    Why Multiple Analyzers:
        - VADER: Fast, rule-based, optimized for social media/reviews
        - TextBlob: Pattern-based, provides subjectivity scores
        - Transformer: Deep learning, most accurate for complex patterns
        - Ensemble of all three provides robust results

    Returns:
        dict: Initialized analyzers (None if unavailable)
    """
    analyzers = {}

    # VADER Initialization
    try:
        import nltk
        nltk.download('vader_lexicon', quiet=True)
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        analyzers['vader'] = SentimentIntensityAnalyzer()
        print("✅ VADER sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ VADER not available: {e}")
        analyzers['vader'] = None

    # TextBlob Initialization
    try:
        from textblob import TextBlob
        analyzers['textblob'] = TextBlob
        print("✅ TextBlob sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ TextBlob not available: {e}")
        analyzers['textblob'] = None

    # Transformer Initialization (Optional)
    try:
        from transformers import pipeline
        analyzers['transformer'] = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=-1
        )
        print("✅ Transformer sentiment analyzer loaded")
    except Exception as e:
        print(f"ℹ️ Transformer not available (optional): {e}")
        analyzers['transformer'] = None

    return analyzers


def get_sentiment_analyzers():
    """Get or initialize sentiment analyzers (singleton pattern)."""
    global SENTIMENT_ANALYZERS
    if SENTIMENT_ANALYZERS is None:
        SENTIMENT_ANALYZERS = initialize_sentiment_analyzers()
    return SENTIMENT_ANALYZERS


# ============================================
# SECTION 4: UNIVERSAL RESTAURANT VOCABULARY
# ============================================
"""
Universal Vocabulary Design:
    These word lists are designed to work across ALL restaurant types.
    They cover common aspects, opinions, and patterns found in reviews
    regardless of cuisine or service model.
"""


def get_common_english_names():
    """
    Common first names to filter from opinions.

    Rationale:
        Reviews often mention server names: "Sarah was great"
        Names alone don't provide sentiment - the opinion does.

    Returns:
        set: Common English first names (lowercase)
    """
    male_names = {
        'james', 'john', 'robert', 'michael', 'william', 'david', 'richard',
        'joseph', 'thomas', 'charles', 'christopher', 'daniel', 'matthew',
        'anthony', 'mark', 'donald', 'steven', 'paul', 'andrew', 'joshua',
        'kenneth', 'kevin', 'brian', 'george', 'timothy', 'ronald', 'edward',
        'jason', 'jeffrey', 'ryan', 'jacob', 'gary', 'nicholas', 'eric',
        'jonathan', 'stephen', 'larry', 'justin', 'scott', 'brandon', 'benjamin',
        'samuel', 'raymond', 'gregory', 'frank', 'alexander', 'patrick', 'jack',
        'joel', 'paul', 'morgan', 'jordan', 'chris', 'matt', 'andy', 'ray',
        'nick', 'greg', 'mike', 'steve', 'tom', 'bob', 'jim', 'dan'
    }

    female_names = {
        'mary', 'patricia', 'jennifer', 'linda', 'barbara', 'elizabeth', 'susan',
        'jessica', 'sarah', 'karen', 'lisa', 'nancy', 'betty', 'margaret', 'sandra',
        'ashley', 'kimberly', 'emily', 'donna', 'michelle', 'dorothy', 'carol',
        'amanda', 'melissa', 'deborah', 'stephanie', 'rebecca', 'sharon', 'laura',
        'sara', 'fatima', 'kaitlin', 'kaitlyn', 'natalie', 'melanie', 'christina',
        'veronica', 'morgan', 'jordan', 'gabby', 'tina', 'cameron'
    }

    return male_names | female_names


def get_positive_sentiment_words():
    """
    Universal positive sentiment words for restaurant reviews.

    Categories:
        - General quality descriptors
        - Food-specific positive terms
        - Service-related positive terms
        - Atmosphere/environment terms
        - Value-related terms

    Returns:
        set: Positive sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic',
        'outstanding', 'superb', 'perfect', 'incredible', 'awesome',
        'terrific', 'fabulous', 'marvelous', 'exceptional', 'best',
        'impressive', 'remarkable', 'magnificent', 'splendid', 'brilliant',
        'stellar', 'phenomenal', 'spectacular', 'extraordinary', 'superb',

        # ===== FOOD-SPECIFIC (Universal across cuisines) =====
        'delicious', 'tasty', 'yummy', 'flavorful', 'savory', 'fresh',
        'tender', 'juicy', 'crispy', 'creamy', 'rich', 'light',
        'authentic', 'homemade', 'seasoned', 'aromatic', 'scrumptious',
        'divine', 'heavenly', 'mouthwatering', 'succulent', 'zesty',
        'perfectly cooked', 'well-seasoned', 'flavorsome',

        # ===== SERVICE-RELATED =====
        'friendly', 'helpful', 'attentive', 'professional', 'courteous',
        'prompt', 'efficient', 'welcoming', 'accommodating', 'polite',
        'responsive', 'knowledgeable', 'patient', 'thorough', 'dedicated',
        'personable', 'warm', 'gracious', 'sweet', 'caring',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'clean', 'comfortable', 'cozy', 'spacious', 'nice', 'lovely',
        'beautiful', 'charming', 'pleasant', 'relaxing', 'quiet',
        'modern', 'elegant', 'stylish', 'inviting', 'cute',
        'trendy', 'vibrant', 'lively', 'romantic', 'intimate',

        # ===== VALUE-RELATED =====
        'reasonable', 'affordable', 'worth', 'value', 'cheap', 'bargain',
        'fair', 'inexpensive', 'economical', 'generous'
    }


def get_negative_sentiment_words():
    """
    Universal negative sentiment words for restaurant reviews.

    Returns:
        set: Negative sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'bad', 'terrible', 'awful', 'horrible', 'poor', 'disappointing',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross', 'worst',
        'inferior', 'subpar', 'unacceptable', 'dreadful', 'appalling',
        'atrocious', 'abysmal', 'horrendous', 'disgusting',

        # ===== FOOD-SPECIFIC =====
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty', 'stale',
        'cold', 'lukewarm', 'frozen', 'rubbery', 'tough', 'chewy',
        'flavorless', 'mushy', 'bitter', 'sour', 'spoiled',
        'unseasoned', 'unappetizing', 'oily', 'oversalted',

        # ===== SERVICE-RELATED =====
        'rude', 'slow', 'unfriendly', 'unhelpful', 'inattentive',
        'unprofessional', 'careless', 'dismissive', 'indifferent',
        'incompetent', 'negligent', 'impolite', 'disrespectful',
        'ignored', 'forgotten', 'invisible',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'dirty', 'messy', 'noisy', 'crowded', 'cramped', 'uncomfortable',
        'dark', 'smelly', 'hot', 'stuffy', 'outdated', 'dingy',
        'filthy', 'cluttered', 'rundown', 'shabby', 'sticky',

        # ===== VALUE-RELATED =====
        'expensive', 'overpriced', 'costly', 'pricey', 'ripoff',
        'unreasonable', 'exorbitant', 'small portions'
    }


def get_action_verbs():
    """
    Action verbs that should NOT be treated as opinions.

    Rationale:
        "I ordered the pasta" - "ordered" is action, not opinion
        "The waiter served us" - "served" is action, not opinion

    Returns:
        set: Action verbs in various forms
    """
    return {
        'order', 'ordered', 'ordering', 'orders',
        'ask', 'asked', 'asking', 'asks',
        'request', 'requested', 'requesting', 'requests',
        'come', 'came', 'coming', 'comes',
        'go', 'went', 'going', 'goes', 'gone',
        'arrive', 'arrived', 'arriving', 'arrives',
        'leave', 'left', 'leaving', 'leaves',
        'walk', 'walked', 'walking', 'walks',
        'enter', 'entered', 'entering', 'enters',
        'visit', 'visited', 'visiting', 'visits',
        'get', 'got', 'getting', 'gets',
        'take', 'took', 'taking', 'takes', 'taken',
        'bring', 'brought', 'bringing', 'brings',
        'receive', 'received', 'receiving', 'receives',
        'make', 'made', 'making', 'makes',
        'do', 'did', 'doing', 'does', 'done',
        'prepare', 'prepared', 'preparing', 'prepares',
        'cook', 'cooked', 'cooking', 'cooks',
        'say', 'said', 'saying', 'says',
        'tell', 'told', 'telling', 'tells',
        'call', 'called', 'calling', 'calls',
        'serve', 'served', 'serving', 'serves',
        'show', 'showed', 'showing', 'shows', 'shown',
        'give', 'gave', 'giving', 'gives', 'given',
        'seat', 'seated', 'seating', 'seats',
        'try', 'tried', 'trying', 'tries',
        'want', 'wanted', 'wanting', 'wants',
        'need', 'needed', 'needing', 'needs',
        'hope', 'hoped', 'hoping', 'hopes',
        'expect', 'expected', 'expecting', 'expects',
        'enjoy', 'enjoyed', 'enjoying', 'enjoys',
        'pay', 'paid', 'paying', 'pays',
        'wait', 'waited', 'waiting', 'waits',
        'sit', 'sat', 'sitting', 'sits',
        'eat', 'ate', 'eating', 'eats', 'eaten',
        'drink', 'drank', 'drinking', 'drinks', 'drunk'
    }


def get_function_words():
    """
    Function words that carry no sentiment meaning.

    Returns:
        set: Articles, prepositions, pronouns, etc.
    """
    return {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'i', 'me', 'my', 'mine', 'myself',
        'you', 'your', 'yours', 'yourself',
        'he', 'him', 'his', 'himself',
        'she', 'her', 'hers', 'herself',
        'it', 'its', 'itself',
        'we', 'us', 'our', 'ours', 'ourselves',
        'they', 'them', 'their', 'theirs', 'themselves',
        'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
        'from', 'about', 'into', 'through', 'during', 'before',
        'after', 'above', 'below', 'between', 'under', 'over',
        'and', 'or', 'but', 'so', 'yet', 'nor',
        'if', 'when', 'where', 'while', 'as', 'because', 'since',
        'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'has', 'have', 'had', 'having',
        'do', 'does', 'did',
        'will', 'would', 'shall', 'should',
        'can', 'could', 'may', 'might', 'must',
        'here', 'there', 'some', 'any', 'no', 'every'
    }


def get_intensifiers():
    """
    Intensifier words that modify sentiment strength.

    Returns:
        set: Intensifier words
    """
    return {
        'very', 'extremely', 'incredibly', 'amazingly', 'exceptionally',
        'remarkably', 'absolutely', 'totally', 'completely', 'utterly',
        'entirely', 'thoroughly', 'perfectly', 'highly', 'deeply',
        'truly', 'really', 'genuinely', 'seriously',
        'quite', 'rather', 'fairly', 'pretty', 'somewhat',
        'reasonably', 'moderately', 'relatively',
        'slightly', 'a bit', 'a little', 'mildly', 'barely',
        'too', 'overly', 'excessively',
        'so', 'such', 'super', 'especially', 'particularly'
    }


# ============================================
# SECTION 5: IDIOM DETECTION FUNCTIONS
# ============================================

def check_for_idiom(text):
    """
    Check if text contains any known idiom.

    PRIORITY: This check runs BEFORE negation detection.

    Args:
        text (str): Text to check

    Returns:
        tuple: (is_idiom: bool, sentiment: str or None, matched_idiom: str or None)

    Example:
        >>> check_for_idiom("The food did not disappoint")
        (True, "Positive", "did not disappoint")

        >>> check_for_idiom("The food was cold")
        (False, None, None)
    """
    text_lower = text.lower()

    # Check positive idioms
    positive_idioms = get_positive_idioms()
    for idiom, sentiment in positive_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    # Check negative idioms
    negative_idioms = get_negative_idioms()
    for idiom, sentiment in negative_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    return False, None, None


# ============================================
# SECTION 6: ADJECTIVE/ADVERB DETECTION
# ============================================

def is_adjective_or_adverb(word):
    """
    Determine if a word is an adjective or adverb.

    Uses two approaches:
        1. Dictionary lookup (known sentiment words)
        2. Morphological rules (suffix patterns)

    Args:
        word (str): Word to check

    Returns:
        tuple: (is_adj_or_adv: bool, word_type: str)
    """
    word_lower = word.lower().strip('.,!?;:\'"')

    if not word_lower:
        return False, 'none'

    # Check sentiment dictionaries
    if word_lower in get_positive_sentiment_words():
        return True, 'positive_adj'

    if word_lower in get_negative_sentiment_words():
        return True, 'negative_adj'

    if word_lower in get_intensifiers():
        return True, 'intensifier'

    # Morphological rules
    adj_suffixes = ['ful', 'less', 'ous', 'ive', 'able', 'ible',
                    'al', 'ic', 'ish', 'ent', 'ant', 'ory', 'ary']

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    # Adverb suffix
    if word_lower.endswith('ly') and len(word_lower) > 4:
        non_adverb_ly = {'family', 'only', 'early', 'likely', 'friendly', 'lonely'}
        if word_lower not in non_adverb_ly:
            return True, 'suffix_adv'

    # Past participles as adjectives
    sentiment_participles = {
        'disappointed', 'satisfied', 'pleased', 'impressed', 'amazed',
        'surprised', 'disgusted', 'frustrated', 'annoyed', 'delighted',
        'thrilled', 'excited', 'bored', 'tired', 'exhausted',
        'overwhelmed', 'underwhelmed', 'overpriced'
    }
    if word_lower in sentiment_participles:
        return True, 'suffix_adj'

    # Present participles as adjectives
    sentiment_ing = {
        'amazing', 'disappointing', 'disgusting', 'interesting', 'boring',
        'exciting', 'frustrating', 'annoying', 'satisfying', 'refreshing',
        'relaxing', 'welcoming', 'inviting', 'appealing', 'appalling'
    }
    if word_lower in sentiment_ing:
        return True, 'suffix_adj'

    return False, 'none'


# ============================================
# SECTION 7: NEGATION DETECTION
# ============================================

def detect_negation_in_opinion(opinion):
    """
    Detect negation in opinion text.

    IMPORTANT: Check idioms FIRST before calling this function.

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (has_negation: bool, negation_type: str or None)

    Example:
        >>> detect_negation_in_opinion("not good")
        (True, 'direct')

        >>> detect_negation_in_opinion("wasn't fresh")
        (True, 'contraction')
    """
    opinion_lower = opinion.lower()

    # SAFETY CHECK: If this is an idiom, don't treat as simple negation
    is_idiom, _, _ = check_for_idiom(opinion_lower)
    if is_idiom:
        return False, 'idiom_detected'

    # Direct negation words
    direct_negations = [
        'not ', "n't ", 'no ', 'never ', 'none ', 'nothing ',
        'neither ', 'nobody ', 'nowhere ', 'cannot '
    ]

    for neg in direct_negations:
        if neg in opinion_lower or opinion_lower.startswith(neg.strip()):
            return True, 'direct'

    # Contracted negations
    contracted_negations = [
        "didn't", "wasn't", "weren't", "isn't", "aren't",
        "don't", "doesn't", "won't", "wouldn't", "couldn't",
        "shouldn't", "can't", "haven't", "hasn't", "hadn't"
    ]

    for neg in contracted_negations:
        if neg in opinion_lower:
            return True, 'contraction'

    return False, None


# ============================================
# SECTION 8: BOUNDARY DETECTION
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """Find sentence boundaries containing the aspect."""
    sentence_enders = {'.', '!', '?'}

    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in sentence_enders:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in sentence_enders:
            sentence_end = i
            break

    return sentence_start, sentence_end


def find_previous_aspect_end(current_start_idx, all_aspect_positions):
    """Find end position of previous aspect."""
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(current_end_idx, all_aspect_positions):
    """Find start position of next aspect."""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# SECTION 9: BE-VERB DETECTION
# ============================================

def detect_be_verb(tokens, start_idx):
    """Detect be-verb at given position."""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    be_verbs = {'is', 'was', 'were', 'are', 'be', 'been', "'s", "s"}
    if word in be_verbs:
        return True, 1, False

    negated_be_verbs = {
        "isn't", "isnt", "wasn't", "wasnt",
        "weren't", "werent", "aren't", "arent"
    }
    if word in negated_be_verbs:
        return True, 1, True

    return False, 0, False


# ============================================
# SECTION 10: OPINION VALIDATION
# ============================================

def validate_opinion_completeness(opinion):
    """
    Validate that opinion is complete and meaningful.

    Rules:
        1. Cannot start with conjunction (incomplete phrase)
        2. Single intensifier alone is invalid
        3. Cannot be factual statement (reporting, not opinion)
        4. Must contain sentiment signal

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (is_valid: bool, reason: str or None)

    Examples:
        >>> validate_opinion_completeness("and was delicious")
        (False, "starts_with_conjunction")

        >>> validate_opinion_completeness("she said yes")
        (False, "factual_statement")

        >>> validate_opinion_completeness("absolutely delicious")
        (True, None)
    """
    if not opinion or len(opinion.strip()) < 2:
        return False, 'empty'

    words = opinion.lower().strip().split()
    if not words:
        return False, 'empty'

    # Rule 1: Cannot start with conjunction
    conjunctions = {'and', 'or', 'but', 'so', 'yet', 'nor', 'for'}
    if words[0] in conjunctions:
        return False, 'starts_with_conjunction'

    # Rule 2: Single intensifier is invalid
    if len(words) == 1:
        intensifiers = get_intensifiers()
        if words[0] in intensifiers:
            return False, 'only_intensifier'

        function_words = get_function_words()
        if words[0] in function_words:
            return False, 'only_function_word'

    # Rule 3: Cannot be factual statement
    factual_patterns = [
        r'^(he|she|they|it|we|i)\s+(said|told|asked|mentioned|replied)',
        r'^(was|were|is|are)\s+(out of|available|unavailable)',
        r'^if\s+(it|they|she|he|we)\s+(was|were|is|are)',
        r'^that\s+(it|they)',
        r'^\d+\s*/',
        r'^with\s+(a\s+)?side',
    ]

    opinion_text = opinion.lower()
    for pattern in factual_patterns:
        if re.search(pattern, opinion_text):
            return False, 'factual_statement'

    # Rule 4: Must contain sentiment signal (for short opinions)
    if len(words) <= 4:
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        all_sentiment = positive_words | negative_words

        has_sentiment = False
        for word in words:
            clean_word = word.strip('.,!?;:\'"')
            if clean_word in all_sentiment:
                has_sentiment = True
                break
            is_adj, _ = is_adjective_or_adverb(clean_word)
            if is_adj:
                has_sentiment = True
                break

        if not has_sentiment:
            # Exception: Idioms should pass
            is_idiom, _, _ = check_for_idiom(opinion)
            if not is_idiom:
                return False, 'no_sentiment_signal'

    return True, None


def is_valid_opinion(opinion):
    """
    Check if extracted text is a valid opinion.

    Args:
        opinion (str): Extracted opinion text

    Returns:
        bool: True if valid opinion
    """
    if not opinion or len(opinion.strip()) < 2:
        return False

    # Use completeness validation
    is_complete, _ = validate_opinion_completeness(opinion)
    if not is_complete:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    if not words:
        return False

    action_verbs = get_action_verbs()
    function_words = get_function_words()
    common_names = get_common_english_names()

    # Single word validation
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')
        if clean_word in action_verbs:
            return False
        if clean_word in function_words:
            return False
        if clean_word in common_names:
            return False

    # Invalid patterns
    invalid_patterns = [
        r"^'s\s+",
        r"^\d+\s*/",
        r"^or\s+maybe\s+both$",
        r"^going\s+forward$",
        r"^n\s+cheese",  # Truncated "mac n cheese"
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    return True


# ============================================
# SECTION 11: ASPECT VALIDATION
# ============================================

def is_valid_aspect(aspect):
    """Check if extracted text is a valid aspect."""
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    if not cleaned or cleaned == "-":
        return False

    if all(c in '.,!?;:-_\'"' for c in cleaned):
        return False

    # Invalid single words (prepositions, conjunctions, etc.)
    invalid_single = {'of', 'to', 'for', 'with', 'at', 'in', 'on', 'by',
                      'and', 'or', 'but', 'n', 'the', 'a', 'an'}
    if cleaned in invalid_single:
        return False

    invalid_aspects = {
        # Verbs
        'wait', 'waited', 'waiting', 'order', 'ordered', 'ordering',
        'serve', 'served', 'serving', 'ask', 'asked', 'asking',
        # Pronouns
        'i', 'me', 'my', 'you', 'your', 'he', 'him', 'his',
        'she', 'her', 'it', 'its', 'we', 'us', 'our', 'they', 'them',
        # Time units
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours',
        # Adjectives (opinions, not aspects)
        'good', 'bad', 'great', 'nice', 'poor',
        # Generic
        'thing', 'things', 'stuff', 'way', 'time', 'place'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """Clean and standardize aspect term."""
    if not aspect:
        return ""

    clean_aspect = aspect.strip()
    clean_aspect = re.sub(r'^(and|or|but)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^(the|a|an|this|that|my|our)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^[-:;,.\'"]+\s*', '', clean_aspect)
    clean_aspect = re.sub(r'[-:;,.\'"]+$', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# SECTION 12: OPINION EXTRACTION (ENHANCED)
# ============================================

def check_sentence_completeness(opinion_words, tokens, current_end_idx, sentence_end):
    """
    Check if opinion forms a complete sentence/phrase.
    If incomplete, extend to next sentence boundary.

    Args:
        opinion_words: Current list of opinion words
        tokens: All tokens in the review
        current_end_idx: Current end position
        sentence_end: Maximum sentence boundary

    Returns:
        list: Extended opinion words if needed
    """
    if not opinion_words:
        return opinion_words

    # Join current words to check completeness
    opinion_text = ' '.join(opinion_words).lower().strip()

    # Patterns indicating incomplete sentences
    incomplete_patterns = [
        r"(wasn|isn|aren|weren|didn|doesn|hasn|haven|hadn|wouldn|couldn|shouldn|won|can|don)\s*'\s*t\s*$",  # ends with contraction
        r"\s+(was|is|are|were|did|does|has|have|had|would|could|should|will|can|do)\s*$",  # ends with auxiliary verb
        r"\s+(he|she|they|it|we|i)\s+(said|told|asked)\s+.{0,20}$",  # incomplete quote
        r",\s*$",  # ends with comma
        r"\s+that\s*$",  # ends with "that"
        r"\s+which\s*$",  # ends with "which"
        r"\s+who\s*$",  # ends with "who"
        r"\s+when\s*$",  # ends with "when"
        r"\s+where\s*$",  # ends with "where"
        r"\s+if\s*$",  # ends with "if"
        r"\s+because\s*$",  # ends with "because"
        r"\s+although\s*$",  # ends with "although"
        r"\s+their\s*$",  # ends with possessive
        r"\s+his\s*$",
        r"\s+her\s*$",
        r"\s+our\s*$",
        r"\s+my\s*$",
    ]

    is_incomplete = False
    for pattern in incomplete_patterns:
        if re.search(pattern, opinion_text):
            is_incomplete = True
            break

    # Also check if last word is a verb form without object
    if opinion_words:
        last_word = opinion_words[-1].lower().strip('.,!?;:\'"')
        incomplete_endings = {
            'was', 'is', 'are', 'were', 'did', 'does', 'has', 'have', 'had',
            'would', 'could', 'should', 'will', 'can', 'do', 'said', 'told',
            'their', 'his', 'her', 'our', 'my', 'your', 'its', 'the', 'a', 'an',
            "wasn", "isn", "aren", "weren", "didn", "doesn", "hasn",
            "haven", "hadn", "wouldn", "couldn", "shouldn", "won", "don"
        }
        if last_word in incomplete_endings:
            is_incomplete = True

    # If incomplete, extend to next sentence boundary
    if is_incomplete and current_end_idx < sentence_end:
        extended_words = list(opinion_words)
        for i in range(current_end_idx, sentence_end):
            if i < len(tokens):
                word = tokens[i]
                if word in {'.', '!', '?'}:
                    break
                extended_words.append(word)
        return extended_words

    return opinion_words


def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """
    Extract opinion associated with aspect, respecting boundaries.
    Enhanced to ensure complete sentences.

    Extraction Patterns (Priority Order):
        1. Pre-modifier: "delicious food" → opinion before aspect
        2. Be-adjective: "food is delicious" → opinion after be-verb
        3. Verb phrase: "food tastes great" → opinion in verb phrase
        4. Context search: Fallback for complex sentences

    Args:
        tokens: Tokenized review
        aspect_positions: Position indices of aspect
        aspect_text: Aspect text (for reference)
        all_aspect_positions: All aspect positions (for boundaries)

    Returns:
        tuple: (opinion_phrase: str, pattern_type: str)
    """
    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1] if len(aspect_positions) > 1 else aspect_positions[0]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)
    prev_aspect_end = find_previous_aspect_end(start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(end_idx, all_aspect_positions)

    # Relaxed boundary - allow extending to sentence end for completeness
    original_sentence_end = sentence_end
    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # PATTERN 1: Pre-modifier
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, _ = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in {',', 'and', 'with', 'or', 'but'}:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

    # PATTERN 2: Be-verb + Adjective
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []
        if is_negative:
            post_modifiers.append("not")

        for i in range(opinion_start, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            post_modifiers.append(word)

        if post_modifiers:
            # Check for completeness and extend if needed
            post_modifiers = check_sentence_completeness(
                post_modifiers, tokens, opinion_end, original_sentence_end
            )
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # PATTERN 3: Verb Phrase
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            temp_words.append(word)

        if temp_words:
            # Check for completeness and extend if needed
            temp_words = check_sentence_completeness(
                temp_words, tokens, opinion_end, original_sentence_end
            )
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # PATTERN 4: Context Search
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, _ = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                for j in range(context_start, context_end):
                    if tokens[j] not in {'.', '!', '?', ';'}:
                        opinion_words.append(tokens[j])

                # Check for completeness
                opinion_words = check_sentence_completeness(
                    opinion_words, tokens, context_end, original_sentence_end
                )
                pattern_type = "context_search"
                break

    opinion_phrase = ' '.join(opinion_words).strip()
    return opinion_phrase, pattern_type


# ============================================
# SECTION 13: OPINION FORMATTING (ENHANCED)
# ============================================

def format_opinion_for_display(opinion):
    """
    Format and clean extracted opinion for display.
    Enhanced to remove leading punctuation and ensure completeness.

    Processing Steps:
        1. Remove leading punctuation (comma, parenthesis, etc.)
        2. Remove leading conjunctions ("and", "or", "but")
        3. Remove leading function words
        4. Handle hyphenation
        5. Truncate at cutoff words
        6. Remove trailing garbage
        7. Enforce length limit (extended to 20 words)

    Args:
        opinion (str): Raw extracted opinion

    Returns:
        str: Cleaned and formatted opinion
    """
    if not opinion:
        return ""

    # ENHANCED: Remove leading punctuation (comma, parenthesis, colon, etc.)
    opinion = re.sub(r'^[\s,;:\)\(\-\'"\.]+', '', opinion)

    # Remove leading conjunctions
    opinion = re.sub(r'^(and|or|but|so)\s+', '', opinion, flags=re.IGNORECASE)

    # Fix punctuation spacing
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    # Fix tokenization issues with contractions
    opinion = re.sub(r"\s+'\s*t\b", "'t", opinion)  # wasn ' t -> wasn't
    opinion = re.sub(r"\s+'\s*s\b", "'s", opinion)  # he ' s -> he's
    opinion = re.sub(r"\s+'\s*re\b", "'re", opinion)  # they ' re -> they're
    opinion = re.sub(r"\s+'\s*ve\b", "'ve", opinion)  # we ' ve -> we've
    opinion = re.sub(r"\s+'\s*ll\b", "'ll", opinion)  # we ' ll -> we'll
    opinion = re.sub(r"\s+'\s*d\b", "'d", opinion)  # he ' d -> he'd
    opinion = re.sub(r"\s+'\s*m\b", "'m", opinion)  # I ' m -> I'm

    words = opinion.split()
    if not words:
        return ""

    # Remove leading fillers
    leading_fillers = {
        'a', 'an', 'the', 'this', 'that', 'also', 'too',
        'has', 'have', 'had', 'is', 'are', 'was', 'were',
        'it', 'itself', 'they', 'themselves', 'we', 'i',
        'and', 'or', 'but', 'so'
    }

    while words and words[0].lower() in leading_fillers:
        words.pop(0)

    # ENHANCED: Also remove leading punctuation words
    while words and words[0] in {',', ';', ':', ')', '(', '-', '.', "'", '"'}:
        words.pop(0)

    if not words:
        return ""

    # Truncate at cutoff words (but allow more context)
    cutoff_words = {
        'making', 'causing', 'forcing', 'leaving',
        'unless', 'except', 'despite', 'although',
    }

    truncated = []
    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')
        if word_lower in cutoff_words and i >= 4:  # Allow more words before cutoff
            break
        truncated.append(word)

    words = truncated

    # Remove trailing garbage
    trailing_garbage = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'a', 'an', 'the', ',', '-',
        'is', 'are', 'was', 'were',
        'it', 'i', 'we', 'they'
    }

    while words:
        last_word = words[-1].lower().strip('.,!?;:')
        if last_word in trailing_garbage or words[-1].endswith('-'):
            words.pop()
        else:
            break

    result = ' '.join(words)

    # ENHANCED: Extended length limit to 20 words for completeness
    words = result.split()
    if len(words) > 20:
        words = words[:20]
        result = ' '.join(words)

    # Final cleanup
    result = result.strip().rstrip('.,;-\'"(')
    result = re.sub(r'^[\s,;:\)\(\-\'"\.]+', '', result)  # Remove leading punctuation again
    result = re.sub(r'^(and|or|but|so)\s+', '', result, flags=re.IGNORECASE)

    return result


# ============================================
# SECTION 14: SENTIMENT ANALYSIS
# ============================================

def analyze_sentiment_scores(text):
    """
    Analyze text sentiment using multiple tools.

    Returns ensemble score combining:
        - VADER (weight: 0.4)
        - TextBlob (weight: 0.3)
        - Transformer (weight: 0.3)

    Args:
        text (str): Text to analyze

    Returns:
        dict: Sentiment scores from all analyzers and ensemble
    """
    analyzers = get_sentiment_analyzers()
    results = {}

    if not text or len(text.strip()) < 2:
        return {
            'vader': None,
            'textblob': None,
            'transformer': None,
            'ensemble': {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}
        }

    # VADER
    if analyzers.get('vader'):
        try:
            vader_scores = analyzers['vader'].polarity_scores(text)
            results['vader'] = {'compound': vader_scores['compound']}
        except:
            results['vader'] = None
    else:
        results['vader'] = None

    # TextBlob
    if analyzers.get('textblob'):
        try:
            blob = analyzers['textblob'](text)
            results['textblob'] = {'polarity': blob.sentiment.polarity}
        except:
            results['textblob'] = None
    else:
        results['textblob'] = None

    # Transformer
    if analyzers.get('transformer'):
        try:
            truncated = text[:500] if len(text) > 500 else text
            trans_result = analyzers['transformer'](truncated)[0]
            normalized = trans_result['score'] if trans_result['label'] == 'POSITIVE' else -trans_result['score']
            results['transformer'] = {'normalized_score': normalized}
        except:
            results['transformer'] = None
    else:
        results['transformer'] = None

    # Calculate ensemble
    results['ensemble'] = calculate_ensemble_score(results)

    return results


def calculate_ensemble_score(sentiment_results):
    """Calculate weighted ensemble sentiment score."""
    scores = []
    weights = []

    if sentiment_results.get('vader') and sentiment_results['vader'].get('compound') is not None:
        scores.append(sentiment_results['vader']['compound'])
        weights.append(0.4)

    if sentiment_results.get('textblob') and sentiment_results['textblob'].get('polarity') is not None:
        scores.append(sentiment_results['textblob']['polarity'])
        weights.append(0.3)

    if sentiment_results.get('transformer') and sentiment_results['transformer'].get('normalized_score') is not None:
        scores.append(sentiment_results['transformer']['normalized_score'])
        weights.append(0.3)

    if not scores:
        return {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}

    total_weight = sum(weights)
    normalized_weights = [w / total_weight for w in weights]
    ensemble_score = sum(s * w for s, w in zip(scores, normalized_weights))

    # Calculate confidence (agreement)
    if len(scores) > 1:
        variance = sum((s - ensemble_score) ** 2 for s in scores) / len(scores)
        confidence = max(0, 1 - variance)
    else:
        confidence = 0.7

    # Determine label
    if ensemble_score >= 0.05:
        label = 'Positive'
    elif ensemble_score <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return {
        'score': round(ensemble_score, 4),
        'label': label,
        'confidence': round(confidence, 4)
    }


# ============================================
# SECTION 15: ENHANCED SENTIMENT CORRECTION
# ============================================

# Strong sentiment indicators for sanity checking
OBVIOUS_NEGATIVE_INDICATORS = {
    'terrible', 'horrible', 'awful', 'worst', 'disgusting', 'nasty',
    'gross', 'pathetic', 'abysmal', 'dreadful', 'atrocious', 'appalling',
    'incompetent', 'useless', 'unacceptable', 'inexcusable', 'nightmare'
}

OBVIOUS_POSITIVE_INDICATORS = {
    'excellent', 'amazing', 'wonderful', 'fantastic', 'outstanding',
    'superb', 'incredible', 'phenomenal', 'magnificent', 'exceptional',
    'perfect', 'best', 'love', 'loved', 'brilliant', 'marvelous'
}


def sanity_check_sentiment(opinion, predicted_sentiment):
    """
    Sanity check to catch obvious sentiment misclassifications.

    Example:
        "TERRIBLE service" should NEVER be classified as Positive
        "EXCELLENT food" should NEVER be classified as Negative

    This is a safety net for when other correction logic fails.

    Args:
        opinion (str): Opinion text
        predicted_sentiment (str): Current sentiment prediction

    Returns:
        tuple: (corrected_sentiment, correction_reason) or (None, None) if no change
    """
    opinion_lower = opinion.lower()
    words = opinion_lower.split()

    # Check for obvious negative indicators
    for neg_word in OBVIOUS_NEGATIVE_INDICATORS:
        if neg_word in words or neg_word in opinion_lower:
            # Check if negated (e.g., "not terrible")
            neg_index = opinion_lower.find(neg_word)
            prefix = opinion_lower[max(0, neg_index-10):neg_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Positive':
                    return 'Negative', f'sanity_check_negative ({neg_word})'

    # Check for obvious positive indicators
    for pos_word in OBVIOUS_POSITIVE_INDICATORS:
        if pos_word in words or pos_word in opinion_lower:
            # Check if negated
            pos_index = opinion_lower.find(pos_word)
            prefix = opinion_lower[max(0, pos_index-10):pos_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Negative':
                    return 'Positive', f'sanity_check_positive ({pos_word})'

    return None, None


def correct_sentiment_with_analysis(aspect, opinion, predicted_sentiment, sentiment_scores=None):
    """
    Correct sentiment prediction using rules and analysis.

    PRIORITY ORDER (Critical for accuracy):
        0. SANITY CHECK (NEW) - Catch obvious misclassifications like "TERRIBLE" → Positive
        1. Idiom detection - "did not disappoint" → Positive
        2. Superlative patterns - "could not have been better" → Positive
        3. Negation + sentiment word
        4. Score override (when highly confident)
        5. Keyword detection

    Args:
        aspect (str): Aspect term
        opinion (str): Opinion text
        predicted_sentiment (str): Model's prediction
        sentiment_scores (dict): Sentiment analysis scores

    Returns:
        tuple: (corrected_sentiment, correction_reason)
    """
    opinion_lower = opinion.lower()

    # ========== PRIORITY 0: SANITY CHECK (NEW - HIGHEST) ==========
    # Catch obvious cases like "TERRIBLE" classified as Positive
    sanity_result, sanity_reason = sanity_check_sentiment(opinion, predicted_sentiment)
    if sanity_result:
        return sanity_result, sanity_reason

    # ========== PRIORITY 1: IDIOM DETECTION ==========
    is_idiom, idiom_sentiment, idiom_matched = check_for_idiom(opinion_lower)
    if is_idiom:
        if predicted_sentiment != idiom_sentiment:
            return idiom_sentiment, f'idiom_override ({idiom_matched})'
        return predicted_sentiment, None

    # ========== PRIORITY 2: SUPERLATIVE PATTERNS ==========
    superlative_patterns = [
        r"could(n't| not) have been (more )?(friendly|helpful|better|nicer|attentive|professional)",
        r"could(n't| not) ask for (more|better)",
        r"could(n't| not) be (happier|better)",
    ]
    for pattern in superlative_patterns:
        if re.search(pattern, opinion_lower):
            if predicted_sentiment != 'Positive':
                return 'Positive', 'superlative_positive'
            return predicted_sentiment, None

    # Get sentiment word lists
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()

    # ========== PRIORITY 3: NEGATION LOGIC ==========
    has_negation, neg_type = detect_negation_in_opinion(opinion_lower)

    if has_negation and neg_type != 'idiom_detected':
        # Negating positive word → Negative
        for pos_word in positive_words:
            if pos_word in opinion_lower:
                if "not only" not in opinion_lower:  # Exception
                    return 'Negative', f'negation_positive ({pos_word})'

        # Negating negative word → Positive (double negative)
        for neg_word in negative_words:
            if neg_word in opinion_lower:
                return 'Positive', f'double_negation ({neg_word})'

        # General negation with no clear word
        if sentiment_scores:
            score = sentiment_scores.get('ensemble', {}).get('score', 0)
            if score > 0.3:
                return predicted_sentiment, None
        return 'Negative', f'negation_general ({neg_type})'

    # ========== PRIORITY 4: SCORE OVERRIDE ==========
    if sentiment_scores:
        ensemble = sentiment_scores.get('ensemble', {})
        ensemble_score = ensemble.get('score', 0)
        confidence = ensemble.get('confidence', 0)

        if confidence >= SENTIMENT_CONFIDENCE_THRESHOLD:
            if predicted_sentiment == 'Positive' and ensemble_score < -SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(pw in opinion_lower for pw in positive_words):
                    return 'Negative', f'sentiment_override ({ensemble_score:.2f})'

            if predicted_sentiment == 'Negative' and ensemble_score > SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(nw in opinion_lower for nw in negative_words):
                    return 'Positive', f'sentiment_override ({ensemble_score:.2f})'

    # ========== PRIORITY 5: KEYWORD DETECTION ==========
    for pos_word in positive_words:
        if pos_word in opinion_lower and predicted_sentiment != 'Positive':
            return 'Positive', f'positive_keyword ({pos_word})'

    for neg_word in negative_words:
        if neg_word in opinion_lower and predicted_sentiment != 'Negative':
            return 'Negative', f'negative_keyword ({neg_word})'

    return predicted_sentiment, None


# ============================================
# SECTION 16: CONFIDENCE-BASED FILTERING
# ============================================

def should_include_extraction(sentiment_scores, opinion, predicted_sentiment):
    """
    Determine if extraction should be included based on confidence.

    Exclusion Rules:
        1. |score| < MINIMUM_SENTIMENT_SCORE_THRESHOLD (unclear sentiment)
        2. confidence < MINIMUM_AGREEMENT_THRESHOLD (analyzers disagree)

    Exceptions:
        - Known idioms are always included
        - Clear sentiment words override low scores

    Args:
        sentiment_scores (dict): Sentiment analysis results
        opinion (str): Opinion text
        predicted_sentiment (str): Model prediction

    Returns:
        tuple: (include: bool, reason: str or None)
    """
    if not sentiment_scores:
        return True, None

    ensemble = sentiment_scores.get('ensemble', {})
    score = ensemble.get('score', 0)
    confidence = ensemble.get('confidence', 0)

    # Rule 1: Score too close to zero
    if abs(score) < MINIMUM_SENTIMENT_SCORE_THRESHOLD:
        # Exception: Idioms should be included
        is_idiom, _, _ = check_for_idiom(opinion.lower())
        if not is_idiom:
            return False, f'low_score (|{score:.2f}| < {MINIMUM_SENTIMENT_SCORE_THRESHOLD})'

    # Rule 2: Low agreement between analyzers
    if confidence < MINIMUM_AGREEMENT_THRESHOLD:
        # Exception: Clear sentiment words
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        has_clear_word = any(w in opinion.lower() for w in positive_words | negative_words)
        if not has_clear_word:
            return False, f'low_agreement ({confidence:.2f})'

    return True, None


# ============================================
# SECTION 17: RESULT STRUCTURES
# ============================================

def create_aspect_opinion_pair(aspect, opinion, raw_opinion, sentiment,
                                original_sentiment, pattern, chunk_id,
                                sentiment_scores, correction_reason):
    """Create standardized result dictionary."""

    scores_dict = {
        'vader_compound': None,
        'textblob_polarity': None,
        'transformer_score': None,
        'ensemble_score': None,
        'ensemble_confidence': None
    }

    if sentiment_scores:
        if sentiment_scores.get('vader'):
            scores_dict['vader_compound'] = sentiment_scores['vader'].get('compound')
        if sentiment_scores.get('textblob'):
            scores_dict['textblob_polarity'] = sentiment_scores['textblob'].get('polarity')
        if sentiment_scores.get('transformer'):
            scores_dict['transformer_score'] = sentiment_scores['transformer'].get('normalized_score')
        if sentiment_scores.get('ensemble'):
            scores_dict['ensemble_score'] = sentiment_scores['ensemble'].get('score')
            scores_dict['ensemble_confidence'] = sentiment_scores['ensemble'].get('confidence')

    return {
        'aspect': aspect,
        'opinion': opinion,
        'raw_opinion': raw_opinion,
        'formatted': f"{aspect}: {opinion}",
        'pattern': pattern,
        'chunk': chunk_id,
        'original_sentiment': original_sentiment,
        'sentiment': sentiment,
        'correction_reason': correction_reason,
        'sentiment_scores': scores_dict
    }


# ============================================
# SECTION 18: FILE LOGGING
# ============================================

class FileLogger:
    """Logger for both console and file output."""

    def __init__(self, filename):
        self.filename = filename
        self.content = []

    def log(self, message=""):
        self.content.append(message)
        print(message)

    def save(self):
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ Results saved to: {self.filename}")


# ============================================
# SECTION 19: REVIEW CHUNKING
# ============================================

def split_long_review(text, max_words=MAX_WORDS_PER_CHUNK):
    """Split long reviews into sentence-preserving chunks."""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub = ' '.join(words[i:i + max_words])
                chunks.append(sub + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# SECTION 20: MAIN ANALYSIS FUNCTION
# ============================================

def comprehensive_aspect_opinion_analysis(reviews, aspect_extractor,
                                          max_words=MAX_WORDS_PER_CHUNK,
                                          verbose=True, logger=None,
                                          use_sentiment_analysis=True,
                                          apply_confidence_filter=True):
    """
    Run complete ABSA analysis on restaurant reviews.

    Args:
        reviews: Iterable of review texts
        aspect_extractor: PyABSA aspect extractor
        max_words: Max words per chunk
        verbose: Enable detailed logging
        logger: FileLogger instance
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence-based filtering

    Returns:
        tuple: (results_list, statistics_dict)
    """
    if logger:
        logger.log("=" * 100)
        logger.log("ABSA Analysis System - V5 Enhanced Universal Edition")
        logger.log(f"Chunk size: {max_words} words")
        logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
        logger.log(f"Confidence filter: {'Enabled' if apply_confidence_filter else 'Disabled'}")
        logger.log(f"Min score threshold: {MINIMUM_SENTIMENT_SCORE_THRESHOLD}")
        logger.log(f"Analysis time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 100)

    if use_sentiment_analysis:
        _ = get_sentiment_analyzers()

    all_results = []
    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'filtered_low_confidence': 0,
        'sentiment_corrections': 0,
        'correction_reasons': Counter(),
        'idiom_detections': 0,
        'sanity_check_corrections': 0,  # NEW: Track sanity check fixes
    }

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'=' * 100}")
            logger.log(f"Review #{idx} ({word_count} words)")
            logger.log("=" * 100)

        # Chunking
        if word_count <= max_words:
            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]
            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if logger and verbose:
                logger.log("📄 Splitting long review into chunks...")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   Split into {len(chunks)} chunks")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = [
                {'chunk_id': i + 1, 'result': r}
                for i, r in enumerate(chunk_results)
            ]

        # Process extractions
        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0
        filtered_confidence_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, original_sentiment in zip(aspects, positions, sentiments):

                # Validate aspect
                aspect = refine_aspect_term(aspect)
                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Extract opinion
                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Format and validate opinion
                display_opinion = format_opinion_for_display(raw_opinion)

                if not display_opinion or not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                # Sentiment analysis
                sentiment_scores = None
                if use_sentiment_analysis:
                    sentiment_scores = analyze_sentiment_scores(display_opinion)

                # Confidence filtering
                if apply_confidence_filter and sentiment_scores:
                    should_include, _ = should_include_extraction(
                        sentiment_scores, display_opinion, original_sentiment
                    )
                    if not should_include:
                        filtered_confidence_count += 1
                        stats['filtered_low_confidence'] += 1
                        continue

                # Sentiment correction
                corrected_sentiment, correction_reason = correct_sentiment_with_analysis(
                    aspect, raw_opinion, original_sentiment, sentiment_scores
                )

                if corrected_sentiment != original_sentiment:
                    stats['sentiment_corrections'] += 1
                    if correction_reason:
                        stats['correction_reasons'][correction_reason] += 1
                        if 'idiom' in correction_reason:
                            stats['idiom_detections'] += 1
                        if 'sanity_check' in correction_reason:
                            stats['sanity_check_corrections'] += 1

                stats['total_aspects'] += 1

                # Create result
                pair = create_aspect_opinion_pair(
                    aspect=aspect,
                    opinion=display_opinion,
                    raw_opinion=raw_opinion,
                    sentiment=corrected_sentiment,
                    original_sentiment=original_sentiment,
                    pattern=pattern,
                    chunk_id=chunk_data['chunk_id'] if len(analysis_results) > 1 else None,
                    sentiment_scores=sentiment_scores,
                    correction_reason=correction_reason
                )
                aspect_opinion_pairs.append(pair)

        # ENHANCED: Log results with wider columns and full content
        if logger and verbose:
            logger.log(f"\n🎯 Extraction Results:")
            # ENHANCED: Wider opinion column (60 chars instead of 35)
            header = f"{'#':<3} {'Aspect':<20} {'Opinion':<60} {'Orig':<10} {'Final':<10} {'Score':<8}"
            logger.log(header)
            logger.log("-" * 115)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    opinion_text = pair['opinion']

                    # ENHANCED: Show full opinion, use multiple lines if needed
                    if len(opinion_text) > 60:
                        # First line with all columns
                        opinion_line1 = opinion_text[:60]
                        scores = pair['sentiment_scores']
                        score_str = f"{scores['ensemble_score']:+.2f}" if scores.get('ensemble_score') else "N/A"

                        final = pair['sentiment']
                        if pair['correction_reason']:
                            final += "*"

                        logger.log(f"{i:<3} {pair['aspect']:<20} {opinion_line1:<60} "
                                  f"{pair['original_sentiment']:<10} {final:<10} {score_str:<8}")

                        # Additional lines for remaining opinion text
                        remaining = opinion_text[60:]
                        while remaining:
                            chunk = remaining[:60]
                            remaining = remaining[60:]
                            logger.log(f"{'':3} {'':20} {chunk:<60}")
                    else:
                        scores = pair['sentiment_scores']
                        score_str = f"{scores['ensemble_score']:+.2f}" if scores.get('ensemble_score') else "N/A"

                        final = pair['sentiment']
                        if pair['correction_reason']:
                            final += "*"

                        logger.log(f"{i:<3} {pair['aspect']:<20} {opinion_text:<60} "
                                  f"{pair['original_sentiment']:<10} {final:<10} {score_str:<8}")

                total_filtered = filtered_aspect_count + filtered_opinion_count + filtered_confidence_count
                if total_filtered > 0:
                    logger.log(f"\n   ℹ️ Filtered: {filtered_aspect_count} aspects, "
                              f"{filtered_opinion_count} opinions, "
                              f"{filtered_confidence_count} low confidence")
            else:
                logger.log("   ⚠️ No valid aspect-opinion pairs found")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
        })

    return all_results, stats


# ============================================
# SECTION 21: ENHANCED REPORT GENERATION
# ============================================

def select_representative_opinions(pairs, n=5, target_sentiment=None):
    """
    Select high-quality representative opinions.
    ENHANCED: Now returns 5 opinions by default (increased from 3).

    Selection Criteria:
        1. Sentiment matches target (CRITICAL: positive section shows only positive opinions)
        2. High absolute sentiment score (clear sentiment)
        3. Sufficient length (not fragments)
        4. Passes completeness validation
        5. Diverse (not similar to already selected)
        6. No negative words in positive section (and vice versa)

    Args:
        pairs: List of aspect-opinion pairs
        n: Number of opinions to select (default: 5)
        target_sentiment: 'Positive' or 'Negative' - MUST match this sentiment

    Returns:
        list: Selected representative opinions
    """
    # CRITICAL FIX: Filter by target sentiment FIRST
    if target_sentiment:
        pairs = [p for p in pairs if p.get('sentiment') == target_sentiment]

    # Sort by score magnitude (clearest sentiment first)
    sorted_pairs = sorted(
        pairs,
        key=lambda p: abs(p['sentiment_scores'].get('ensemble_score', 0) if p['sentiment_scores'] else 0),
        reverse=True
    )

    # Define words that indicate opposite sentiment
    strong_negative_words = {
        'terrible', 'horrible', 'awful', 'worst', 'bad', 'poor', 'disgusting',
        'rude', 'slow', 'cold', 'bland', 'disappointing', 'mediocre', 'nasty',
        'gross', 'dirty', 'stale', 'overpriced', 'incompetent', 'never'
    }
    strong_positive_words = {
        'great', 'excellent', 'amazing', 'wonderful', 'fantastic', 'perfect',
        'delicious', 'friendly', 'awesome', 'best', 'love', 'outstanding',
        'superb', 'incredible', 'fresh', 'tasty', 'beautiful', 'attentive'
    }

    selected = []
    seen_stems = set()

    for pair in sorted_pairs:
        opinion = pair['opinion']
        opinion_lower = opinion.lower()

        # CRITICAL: Skip if opinion contains strong opposite-sentiment words
        if target_sentiment == 'Positive':
            # For positive section, skip opinions with strong negative words
            if any(neg_word in opinion_lower for neg_word in strong_negative_words):
                continue
        elif target_sentiment == 'Negative':
            # For negative section, verify it actually sounds negative
            # (optional: could skip if too many positive words)
            pass

        # Quality checks
        if len(opinion.split()) < 2:
            continue

        if opinion_lower.startswith(('and ', 'or ', 'but ')):
            continue

        is_valid, _ = validate_opinion_completeness(opinion)
        if not is_valid:
            continue

        # Diversity check (avoid similar opinions)
        stem = ' '.join(opinion_lower.split()[:3])
        if stem in seen_stems:
            continue

        seen_stems.add(stem)
        selected.append(pair)

        if len(selected) >= n:
            break

    return selected


def generate_management_report(results, stats, logger=None):
    """
    Generate enhanced management report with multiple representative opinions.
    ENHANCED: Shows 5 examples per aspect (increased from 3) with full content.

    Report Sections:
        1. Overall Statistics
        2. Correction Breakdown
        3. Competitive Advantages (with 5 example opinions each)
        4. Areas for Improvement (with 5 example complaints each)
        5. Recommendations
        6. Quality Metrics
    """
    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 100)
    log("MANAGEMENT ANALYSIS REPORT")
    log("Universal Restaurant Review Analysis")
    log("=" * 100)

    # Aggregate by sentiment
    positive_pairs = []
    negative_pairs = []

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

    total_pairs = len(positive_pairs) + len(negative_pairs)

    # ===== STATISTICS =====
    log(f"\n📊 OVERALL STATISTICS")
    log(f"   • Total reviews analyzed: {len(results)}")
    log(f"   • Total aspects extracted: {total_pairs}")
    log(f"   • Invalid aspects filtered: {stats['filtered_invalid_aspects']}")
    log(f"   • Invalid opinions filtered: {stats['filtered_invalid_opinions']}")
    log(f"   • Low confidence filtered: {stats.get('filtered_low_confidence', 0)}")

    if stats['sentiment_corrections'] > 0:
        log(f"   • Sentiments corrected: {stats['sentiment_corrections']}")
        log(f"   • Idiom detections: {stats.get('idiom_detections', 0)}")
        log(f"   • Sanity check fixes: {stats.get('sanity_check_corrections', 0)}")

    if total_pairs > 0:
        pos_pct = len(positive_pairs) / total_pairs * 100
        neg_pct = len(negative_pairs) / total_pairs * 100
        log(f"   • Positive mentions: {len(positive_pairs)} ({pos_pct:.1f}%)")
        log(f"   • Negative mentions: {len(negative_pairs)} ({neg_pct:.1f}%)")

    # ===== CORRECTION BREAKDOWN =====
    if stats['correction_reasons']:
        log(f"\n📈 CORRECTION BREAKDOWN")
        for reason, count in stats['correction_reasons'].most_common(10):
            log(f"   • {reason}: {count}")

    # Aggregate by aspect
    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair)

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair)

    # ===== COMPETITIVE ADVANTAGES =====
    if positive_aspects:
        log(f"\n✅ COMPETITIVE ADVANTAGES (Top Positive Aspects)")
        log("-" * 100)

        sorted_pos = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_pos[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # ENHANCED: 5 representative opinions (increased from 3) - MUST be positive sentiment
            examples = select_representative_opinions(pairs, n=5, target_sentiment='Positive')
            for ex in examples:
                # ENHANCED: Show full opinion text without truncation
                opinion_text = ex['opinion']
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0

                # Use multiple lines if opinion is long
                if len(opinion_text) > 70:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text[:70]}"')
                    remaining = opinion_text[70:]
                    while remaining:
                        chunk = remaining[:70]
                        remaining = remaining[70:]
                        log(f'              "{chunk}"')
                else:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid positive examples found, note it
            if not examples:
                log(f'   • [No clear positive examples available]')

    # ===== AREAS FOR IMPROVEMENT =====
    if negative_aspects:
        log(f"\n⚠️ AREAS FOR IMPROVEMENT (Top Negative Aspects)")
        log("-" * 100)

        sorted_neg = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_neg[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # ENHANCED: 5 representative opinions (increased from 3) - MUST be negative sentiment
            examples = select_representative_opinions(pairs, n=5, target_sentiment='Negative')
            for ex in examples:
                # ENHANCED: Show full opinion text without truncation
                opinion_text = ex['opinion']
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0

                # Use multiple lines if opinion is long
                if len(opinion_text) > 70:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text[:70]}"')
                    remaining = opinion_text[70:]
                    while remaining:
                        chunk = remaining[:70]
                        remaining = remaining[70:]
                        log(f'              "{chunk}"')
                else:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid negative examples found, note it
            if not examples:
                log(f'   • [No clear negative examples available]')

    # ===== RECOMMENDATIONS =====
    log(f"\n💡 RECOMMENDATIONS")

    if negative_aspects:
        worst = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   1. PRIORITY: Address '{worst[0]}' ({len(worst[1])} negative mentions)")

    if positive_aspects:
        best = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   2. LEVERAGE: Promote '{best[0]}' ({len(best[1])} positive mentions)")

    log(f"   3. MONITOR: Track sentiment trends over time")
    log(f"   4. INVESTIGATE: Review corrected sentiments for accuracy")

    # ===== QUALITY METRICS =====
    log(f"\n📉 EXTRACTION QUALITY METRICS")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   • Extraction success rate: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   • Average aspects per review: {avg_pairs:.2f}")

    total_filtered = (stats['filtered_invalid_aspects'] +
                      stats['filtered_invalid_opinions'] +
                      stats.get('filtered_low_confidence', 0))
    total_extracted = total_pairs + total_filtered
    if total_extracted > 0:
        quality_rate = total_pairs / total_extracted * 100
        log(f"   • Quality pass rate: {total_pairs}/{total_extracted} ({quality_rate:.1f}%)")


# ============================================
# SECTION 22: DATA EXPORT
# ============================================

def export_results_to_dataframe(results):
    """Export results to pandas DataFrame."""
    import pandas as pd

    rows = []
    for result in results:
        review_id = result['review_id']
        for pair in result['pairs']:
            scores = pair.get('sentiment_scores', {})
            rows.append({
                'review_id': review_id,
                'aspect': pair['aspect'],
                'opinion': pair['opinion'],
                'original_sentiment': pair['original_sentiment'],
                'corrected_sentiment': pair['sentiment'],
                'correction_reason': pair.get('correction_reason'),
                'ensemble_score': scores.get('ensemble_score'),
                'ensemble_confidence': scores.get('ensemble_confidence')
            })

    return pd.DataFrame(rows)


# ============================================
# SECTION 23: MAIN EXECUTION
# ============================================

def run_analysis(reviews, aspect_extractor,
                 output_file="absa_results_v5_universal.txt",
                 use_sentiment_analysis=True,
                 apply_confidence_filter=True):
    """
    Run complete ABSA analysis pipeline.

    Args:
        reviews: Review texts (list or Series)
        aspect_extractor: PyABSA extractor
        output_file: Output log file path
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence filtering

    Returns:
        tuple: (results, stats, dataframe)
    """
    logger = FileLogger(output_file)

    logger.log("=" * 100)
    logger.log("ABSA ANALYSIS SYSTEM - V5 ENHANCED UNIVERSAL EDITION")
    logger.log(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"Reviews to analyze: {len(reviews)}")
    logger.log("=" * 100)

    results, stats = comprehensive_aspect_opinion_analysis(
        reviews,
        aspect_extractor,
        max_words=MAX_WORDS_PER_CHUNK,
        verbose=True,
        logger=logger,
        use_sentiment_analysis=use_sentiment_analysis,
        apply_confidence_filter=apply_confidence_filter
    )

    logger.log("\n" + "=" * 100)
    logger.log("✅ ANALYSIS COMPLETE")
    logger.log("=" * 100)

    generate_management_report(results, stats, logger)
    logger.save()

    df_results = export_results_to_dataframe(results)

    return results, stats, df_results


if __name__ == "__main__":
    print("=" * 70)
    print("ABSA Analysis System - V5 Enhanced Universal Edition")
    print("=" * 70)
    print("\nDesigned for UNIVERSAL restaurant review analysis:")
    print("  • All restaurant types (fast food to fine dining)")
    print("  • All cuisines (American, Asian, Mexican, etc.)")
    print("  • All service models (dine-in, takeout, delivery)")
    print("\nKey Enhancements:")
    print("  1. Idiom/Slang Dictionary (~80 expressions)")
    print("  2. Confidence-Based Filtering")
    print("  3. Improved Opinion Boundary Detection")
    print("  4. Enhanced Management Report (5 examples per aspect)")
    print("  5. Complete sentence extraction")
    print("  6. Removed leading punctuation (comma, parenthesis)")
    print("\nUsage:")
    print("  results, stats, df = run_analysis(reviews, aspect_extractor)")

In [ ]:
results, stats, df = run_analysis(test_reviews, aspect_extractor)

In [ ]:
"""
================================================================================
ABSA (Aspect-Based Sentiment Analysis) System - V5 Enhanced Universal Edition
================================================================================

Purpose:
    Extract aspect-opinion-sentiment triplets from restaurant reviews.
    Designed for UNIVERSAL application across ALL restaurant types:
    - Fast food, casual dining, fine dining
    - All cuisines (American, Asian, Mexican, Italian, etc.)
    - All service models (dine-in, takeout, delivery, food trucks)

Key Enhancements in V5:
    1. Idiom/Slang Dictionary - Handles "did not disappoint", "killed it", etc.
    2. Confidence-Based Filtering - Excludes low-confidence extractions
    3. Improved Opinion Boundary Detection - Avoids fragments and factual statements
    4. Enhanced Management Report - Multiple representative opinions per aspect
    5. Universal Restaurant Vocabulary - Multi-cuisine support

Version: 5.0 Enhanced Universal
Author: HAOS Framework Research Team
Date: 2025
================================================================================
"""

import warnings
warnings.filterwarnings('ignore')

import re
from collections import Counter
from datetime import datetime

# ============================================
# SECTION 1: GLOBAL CONFIGURATION
# ============================================

OUTPUT_FILE = "absa_analysis_results_v5_universal.txt"
MAX_WORDS_PER_CHUNK = 80

# Sentiment correction thresholds
SENTIMENT_CONFIDENCE_THRESHOLD = 0.6
SENTIMENT_OVERRIDE_THRESHOLD = 0.4

# NEW: Confidence-based filtering thresholds
# Extractions with |score| < this threshold will be excluded
MINIMUM_SENTIMENT_SCORE_THRESHOLD = 0.25

# Minimum agreement level between sentiment analyzers
MINIMUM_AGREEMENT_THRESHOLD = 0.40


# ============================================
# SECTION 2: IDIOM/SLANG DICTIONARIES
# ============================================
"""
CRITICAL SECTION: Idiom Detection

Problem Solved:
    Many English expressions use negation words but convey POSITIVE meaning.
    Example: "The service did not disappoint" = POSITIVE (not negative!)

These idioms are universal across ALL restaurant types:
    - Fast food: "The fries hit the spot"
    - Fine dining: "The tasting menu was to die for"
    - BBQ: "Ribs fall off the bone"
    - Any cuisine: "Our server killed it tonight"
"""


def get_positive_idioms():
    """
    Dictionary of positive idioms/slang expressions.

    Categories:
    1. Double Negative = Positive ("did not disappoint")
    2. Superlative Negation = Maximum Positive ("could not have been better")
    3. Slang = Positive ("killed it", "fire", "bussin")
    4. Food-Specific = Positive ("melt in your mouth")

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DOUBLE NEGATIVE = POSITIVE ==========
        # Pattern: "did not [negative_word]" → Positive
        # Logic: NOT disappointing = satisfying
        "did not disappoint": "Positive",
        "didn't disappoint": "Positive",
        "does not disappoint": "Positive",
        "doesn't disappoint": "Positive",
        "never disappoints": "Positive",
        "never disappointed": "Positive",
        "not disappointed": "Positive",
        "wasn't disappointed": "Positive",
        "weren't disappointed": "Positive",
        "not let down": "Positive",
        "didn't let us down": "Positive",

        # ========== SUPERLATIVE NEGATION = MAXIMUM POSITIVE ==========
        # Pattern: "could not have been more [positive_adj]"
        # Logic: Impossible to be MORE positive = maximum level achieved
        "could not have been better": "Positive",
        "couldn't have been better": "Positive",
        "could not have been nicer": "Positive",
        "couldn't have been nicer": "Positive",
        "could not have been friendlier": "Positive",
        "couldn't have been friendlier": "Positive",
        "could not have been more helpful": "Positive",
        "couldn't have been more helpful": "Positive",
        "could not have been more attentive": "Positive",
        "couldn't have been more attentive": "Positive",
        "could not have been more professional": "Positive",
        "couldn't have been more professional": "Positive",
        "could not ask for more": "Positive",
        "couldn't ask for more": "Positive",
        "could not ask for better": "Positive",
        "couldn't ask for better": "Positive",
        "could not be happier": "Positive",
        "couldn't be happier": "Positive",
        "could not be more satisfied": "Positive",
        "couldn't be more satisfied": "Positive",

        # ========== NO COMPLAINTS = POSITIVE ==========
        "can't complain": "Positive",
        "cannot complain": "Positive",
        "nothing to complain about": "Positive",
        "no complaints": "Positive",
        "zero complaints": "Positive",
        "left nothing to be desired": "Positive",
        "leaves nothing to be desired": "Positive",

        # ========== SLANG/INFORMAL = POSITIVE ==========
        # Common in casual reviews, especially younger demographics
        "killed it": "Positive",           # Performed excellently
        "nailed it": "Positive",           # Got it exactly right
        "crushed it": "Positive",          # Exceeded expectations
        "smashed it": "Positive",          # Did amazingly well
        "hit the spot": "Positive",        # Perfectly satisfying
        "hits the spot": "Positive",
        "hit different": "Positive",       # Especially good
        "hits different": "Positive",
        "on point": "Positive",            # Exactly right
        "on fire": "Positive",             # Performing excellently
        "off the hook": "Positive",        # Extremely good
        "off the chain": "Positive",       # Amazing
        "off the charts": "Positive",      # Exceptionally high quality
        "out of this world": "Positive",   # Extraordinary
        "to die for": "Positive",          # Extremely desirable
        "die for": "Positive",
        "the bomb": "Positive",            # Excellent
        "da bomb": "Positive",
        "was bomb": "Positive",            # "The pizza was bomb"
        "is bomb": "Positive",
        "is fire": "Positive",             # "These wings are fire"
        "was fire": "Positive",
        "straight fire": "Positive",
        "chef's kiss": "Positive",         # Perfect
        "chefs kiss": "Positive",
        "slaps": "Positive",               # "This mac and cheese slaps"
        "bussin": "Positive",              # "Food was bussin"
        "bussin'": "Positive",
        "top notch": "Positive",
        "top-notch": "Positive",
        "first rate": "Positive",
        "first-rate": "Positive",
        "A1": "Positive",                  # Top quality
        "a-1": "Positive",
        "10/10": "Positive",
        "10 out of 10": "Positive",
        "five stars": "Positive",
        "5 stars": "Positive",

        # ========== FOOD-SPECIFIC POSITIVE IDIOMS ==========
        # Universal across cuisines
        "melt in your mouth": "Positive",
        "melts in your mouth": "Positive",
        "melted in my mouth": "Positive",
        "fall off the bone": "Positive",   # Tender meat
        "falls off the bone": "Positive",
        "fell off the bone": "Positive",
        "cooked to perfection": "Positive",
        "done to perfection": "Positive",
        "seasoned to perfection": "Positive",
        "finger licking good": "Positive",
        "finger-licking good": "Positive",
        "fresh out of the oven": "Positive",
        "fresh off the grill": "Positive",
        "made with love": "Positive",
        "like grandma used to make": "Positive",
        "like mama used to make": "Positive",
        "home cooked": "Positive",
        "homemade taste": "Positive",

        # ========== RECOMMENDATION POSITIVE ==========
        "a must try": "Positive",
        "must try": "Positive",
        "a must": "Positive",
        "a must visit": "Positive",
        "must visit": "Positive",
        "a gem": "Positive",
        "hidden gem": "Positive",
        "best kept secret": "Positive",
        "saved the day": "Positive",
        "made my day": "Positive",
        "worth the wait": "Positive",
        "worth the drive": "Positive",
        "worth every penny": "Positive",
        "worth the price": "Positive",
        "bang for your buck": "Positive",
        "will be back": "Positive",
        "coming back": "Positive",
        "definitely returning": "Positive",
        "can't wait to come back": "Positive",
    }


def get_negative_idioms():
    """
    Dictionary of negative idioms/slang expressions.

    Categories:
    1. Disappointment expressions
    2. Mediocrity expressions
    3. Poor value expressions
    4. Food-specific negative idioms

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DISAPPOINTMENT ==========
        "left a lot to be desired": "Negative",
        "leaves a lot to be desired": "Negative",
        "left much to be desired": "Negative",
        "leaves much to be desired": "Negative",
        "nothing to write home about": "Negative",
        "not my cup of tea": "Negative",
        "wasn't my cup of tea": "Negative",
        "seen better days": "Negative",
        "has seen better days": "Negative",
        "let down": "Negative",
        "let us down": "Negative",
        "fell short": "Negative",
        "falls short": "Negative",
        "missed the mark": "Negative",
        "misses the mark": "Negative",

        # ========== MEDIOCRITY ==========
        "hit or miss": "Negative",
        "meh": "Negative",
        "just ok": "Negative",
        "just okay": "Negative",
        "nothing special": "Negative",
        "middle of the road": "Negative",
        "run of the mill": "Negative",
        "average at best": "Negative",
        "could be better": "Negative",
        "room for improvement": "Negative",
        "needs work": "Negative",

        # ========== POOR VALUE ==========
        "waste of money": "Negative",
        "waste of time": "Negative",
        "rip off": "Negative",
        "rip-off": "Negative",
        "ripoff": "Negative",
        "not worth it": "Negative",
        "not worth the price": "Negative",
        "not worth the money": "Negative",
        "not worth the wait": "Negative",
        "not worth the hype": "Negative",
        "overpriced": "Negative",
        "over priced": "Negative",
        "overhyped": "Negative",
        "over-hyped": "Negative",
        "overrated": "Negative",
        "over-rated": "Negative",

        # ========== FOOD-SPECIFIC NEGATIVE ==========
        "tasted like cardboard": "Negative",
        "tastes like cardboard": "Negative",
        "like eating cardboard": "Negative",
        "rubber chicken": "Negative",
        "hockey puck": "Negative",         # Overcooked burger
        "sat under a heat lamp": "Negative",
        "sitting under a heat lamp": "Negative",
        "straight from the freezer": "Negative",
        "microwaved": "Negative",
        "reheated": "Negative",
        "day old": "Negative",
        "stale": "Negative",

        # ========== SERVICE NEGATIVE ==========
        "worst service ever": "Negative",
        "worst experience ever": "Negative",
        "never coming back": "Negative",
        "will not return": "Negative",
        "won't be back": "Negative",
        "avoid this place": "Negative",
        "stay away": "Negative",
        "save your money": "Negative",
    }


# ============================================
# SECTION 3: SENTIMENT ANALYZER INITIALIZATION
# ============================================

SENTIMENT_ANALYZERS = None


def initialize_sentiment_analyzers():
    """
    Initialize multiple sentiment analysis tools for ensemble approach.

    Why Multiple Analyzers:
        - VADER: Fast, rule-based, optimized for social media/reviews
        - TextBlob: Pattern-based, provides subjectivity scores
        - Transformer: Deep learning, most accurate for complex patterns
        - Ensemble of all three provides robust results

    Returns:
        dict: Initialized analyzers (None if unavailable)
    """
    analyzers = {}

    # VADER Initialization
    try:
        import nltk
        nltk.download('vader_lexicon', quiet=True)
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        analyzers['vader'] = SentimentIntensityAnalyzer()
        print("✅ VADER sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ VADER not available: {e}")
        analyzers['vader'] = None

    # TextBlob Initialization
    try:
        from textblob import TextBlob
        analyzers['textblob'] = TextBlob
        print("✅ TextBlob sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ TextBlob not available: {e}")
        analyzers['textblob'] = None

    # Transformer Initialization (Optional)
    try:
        from transformers import pipeline
        analyzers['transformer'] = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=-1
        )
        print("✅ Transformer sentiment analyzer loaded")
    except Exception as e:
        print(f"ℹ️ Transformer not available (optional): {e}")
        analyzers['transformer'] = None

    return analyzers


def get_sentiment_analyzers():
    """Get or initialize sentiment analyzers (singleton pattern)."""
    global SENTIMENT_ANALYZERS
    if SENTIMENT_ANALYZERS is None:
        SENTIMENT_ANALYZERS = initialize_sentiment_analyzers()
    return SENTIMENT_ANALYZERS


# ============================================
# SECTION 4: UNIVERSAL RESTAURANT VOCABULARY
# ============================================
"""
Universal Vocabulary Design:
    These word lists are designed to work across ALL restaurant types.
    They cover common aspects, opinions, and patterns found in reviews
    regardless of cuisine or service model.
"""


def get_common_english_names():
    """
    Common first names to filter from opinions.

    Rationale:
        Reviews often mention server names: "Sarah was great"
        Names alone don't provide sentiment - the opinion does.

    Returns:
        set: Common English first names (lowercase)
    """
    male_names = {
        'james', 'john', 'robert', 'michael', 'william', 'david', 'richard',
        'joseph', 'thomas', 'charles', 'christopher', 'daniel', 'matthew',
        'anthony', 'mark', 'donald', 'steven', 'paul', 'andrew', 'joshua',
        'kenneth', 'kevin', 'brian', 'george', 'timothy', 'ronald', 'edward',
        'jason', 'jeffrey', 'ryan', 'jacob', 'gary', 'nicholas', 'eric',
        'jonathan', 'stephen', 'larry', 'justin', 'scott', 'brandon', 'benjamin',
        'samuel', 'raymond', 'gregory', 'frank', 'alexander', 'patrick', 'jack',
        'joel', 'paul', 'morgan', 'jordan', 'chris', 'matt', 'andy', 'ray',
        'nick', 'greg', 'mike', 'steve', 'tom', 'bob', 'jim', 'dan'
    }

    female_names = {
        'mary', 'patricia', 'jennifer', 'linda', 'barbara', 'elizabeth', 'susan',
        'jessica', 'sarah', 'karen', 'lisa', 'nancy', 'betty', 'margaret', 'sandra',
        'ashley', 'kimberly', 'emily', 'donna', 'michelle', 'dorothy', 'carol',
        'amanda', 'melissa', 'deborah', 'stephanie', 'rebecca', 'sharon', 'laura',
        'sara', 'fatima', 'kaitlin', 'kaitlyn', 'natalie', 'melanie', 'christina',
        'veronica', 'morgan', 'jordan', 'gabby', 'tina', 'cameron'
    }

    return male_names | female_names


def get_positive_sentiment_words():
    """
    Universal positive sentiment words for restaurant reviews.

    Categories:
        - General quality descriptors
        - Food-specific positive terms
        - Service-related positive terms
        - Atmosphere/environment terms
        - Value-related terms

    Returns:
        set: Positive sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic',
        'outstanding', 'superb', 'perfect', 'incredible', 'awesome',
        'terrific', 'fabulous', 'marvelous', 'exceptional', 'best',
        'impressive', 'remarkable', 'magnificent', 'splendid', 'brilliant',
        'stellar', 'phenomenal', 'spectacular', 'extraordinary', 'superb',

        # ===== FOOD-SPECIFIC (Universal across cuisines) =====
        'delicious', 'tasty', 'yummy', 'flavorful', 'savory', 'fresh',
        'tender', 'juicy', 'crispy', 'creamy', 'rich', 'light',
        'authentic', 'homemade', 'seasoned', 'aromatic', 'scrumptious',
        'divine', 'heavenly', 'mouthwatering', 'succulent', 'zesty',
        'perfectly cooked', 'well-seasoned', 'flavorsome',

        # ===== SERVICE-RELATED =====
        'friendly', 'helpful', 'attentive', 'professional', 'courteous',
        'prompt', 'efficient', 'welcoming', 'accommodating', 'polite',
        'responsive', 'knowledgeable', 'patient', 'thorough', 'dedicated',
        'personable', 'warm', 'gracious', 'sweet', 'caring',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'clean', 'comfortable', 'cozy', 'spacious', 'nice', 'lovely',
        'beautiful', 'charming', 'pleasant', 'relaxing', 'quiet',
        'modern', 'elegant', 'stylish', 'inviting', 'cute',
        'trendy', 'vibrant', 'lively', 'romantic', 'intimate',

        # ===== VALUE-RELATED =====
        'reasonable', 'affordable', 'worth', 'value', 'cheap', 'bargain',
        'fair', 'inexpensive', 'economical', 'generous'
    }


def get_negative_sentiment_words():
    """
    Universal negative sentiment words for restaurant reviews.

    Returns:
        set: Negative sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'bad', 'terrible', 'awful', 'horrible', 'poor', 'disappointing',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross', 'worst',
        'inferior', 'subpar', 'unacceptable', 'dreadful', 'appalling',
        'atrocious', 'abysmal', 'horrendous', 'disgusting',

        # ===== FOOD-SPECIFIC =====
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty', 'stale',
        'cold', 'lukewarm', 'frozen', 'rubbery', 'tough', 'chewy',
        'flavorless', 'mushy', 'bitter', 'sour', 'spoiled',
        'unseasoned', 'unappetizing', 'oily', 'oversalted',

        # ===== SERVICE-RELATED =====
        'rude', 'slow', 'unfriendly', 'unhelpful', 'inattentive',
        'unprofessional', 'careless', 'dismissive', 'indifferent',
        'incompetent', 'negligent', 'impolite', 'disrespectful',
        'ignored', 'forgotten', 'invisible',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'dirty', 'messy', 'noisy', 'crowded', 'cramped', 'uncomfortable',
        'dark', 'smelly', 'hot', 'stuffy', 'outdated', 'dingy',
        'filthy', 'cluttered', 'rundown', 'shabby', 'sticky',

        # ===== VALUE-RELATED =====
        'expensive', 'overpriced', 'costly', 'pricey', 'ripoff',
        'unreasonable', 'exorbitant', 'small portions'
    }


def get_action_verbs():
    """
    Action verbs that should NOT be treated as opinions.

    Rationale:
        "I ordered the pasta" - "ordered" is action, not opinion
        "The waiter served us" - "served" is action, not opinion

    Returns:
        set: Action verbs in various forms
    """
    return {
        'order', 'ordered', 'ordering', 'orders',
        'ask', 'asked', 'asking', 'asks',
        'request', 'requested', 'requesting', 'requests',
        'come', 'came', 'coming', 'comes',
        'go', 'went', 'going', 'goes', 'gone',
        'arrive', 'arrived', 'arriving', 'arrives',
        'leave', 'left', 'leaving', 'leaves',
        'walk', 'walked', 'walking', 'walks',
        'enter', 'entered', 'entering', 'enters',
        'visit', 'visited', 'visiting', 'visits',
        'get', 'got', 'getting', 'gets',
        'take', 'took', 'taking', 'takes', 'taken',
        'bring', 'brought', 'bringing', 'brings',
        'receive', 'received', 'receiving', 'receives',
        'make', 'made', 'making', 'makes',
        'do', 'did', 'doing', 'does', 'done',
        'prepare', 'prepared', 'preparing', 'prepares',
        'cook', 'cooked', 'cooking', 'cooks',
        'say', 'said', 'saying', 'says',
        'tell', 'told', 'telling', 'tells',
        'call', 'called', 'calling', 'calls',
        'serve', 'served', 'serving', 'serves',
        'show', 'showed', 'showing', 'shows', 'shown',
        'give', 'gave', 'giving', 'gives', 'given',
        'seat', 'seated', 'seating', 'seats',
        'try', 'tried', 'trying', 'tries',
        'want', 'wanted', 'wanting', 'wants',
        'need', 'needed', 'needing', 'needs',
        'hope', 'hoped', 'hoping', 'hopes',
        'expect', 'expected', 'expecting', 'expects',
        'enjoy', 'enjoyed', 'enjoying', 'enjoys',
        'pay', 'paid', 'paying', 'pays',
        'wait', 'waited', 'waiting', 'waits',
        'sit', 'sat', 'sitting', 'sits',
        'eat', 'ate', 'eating', 'eats', 'eaten',
        'drink', 'drank', 'drinking', 'drinks', 'drunk'
    }


def get_function_words():
    """
    Function words that carry no sentiment meaning.

    Returns:
        set: Articles, prepositions, pronouns, etc.
    """
    return {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'i', 'me', 'my', 'mine', 'myself',
        'you', 'your', 'yours', 'yourself',
        'he', 'him', 'his', 'himself',
        'she', 'her', 'hers', 'herself',
        'it', 'its', 'itself',
        'we', 'us', 'our', 'ours', 'ourselves',
        'they', 'them', 'their', 'theirs', 'themselves',
        'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
        'from', 'about', 'into', 'through', 'during', 'before',
        'after', 'above', 'below', 'between', 'under', 'over',
        'and', 'or', 'but', 'so', 'yet', 'nor',
        'if', 'when', 'where', 'while', 'as', 'because', 'since',
        'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'has', 'have', 'had', 'having',
        'do', 'does', 'did',
        'will', 'would', 'shall', 'should',
        'can', 'could', 'may', 'might', 'must',
        'here', 'there', 'some', 'any', 'no', 'every'
    }


def get_intensifiers():
    """
    Intensifier words that modify sentiment strength.

    Returns:
        set: Intensifier words
    """
    return {
        'very', 'extremely', 'incredibly', 'amazingly', 'exceptionally',
        'remarkably', 'absolutely', 'totally', 'completely', 'utterly',
        'entirely', 'thoroughly', 'perfectly', 'highly', 'deeply',
        'truly', 'really', 'genuinely', 'seriously',
        'quite', 'rather', 'fairly', 'pretty', 'somewhat',
        'reasonably', 'moderately', 'relatively',
        'slightly', 'a bit', 'a little', 'mildly', 'barely',
        'too', 'overly', 'excessively',
        'so', 'such', 'super', 'especially', 'particularly'
    }


# ============================================
# SECTION 5: IDIOM DETECTION FUNCTIONS
# ============================================

def check_for_idiom(text):
    """
    Check if text contains any known idiom.

    PRIORITY: This check runs BEFORE negation detection.

    Args:
        text (str): Text to check

    Returns:
        tuple: (is_idiom: bool, sentiment: str or None, matched_idiom: str or None)

    Example:
        >>> check_for_idiom("The food did not disappoint")
        (True, "Positive", "did not disappoint")

        >>> check_for_idiom("The food was cold")
        (False, None, None)
    """
    text_lower = text.lower()

    # Check positive idioms
    positive_idioms = get_positive_idioms()
    for idiom, sentiment in positive_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    # Check negative idioms
    negative_idioms = get_negative_idioms()
    for idiom, sentiment in negative_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    return False, None, None


# ============================================
# SECTION 6: ADJECTIVE/ADVERB DETECTION
# ============================================

def is_adjective_or_adverb(word):
    """
    Determine if a word is an adjective or adverb.

    Uses two approaches:
        1. Dictionary lookup (known sentiment words)
        2. Morphological rules (suffix patterns)

    Args:
        word (str): Word to check

    Returns:
        tuple: (is_adj_or_adv: bool, word_type: str)
    """
    word_lower = word.lower().strip('.,!?;:\'"')

    if not word_lower:
        return False, 'none'

    # Check sentiment dictionaries
    if word_lower in get_positive_sentiment_words():
        return True, 'positive_adj'

    if word_lower in get_negative_sentiment_words():
        return True, 'negative_adj'

    if word_lower in get_intensifiers():
        return True, 'intensifier'

    # Morphological rules
    adj_suffixes = ['ful', 'less', 'ous', 'ive', 'able', 'ible',
                    'al', 'ic', 'ish', 'ent', 'ant', 'ory', 'ary']

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    # Adverb suffix
    if word_lower.endswith('ly') and len(word_lower) > 4:
        non_adverb_ly = {'family', 'only', 'early', 'likely', 'friendly', 'lonely'}
        if word_lower not in non_adverb_ly:
            return True, 'suffix_adv'

    # Past participles as adjectives
    sentiment_participles = {
        'disappointed', 'satisfied', 'pleased', 'impressed', 'amazed',
        'surprised', 'disgusted', 'frustrated', 'annoyed', 'delighted',
        'thrilled', 'excited', 'bored', 'tired', 'exhausted',
        'overwhelmed', 'underwhelmed', 'overpriced'
    }
    if word_lower in sentiment_participles:
        return True, 'suffix_adj'

    # Present participles as adjectives
    sentiment_ing = {
        'amazing', 'disappointing', 'disgusting', 'interesting', 'boring',
        'exciting', 'frustrating', 'annoying', 'satisfying', 'refreshing',
        'relaxing', 'welcoming', 'inviting', 'appealing', 'appalling'
    }
    if word_lower in sentiment_ing:
        return True, 'suffix_adj'

    return False, 'none'


# ============================================
# SECTION 7: NEGATION DETECTION
# ============================================

def detect_negation_in_opinion(opinion):
    """
    Detect negation in opinion text.

    IMPORTANT: Check idioms FIRST before calling this function.

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (has_negation: bool, negation_type: str or None)

    Example:
        >>> detect_negation_in_opinion("not good")
        (True, 'direct')

        >>> detect_negation_in_opinion("wasn't fresh")
        (True, 'contraction')
    """
    opinion_lower = opinion.lower()

    # SAFETY CHECK: If this is an idiom, don't treat as simple negation
    is_idiom, _, _ = check_for_idiom(opinion_lower)
    if is_idiom:
        return False, 'idiom_detected'

    # Direct negation words
    direct_negations = [
        'not ', "n't ", 'no ', 'never ', 'none ', 'nothing ',
        'neither ', 'nobody ', 'nowhere ', 'cannot '
    ]

    for neg in direct_negations:
        if neg in opinion_lower or opinion_lower.startswith(neg.strip()):
            return True, 'direct'

    # Contracted negations
    contracted_negations = [
        "didn't", "wasn't", "weren't", "isn't", "aren't",
        "don't", "doesn't", "won't", "wouldn't", "couldn't",
        "shouldn't", "can't", "haven't", "hasn't", "hadn't"
    ]

    for neg in contracted_negations:
        if neg in opinion_lower:
            return True, 'contraction'

    return False, None


# ============================================
# SECTION 8: BOUNDARY DETECTION
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """Find sentence boundaries containing the aspect."""
    sentence_enders = {'.', '!', '?'}

    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in sentence_enders:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in sentence_enders:
            sentence_end = i
            break

    return sentence_start, sentence_end


def find_previous_aspect_end(current_start_idx, all_aspect_positions):
    """Find end position of previous aspect."""
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(current_end_idx, all_aspect_positions):
    """Find start position of next aspect."""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# SECTION 9: BE-VERB DETECTION
# ============================================

def detect_be_verb(tokens, start_idx):
    """Detect be-verb at given position."""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    be_verbs = {'is', 'was', 'were', 'are', 'be', 'been', "'s", "s"}
    if word in be_verbs:
        return True, 1, False

    negated_be_verbs = {
        "isn't", "isnt", "wasn't", "wasnt",
        "weren't", "werent", "aren't", "arent"
    }
    if word in negated_be_verbs:
        return True, 1, True

    return False, 0, False


# ============================================
# SECTION 10: OPINION VALIDATION
# ============================================

def validate_opinion_completeness(opinion):
    """
    Validate that opinion is complete and meaningful.

    Rules:
        1. Cannot start with conjunction (incomplete phrase)
        2. Single intensifier alone is invalid
        3. Cannot be factual statement (reporting, not opinion)
        4. Must contain sentiment signal

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (is_valid: bool, reason: str or None)

    Examples:
        >>> validate_opinion_completeness("and was delicious")
        (False, "starts_with_conjunction")

        >>> validate_opinion_completeness("she said yes")
        (False, "factual_statement")

        >>> validate_opinion_completeness("absolutely delicious")
        (True, None)
    """
    if not opinion or len(opinion.strip()) < 2:
        return False, 'empty'

    words = opinion.lower().strip().split()
    if not words:
        return False, 'empty'

    # Rule 1: Cannot start with conjunction
    conjunctions = {'and', 'or', 'but', 'so', 'yet', 'nor', 'for'}
    if words[0] in conjunctions:
        return False, 'starts_with_conjunction'

    # Rule 2: Single intensifier is invalid
    if len(words) == 1:
        intensifiers = get_intensifiers()
        if words[0] in intensifiers:
            return False, 'only_intensifier'

        function_words = get_function_words()
        if words[0] in function_words:
            return False, 'only_function_word'

    # Rule 3: Cannot be factual statement
    factual_patterns = [
        r'^(he|she|they|it|we|i)\s+(said|told|asked|mentioned|replied)',
        r'^(was|were|is|are)\s+(out of|available|unavailable)',
        r'^if\s+(it|they|she|he|we)\s+(was|were|is|are)',
        r'^that\s+(it|they)',
        r'^\d+\s*/',
        r'^with\s+(a\s+)?side',
    ]

    opinion_text = opinion.lower()
    for pattern in factual_patterns:
        if re.search(pattern, opinion_text):
            return False, 'factual_statement'

    # Rule 4: Must contain sentiment signal (for short opinions)
    if len(words) <= 4:
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        all_sentiment = positive_words | negative_words

        has_sentiment = False
        for word in words:
            clean_word = word.strip('.,!?;:\'"')
            if clean_word in all_sentiment:
                has_sentiment = True
                break
            is_adj, _ = is_adjective_or_adverb(clean_word)
            if is_adj:
                has_sentiment = True
                break

        if not has_sentiment:
            # Exception: Idioms should pass
            is_idiom, _, _ = check_for_idiom(opinion)
            if not is_idiom:
                return False, 'no_sentiment_signal'

    return True, None


def is_valid_opinion(opinion):
    """
    Check if extracted text is a valid opinion.

    Args:
        opinion (str): Extracted opinion text

    Returns:
        bool: True if valid opinion
    """
    if not opinion or len(opinion.strip()) < 2:
        return False

    # Use completeness validation
    is_complete, _ = validate_opinion_completeness(opinion)
    if not is_complete:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    if not words:
        return False

    action_verbs = get_action_verbs()
    function_words = get_function_words()
    common_names = get_common_english_names()

    # Single word validation
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')
        if clean_word in action_verbs:
            return False
        if clean_word in function_words:
            return False
        if clean_word in common_names:
            return False

    # Invalid patterns
    invalid_patterns = [
        r"^'s\s+",
        r"^\d+\s*/",
        r"^or\s+maybe\s+both$",
        r"^going\s+forward$",
        r"^n\s+cheese",  # Truncated "mac n cheese"
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    return True


# ============================================
# SECTION 11: ASPECT VALIDATION
# ============================================

def is_valid_aspect(aspect):
    """Check if extracted text is a valid aspect."""
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    if not cleaned or cleaned == "-":
        return False

    if all(c in '.,!?;:-_\'"' for c in cleaned):
        return False

    # Invalid single words (prepositions, conjunctions, etc.)
    invalid_single = {'of', 'to', 'for', 'with', 'at', 'in', 'on', 'by',
                      'and', 'or', 'but', 'n', 'the', 'a', 'an'}
    if cleaned in invalid_single:
        return False

    invalid_aspects = {
        # Verbs
        'wait', 'waited', 'waiting', 'order', 'ordered', 'ordering',
        'serve', 'served', 'serving', 'ask', 'asked', 'asking',
        # Pronouns
        'i', 'me', 'my', 'you', 'your', 'he', 'him', 'his',
        'she', 'her', 'it', 'its', 'we', 'us', 'our', 'they', 'them',
        # Time units
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours',
        # Adjectives (opinions, not aspects)
        'good', 'bad', 'great', 'nice', 'poor',
        # Generic
        'thing', 'things', 'stuff', 'way', 'time', 'place'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """Clean and standardize aspect term."""
    if not aspect:
        return ""

    clean_aspect = aspect.strip()
    clean_aspect = re.sub(r'^(and|or|but)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^(the|a|an|this|that|my|our)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^[-:;,.\'"]+\s*', '', clean_aspect)
    clean_aspect = re.sub(r'[-:;,.\'"]+$', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# SECTION 12: OPINION EXTRACTION (ENHANCED)
# ============================================

def check_sentence_completeness(opinion_words, tokens, current_end_idx, sentence_end):
    """
    Check if opinion forms a complete sentence/phrase.
    If incomplete, extend to next sentence boundary.

    Args:
        opinion_words: Current list of opinion words
        tokens: All tokens in the review
        current_end_idx: Current end position
        sentence_end: Maximum sentence boundary

    Returns:
        list: Extended opinion words if needed
    """
    if not opinion_words:
        return opinion_words

    # Join current words to check completeness
    opinion_text = ' '.join(opinion_words).lower().strip()

    # Patterns indicating incomplete sentences
    incomplete_patterns = [
        r"(wasn|isn|aren|weren|didn|doesn|hasn|haven|hadn|wouldn|couldn|shouldn|won|can|don)\s*'\s*t\s*$",  # ends with contraction
        r"\s+(was|is|are|were|did|does|has|have|had|would|could|should|will|can|do)\s*$",  # ends with auxiliary verb
        r"\s+(he|she|they|it|we|i)\s+(said|told|asked)\s+.{0,20}$",  # incomplete quote
        r",\s*$",  # ends with comma
        r"\s+that\s*$",  # ends with "that"
        r"\s+which\s*$",  # ends with "which"
        r"\s+who\s*$",  # ends with "who"
        r"\s+when\s*$",  # ends with "when"
        r"\s+where\s*$",  # ends with "where"
        r"\s+if\s*$",  # ends with "if"
        r"\s+because\s*$",  # ends with "because"
        r"\s+although\s*$",  # ends with "although"
        r"\s+their\s*$",  # ends with possessive
        r"\s+his\s*$",
        r"\s+her\s*$",
        r"\s+our\s*$",
        r"\s+my\s*$",
    ]

    is_incomplete = False
    for pattern in incomplete_patterns:
        if re.search(pattern, opinion_text):
            is_incomplete = True
            break

    # Also check if last word is a verb form without object
    if opinion_words:
        last_word = opinion_words[-1].lower().strip('.,!?;:\'"')
        incomplete_endings = {
            'was', 'is', 'are', 'were', 'did', 'does', 'has', 'have', 'had',
            'would', 'could', 'should', 'will', 'can', 'do', 'said', 'told',
            'their', 'his', 'her', 'our', 'my', 'your', 'its', 'the', 'a', 'an',
            "wasn", "isn", "aren", "weren", "didn", "doesn", "hasn",
            "haven", "hadn", "wouldn", "couldn", "shouldn", "won", "don"
        }
        if last_word in incomplete_endings:
            is_incomplete = True

    # If incomplete, extend to next sentence boundary
    if is_incomplete and current_end_idx < sentence_end:
        extended_words = list(opinion_words)
        for i in range(current_end_idx, sentence_end):
            if i < len(tokens):
                word = tokens[i]
                if word in {'.', '!', '?'}:
                    break
                extended_words.append(word)
        return extended_words

    return opinion_words


def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """
    Extract opinion associated with aspect, respecting boundaries.
    Enhanced to ensure complete sentences.

    Extraction Patterns (Priority Order):
        1. Pre-modifier: "delicious food" → opinion before aspect
        2. Be-adjective: "food is delicious" → opinion after be-verb
        3. Verb phrase: "food tastes great" → opinion in verb phrase
        4. Context search: Fallback for complex sentences

    Args:
        tokens: Tokenized review
        aspect_positions: Position indices of aspect
        aspect_text: Aspect text (for reference)
        all_aspect_positions: All aspect positions (for boundaries)

    Returns:
        tuple: (opinion_phrase: str, pattern_type: str)
    """
    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1] if len(aspect_positions) > 1 else aspect_positions[0]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)
    prev_aspect_end = find_previous_aspect_end(start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(end_idx, all_aspect_positions)

    # Relaxed boundary - allow extending to sentence end for completeness
    original_sentence_end = sentence_end
    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # PATTERN 1: Pre-modifier
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, _ = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in {',', 'and', 'with', 'or', 'but'}:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

    # PATTERN 2: Be-verb + Adjective
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []
        if is_negative:
            post_modifiers.append("not")

        # ENHANCED: Skip punctuation tokens at the start
        skip_punct = {',', ';', ':', '(', ')', '-', '"', "'"}

        for i in range(opinion_start, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            # Skip leading punctuation tokens
            if not post_modifiers and word.strip() in skip_punct:
                continue
            post_modifiers.append(word)

        if post_modifiers:
            # Check for completeness and extend if needed
            post_modifiers = check_sentence_completeness(
                post_modifiers, tokens, opinion_end, original_sentence_end
            )
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # PATTERN 3: Verb Phrase
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        # ENHANCED: Skip punctuation tokens
        skip_punct = {',', ';', ':', '(', ')', '-', '"', "'"}

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            # Skip leading punctuation tokens
            if not temp_words and word.strip() in skip_punct:
                continue
            temp_words.append(word)

        if temp_words:
            # Check for completeness and extend if needed
            temp_words = check_sentence_completeness(
                temp_words, tokens, opinion_end, original_sentence_end
            )
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # PATTERN 4: Context Search
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        # ENHANCED: Skip punctuation tokens
        skip_punct = {',', ';', ':', '(', ')', '-', '"', "'"}

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, _ = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                for j in range(context_start, context_end):
                    token_word = tokens[j]
                    if token_word not in {'.', '!', '?', ';'}:
                        # Skip leading punctuation tokens
                        if not opinion_words and token_word.strip() in skip_punct:
                            continue
                        opinion_words.append(token_word)

                # Check for completeness
                opinion_words = check_sentence_completeness(
                    opinion_words, tokens, context_end, original_sentence_end
                )
                pattern_type = "context_search"
                break

    opinion_phrase = ' '.join(opinion_words).strip()
    return opinion_phrase, pattern_type


# ============================================
# SECTION 13: OPINION FORMATTING (ENHANCED)
# ============================================

def format_opinion_for_display(opinion):
    """
    Format and clean extracted opinion for display.
    Enhanced to remove leading punctuation and ensure completeness.

    Processing Steps:
        1. Remove leading punctuation (comma, parenthesis, etc.)
        2. Remove leading conjunctions ("and", "or", "but")
        3. Remove leading function words
        4. Handle hyphenation
        5. Truncate at cutoff words
        6. Remove trailing garbage
        7. Enforce length limit (extended to 20 words)

    Args:
        opinion (str): Raw extracted opinion

    Returns:
        str: Cleaned and formatted opinion
    """
    if not opinion:
        return ""

    # ENHANCED: Remove leading punctuation with spaces (handles " , " or ", " or " ,")
    # This pattern matches any combination of spaces and punctuation at the start
    opinion = re.sub(r'^[\s,;:\)\(\-\'\"\.\!\?]+', '', opinion)

    # Also handle cases where punctuation has spaces around it: " , just" -> "just"
    opinion = re.sub(r'^\s*[,;:\)\(\-]+\s*', '', opinion)

    # Remove leading conjunctions
    opinion = re.sub(r'^(and|or|but|so)\s+', '', opinion, flags=re.IGNORECASE)

    # Fix punctuation spacing
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    # Fix tokenization issues with contractions
    opinion = re.sub(r"\s+'\s*t\b", "'t", opinion)  # wasn ' t -> wasn't
    opinion = re.sub(r"\s+'\s*s\b", "'s", opinion)  # he ' s -> he's
    opinion = re.sub(r"\s+'\s*re\b", "'re", opinion)  # they ' re -> they're
    opinion = re.sub(r"\s+'\s*ve\b", "'ve", opinion)  # we ' ve -> we've
    opinion = re.sub(r"\s+'\s*ll\b", "'ll", opinion)  # we ' ll -> we'll
    opinion = re.sub(r"\s+'\s*d\b", "'d", opinion)  # he ' d -> he'd
    opinion = re.sub(r"\s+'\s*m\b", "'m", opinion)  # I ' m -> I'm

    words = opinion.split()
    if not words:
        return ""

    # ENHANCED: First remove any leading punctuation tokens (with strip to handle spaces)
    leading_punct = {',', ';', ':', ')', '(', '-', '.', "'", '"', '!', '?'}
    while words and words[0].strip() in leading_punct:
        words.pop(0)

    if not words:
        return ""

    # Remove leading fillers
    leading_fillers = {
        'a', 'an', 'the', 'this', 'that', 'also', 'too',
        'has', 'have', 'had', 'is', 'are', 'was', 'were',
        'it', 'itself', 'they', 'themselves', 'we', 'i',
        'and', 'or', 'but', 'so'
    }

    while words and words[0].lower().strip('.,;:') in leading_fillers:
        words.pop(0)

    # Check again for punctuation after removing fillers
    while words and words[0].strip() in leading_punct:
        words.pop(0)

    if not words:
        return ""

    # Truncate at cutoff words (but allow more context)
    cutoff_words = {
        'making', 'causing', 'forcing', 'leaving',
        'unless', 'except', 'despite', 'although',
    }

    truncated = []
    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')
        if word_lower in cutoff_words and i >= 4:  # Allow more words before cutoff
            break
        truncated.append(word)

    words = truncated

    # Remove trailing garbage
    trailing_garbage = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'a', 'an', 'the', ',', '-',
        'is', 'are', 'was', 'were',
        'it', 'i', 'we', 'they'
    }

    while words:
        last_word = words[-1].lower().strip('.,!?;:')
        if last_word in trailing_garbage or words[-1].endswith('-'):
            words.pop()
        else:
            break

    result = ' '.join(words)

    # ENHANCED: Extended length limit to 20 words for completeness
    words = result.split()
    if len(words) > 20:
        words = words[:20]
        result = ' '.join(words)

    # Final cleanup
    result = result.strip().rstrip('.,;-\'"(')
    result = re.sub(r'^[\s,;:\)\(\-\'"\.]+', '', result)  # Remove leading punctuation again
    result = re.sub(r'^(and|or|but|so)\s+', '', result, flags=re.IGNORECASE)

    return result


# ============================================
# SECTION 14: SENTIMENT ANALYSIS
# ============================================

def analyze_sentiment_scores(text):
    """
    Analyze text sentiment using multiple tools.

    Returns ensemble score combining:
        - VADER (weight: 0.4)
        - TextBlob (weight: 0.3)
        - Transformer (weight: 0.3)

    Args:
        text (str): Text to analyze

    Returns:
        dict: Sentiment scores from all analyzers and ensemble
    """
    analyzers = get_sentiment_analyzers()
    results = {}

    if not text or len(text.strip()) < 2:
        return {
            'vader': None,
            'textblob': None,
            'transformer': None,
            'ensemble': {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}
        }

    # VADER
    if analyzers.get('vader'):
        try:
            vader_scores = analyzers['vader'].polarity_scores(text)
            results['vader'] = {'compound': vader_scores['compound']}
        except:
            results['vader'] = None
    else:
        results['vader'] = None

    # TextBlob
    if analyzers.get('textblob'):
        try:
            blob = analyzers['textblob'](text)
            results['textblob'] = {'polarity': blob.sentiment.polarity}
        except:
            results['textblob'] = None
    else:
        results['textblob'] = None

    # Transformer
    if analyzers.get('transformer'):
        try:
            truncated = text[:500] if len(text) > 500 else text
            trans_result = analyzers['transformer'](truncated)[0]
            normalized = trans_result['score'] if trans_result['label'] == 'POSITIVE' else -trans_result['score']
            results['transformer'] = {'normalized_score': normalized}
        except:
            results['transformer'] = None
    else:
        results['transformer'] = None

    # Calculate ensemble
    results['ensemble'] = calculate_ensemble_score(results)

    return results


def calculate_ensemble_score(sentiment_results):
    """Calculate weighted ensemble sentiment score."""
    scores = []
    weights = []

    if sentiment_results.get('vader') and sentiment_results['vader'].get('compound') is not None:
        scores.append(sentiment_results['vader']['compound'])
        weights.append(0.4)

    if sentiment_results.get('textblob') and sentiment_results['textblob'].get('polarity') is not None:
        scores.append(sentiment_results['textblob']['polarity'])
        weights.append(0.3)

    if sentiment_results.get('transformer') and sentiment_results['transformer'].get('normalized_score') is not None:
        scores.append(sentiment_results['transformer']['normalized_score'])
        weights.append(0.3)

    if not scores:
        return {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}

    total_weight = sum(weights)
    normalized_weights = [w / total_weight for w in weights]
    ensemble_score = sum(s * w for s, w in zip(scores, normalized_weights))

    # Calculate confidence (agreement)
    if len(scores) > 1:
        variance = sum((s - ensemble_score) ** 2 for s in scores) / len(scores)
        confidence = max(0, 1 - variance)
    else:
        confidence = 0.7

    # Determine label
    if ensemble_score >= 0.05:
        label = 'Positive'
    elif ensemble_score <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return {
        'score': round(ensemble_score, 4),
        'label': label,
        'confidence': round(confidence, 4)
    }


# ============================================
# SECTION 15: ENHANCED SENTIMENT CORRECTION
# ============================================

# Strong sentiment indicators for sanity checking
OBVIOUS_NEGATIVE_INDICATORS = {
    'terrible', 'horrible', 'awful', 'worst', 'disgusting', 'nasty',
    'gross', 'pathetic', 'abysmal', 'dreadful', 'atrocious', 'appalling',
    'incompetent', 'useless', 'unacceptable', 'inexcusable', 'nightmare'
}

OBVIOUS_POSITIVE_INDICATORS = {
    'excellent', 'amazing', 'wonderful', 'fantastic', 'outstanding',
    'superb', 'incredible', 'phenomenal', 'magnificent', 'exceptional',
    'perfect', 'best', 'love', 'loved', 'brilliant', 'marvelous'
}


def sanity_check_sentiment(opinion, predicted_sentiment):
    """
    Sanity check to catch obvious sentiment misclassifications.

    Example:
        "TERRIBLE service" should NEVER be classified as Positive
        "EXCELLENT food" should NEVER be classified as Negative

    This is a safety net for when other correction logic fails.

    Args:
        opinion (str): Opinion text
        predicted_sentiment (str): Current sentiment prediction

    Returns:
        tuple: (corrected_sentiment, correction_reason) or (None, None) if no change
    """
    opinion_lower = opinion.lower()
    words = opinion_lower.split()

    # Check for obvious negative indicators
    for neg_word in OBVIOUS_NEGATIVE_INDICATORS:
        if neg_word in words or neg_word in opinion_lower:
            # Check if negated (e.g., "not terrible")
            neg_index = opinion_lower.find(neg_word)
            prefix = opinion_lower[max(0, neg_index-10):neg_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Positive':
                    return 'Negative', f'sanity_check_negative ({neg_word})'

    # Check for obvious positive indicators
    for pos_word in OBVIOUS_POSITIVE_INDICATORS:
        if pos_word in words or pos_word in opinion_lower:
            # Check if negated
            pos_index = opinion_lower.find(pos_word)
            prefix = opinion_lower[max(0, pos_index-10):pos_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Negative':
                    return 'Positive', f'sanity_check_positive ({pos_word})'

    return None, None


def correct_sentiment_with_analysis(aspect, opinion, predicted_sentiment, sentiment_scores=None):
    """
    Correct sentiment prediction using rules and analysis.

    PRIORITY ORDER (Critical for accuracy):
        0. SANITY CHECK (NEW) - Catch obvious misclassifications like "TERRIBLE" → Positive
        1. Idiom detection - "did not disappoint" → Positive
        2. Superlative patterns - "could not have been better" → Positive
        3. Negation + sentiment word
        4. Score override (when highly confident)
        5. Keyword detection

    Args:
        aspect (str): Aspect term
        opinion (str): Opinion text
        predicted_sentiment (str): Model's prediction
        sentiment_scores (dict): Sentiment analysis scores

    Returns:
        tuple: (corrected_sentiment, correction_reason)
    """
    opinion_lower = opinion.lower()

    # ========== PRIORITY 0: SANITY CHECK (NEW - HIGHEST) ==========
    # Catch obvious cases like "TERRIBLE" classified as Positive
    sanity_result, sanity_reason = sanity_check_sentiment(opinion, predicted_sentiment)
    if sanity_result:
        return sanity_result, sanity_reason

    # ========== PRIORITY 1: IDIOM DETECTION ==========
    is_idiom, idiom_sentiment, idiom_matched = check_for_idiom(opinion_lower)
    if is_idiom:
        if predicted_sentiment != idiom_sentiment:
            return idiom_sentiment, f'idiom_override ({idiom_matched})'
        return predicted_sentiment, None

    # ========== PRIORITY 2: SUPERLATIVE PATTERNS ==========
    superlative_patterns = [
        r"could(n't| not) have been (more )?(friendly|helpful|better|nicer|attentive|professional)",
        r"could(n't| not) ask for (more|better)",
        r"could(n't| not) be (happier|better)",
    ]
    for pattern in superlative_patterns:
        if re.search(pattern, opinion_lower):
            if predicted_sentiment != 'Positive':
                return 'Positive', 'superlative_positive'
            return predicted_sentiment, None

    # Get sentiment word lists
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()

    # ========== PRIORITY 3: NEGATION LOGIC ==========
    has_negation, neg_type = detect_negation_in_opinion(opinion_lower)

    if has_negation and neg_type != 'idiom_detected':
        # Negating positive word → Negative
        for pos_word in positive_words:
            if pos_word in opinion_lower:
                if "not only" not in opinion_lower:  # Exception
                    return 'Negative', f'negation_positive ({pos_word})'

        # Negating negative word → Positive (double negative)
        for neg_word in negative_words:
            if neg_word in opinion_lower:
                return 'Positive', f'double_negation ({neg_word})'

        # General negation with no clear word
        if sentiment_scores:
            score = sentiment_scores.get('ensemble', {}).get('score', 0)
            if score > 0.3:
                return predicted_sentiment, None
        return 'Negative', f'negation_general ({neg_type})'

    # ========== PRIORITY 4: SCORE OVERRIDE ==========
    if sentiment_scores:
        ensemble = sentiment_scores.get('ensemble', {})
        ensemble_score = ensemble.get('score', 0)
        confidence = ensemble.get('confidence', 0)

        if confidence >= SENTIMENT_CONFIDENCE_THRESHOLD:
            if predicted_sentiment == 'Positive' and ensemble_score < -SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(pw in opinion_lower for pw in positive_words):
                    return 'Negative', f'sentiment_override ({ensemble_score:.2f})'

            if predicted_sentiment == 'Negative' and ensemble_score > SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(nw in opinion_lower for nw in negative_words):
                    return 'Positive', f'sentiment_override ({ensemble_score:.2f})'

    # ========== PRIORITY 5: KEYWORD DETECTION ==========
    for pos_word in positive_words:
        if pos_word in opinion_lower and predicted_sentiment != 'Positive':
            return 'Positive', f'positive_keyword ({pos_word})'

    for neg_word in negative_words:
        if neg_word in opinion_lower and predicted_sentiment != 'Negative':
            return 'Negative', f'negative_keyword ({neg_word})'

    return predicted_sentiment, None


# ============================================
# SECTION 16: CONFIDENCE-BASED FILTERING
# ============================================

def should_include_extraction(sentiment_scores, opinion, predicted_sentiment):
    """
    Determine if extraction should be included based on confidence.

    Exclusion Rules:
        1. |score| < MINIMUM_SENTIMENT_SCORE_THRESHOLD (unclear sentiment)
        2. confidence < MINIMUM_AGREEMENT_THRESHOLD (analyzers disagree)

    Exceptions:
        - Known idioms are always included
        - Clear sentiment words override low scores

    Args:
        sentiment_scores (dict): Sentiment analysis results
        opinion (str): Opinion text
        predicted_sentiment (str): Model prediction

    Returns:
        tuple: (include: bool, reason: str or None)
    """
    if not sentiment_scores:
        return True, None

    ensemble = sentiment_scores.get('ensemble', {})
    score = ensemble.get('score', 0)
    confidence = ensemble.get('confidence', 0)

    # Rule 1: Score too close to zero
    if abs(score) < MINIMUM_SENTIMENT_SCORE_THRESHOLD:
        # Exception: Idioms should be included
        is_idiom, _, _ = check_for_idiom(opinion.lower())
        if not is_idiom:
            return False, f'low_score (|{score:.2f}| < {MINIMUM_SENTIMENT_SCORE_THRESHOLD})'

    # Rule 2: Low agreement between analyzers
    if confidence < MINIMUM_AGREEMENT_THRESHOLD:
        # Exception: Clear sentiment words
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        has_clear_word = any(w in opinion.lower() for w in positive_words | negative_words)
        if not has_clear_word:
            return False, f'low_agreement ({confidence:.2f})'

    return True, None


# ============================================
# SECTION 17: RESULT STRUCTURES
# ============================================

def create_aspect_opinion_pair(aspect, opinion, raw_opinion, sentiment,
                                original_sentiment, pattern, chunk_id,
                                sentiment_scores, correction_reason):
    """Create standardized result dictionary."""

    scores_dict = {
        'vader_compound': None,
        'textblob_polarity': None,
        'transformer_score': None,
        'ensemble_score': None,
        'ensemble_confidence': None
    }

    if sentiment_scores:
        if sentiment_scores.get('vader'):
            scores_dict['vader_compound'] = sentiment_scores['vader'].get('compound')
        if sentiment_scores.get('textblob'):
            scores_dict['textblob_polarity'] = sentiment_scores['textblob'].get('polarity')
        if sentiment_scores.get('transformer'):
            scores_dict['transformer_score'] = sentiment_scores['transformer'].get('normalized_score')
        if sentiment_scores.get('ensemble'):
            scores_dict['ensemble_score'] = sentiment_scores['ensemble'].get('score')
            scores_dict['ensemble_confidence'] = sentiment_scores['ensemble'].get('confidence')

    return {
        'aspect': aspect,
        'opinion': opinion,
        'raw_opinion': raw_opinion,
        'formatted': f"{aspect}: {opinion}",
        'pattern': pattern,
        'chunk': chunk_id,
        'original_sentiment': original_sentiment,
        'sentiment': sentiment,
        'correction_reason': correction_reason,
        'sentiment_scores': scores_dict
    }


# ============================================
# SECTION 18: FILE LOGGING
# ============================================

class FileLogger:
    """Logger for both console and file output."""

    def __init__(self, filename):
        self.filename = filename
        self.content = []

    def log(self, message=""):
        self.content.append(message)
        print(message)

    def save(self):
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ Results saved to: {self.filename}")


# ============================================
# SECTION 19: REVIEW CHUNKING
# ============================================

def split_long_review(text, max_words=MAX_WORDS_PER_CHUNK):
    """Split long reviews into sentence-preserving chunks."""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub = ' '.join(words[i:i + max_words])
                chunks.append(sub + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# SECTION 20: MAIN ANALYSIS FUNCTION
# ============================================

def comprehensive_aspect_opinion_analysis(reviews, aspect_extractor,
                                          max_words=MAX_WORDS_PER_CHUNK,
                                          verbose=True, logger=None,
                                          use_sentiment_analysis=True,
                                          apply_confidence_filter=True):
    """
    Run complete ABSA analysis on restaurant reviews.

    Args:
        reviews: Iterable of review texts
        aspect_extractor: PyABSA aspect extractor
        max_words: Max words per chunk
        verbose: Enable detailed logging
        logger: FileLogger instance
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence-based filtering

    Returns:
        tuple: (results_list, statistics_dict)
    """
    if logger:
        logger.log("=" * 100)
        logger.log("ABSA Analysis System - V5 Enhanced Universal Edition")
        logger.log(f"Chunk size: {max_words} words")
        logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
        logger.log(f"Confidence filter: {'Enabled' if apply_confidence_filter else 'Disabled'}")
        logger.log(f"Min score threshold: {MINIMUM_SENTIMENT_SCORE_THRESHOLD}")
        logger.log(f"Analysis time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 100)

    if use_sentiment_analysis:
        _ = get_sentiment_analyzers()

    all_results = []
    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'filtered_low_confidence': 0,
        'sentiment_corrections': 0,
        'correction_reasons': Counter(),
        'idiom_detections': 0,
        'sanity_check_corrections': 0,  # NEW: Track sanity check fixes
    }

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'=' * 100}")
            logger.log(f"Review #{idx} ({word_count} words)")
            logger.log("=" * 100)

        # Chunking
        if word_count <= max_words:
            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]
            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if logger and verbose:
                logger.log("📄 Splitting long review into chunks...")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   Split into {len(chunks)} chunks")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = [
                {'chunk_id': i + 1, 'result': r}
                for i, r in enumerate(chunk_results)
            ]

        # Process extractions
        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0
        filtered_confidence_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, original_sentiment in zip(aspects, positions, sentiments):

                # Validate aspect
                aspect = refine_aspect_term(aspect)
                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Extract opinion
                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Format and validate opinion
                display_opinion = format_opinion_for_display(raw_opinion)

                if not display_opinion or not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                # Sentiment analysis
                sentiment_scores = None
                if use_sentiment_analysis:
                    sentiment_scores = analyze_sentiment_scores(display_opinion)

                # Confidence filtering
                if apply_confidence_filter and sentiment_scores:
                    should_include, _ = should_include_extraction(
                        sentiment_scores, display_opinion, original_sentiment
                    )
                    if not should_include:
                        filtered_confidence_count += 1
                        stats['filtered_low_confidence'] += 1
                        continue

                # Sentiment correction
                corrected_sentiment, correction_reason = correct_sentiment_with_analysis(
                    aspect, raw_opinion, original_sentiment, sentiment_scores
                )

                if corrected_sentiment != original_sentiment:
                    stats['sentiment_corrections'] += 1
                    if correction_reason:
                        stats['correction_reasons'][correction_reason] += 1
                        if 'idiom' in correction_reason:
                            stats['idiom_detections'] += 1
                        if 'sanity_check' in correction_reason:
                            stats['sanity_check_corrections'] += 1

                stats['total_aspects'] += 1

                # Create result
                pair = create_aspect_opinion_pair(
                    aspect=aspect,
                    opinion=display_opinion,
                    raw_opinion=raw_opinion,
                    sentiment=corrected_sentiment,
                    original_sentiment=original_sentiment,
                    pattern=pattern,
                    chunk_id=chunk_data['chunk_id'] if len(analysis_results) > 1 else None,
                    sentiment_scores=sentiment_scores,
                    correction_reason=correction_reason
                )
                aspect_opinion_pairs.append(pair)

        # ENHANCED: Log results with wider columns and full content
        if logger and verbose:
            logger.log(f"\n🎯 Extraction Results:")
            # ENHANCED: Wider opinion column (60 chars instead of 35)
            header = f"{'#':<3} {'Aspect':<20} {'Opinion':<60} {'Orig':<10} {'Final':<10} {'Score':<8}"
            logger.log(header)
            logger.log("-" * 115)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    opinion_text = pair['opinion']

                    # ENHANCED: Show full opinion, use multiple lines if needed
                    if len(opinion_text) > 60:
                        # First line with all columns
                        opinion_line1 = opinion_text[:60]
                        scores = pair['sentiment_scores']
                        score_str = f"{scores['ensemble_score']:+.2f}" if scores.get('ensemble_score') else "N/A"

                        final = pair['sentiment']
                        if pair['correction_reason']:
                            final += "*"

                        logger.log(f"{i:<3} {pair['aspect']:<20} {opinion_line1:<60} "
                                  f"{pair['original_sentiment']:<10} {final:<10} {score_str:<8}")

                        # Additional lines for remaining opinion text
                        remaining = opinion_text[60:]
                        while remaining:
                            chunk = remaining[:60]
                            remaining = remaining[60:]
                            logger.log(f"{'':3} {'':20} {chunk:<60}")
                    else:
                        scores = pair['sentiment_scores']
                        score_str = f"{scores['ensemble_score']:+.2f}" if scores.get('ensemble_score') else "N/A"

                        final = pair['sentiment']
                        if pair['correction_reason']:
                            final += "*"

                        logger.log(f"{i:<3} {pair['aspect']:<20} {opinion_text:<60} "
                                  f"{pair['original_sentiment']:<10} {final:<10} {score_str:<8}")

                total_filtered = filtered_aspect_count + filtered_opinion_count + filtered_confidence_count
                if total_filtered > 0:
                    logger.log(f"\n   ℹ️ Filtered: {filtered_aspect_count} aspects, "
                              f"{filtered_opinion_count} opinions, "
                              f"{filtered_confidence_count} low confidence")
            else:
                logger.log("   ⚠️ No valid aspect-opinion pairs found")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
        })

    return all_results, stats


# ============================================
# SECTION 21: ENHANCED REPORT GENERATION
# ============================================

def select_representative_opinions(pairs, n=5, target_sentiment=None):
    """
    Select high-quality representative opinions.
    ENHANCED: Now returns 5 opinions by default (increased from 3).

    Selection Criteria:
        1. Sentiment matches target (CRITICAL: positive section shows only positive opinions)
        2. High absolute sentiment score (clear sentiment)
        3. Sufficient length (not fragments)
        4. Passes completeness validation
        5. Diverse (not similar to already selected)
        6. No negative words in positive section (and vice versa)

    Args:
        pairs: List of aspect-opinion pairs
        n: Number of opinions to select (default: 5)
        target_sentiment: 'Positive' or 'Negative' - MUST match this sentiment

    Returns:
        list: Selected representative opinions
    """
    # CRITICAL FIX: Filter by target sentiment FIRST
    if target_sentiment:
        pairs = [p for p in pairs if p.get('sentiment') == target_sentiment]

    # Sort by score magnitude (clearest sentiment first)
    sorted_pairs = sorted(
        pairs,
        key=lambda p: abs(p['sentiment_scores'].get('ensemble_score', 0) if p['sentiment_scores'] else 0),
        reverse=True
    )

    # Define words that indicate opposite sentiment
    strong_negative_words = {
        'terrible', 'horrible', 'awful', 'worst', 'bad', 'poor', 'disgusting',
        'rude', 'slow', 'cold', 'bland', 'disappointing', 'mediocre', 'nasty',
        'gross', 'dirty', 'stale', 'overpriced', 'incompetent', 'never'
    }
    strong_positive_words = {
        'great', 'excellent', 'amazing', 'wonderful', 'fantastic', 'perfect',
        'delicious', 'friendly', 'awesome', 'best', 'love', 'outstanding',
        'superb', 'incredible', 'fresh', 'tasty', 'beautiful', 'attentive'
    }

    selected = []
    seen_stems = set()

    for pair in sorted_pairs:
        opinion = pair['opinion']
        opinion_lower = opinion.lower()

        # CRITICAL: Skip if opinion contains strong opposite-sentiment words
        if target_sentiment == 'Positive':
            # For positive section, skip opinions with strong negative words
            if any(neg_word in opinion_lower for neg_word in strong_negative_words):
                continue
        elif target_sentiment == 'Negative':
            # For negative section, verify it actually sounds negative
            # (optional: could skip if too many positive words)
            pass

        # Quality checks
        if len(opinion.split()) < 2:
            continue

        if opinion_lower.startswith(('and ', 'or ', 'but ')):
            continue

        is_valid, _ = validate_opinion_completeness(opinion)
        if not is_valid:
            continue

        # Diversity check (avoid similar opinions)
        stem = ' '.join(opinion_lower.split()[:3])
        if stem in seen_stems:
            continue

        seen_stems.add(stem)
        selected.append(pair)

        if len(selected) >= n:
            break

    return selected


def generate_management_report(results, stats, logger=None):
    """
    Generate enhanced management report with multiple representative opinions.
    ENHANCED: Shows 5 examples per aspect (increased from 3) with full content.

    Report Sections:
        1. Overall Statistics
        2. Correction Breakdown
        3. Competitive Advantages (with 5 example opinions each)
        4. Areas for Improvement (with 5 example complaints each)
        5. Recommendations
        6. Quality Metrics
    """
    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 100)
    log("MANAGEMENT ANALYSIS REPORT")
    log("Universal Restaurant Review Analysis")
    log("=" * 100)

    # Aggregate by sentiment
    positive_pairs = []
    negative_pairs = []

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

    total_pairs = len(positive_pairs) + len(negative_pairs)

    # ===== STATISTICS =====
    log(f"\n📊 OVERALL STATISTICS")
    log(f"   • Total reviews analyzed: {len(results)}")
    log(f"   • Total aspects extracted: {total_pairs}")
    log(f"   • Invalid aspects filtered: {stats['filtered_invalid_aspects']}")
    log(f"   • Invalid opinions filtered: {stats['filtered_invalid_opinions']}")
    log(f"   • Low confidence filtered: {stats.get('filtered_low_confidence', 0)}")

    if stats['sentiment_corrections'] > 0:
        log(f"   • Sentiments corrected: {stats['sentiment_corrections']}")
        log(f"   • Idiom detections: {stats.get('idiom_detections', 0)}")
        log(f"   • Sanity check fixes: {stats.get('sanity_check_corrections', 0)}")

    if total_pairs > 0:
        pos_pct = len(positive_pairs) / total_pairs * 100
        neg_pct = len(negative_pairs) / total_pairs * 100
        log(f"   • Positive mentions: {len(positive_pairs)} ({pos_pct:.1f}%)")
        log(f"   • Negative mentions: {len(negative_pairs)} ({neg_pct:.1f}%)")

    # ===== CORRECTION BREAKDOWN =====
    if stats['correction_reasons']:
        log(f"\n📈 CORRECTION BREAKDOWN")
        for reason, count in stats['correction_reasons'].most_common(10):
            log(f"   • {reason}: {count}")

    # Aggregate by aspect
    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair)

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair)

    # ===== COMPETITIVE ADVANTAGES =====
    if positive_aspects:
        log(f"\n✅ COMPETITIVE ADVANTAGES (Top Positive Aspects)")
        log("-" * 100)

        sorted_pos = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_pos[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # ENHANCED: 5 representative opinions (increased from 3) - MUST be positive sentiment
            examples = select_representative_opinions(pairs, n=5, target_sentiment='Positive')
            for ex in examples:
                # ENHANCED: Show full opinion text without truncation
                opinion_text = ex['opinion']
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0

                # Use multiple lines if opinion is long
                if len(opinion_text) > 70:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text[:70]}"')
                    remaining = opinion_text[70:]
                    while remaining:
                        chunk = remaining[:70]
                        remaining = remaining[70:]
                        log(f'              "{chunk}"')
                else:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid positive examples found, note it
            if not examples:
                log(f'   • [No clear positive examples available]')

    # ===== AREAS FOR IMPROVEMENT =====
    if negative_aspects:
        log(f"\n⚠️ AREAS FOR IMPROVEMENT (Top Negative Aspects)")
        log("-" * 100)

        sorted_neg = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_neg[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # ENHANCED: 5 representative opinions (increased from 3) - MUST be negative sentiment
            examples = select_representative_opinions(pairs, n=5, target_sentiment='Negative')
            for ex in examples:
                # ENHANCED: Show full opinion text without truncation
                opinion_text = ex['opinion']
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0

                # Use multiple lines if opinion is long
                if len(opinion_text) > 70:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text[:70]}"')
                    remaining = opinion_text[70:]
                    while remaining:
                        chunk = remaining[:70]
                        remaining = remaining[70:]
                        log(f'              "{chunk}"')
                else:
                    log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid negative examples found, note it
            if not examples:
                log(f'   • [No clear negative examples available]')

    # ===== RECOMMENDATIONS =====
    log(f"\n💡 RECOMMENDATIONS")

    if negative_aspects:
        worst = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   1. PRIORITY: Address '{worst[0]}' ({len(worst[1])} negative mentions)")

    if positive_aspects:
        best = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   2. LEVERAGE: Promote '{best[0]}' ({len(best[1])} positive mentions)")

    log(f"   3. MONITOR: Track sentiment trends over time")
    log(f"   4. INVESTIGATE: Review corrected sentiments for accuracy")

    # ===== QUALITY METRICS =====
    log(f"\n📉 EXTRACTION QUALITY METRICS")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   • Extraction success rate: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   • Average aspects per review: {avg_pairs:.2f}")

    total_filtered = (stats['filtered_invalid_aspects'] +
                      stats['filtered_invalid_opinions'] +
                      stats.get('filtered_low_confidence', 0))
    total_extracted = total_pairs + total_filtered
    if total_extracted > 0:
        quality_rate = total_pairs / total_extracted * 100
        log(f"   • Quality pass rate: {total_pairs}/{total_extracted} ({quality_rate:.1f}%)")


# ============================================
# SECTION 22: DATA EXPORT
# ============================================

def export_results_to_dataframe(results):
    """Export results to pandas DataFrame."""
    import pandas as pd

    rows = []
    for result in results:
        review_id = result['review_id']
        for pair in result['pairs']:
            scores = pair.get('sentiment_scores', {})
            rows.append({
                'review_id': review_id,
                'aspect': pair['aspect'],
                'opinion': pair['opinion'],
                'original_sentiment': pair['original_sentiment'],
                'corrected_sentiment': pair['sentiment'],
                'correction_reason': pair.get('correction_reason'),
                'ensemble_score': scores.get('ensemble_score'),
                'ensemble_confidence': scores.get('ensemble_confidence')
            })

    return pd.DataFrame(rows)


# ============================================
# SECTION 23: MAIN EXECUTION
# ============================================

def run_analysis(reviews, aspect_extractor,
                 output_file="absa_results_v5_universal.txt",
                 use_sentiment_analysis=True,
                 apply_confidence_filter=True):
    """
    Run complete ABSA analysis pipeline.

    Args:
        reviews: Review texts (list or Series)
        aspect_extractor: PyABSA extractor
        output_file: Output log file path
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence filtering

    Returns:
        tuple: (results, stats, dataframe)
    """
    logger = FileLogger(output_file)

    logger.log("=" * 100)
    logger.log("ABSA ANALYSIS SYSTEM - V5 ENHANCED UNIVERSAL EDITION")
    logger.log(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"Reviews to analyze: {len(reviews)}")
    logger.log("=" * 100)

    results, stats = comprehensive_aspect_opinion_analysis(
        reviews,
        aspect_extractor,
        max_words=MAX_WORDS_PER_CHUNK,
        verbose=True,
        logger=logger,
        use_sentiment_analysis=use_sentiment_analysis,
        apply_confidence_filter=apply_confidence_filter
    )

    logger.log("\n" + "=" * 100)
    logger.log("✅ ANALYSIS COMPLETE")
    logger.log("=" * 100)

    generate_management_report(results, stats, logger)
    logger.save()

    df_results = export_results_to_dataframe(results)

    return results, stats, df_results


if __name__ == "__main__":
    print("=" * 70)
    print("ABSA Analysis System - V5 Enhanced Universal Edition")
    print("=" * 70)
    print("\nDesigned for UNIVERSAL restaurant review analysis:")
    print("  • All restaurant types (fast food to fine dining)")
    print("  • All cuisines (American, Asian, Mexican, etc.)")
    print("  • All service models (dine-in, takeout, delivery)")
    print("\nKey Enhancements:")
    print("  1. Idiom/Slang Dictionary (~80 expressions)")
    print("  2. Confidence-Based Filtering")
    print("  3. Improved Opinion Boundary Detection")
    print("  4. Enhanced Management Report (5 examples per aspect)")
    print("  5. Complete sentence extraction")
    print("  6. Removed leading punctuation (comma, parenthesis)")
    print("\nUsage:")
    print("  results, stats, df = run_analysis(reviews, aspect_extractor)")

In [ ]:
 results, stats, df = run_analysis(test_reviews, aspect_extractor)

In [ ]:
[2026-01-16 17:07:26] (2.4.2) Example 0: Customer Service could have much been better . <Waiter:Negative Confidence:0.9953> forgot items that we ordered . He also charged my credit card for the wrong check . The <staff:Negative Confidence:0.9888> acted like it was my fault was I told them about it . No apologies were given . Had to wait 3 days for the credit to appear back into my account . <Salads:Negative Confidence:0.9925> were not fresh as <lettuce:Negative Confidence:0.9888> was brown . Items on the <nachos:Negative Confidence:0.9919> was cold . Overall not a good experience at all . [2026-01-16 17:07:26] (2.4.2) Example 1: The only thing that was good was the <music:Positive Confidence:0.9689> being played . The one star I gave was for the music .  🎯 Extraction Results: #   Aspect               Opinion                                                      Orig       Final      Score    ------------------------------------------------------------------------------------------------------------------- 1   Waiter               forgot items that we ordered                                 Negative   Negative   -0.30    2   staff                acted like it was my fault                                   Negative   Negative   -0.32    3   Salads               not fresh                                                    Negative   Negative*  -0.44    4   lettuce              fresh                                                        Negative   Positive*  +0.52    5   nachos               cold                                                         Negative   Negative   -0.48    6   music                good                                                         Positive   Positive   +0.69       正面也應該知道為何正面   長一點可避面誤判  例如 這個案例   提到沙拉不新鮮  萵苣不如剛摘下那麼新鮮 但卻誤判為新鮮  因此 請深度思考 如何擷取更完整的意見 例如 一整句話或一整個意見段落   請深度思考   避免誤判

In [ ]:
# =============================================================================
# ABSA 深度洞察分析系統 V4
# =============================================================================
# 功能：
# 1. 解析 V3 結果並去重
# 2. 屬性語義聚類（將相似屬性歸類到顧客旅程階段）
# 3. 深度洞察報告生成
# 4. 輸出 Excel 報告
# =============================================================================

import re
import pandas as pd
from collections import defaultdict, Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 第一部分：解析 V3 結果文件
# =============================================================================

def parse_v3_results(file_path):
    """解析 V3 結果文件，提取所有屬性-意見-情感三元組"""

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 提取所有評論區塊
    review_pattern = r'評論 #(\d+) \((\d+) 字\)'
    pair_pattern = r'(\d+)\.\s+([^:]+):\s+(.+?)\s+(😊|😞)(?:\s+\[段\d+\])?(?:\s+\[糾正\])?'

    all_pairs = []
    current_review_id = None

    lines = content.split('\n')

    for i, line in enumerate(lines):
        # 檢查是否是評論標題
        review_match = re.search(review_pattern, line)
        if review_match:
            current_review_id = int(review_match.group(1))
            continue

        # 檢查是否是屬性-意見對
        pair_match = re.search(pair_pattern, line.strip())
        if pair_match and current_review_id:
            idx = pair_match.group(1)
            aspect = pair_match.group(2).strip()
            opinion = pair_match.group(3).strip()
            emoji = pair_match.group(4)
            sentiment = 'Positive' if emoji == '😊' else 'Negative'

            all_pairs.append({
                'review_id': current_review_id,
                'aspect': aspect,
                'opinion': opinion,
                'sentiment': sentiment,
                'aspect_lower': aspect.lower().strip(),
                'opinion_lower': opinion.lower().strip()
            })

    return pd.DataFrame(all_pairs)


# =============================================================================
# 第二部分：資料清理與去重
# =============================================================================

def clean_and_deduplicate(df):
    """清理資料並去除重複"""

    print("=" * 60)
    print("資料清理與去重")
    print("=" * 60)

    original_count = len(df)
    print(f"原始資料筆數: {original_count}")

    # 1. 移除完全重複的記錄（同評論、同屬性、同意見）
    df_dedup = df.drop_duplicates(subset=['review_id', 'aspect_lower', 'opinion_lower'])
    after_exact_dedup = len(df_dedup)
    print(f"移除完全重複後: {after_exact_dedup} (移除 {original_count - after_exact_dedup} 筆)")

    # 2. 移除同評論中相同屬性的重複（保留第一個）
    df_dedup = df_dedup.drop_duplicates(subset=['review_id', 'aspect_lower'], keep='first')
    after_aspect_dedup = len(df_dedup)
    print(f"移除同評論重複屬性後: {after_aspect_dedup} (移除 {after_exact_dedup - after_aspect_dedup} 筆)")

    # 3. 清理屬性名稱
    def clean_aspect(aspect):
        # 移除開頭的 and/or/the/a
        cleaned = re.sub(r'^(and|or|the|a|an)\s+', '', aspect.strip(), flags=re.IGNORECASE)
        # 移除特殊字符
        cleaned = re.sub(r'^[-:;]\s*', '', cleaned)
        return cleaned.strip()

    df_dedup['aspect_cleaned'] = df_dedup['aspect'].apply(clean_aspect)

    # 4. 移除無效屬性
    invalid_aspects = {
        'waited', 'waiting', 'ordering', 'making', 'having',
        'she', 'he', 'they', 'it', 'we', 'i', 'you', 'someone',
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours'
    }

    df_dedup = df_dedup[~df_dedup['aspect_cleaned'].str.lower().isin(invalid_aspects)]
    df_dedup = df_dedup[df_dedup['aspect_cleaned'].str.len() > 1]

    final_count = len(df_dedup)
    print(f"移除無效屬性後: {final_count} (移除 {after_aspect_dedup - final_count} 筆)")
    print(f"\n最終有效資料: {final_count} 筆")

    return df_dedup


# =============================================================================
# 第三部分：屬性語義聚類（顧客旅程框架）
# =============================================================================

# 定義顧客旅程階段與屬性映射
CUSTOMER_JOURNEY_MAPPING = {
    # 階段 1: 來店動機 (Why they come)
    'visit_motivation': {
        'keywords': ['location', 'downtown', 'hotel', 'game', 'astros', 'event',
                     'recommendation', 'reviews', 'rating', 'yelp', 'google'],
        'aspects': []
    },

    # 階段 2: 第一印象 (First Impression)
    'first_impression': {
        'keywords': ['atmosphere', 'vibe', 'ambiance', 'decor', 'decoration',
                     'interior', 'exterior', 'entrance', 'space', 'place',
                     'restaurant', 'bar', 'patio', 'seating', 'music', 'noise',
                     'crowd', 'busy', 'wait', 'line', 'parking'],
        'aspects': []
    },

    # 階段 3: 服務體驗 (Service Experience)
    'service_experience': {
        'keywords': ['service', 'server', 'waiter', 'waitress', 'staff',
                     'manager', 'host', 'hostess', 'bartender', 'busser',
                     'attentive', 'friendly', 'rude', 'slow', 'fast', 'quick'],
        'person_names': ['joel', 'paul', 'kaitlyn', 'kaitlin', 'christina',
                        'natalie', 'melanie', 'sara', 'morgan', 'fatima',
                        'andy', 'nick', 'alyssa', 'janelle', 'kiara'],
        'aspects': []
    },

    # 階段 4: 食物品質 (Food Quality)
    'food_quality': {
        'keywords': ['food', 'meal', 'dish', 'taste', 'flavor', 'texture',
                     'portion', 'size', 'presentation', 'quality', 'fresh',
                     'cooked', 'seasoning', 'sauce', 'spicy', 'bland'],
        'aspects': []
    },

    # 階段 5: 具體菜品 (Specific Menu Items)
    'menu_items': {
        'keywords': ['chicken', 'waffles', 'brisket', 'nachos', 'burger',
                     'wings', 'fries', 'tacos', 'sandwich', 'salad', 'soup',
                     'steak', 'ribs', 'pork', 'bacon', 'eggs', 'pancakes',
                     'mac', 'cheese', 'grits', 'coleslaw', 'beans', 'corn',
                     'bread', 'biscuit', 'dessert', 'pie', 'cake'],
        'aspects': []
    },

    # 階段 6: 飲品 (Beverages)
    'beverages': {
        'keywords': ['drink', 'drinks', 'cocktail', 'cocktails', 'beer', 'beers',
                     'wine', 'moonshine', 'margarita', 'mimosa', 'bloody mary',
                     'coffee', 'tea', 'juice', 'water', 'soda', 'lemonade',
                     'bar', 'happy hour'],
        'aspects': []
    },

    # 階段 7: 價值感知 (Value Perception)
    'value_perception': {
        'keywords': ['price', 'prices', 'cost', 'value', 'money', 'worth',
                     'expensive', 'cheap', 'affordable', 'reasonable',
                     'overpriced', 'deal', 'discount', 'happy hour'],
        'aspects': []
    },

    # 階段 8: 環境設施 (Facilities)
    'facilities': {
        'keywords': ['table', 'tables', 'chair', 'booth', 'restroom', 'bathroom',
                     'clean', 'dirty', 'sticky', 'menu', 'utensils', 'napkin',
                     'plate', 'glass', 'air conditioning', 'temperature'],
        'aspects': []
    },

    # 階段 9: 整體體驗 (Overall Experience)
    'overall_experience': {
        'keywords': ['experience', 'overall', 'visit', 'time', 'night',
                     'lunch', 'dinner', 'brunch', 'breakfast', 'return',
                     'recommend', 'back', 'again'],
        'aspects': []
    }
}

# 情感細分類別
SENTIMENT_SUBCATEGORIES = {
    'highly_positive': ['excellent', 'amazing', 'fantastic', 'perfect', 'outstanding',
                        'incredible', 'best', 'wonderful', 'awesome', 'superb'],
    'positive': ['good', 'great', 'nice', 'lovely', 'friendly', 'helpful',
                 'tasty', 'delicious', 'fresh', 'solid'],
    'neutral_positive': ['ok', 'okay', 'decent', 'fine', 'average', 'standard'],
    'neutral_negative': ['mediocre', 'disappointing', 'underwhelming'],
    'negative': ['bad', 'poor', 'slow', 'cold', 'bland', 'dry', 'soggy',
                 'rude', 'unfriendly', 'dirty'],
    'highly_negative': ['terrible', 'awful', 'horrible', 'worst', 'disgusting',
                        'pathetic', 'nasty', 'gross']
}


def classify_aspect_to_journey(aspect, opinion):
    """將屬性分類到顧客旅程階段"""

    aspect_lower = aspect.lower()
    opinion_lower = opinion.lower()
    combined = f"{aspect_lower} {opinion_lower}"

    # 優先檢查具體菜品
    menu_keywords = CUSTOMER_JOURNEY_MAPPING['menu_items']['keywords']
    for keyword in menu_keywords:
        if keyword in aspect_lower:
            return 'menu_items'

    # 檢查飲品
    beverage_keywords = CUSTOMER_JOURNEY_MAPPING['beverages']['keywords']
    for keyword in beverage_keywords:
        if keyword in aspect_lower:
            return 'beverages'

    # 檢查服務相關（包含人名）
    service_keywords = CUSTOMER_JOURNEY_MAPPING['service_experience']['keywords']
    person_names = CUSTOMER_JOURNEY_MAPPING['service_experience'].get('person_names', [])
    for keyword in service_keywords:
        if keyword in aspect_lower:
            return 'service_experience'
    for name in person_names:
        if name in combined:
            return 'service_experience'

    # 檢查其他階段
    for stage, config in CUSTOMER_JOURNEY_MAPPING.items():
        if stage in ['menu_items', 'beverages', 'service_experience']:
            continue
        for keyword in config['keywords']:
            if keyword in aspect_lower or keyword in opinion_lower:
                return stage

    # 默認歸類到食物品質
    return 'food_quality'


def get_sentiment_intensity(opinion):
    """獲取情感強度"""

    opinion_lower = opinion.lower()

    for intensity, keywords in SENTIMENT_SUBCATEGORIES.items():
        for keyword in keywords:
            if keyword in opinion_lower:
                return intensity

    return 'neutral'


def add_journey_classification(df):
    """為資料添加顧客旅程分類"""

    df['journey_stage'] = df.apply(
        lambda row: classify_aspect_to_journey(row['aspect_cleaned'], row['opinion']),
        axis=1
    )

    df['sentiment_intensity'] = df['opinion'].apply(get_sentiment_intensity)

    return df


# =============================================================================
# 第四部分：屬性標準化與聚合
# =============================================================================

# 屬性同義詞映射
ASPECT_SYNONYMS = {
    # 服務人員
    'server': ['server', 'waiter', 'waitress', 'waitstaff'],
    'staff': ['staff', 'employee', 'team', 'crew'],
    'manager': ['manager', 'gm', 'general manager'],
    'bartender': ['bartender', 'barman', 'barmaid'],
    'host': ['host', 'hostess', 'greeter'],

    # 食物總類
    'food': ['food', 'meal', 'dish', 'cuisine'],

    # 具體菜品
    'chicken_and_waffles': ['chicken and waffles', 'waffles', 'chicken waffles',
                            'waffle', 'hot chicken'],
    'brisket': ['brisket', 'beef brisket', 'smoked brisket'],
    'nachos': ['nachos', 'brisket nachos', 'nacho'],
    'wings': ['wings', 'chicken wings', 'wing'],
    'burger': ['burger', 'hamburger', 'cheeseburger'],
    'fries': ['fries', 'french fries', 'fry'],
    'tacos': ['tacos', 'taco', 'pulled pork tacos'],
    'mac_and_cheese': ['mac', 'mac and cheese', 'macaroni'],

    # 飲品
    'drinks': ['drinks', 'drink', 'beverages', 'beverage'],
    'cocktails': ['cocktail', 'cocktails', 'mixed drink'],
    'beer': ['beer', 'beers', 'draft', 'ale'],
    'moonshine': ['moonshine', 'shine'],
    'mimosa': ['mimosa', 'mimosas'],

    # 環境
    'atmosphere': ['atmosphere', 'vibe', 'ambiance', 'ambience', 'environment'],
    'decor': ['decor', 'decoration', 'interior', 'design'],
    'seating': ['seating', 'seat', 'seats', 'booth', 'table', 'tables'],

    # 服務屬性
    'service': ['service'],
    'wait_time': ['wait', 'waiting', 'wait time', 'waited'],

    # 價格
    'price': ['price', 'prices', 'cost', 'pricing'],
    'value': ['value', 'worth', 'deal'],

    # 其他
    'location': ['location', 'place', 'spot'],
    'parking': ['parking', 'park'],
    'cleanliness': ['clean', 'cleanliness', 'dirty', 'sticky']
}


def standardize_aspect(aspect):
    """標準化屬性名稱"""

    aspect_lower = aspect.lower().strip()

    for standard_name, synonyms in ASPECT_SYNONYMS.items():
        for synonym in synonyms:
            if synonym in aspect_lower or aspect_lower in synonym:
                return standard_name

    return aspect_lower


def aggregate_aspects(df):
    """聚合相似屬性"""

    df['aspect_standardized'] = df['aspect_cleaned'].apply(standardize_aspect)

    return df


# =============================================================================
# 第五部分：深度洞察分析
# =============================================================================

def generate_deep_insights(df):
    """生成深度洞察報告"""

    insights = {}

    # 1. 整體統計
    insights['overall'] = {
        'total_reviews': df['review_id'].nunique(),
        'total_mentions': len(df),
        'positive_count': len(df[df['sentiment'] == 'Positive']),
        'negative_count': len(df[df['sentiment'] == 'Negative']),
        'positive_rate': len(df[df['sentiment'] == 'Positive']) / len(df) * 100
    }

    # 2. 顧客旅程階段分析
    journey_analysis = {}
    for stage in CUSTOMER_JOURNEY_MAPPING.keys():
        stage_df = df[df['journey_stage'] == stage]
        if len(stage_df) > 0:
            journey_analysis[stage] = {
                'total_mentions': len(stage_df),
                'positive_count': len(stage_df[stage_df['sentiment'] == 'Positive']),
                'negative_count': len(stage_df[stage_df['sentiment'] == 'Negative']),
                'satisfaction_rate': len(stage_df[stage_df['sentiment'] == 'Positive']) / len(stage_df) * 100,
                'top_positive_aspects': stage_df[stage_df['sentiment'] == 'Positive']['aspect_standardized'].value_counts().head(5).to_dict(),
                'top_negative_aspects': stage_df[stage_df['sentiment'] == 'Negative']['aspect_standardized'].value_counts().head(5).to_dict(),
                'sample_positive_opinions': stage_df[stage_df['sentiment'] == 'Positive'][['aspect_cleaned', 'opinion']].head(3).values.tolist(),
                'sample_negative_opinions': stage_df[stage_df['sentiment'] == 'Negative'][['aspect_cleaned', 'opinion']].head(3).values.tolist()
            }
    insights['journey_analysis'] = journey_analysis

    # 3. 熱門菜品分析
    menu_df = df[df['journey_stage'] == 'menu_items']
    menu_analysis = {}

    for aspect in menu_df['aspect_standardized'].unique():
        aspect_df = menu_df[menu_df['aspect_standardized'] == aspect]
        if len(aspect_df) >= 3:  # 至少3次提及
            pos_count = len(aspect_df[aspect_df['sentiment'] == 'Positive'])
            neg_count = len(aspect_df[aspect_df['sentiment'] == 'Negative'])

            menu_analysis[aspect] = {
                'total_mentions': len(aspect_df),
                'positive_count': pos_count,
                'negative_count': neg_count,
                'satisfaction_rate': pos_count / len(aspect_df) * 100 if len(aspect_df) > 0 else 0,
                'positive_opinions': aspect_df[aspect_df['sentiment'] == 'Positive']['opinion'].tolist()[:5],
                'negative_opinions': aspect_df[aspect_df['sentiment'] == 'Negative']['opinion'].tolist()[:5]
            }

    insights['menu_analysis'] = dict(sorted(menu_analysis.items(),
                                            key=lambda x: x[1]['total_mentions'],
                                            reverse=True))

    # 4. 服務人員分析
    service_df = df[df['journey_stage'] == 'service_experience']

    # 提取提到的服務人員名字
    person_names = CUSTOMER_JOURNEY_MAPPING['service_experience'].get('person_names', [])
    staff_mentions = defaultdict(lambda: {'positive': 0, 'negative': 0, 'opinions': []})

    for _, row in service_df.iterrows():
        combined = f"{row['aspect_cleaned']} {row['opinion']}".lower()
        for name in person_names:
            if name in combined:
                if row['sentiment'] == 'Positive':
                    staff_mentions[name]['positive'] += 1
                else:
                    staff_mentions[name]['negative'] += 1
                staff_mentions[name]['opinions'].append(row['opinion'][:50])

    insights['staff_mentions'] = dict(staff_mentions)

    # 5. 問題熱點分析
    negative_df = df[df['sentiment'] == 'Negative']

    # 按旅程階段統計負面評價
    problem_hotspots = {}
    for stage in CUSTOMER_JOURNEY_MAPPING.keys():
        stage_neg = negative_df[negative_df['journey_stage'] == stage]
        if len(stage_neg) > 0:
            problem_hotspots[stage] = {
                'count': len(stage_neg),
                'percentage': len(stage_neg) / len(negative_df) * 100,
                'top_issues': stage_neg['aspect_standardized'].value_counts().head(5).to_dict(),
                'sample_complaints': stage_neg[['aspect_cleaned', 'opinion']].head(5).values.tolist()
            }

    insights['problem_hotspots'] = dict(sorted(problem_hotspots.items(),
                                               key=lambda x: x[1]['count'],
                                               reverse=True))

    # 6. 情感強度分析
    intensity_analysis = df.groupby(['sentiment_intensity', 'sentiment']).size().unstack(fill_value=0)
    insights['sentiment_intensity'] = intensity_analysis.to_dict()

    return insights


# =============================================================================
# 第六部分：報告生成
# =============================================================================

def generate_text_report(df, insights, output_file):
    """生成詳細文字報告"""

    report = []

    def add_line(text=""):
        report.append(text)

    # 標題
    add_line("=" * 80)
    add_line("餐廳評論深度洞察分析報告")
    add_line(f"分析時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    add_line("=" * 80)

    # 1. 執行摘要
    add_line("\n" + "=" * 80)
    add_line("📊 執行摘要 (Executive Summary)")
    add_line("=" * 80)

    overall = insights['overall']
    add_line(f"\n總評論數: {overall['total_reviews']} 則")
    add_line(f"有效屬性-意見對: {overall['total_mentions']} 組")
    add_line(f"正面評價: {overall['positive_count']} ({overall['positive_rate']:.1f}%)")
    add_line(f"負面評價: {overall['negative_count']} ({100 - overall['positive_rate']:.1f}%)")

    # 關鍵發現
    add_line("\n🔑 關鍵發現:")

    # 找出最佳和最差的旅程階段
    journey = insights['journey_analysis']
    best_stage = max(journey.items(), key=lambda x: x[1]['satisfaction_rate'])
    worst_stage = min(journey.items(), key=lambda x: x[1]['satisfaction_rate'])

    stage_names_zh = {
        'visit_motivation': '來店動機',
        'first_impression': '第一印象',
        'service_experience': '服務體驗',
        'food_quality': '食物品質',
        'menu_items': '具體菜品',
        'beverages': '飲品',
        'value_perception': '價值感知',
        'facilities': '環境設施',
        'overall_experience': '整體體驗'
    }

    add_line(f"   ✅ 最佳體驗環節: {stage_names_zh.get(best_stage[0], best_stage[0])} "
             f"(滿意度 {best_stage[1]['satisfaction_rate']:.1f}%)")
    add_line(f"   ⚠️ 最需改進環節: {stage_names_zh.get(worst_stage[0], worst_stage[0])} "
             f"(滿意度 {worst_stage[1]['satisfaction_rate']:.1f}%)")

    # 2. 顧客旅程分析
    add_line("\n" + "=" * 80)
    add_line("🚶 顧客旅程分析 (Customer Journey Analysis)")
    add_line("=" * 80)

    for stage, data in sorted(journey.items(), key=lambda x: x[1]['total_mentions'], reverse=True):
        stage_name = stage_names_zh.get(stage, stage)
        add_line(f"\n{'─' * 60}")
        add_line(f"📍 {stage_name}")
        add_line(f"{'─' * 60}")
        add_line(f"   提及次數: {data['total_mentions']} | "
                 f"正面: {data['positive_count']} | "
                 f"負面: {data['negative_count']} | "
                 f"滿意度: {data['satisfaction_rate']:.1f}%")

        if data['top_positive_aspects']:
            add_line(f"\n   ✅ 正面評價焦點:")
            for aspect, count in list(data['top_positive_aspects'].items())[:3]:
                add_line(f"      • {aspect}: {count}次")

        if data['sample_positive_opinions']:
            add_line(f"   📝 正面評價範例:")
            for aspect, opinion in data['sample_positive_opinions'][:2]:
                add_line(f"      「{aspect}: {opinion[:60]}...」" if len(opinion) > 60 else f"      「{aspect}: {opinion}」")

        if data['top_negative_aspects']:
            add_line(f"\n   ⚠️ 負面評價焦點:")
            for aspect, count in list(data['top_negative_aspects'].items())[:3]:
                add_line(f"      • {aspect}: {count}次")

        if data['sample_negative_opinions']:
            add_line(f"   📝 負面評價範例:")
            for aspect, opinion in data['sample_negative_opinions'][:2]:
                add_line(f"      「{aspect}: {opinion[:60]}...」" if len(opinion) > 60 else f"      「{aspect}: {opinion}」")

    # 3. 菜品詳細分析
    add_line("\n" + "=" * 80)
    add_line("🍽️ 菜品詳細分析 (Menu Items Analysis)")
    add_line("=" * 80)

    menu = insights['menu_analysis']

    add_line(f"\n{'菜品':<25} {'提及':<8} {'正面':<8} {'負面':<8} {'滿意度':<10}")
    add_line("-" * 60)

    for item, data in list(menu.items())[:15]:
        sat_rate = f"{data['satisfaction_rate']:.1f}%"
        add_line(f"{item:<25} {data['total_mentions']:<8} {data['positive_count']:<8} "
                 f"{data['negative_count']:<8} {sat_rate:<10}")

    # 菜品深度洞察
    add_line("\n📋 菜品深度洞察:")

    for item, data in list(menu.items())[:8]:
        if data['total_mentions'] >= 5:
            add_line(f"\n   🍴 {item.upper().replace('_', ' ')}")
            add_line(f"      滿意度: {data['satisfaction_rate']:.1f}% ({data['total_mentions']}次提及)")

            if data['positive_opinions']:
                add_line(f"      ✅ 顧客讚賞:")
                for op in data['positive_opinions'][:3]:
                    add_line(f"         • {op[:70]}..." if len(op) > 70 else f"         • {op}")

            if data['negative_opinions']:
                add_line(f"      ⚠️ 顧客抱怨:")
                for op in data['negative_opinions'][:3]:
                    add_line(f"         • {op[:70]}..." if len(op) > 70 else f"         • {op}")

    # 4. 服務人員分析
    add_line("\n" + "=" * 80)
    add_line("👥 服務人員分析 (Staff Analysis)")
    add_line("=" * 80)

    staff = insights['staff_mentions']
    if staff:
        add_line(f"\n{'服務人員':<15} {'正面':<10} {'負面':<10} {'總計':<10}")
        add_line("-" * 50)

        for name, data in sorted(staff.items(), key=lambda x: x[1]['positive'] + x[1]['negative'], reverse=True):
            total = data['positive'] + data['negative']
            if total >= 2:
                add_line(f"{name.title():<15} {data['positive']:<10} {data['negative']:<10} {total:<10}")

        # 表現優異的員工
        top_performers = [(name, data) for name, data in staff.items()
                          if data['positive'] >= 3 and data['negative'] == 0]
        if top_performers:
            add_line("\n   🌟 表現優異員工:")
            for name, data in top_performers:
                add_line(f"      • {name.title()}: {data['positive']} 次正面提及")

    # 5. 問題熱點分析
    add_line("\n" + "=" * 80)
    add_line("🚨 問題熱點分析 (Problem Hotspots)")
    add_line("=" * 80)

    hotspots = insights['problem_hotspots']

    add_line(f"\n{'環節':<20} {'負面數':<10} {'佔比':<10} {'主要問題':<30}")
    add_line("-" * 70)

    for stage, data in list(hotspots.items())[:5]:
        stage_name = stage_names_zh.get(stage, stage)
        top_issue = list(data['top_issues'].keys())[0] if data['top_issues'] else 'N/A'
        add_line(f"{stage_name:<20} {data['count']:<10} {data['percentage']:.1f}%{'':<5} {top_issue:<30}")

    # 詳細問題分析
    add_line("\n📋 詳細問題分析:")

    for stage, data in list(hotspots.items())[:3]:
        stage_name = stage_names_zh.get(stage, stage)
        add_line(f"\n   🔴 {stage_name} ({data['count']}個負面評價)")

        add_line(f"      主要問題:")
        for issue, count in list(data['top_issues'].items())[:5]:
            add_line(f"         • {issue}: {count}次")

        add_line(f"      典型抱怨:")
        for aspect, opinion in data['sample_complaints'][:3]:
            add_line(f"         「{aspect}: {opinion[:60]}...」" if len(opinion) > 60 else f"         「{aspect}: {opinion}」")

    # 6. 行動建議
    add_line("\n" + "=" * 80)
    add_line("💡 行動建議 (Action Recommendations)")
    add_line("=" * 80)

    add_line("\n🔴 緊急改進 (需立即處理):")

    # 基於分析結果生成建議
    if 'service_experience' in hotspots and hotspots['service_experience']['count'] > 20:
        add_line("   1. 【服務速度】")
        add_line("      • 問題: 服務速度慢是最常見抱怨")
        add_line("      • 建議: 增加尖峰時段人手、優化點餐流程、設定服務時間標準")
        add_line("      • KPI: 將平均等待時間降低 30%")

    if 'food_quality' in hotspots:
        add_line("   2. 【食物品質一致性】")
        add_line("      • 問題: 食物品質不穩定（過乾/過淡/過冷）")
        add_line("      • 建議: 加強廚房品管、建立出餐前檢查機制")
        add_line("      • KPI: 將食物相關負評降低 40%")

    add_line("\n🟡 中期優化 (1-3個月):")
    add_line("   3. 【員工培訓】")
    add_line("      • 針對負面提及的員工進行服務培訓")
    add_line("      • 表揚並學習表現優異員工的服務方式")

    add_line("   4. 【菜單優化】")
    add_line("      • 強化高滿意度菜品的行銷")
    add_line("      • 檢討低滿意度菜品的食譜或下架")

    add_line("\n🟢 長期策略 (3-6個月):")
    add_line("   5. 【顧客體驗整體提升】")
    add_line("      • 建立顧客回饋追蹤系統")
    add_line("      • 定期進行滿意度調查")
    add_line("      • 設計忠誠顧客獎勵計畫")

    # 保存報告
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report))

    print(f"\n✅ 報告已保存到: {output_file}")

    return '\n'.join(report)


def generate_excel_report(df, insights, output_file):
    """生成 Excel 報告"""

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

        # Sheet 1: 原始資料（去重後）
        df_export = df[['review_id', 'aspect_cleaned', 'opinion', 'sentiment',
                        'journey_stage', 'aspect_standardized', 'sentiment_intensity']].copy()
        df_export.columns = ['評論ID', '屬性', '意見', '情感', '旅程階段', '標準化屬性', '情感強度']
        df_export.to_excel(writer, sheet_name='原始資料', index=False)

        # Sheet 2: 旅程階段統計
        journey_stats = []
        stage_names_zh = {
            'visit_motivation': '來店動機',
            'first_impression': '第一印象',
            'service_experience': '服務體驗',
            'food_quality': '食物品質',
            'menu_items': '具體菜品',
            'beverages': '飲品',
            'value_perception': '價值感知',
            'facilities': '環境設施',
            'overall_experience': '整體體驗'
        }

        for stage, data in insights['journey_analysis'].items():
            journey_stats.append({
                '旅程階段': stage_names_zh.get(stage, stage),
                '階段代碼': stage,
                '提及次數': data['total_mentions'],
                '正面評價': data['positive_count'],
                '負面評價': data['negative_count'],
                '滿意度(%)': round(data['satisfaction_rate'], 1)
            })

        df_journey = pd.DataFrame(journey_stats)
        df_journey = df_journey.sort_values('提及次數', ascending=False)
        df_journey.to_excel(writer, sheet_name='旅程階段統計', index=False)

        # Sheet 3: 菜品分析
        menu_stats = []
        for item, data in insights['menu_analysis'].items():
            menu_stats.append({
                '菜品': item.replace('_', ' ').title(),
                '提及次數': data['total_mentions'],
                '正面評價': data['positive_count'],
                '負面評價': data['negative_count'],
                '滿意度(%)': round(data['satisfaction_rate'], 1),
                '正面評價範例': ' | '.join(data['positive_opinions'][:3]),
                '負面評價範例': ' | '.join(data['negative_opinions'][:3])
            })

        df_menu = pd.DataFrame(menu_stats)
        df_menu.to_excel(writer, sheet_name='菜品分析', index=False)

        # Sheet 4: 屬性統計
        aspect_stats = df.groupby(['aspect_standardized', 'sentiment']).size().unstack(fill_value=0)
        aspect_stats['總計'] = aspect_stats.sum(axis=1)
        aspect_stats['滿意度(%)'] = (aspect_stats.get('Positive', 0) / aspect_stats['總計'] * 100).round(1)
        aspect_stats = aspect_stats.sort_values('總計', ascending=False)
        aspect_stats.to_excel(writer, sheet_name='屬性統計')

        # Sheet 5: 負面評價詳情
        df_negative = df[df['sentiment'] == 'Negative'][['review_id', 'aspect_cleaned', 'opinion', 'journey_stage']].copy()
        df_negative.columns = ['評論ID', '屬性', '意見', '旅程階段']
        df_negative.to_excel(writer, sheet_name='負面評價詳情', index=False)

        # Sheet 6: 正面評價詳情
        df_positive = df[df['sentiment'] == 'Positive'][['review_id', 'aspect_cleaned', 'opinion', 'journey_stage']].copy()
        df_positive.columns = ['評論ID', '屬性', '意見', '旅程階段']
        df_positive.to_excel(writer, sheet_name='正面評價詳情', index=False)

        # Sheet 7: 摘要統計
        summary_data = {
            '指標': ['總評論數', '有效屬性-意見對', '正面評價數', '負面評價數',
                    '正面評價比例(%)', '負面評價比例(%)', '平均每則評論屬性數'],
            '數值': [
                insights['overall']['total_reviews'],
                insights['overall']['total_mentions'],
                insights['overall']['positive_count'],
                insights['overall']['negative_count'],
                round(insights['overall']['positive_rate'], 1),
                round(100 - insights['overall']['positive_rate'], 1),
                round(insights['overall']['total_mentions'] / insights['overall']['total_reviews'], 2)
            ]
        }
        df_summary = pd.DataFrame(summary_data)
        df_summary.to_excel(writer, sheet_name='摘要統計', index=False)

    print(f"✅ Excel 報告已保存到: {output_file}")


# =============================================================================
# 主程式
# =============================================================================

def main(v3_result_file, output_prefix="absa_deep_analysis"):
    """主程式"""

    print("=" * 60)
    print("ABSA 深度洞察分析系統 V4")
    print("=" * 60)

    # 1. 解析 V3 結果
    print("\n[1/5] 解析 V3 結果文件...")
    df = parse_v3_results(v3_result_file)
    print(f"   解析完成，共 {len(df)} 筆資料")

    # 2. 資料清理與去重
    print("\n[2/5] 資料清理與去重...")
    df = clean_and_deduplicate(df)

    # 3. 顧客旅程分類
    print("\n[3/5] 顧客旅程分類...")
    df = add_journey_classification(df)
    df = aggregate_aspects(df)
    print(f"   分類完成")

    # 4. 深度洞察分析
    print("\n[4/5] 深度洞察分析...")
    insights = generate_deep_insights(df)
    print(f"   分析完成")

    # 5. 生成報告
    print("\n[5/5] 生成報告...")

    text_report_file = f"{output_prefix}_report.txt"
    excel_report_file = f"{output_prefix}_report.xlsx"

    generate_text_report(df, insights, text_report_file)
    generate_excel_report(df, insights, excel_report_file)

    print("\n" + "=" * 60)
    print("✅ 分析完成！")
    print("=" * 60)
    print(f"\n輸出文件:")
    print(f"   📄 文字報告: {text_report_file}")
    print(f"   📊 Excel報告: {excel_report_file}")

    return df, insights


# =============================================================================
# 執行
# =============================================================================

if __name__ == "__main__":
    # 請修改為您的 V3 結果文件路徑
    V3_RESULT_FILE = "absa_analysis_results_v4_universal.txt"
    OUTPUT_PREFIX = "absa_deep_analysis"

    df, insights = main(V3_RESULT_FILE, OUTPUT_PREFIX)

    # 顯示快速摘要
    print("\n" + "=" * 60)
    print("快速摘要")
    print("=" * 60)
    print(f"總評論數: {insights['overall']['total_reviews']}")
    print(f"有效屬性-意見對: {insights['overall']['total_mentions']}")
    print(f"正面評價: {insights['overall']['positive_count']} ({insights['overall']['positive_rate']:.1f}%)")
    print(f"負面評價: {insights['overall']['negative_count']} ({100-insights['overall']['positive_rate']:.1f}%)")

In [ ]:
######  做到此
 [2026-01-16 17:07:26] (2.4.2) Example 0: Customer Service could have much been better . <Waiter:Negative Confidence:0.9953> forgot items that we ordered . He also charged my credit card for the wrong check . The <staff:Negative Confidence:0.9888> acted like it was my fault was I told them about it . No apologies were given . Had to wait 3 days for the credit to appear back into my account . <Salads:Negative Confidence:0.9925> were not fresh as <lettuce:Negative Confidence:0.9888> was brown . Items on the <nachos:Negative Confidence:0.9919> was cold . Overall not a good experience at all . [2026-01-16 17:07:26] (2.4.2) Example 1: The only thing that was good was the <music:Positive Confidence:0.9689> being played . The one star I gave was for the music .  🎯 Extraction Results: #   Aspect               Opinion                                                      Orig       Final      Score    ------------------------------------------------------------------------------------------------------------------- 1   Waiter               forgot items that we ordered                                 Negative   Negative   -0.30    2   staff                acted like it was my fault                                   Negative   Negative   -0.32    3   Salads               not fresh                                                    Negative   Negative*  -0.44    4   lettuce              fresh                                                        Negative   Positive*  +0.52    5   nachos               cold                                                         Negative   Negative   -0.48    6   music                good                                                         Positive   Positive   +0.69       正面也應該知道為何正面   長一點可避面誤判  例如 這個案例   提到沙拉不新鮮  萵苣不如剛摘下那麼新鮮 但卻誤判為新鮮  因此 請深度思考 如何擷取更完整的意見 例如 一整句話或一整個意見段落   請深度思考   避免誤判

In [ ]:
"""
================================================================================
ABSA (Aspect-Based Sentiment Analysis) System - V5 Enhanced Universal Edition
================================================================================

Purpose:
    Extract aspect-opinion-sentiment triplets from restaurant reviews.
    Designed for UNIVERSAL application across ALL restaurant types:
    - Fast food, casual dining, fine dining
    - All cuisines (American, Asian, Mexican, Italian, etc.)
    - All service models (dine-in, takeout, delivery, food trucks)

Key Enhancements in V5:
    1. Idiom/Slang Dictionary - Handles "did not disappoint", "killed it", etc.
    2. Confidence-Based Filtering - Excludes low-confidence extractions
    3. Improved Opinion Boundary Detection - Avoids fragments and factual statements
    4. Enhanced Management Report - Multiple representative opinions per aspect
    5. Universal Restaurant Vocabulary - Multi-cuisine support

Version: 5.0 Enhanced Universal
Author: HAOS Framework Research Team
Date: 2025
================================================================================
"""

import warnings
warnings.filterwarnings('ignore')

import re
from collections import Counter
from datetime import datetime

# ============================================
# SECTION 1: GLOBAL CONFIGURATION
# ============================================

OUTPUT_FILE = "absa_analysis_results_v5_universal.txt"
MAX_WORDS_PER_CHUNK = 80

# Sentiment correction thresholds
SENTIMENT_CONFIDENCE_THRESHOLD = 0.6
SENTIMENT_OVERRIDE_THRESHOLD = 0.4

# NEW: Confidence-based filtering thresholds
# Extractions with |score| < this threshold will be excluded
MINIMUM_SENTIMENT_SCORE_THRESHOLD = 0.25

# Minimum agreement level between sentiment analyzers
MINIMUM_AGREEMENT_THRESHOLD = 0.40


# ============================================
# SECTION 2: IDIOM/SLANG DICTIONARIES
# ============================================
"""
CRITICAL SECTION: Idiom Detection

Problem Solved:
    Many English expressions use negation words but convey POSITIVE meaning.
    Example: "The service did not disappoint" = POSITIVE (not negative!)

These idioms are universal across ALL restaurant types:
    - Fast food: "The fries hit the spot"
    - Fine dining: "The tasting menu was to die for"
    - BBQ: "Ribs fall off the bone"
    - Any cuisine: "Our server killed it tonight"
"""


def get_positive_idioms():
    """
    Dictionary of positive idioms/slang expressions.

    Categories:
    1. Double Negative = Positive ("did not disappoint")
    2. Superlative Negation = Maximum Positive ("could not have been better")
    3. Slang = Positive ("killed it", "fire", "bussin")
    4. Food-Specific = Positive ("melt in your mouth")

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DOUBLE NEGATIVE = POSITIVE ==========
        # Pattern: "did not [negative_word]" → Positive
        # Logic: NOT disappointing = satisfying
        "did not disappoint": "Positive",
        "didn't disappoint": "Positive",
        "does not disappoint": "Positive",
        "doesn't disappoint": "Positive",
        "never disappoints": "Positive",
        "never disappointed": "Positive",
        "not disappointed": "Positive",
        "wasn't disappointed": "Positive",
        "weren't disappointed": "Positive",
        "not let down": "Positive",
        "didn't let us down": "Positive",

        # ========== SUPERLATIVE NEGATION = MAXIMUM POSITIVE ==========
        # Pattern: "could not have been more [positive_adj]"
        # Logic: Impossible to be MORE positive = maximum level achieved
        "could not have been better": "Positive",
        "couldn't have been better": "Positive",
        "could not have been nicer": "Positive",
        "couldn't have been nicer": "Positive",
        "could not have been friendlier": "Positive",
        "couldn't have been friendlier": "Positive",
        "could not have been more helpful": "Positive",
        "couldn't have been more helpful": "Positive",
        "could not have been more attentive": "Positive",
        "couldn't have been more attentive": "Positive",
        "could not have been more professional": "Positive",
        "couldn't have been more professional": "Positive",
        "could not ask for more": "Positive",
        "couldn't ask for more": "Positive",
        "could not ask for better": "Positive",
        "couldn't ask for better": "Positive",
        "could not be happier": "Positive",
        "couldn't be happier": "Positive",
        "could not be more satisfied": "Positive",
        "couldn't be more satisfied": "Positive",

        # ========== NO COMPLAINTS = POSITIVE ==========
        "can't complain": "Positive",
        "cannot complain": "Positive",
        "nothing to complain about": "Positive",
        "no complaints": "Positive",
        "zero complaints": "Positive",
        "left nothing to be desired": "Positive",
        "leaves nothing to be desired": "Positive",

        # ========== SLANG/INFORMAL = POSITIVE ==========
        # Common in casual reviews, especially younger demographics
        "killed it": "Positive",           # Performed excellently
        "nailed it": "Positive",           # Got it exactly right
        "crushed it": "Positive",          # Exceeded expectations
        "smashed it": "Positive",          # Did amazingly well
        "hit the spot": "Positive",        # Perfectly satisfying
        "hits the spot": "Positive",
        "hit different": "Positive",       # Especially good
        "hits different": "Positive",
        "on point": "Positive",            # Exactly right
        "on fire": "Positive",             # Performing excellently
        "off the hook": "Positive",        # Extremely good
        "off the chain": "Positive",       # Amazing
        "off the charts": "Positive",      # Exceptionally high quality
        "out of this world": "Positive",   # Extraordinary
        "to die for": "Positive",          # Extremely desirable
        "die for": "Positive",
        "the bomb": "Positive",            # Excellent
        "da bomb": "Positive",
        "was bomb": "Positive",            # "The pizza was bomb"
        "is bomb": "Positive",
        "is fire": "Positive",             # "These wings are fire"
        "was fire": "Positive",
        "straight fire": "Positive",
        "chef's kiss": "Positive",         # Perfect
        "chefs kiss": "Positive",
        "slaps": "Positive",               # "This mac and cheese slaps"
        "bussin": "Positive",              # "Food was bussin"
        "bussin'": "Positive",
        "top notch": "Positive",
        "top-notch": "Positive",
        "first rate": "Positive",
        "first-rate": "Positive",
        "A1": "Positive",                  # Top quality
        "a-1": "Positive",
        "10/10": "Positive",
        "10 out of 10": "Positive",
        "five stars": "Positive",
        "5 stars": "Positive",

        # ========== FOOD-SPECIFIC POSITIVE IDIOMS ==========
        # Universal across cuisines
        "melt in your mouth": "Positive",
        "melts in your mouth": "Positive",
        "melted in my mouth": "Positive",
        "fall off the bone": "Positive",   # Tender meat
        "falls off the bone": "Positive",
        "fell off the bone": "Positive",
        "cooked to perfection": "Positive",
        "done to perfection": "Positive",
        "seasoned to perfection": "Positive",
        "finger licking good": "Positive",
        "finger-licking good": "Positive",
        "fresh out of the oven": "Positive",
        "fresh off the grill": "Positive",
        "made with love": "Positive",
        "like grandma used to make": "Positive",
        "like mama used to make": "Positive",
        "home cooked": "Positive",
        "homemade taste": "Positive",

        # ========== RECOMMENDATION POSITIVE ==========
        "a must try": "Positive",
        "must try": "Positive",
        "a must": "Positive",
        "a must visit": "Positive",
        "must visit": "Positive",
        "a gem": "Positive",
        "hidden gem": "Positive",
        "best kept secret": "Positive",
        "saved the day": "Positive",
        "made my day": "Positive",
        "worth the wait": "Positive",
        "worth the drive": "Positive",
        "worth every penny": "Positive",
        "worth the price": "Positive",
        "bang for your buck": "Positive",
        "will be back": "Positive",
        "coming back": "Positive",
        "definitely returning": "Positive",
        "can't wait to come back": "Positive",
    }


def get_negative_idioms():
    """
    Dictionary of negative idioms/slang expressions.

    Categories:
    1. Disappointment expressions
    2. Mediocrity expressions
    3. Poor value expressions
    4. Food-specific negative idioms

    Returns:
        dict: Maps idiom phrase to sentiment label
    """
    return {
        # ========== DISAPPOINTMENT ==========
        "left a lot to be desired": "Negative",
        "leaves a lot to be desired": "Negative",
        "left much to be desired": "Negative",
        "leaves much to be desired": "Negative",
        "nothing to write home about": "Negative",
        "not my cup of tea": "Negative",
        "wasn't my cup of tea": "Negative",
        "seen better days": "Negative",
        "has seen better days": "Negative",
        "let down": "Negative",
        "let us down": "Negative",
        "fell short": "Negative",
        "falls short": "Negative",
        "missed the mark": "Negative",
        "misses the mark": "Negative",

        # ========== MEDIOCRITY ==========
        "hit or miss": "Negative",
        "meh": "Negative",
        "just ok": "Negative",
        "just okay": "Negative",
        "nothing special": "Negative",
        "middle of the road": "Negative",
        "run of the mill": "Negative",
        "average at best": "Negative",
        "could be better": "Negative",
        "room for improvement": "Negative",
        "needs work": "Negative",

        # ========== POOR VALUE ==========
        "waste of money": "Negative",
        "waste of time": "Negative",
        "rip off": "Negative",
        "rip-off": "Negative",
        "ripoff": "Negative",
        "not worth it": "Negative",
        "not worth the price": "Negative",
        "not worth the money": "Negative",
        "not worth the wait": "Negative",
        "not worth the hype": "Negative",
        "overpriced": "Negative",
        "over priced": "Negative",
        "overhyped": "Negative",
        "over-hyped": "Negative",
        "overrated": "Negative",
        "over-rated": "Negative",

        # ========== FOOD-SPECIFIC NEGATIVE ==========
        "tasted like cardboard": "Negative",
        "tastes like cardboard": "Negative",
        "like eating cardboard": "Negative",
        "rubber chicken": "Negative",
        "hockey puck": "Negative",         # Overcooked burger
        "sat under a heat lamp": "Negative",
        "sitting under a heat lamp": "Negative",
        "straight from the freezer": "Negative",
        "microwaved": "Negative",
        "reheated": "Negative",
        "day old": "Negative",
        "stale": "Negative",

        # ========== SERVICE NEGATIVE ==========
        "worst service ever": "Negative",
        "worst experience ever": "Negative",
        "never coming back": "Negative",
        "will not return": "Negative",
        "won't be back": "Negative",
        "avoid this place": "Negative",
        "stay away": "Negative",
        "save your money": "Negative",
    }


# ============================================
# SECTION 3: SENTIMENT ANALYZER INITIALIZATION
# ============================================

SENTIMENT_ANALYZERS = None


def initialize_sentiment_analyzers():
    """
    Initialize multiple sentiment analysis tools for ensemble approach.

    Why Multiple Analyzers:
        - VADER: Fast, rule-based, optimized for social media/reviews
        - TextBlob: Pattern-based, provides subjectivity scores
        - Transformer: Deep learning, most accurate for complex patterns
        - Ensemble of all three provides robust results

    Returns:
        dict: Initialized analyzers (None if unavailable)
    """
    analyzers = {}

    # VADER Initialization
    try:
        import nltk
        nltk.download('vader_lexicon', quiet=True)
        from nltk.sentiment.vader import SentimentIntensityAnalyzer
        analyzers['vader'] = SentimentIntensityAnalyzer()
        print("✅ VADER sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ VADER not available: {e}")
        analyzers['vader'] = None

    # TextBlob Initialization
    try:
        from textblob import TextBlob
        analyzers['textblob'] = TextBlob
        print("✅ TextBlob sentiment analyzer loaded")
    except Exception as e:
        print(f"⚠️ TextBlob not available: {e}")
        analyzers['textblob'] = None

    # Transformer Initialization (Optional)
    try:
        from transformers import pipeline
        analyzers['transformer'] = pipeline(
            "sentiment-analysis",
            model="distilbert-base-uncased-finetuned-sst-2-english",
            device=-1
        )
        print("✅ Transformer sentiment analyzer loaded")
    except Exception as e:
        print(f"ℹ️ Transformer not available (optional): {e}")
        analyzers['transformer'] = None

    return analyzers


def get_sentiment_analyzers():
    """Get or initialize sentiment analyzers (singleton pattern)."""
    global SENTIMENT_ANALYZERS
    if SENTIMENT_ANALYZERS is None:
        SENTIMENT_ANALYZERS = initialize_sentiment_analyzers()
    return SENTIMENT_ANALYZERS


# ============================================
# SECTION 4: UNIVERSAL RESTAURANT VOCABULARY
# ============================================
"""
Universal Vocabulary Design:
    These word lists are designed to work across ALL restaurant types.
    They cover common aspects, opinions, and patterns found in reviews
    regardless of cuisine or service model.
"""


def get_common_english_names():
    """
    Common first names to filter from opinions.

    Rationale:
        Reviews often mention server names: "Sarah was great"
        Names alone don't provide sentiment - the opinion does.

    Returns:
        set: Common English first names (lowercase)
    """
    male_names = {
        'james', 'john', 'robert', 'michael', 'william', 'david', 'richard',
        'joseph', 'thomas', 'charles', 'christopher', 'daniel', 'matthew',
        'anthony', 'mark', 'donald', 'steven', 'paul', 'andrew', 'joshua',
        'kenneth', 'kevin', 'brian', 'george', 'timothy', 'ronald', 'edward',
        'jason', 'jeffrey', 'ryan', 'jacob', 'gary', 'nicholas', 'eric',
        'jonathan', 'stephen', 'larry', 'justin', 'scott', 'brandon', 'benjamin',
        'samuel', 'raymond', 'gregory', 'frank', 'alexander', 'patrick', 'jack',
        'joel', 'paul', 'morgan', 'jordan', 'chris', 'matt', 'andy', 'ray',
        'nick', 'greg', 'mike', 'steve', 'tom', 'bob', 'jim', 'dan'
    }

    female_names = {
        'mary', 'patricia', 'jennifer', 'linda', 'barbara', 'elizabeth', 'susan',
        'jessica', 'sarah', 'karen', 'lisa', 'nancy', 'betty', 'margaret', 'sandra',
        'ashley', 'kimberly', 'emily', 'donna', 'michelle', 'dorothy', 'carol',
        'amanda', 'melissa', 'deborah', 'stephanie', 'rebecca', 'sharon', 'laura',
        'sara', 'fatima', 'kaitlin', 'kaitlyn', 'natalie', 'melanie', 'christina',
        'veronica', 'morgan', 'jordan', 'gabby', 'tina', 'cameron'
    }

    return male_names | female_names


def get_positive_sentiment_words():
    """
    Universal positive sentiment words for restaurant reviews.

    Categories:
        - General quality descriptors
        - Food-specific positive terms
        - Service-related positive terms
        - Atmosphere/environment terms
        - Value-related terms

    Returns:
        set: Positive sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'good', 'great', 'excellent', 'amazing', 'wonderful', 'fantastic',
        'outstanding', 'superb', 'perfect', 'incredible', 'awesome',
        'terrific', 'fabulous', 'marvelous', 'exceptional', 'best',
        'impressive', 'remarkable', 'magnificent', 'splendid', 'brilliant',
        'stellar', 'phenomenal', 'spectacular', 'extraordinary', 'superb',

        # ===== FOOD-SPECIFIC (Universal across cuisines) =====
        'delicious', 'tasty', 'yummy', 'flavorful', 'savory', 'fresh',
        'tender', 'juicy', 'crispy', 'creamy', 'rich', 'light',
        'authentic', 'homemade', 'seasoned', 'aromatic', 'scrumptious',
        'divine', 'heavenly', 'mouthwatering', 'succulent', 'zesty',
        'perfectly cooked', 'well-seasoned', 'flavorsome',

        # ===== SERVICE-RELATED =====
        'friendly', 'helpful', 'attentive', 'professional', 'courteous',
        'prompt', 'efficient', 'welcoming', 'accommodating', 'polite',
        'responsive', 'knowledgeable', 'patient', 'thorough', 'dedicated',
        'personable', 'warm', 'gracious', 'sweet', 'caring',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'clean', 'comfortable', 'cozy', 'spacious', 'nice', 'lovely',
        'beautiful', 'charming', 'pleasant', 'relaxing', 'quiet',
        'modern', 'elegant', 'stylish', 'inviting', 'cute',
        'trendy', 'vibrant', 'lively', 'romantic', 'intimate',

        # ===== VALUE-RELATED =====
        'reasonable', 'affordable', 'worth', 'value', 'cheap', 'bargain',
        'fair', 'inexpensive', 'economical', 'generous'
    }


def get_negative_sentiment_words():
    """
    Universal negative sentiment words for restaurant reviews.

    Returns:
        set: Negative sentiment words (lowercase)
    """
    return {
        # ===== GENERAL QUALITY =====
        'bad', 'terrible', 'awful', 'horrible', 'poor', 'disappointing',
        'mediocre', 'pathetic', 'lousy', 'nasty', 'gross', 'worst',
        'inferior', 'subpar', 'unacceptable', 'dreadful', 'appalling',
        'atrocious', 'abysmal', 'horrendous', 'disgusting',

        # ===== FOOD-SPECIFIC =====
        'bland', 'tasteless', 'soggy', 'dry', 'burnt', 'overcooked',
        'undercooked', 'raw', 'fatty', 'greasy', 'salty', 'stale',
        'cold', 'lukewarm', 'frozen', 'rubbery', 'tough', 'chewy',
        'flavorless', 'mushy', 'bitter', 'sour', 'spoiled',
        'unseasoned', 'unappetizing', 'oily', 'oversalted',

        # ===== SERVICE-RELATED =====
        'rude', 'slow', 'unfriendly', 'unhelpful', 'inattentive',
        'unprofessional', 'careless', 'dismissive', 'indifferent',
        'incompetent', 'negligent', 'impolite', 'disrespectful',
        'ignored', 'forgotten', 'invisible',

        # ===== ATMOSPHERE/ENVIRONMENT =====
        'dirty', 'messy', 'noisy', 'crowded', 'cramped', 'uncomfortable',
        'dark', 'smelly', 'hot', 'stuffy', 'outdated', 'dingy',
        'filthy', 'cluttered', 'rundown', 'shabby', 'sticky',

        # ===== VALUE-RELATED =====
        'expensive', 'overpriced', 'costly', 'pricey', 'ripoff',
        'unreasonable', 'exorbitant', 'small portions'
    }


def get_action_verbs():
    """
    Action verbs that should NOT be treated as opinions.

    Rationale:
        "I ordered the pasta" - "ordered" is action, not opinion
        "The waiter served us" - "served" is action, not opinion

    Returns:
        set: Action verbs in various forms
    """
    return {
        'order', 'ordered', 'ordering', 'orders',
        'ask', 'asked', 'asking', 'asks',
        'request', 'requested', 'requesting', 'requests',
        'come', 'came', 'coming', 'comes',
        'go', 'went', 'going', 'goes', 'gone',
        'arrive', 'arrived', 'arriving', 'arrives',
        'leave', 'left', 'leaving', 'leaves',
        'walk', 'walked', 'walking', 'walks',
        'enter', 'entered', 'entering', 'enters',
        'visit', 'visited', 'visiting', 'visits',
        'get', 'got', 'getting', 'gets',
        'take', 'took', 'taking', 'takes', 'taken',
        'bring', 'brought', 'bringing', 'brings',
        'receive', 'received', 'receiving', 'receives',
        'make', 'made', 'making', 'makes',
        'do', 'did', 'doing', 'does', 'done',
        'prepare', 'prepared', 'preparing', 'prepares',
        'cook', 'cooked', 'cooking', 'cooks',
        'say', 'said', 'saying', 'says',
        'tell', 'told', 'telling', 'tells',
        'call', 'called', 'calling', 'calls',
        'serve', 'served', 'serving', 'serves',
        'show', 'showed', 'showing', 'shows', 'shown',
        'give', 'gave', 'giving', 'gives', 'given',
        'seat', 'seated', 'seating', 'seats',
        'try', 'tried', 'trying', 'tries',
        'want', 'wanted', 'wanting', 'wants',
        'need', 'needed', 'needing', 'needs',
        'hope', 'hoped', 'hoping', 'hopes',
        'expect', 'expected', 'expecting', 'expects',
        'enjoy', 'enjoyed', 'enjoying', 'enjoys',
        'pay', 'paid', 'paying', 'pays',
        'wait', 'waited', 'waiting', 'waits',
        'sit', 'sat', 'sitting', 'sits',
        'eat', 'ate', 'eating', 'eats', 'eaten',
        'drink', 'drank', 'drinking', 'drinks', 'drunk'
    }


def get_function_words():
    """
    Function words that carry no sentiment meaning.

    Returns:
        set: Articles, prepositions, pronouns, etc.
    """
    return {
        'a', 'an', 'the', 'this', 'that', 'these', 'those',
        'i', 'me', 'my', 'mine', 'myself',
        'you', 'your', 'yours', 'yourself',
        'he', 'him', 'his', 'himself',
        'she', 'her', 'hers', 'herself',
        'it', 'its', 'itself',
        'we', 'us', 'our', 'ours', 'ourselves',
        'they', 'them', 'their', 'theirs', 'themselves',
        'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by',
        'from', 'about', 'into', 'through', 'during', 'before',
        'after', 'above', 'below', 'between', 'under', 'over',
        'and', 'or', 'but', 'so', 'yet', 'nor',
        'if', 'when', 'where', 'while', 'as', 'because', 'since',
        'is', 'are', 'was', 'were', 'be', 'been', 'being',
        'has', 'have', 'had', 'having',
        'do', 'does', 'did',
        'will', 'would', 'shall', 'should',
        'can', 'could', 'may', 'might', 'must',
        'here', 'there', 'some', 'any', 'no', 'every'
    }


def get_intensifiers():
    """
    Intensifier words that modify sentiment strength.

    Returns:
        set: Intensifier words
    """
    return {
        'very', 'extremely', 'incredibly', 'amazingly', 'exceptionally',
        'remarkably', 'absolutely', 'totally', 'completely', 'utterly',
        'entirely', 'thoroughly', 'perfectly', 'highly', 'deeply',
        'truly', 'really', 'genuinely', 'seriously',
        'quite', 'rather', 'fairly', 'pretty', 'somewhat',
        'reasonably', 'moderately', 'relatively',
        'slightly', 'a bit', 'a little', 'mildly', 'barely',
        'too', 'overly', 'excessively',
        'so', 'such', 'super', 'especially', 'particularly'
    }


# ============================================
# SECTION 5: IDIOM DETECTION FUNCTIONS
# ============================================

def check_for_idiom(text):
    """
    Check if text contains any known idiom.

    PRIORITY: This check runs BEFORE negation detection.

    Args:
        text (str): Text to check

    Returns:
        tuple: (is_idiom: bool, sentiment: str or None, matched_idiom: str or None)

    Example:
        >>> check_for_idiom("The food did not disappoint")
        (True, "Positive", "did not disappoint")

        >>> check_for_idiom("The food was cold")
        (False, None, None)
    """
    text_lower = text.lower()

    # Check positive idioms
    positive_idioms = get_positive_idioms()
    for idiom, sentiment in positive_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    # Check negative idioms
    negative_idioms = get_negative_idioms()
    for idiom, sentiment in negative_idioms.items():
        if idiom in text_lower:
            return True, sentiment, idiom

    return False, None, None


# ============================================
# SECTION 6: ADJECTIVE/ADVERB DETECTION
# ============================================

def is_adjective_or_adverb(word):
    """
    Determine if a word is an adjective or adverb.

    Uses two approaches:
        1. Dictionary lookup (known sentiment words)
        2. Morphological rules (suffix patterns)

    Args:
        word (str): Word to check

    Returns:
        tuple: (is_adj_or_adv: bool, word_type: str)
    """
    word_lower = word.lower().strip('.,!?;:\'"')

    if not word_lower:
        return False, 'none'

    # Check sentiment dictionaries
    if word_lower in get_positive_sentiment_words():
        return True, 'positive_adj'

    if word_lower in get_negative_sentiment_words():
        return True, 'negative_adj'

    if word_lower in get_intensifiers():
        return True, 'intensifier'

    # Morphological rules
    adj_suffixes = ['ful', 'less', 'ous', 'ive', 'able', 'ible',
                    'al', 'ic', 'ish', 'ent', 'ant', 'ory', 'ary']

    for suffix in adj_suffixes:
        if word_lower.endswith(suffix) and len(word_lower) > len(suffix) + 2:
            return True, 'suffix_adj'

    # Adverb suffix
    if word_lower.endswith('ly') and len(word_lower) > 4:
        non_adverb_ly = {'family', 'only', 'early', 'likely', 'friendly', 'lonely'}
        if word_lower not in non_adverb_ly:
            return True, 'suffix_adv'

    # Past participles as adjectives
    sentiment_participles = {
        'disappointed', 'satisfied', 'pleased', 'impressed', 'amazed',
        'surprised', 'disgusted', 'frustrated', 'annoyed', 'delighted',
        'thrilled', 'excited', 'bored', 'tired', 'exhausted',
        'overwhelmed', 'underwhelmed', 'overpriced'
    }
    if word_lower in sentiment_participles:
        return True, 'suffix_adj'

    # Present participles as adjectives
    sentiment_ing = {
        'amazing', 'disappointing', 'disgusting', 'interesting', 'boring',
        'exciting', 'frustrating', 'annoying', 'satisfying', 'refreshing',
        'relaxing', 'welcoming', 'inviting', 'appealing', 'appalling'
    }
    if word_lower in sentiment_ing:
        return True, 'suffix_adj'

    return False, 'none'


# ============================================
# SECTION 7: NEGATION DETECTION
# ============================================

def detect_negation_in_opinion(opinion):
    """
    Detect negation in opinion text.

    IMPORTANT: Check idioms FIRST before calling this function.

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (has_negation: bool, negation_type: str or None)

    Example:
        >>> detect_negation_in_opinion("not good")
        (True, 'direct')

        >>> detect_negation_in_opinion("wasn't fresh")
        (True, 'contraction')
    """
    opinion_lower = opinion.lower()

    # SAFETY CHECK: If this is an idiom, don't treat as simple negation
    is_idiom, _, _ = check_for_idiom(opinion_lower)
    if is_idiom:
        return False, 'idiom_detected'

    # Direct negation words
    direct_negations = [
        'not ', "n't ", 'no ', 'never ', 'none ', 'nothing ',
        'neither ', 'nobody ', 'nowhere ', 'cannot '
    ]

    for neg in direct_negations:
        if neg in opinion_lower or opinion_lower.startswith(neg.strip()):
            return True, 'direct'

    # Contracted negations
    contracted_negations = [
        "didn't", "wasn't", "weren't", "isn't", "aren't",
        "don't", "doesn't", "won't", "wouldn't", "couldn't",
        "shouldn't", "can't", "haven't", "hasn't", "hadn't"
    ]

    for neg in contracted_negations:
        if neg in opinion_lower:
            return True, 'contraction'

    return False, None


# ============================================
# SECTION 8: BOUNDARY DETECTION
# ============================================

def find_sentence_boundaries(tokens, aspect_end_idx):
    """Find sentence boundaries containing the aspect."""
    sentence_enders = {'.', '!', '?'}

    sentence_start = 0
    for i in range(aspect_end_idx, -1, -1):
        if tokens[i] in sentence_enders:
            sentence_start = i + 1
            break

    sentence_end = len(tokens)
    for i in range(aspect_end_idx + 1, len(tokens)):
        if tokens[i] in sentence_enders:
            sentence_end = i
            break

    return sentence_start, sentence_end


def find_previous_aspect_end(current_start_idx, all_aspect_positions):
    """Find end position of previous aspect."""
    prev_end = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                end = positions[0][-1]
            else:
                end = positions[-1] if len(positions) > 1 else positions[0]

            if end < current_start_idx:
                if prev_end is None or end > prev_end:
                    prev_end = end

    return prev_end


def find_next_aspect_position(current_end_idx, all_aspect_positions):
    """Find start position of next aspect."""
    next_start = None

    for positions in all_aspect_positions:
        if isinstance(positions, list) and len(positions) > 0:
            if isinstance(positions[0], list):
                start = positions[0][0]
            else:
                start = positions[0]

            if start > current_end_idx:
                if next_start is None or start < next_start:
                    next_start = start

    return next_start


# ============================================
# SECTION 9: BE-VERB DETECTION
# ============================================

def detect_be_verb(tokens, start_idx):
    """Detect be-verb at given position."""
    if start_idx >= len(tokens):
        return False, 0, False

    word = tokens[start_idx].lower().strip("'")

    be_verbs = {'is', 'was', 'were', 'are', 'be', 'been', "'s", "s"}
    if word in be_verbs:
        return True, 1, False

    negated_be_verbs = {
        "isn't", "isnt", "wasn't", "wasnt",
        "weren't", "werent", "aren't", "arent"
    }
    if word in negated_be_verbs:
        return True, 1, True

    return False, 0, False


# ============================================
# SECTION 10: OPINION VALIDATION
# ============================================

def validate_opinion_completeness(opinion):
    """
    Validate that opinion is complete and meaningful.

    Rules:
        1. Cannot start with conjunction (incomplete phrase)
        2. Single intensifier alone is invalid
        3. Cannot be factual statement (reporting, not opinion)
        4. Must contain sentiment signal

    Args:
        opinion (str): Opinion text

    Returns:
        tuple: (is_valid: bool, reason: str or None)

    Examples:
        >>> validate_opinion_completeness("and was delicious")
        (False, "starts_with_conjunction")

        >>> validate_opinion_completeness("she said yes")
        (False, "factual_statement")

        >>> validate_opinion_completeness("absolutely delicious")
        (True, None)
    """
    if not opinion or len(opinion.strip()) < 2:
        return False, 'empty'

    words = opinion.lower().strip().split()
    if not words:
        return False, 'empty'

    # Rule 1: Cannot start with conjunction
    conjunctions = {'and', 'or', 'but', 'so', 'yet', 'nor', 'for'}
    if words[0] in conjunctions:
        return False, 'starts_with_conjunction'

    # Rule 2: Single intensifier is invalid
    if len(words) == 1:
        intensifiers = get_intensifiers()
        if words[0] in intensifiers:
            return False, 'only_intensifier'

        function_words = get_function_words()
        if words[0] in function_words:
            return False, 'only_function_word'

    # Rule 3: Cannot be factual statement
    factual_patterns = [
        r'^(he|she|they|it|we|i)\s+(said|told|asked|mentioned|replied)',
        r'^(was|were|is|are)\s+(out of|available|unavailable)',
        r'^if\s+(it|they|she|he|we)\s+(was|were|is|are)',
        r'^that\s+(it|they)',
        r'^\d+\s*/',
        r'^with\s+(a\s+)?side',
    ]

    opinion_text = opinion.lower()
    for pattern in factual_patterns:
        if re.search(pattern, opinion_text):
            return False, 'factual_statement'

    # Rule 4: Must contain sentiment signal (for short opinions)
    if len(words) <= 4:
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        all_sentiment = positive_words | negative_words

        has_sentiment = False
        for word in words:
            clean_word = word.strip('.,!?;:\'"')
            if clean_word in all_sentiment:
                has_sentiment = True
                break
            is_adj, _ = is_adjective_or_adverb(clean_word)
            if is_adj:
                has_sentiment = True
                break

        if not has_sentiment:
            # Exception: Idioms should pass
            is_idiom, _, _ = check_for_idiom(opinion)
            if not is_idiom:
                return False, 'no_sentiment_signal'

    return True, None


def is_valid_opinion(opinion):
    """
    Check if extracted text is a valid opinion.

    Args:
        opinion (str): Extracted opinion text

    Returns:
        bool: True if valid opinion
    """
    if not opinion or len(opinion.strip()) < 2:
        return False

    # Use completeness validation
    is_complete, _ = validate_opinion_completeness(opinion)
    if not is_complete:
        return False

    opinion_lower = opinion.lower().strip()
    words = opinion_lower.split()

    if not words:
        return False

    action_verbs = get_action_verbs()
    function_words = get_function_words()
    common_names = get_common_english_names()

    # Single word validation
    if len(words) == 1:
        clean_word = words[0].strip('.,!?;:\'"')
        if clean_word in action_verbs:
            return False
        if clean_word in function_words:
            return False
        if clean_word in common_names:
            return False

    # Invalid patterns
    invalid_patterns = [
        r"^'s\s+",
        r"^\d+\s*/",
        r"^or\s+maybe\s+both$",
        r"^going\s+forward$",
        r"^n\s+cheese",  # Truncated "mac n cheese"
    ]

    for pattern in invalid_patterns:
        if re.search(pattern, opinion_lower):
            return False

    return True


# ============================================
# SECTION 11: ASPECT VALIDATION
# ============================================

def is_valid_aspect(aspect):
    """Check if extracted text is a valid aspect."""
    if not aspect:
        return False

    cleaned = aspect.strip().lower()

    if not cleaned or cleaned == "-":
        return False

    if all(c in '.,!?;:-_\'"' for c in cleaned):
        return False

    # Invalid single words (prepositions, conjunctions, etc.)
    invalid_single = {'of', 'to', 'for', 'with', 'at', 'in', 'on', 'by',
                      'and', 'or', 'but', 'n', 'the', 'a', 'an'}
    if cleaned in invalid_single:
        return False

    invalid_aspects = {
        # Verbs
        'wait', 'waited', 'waiting', 'order', 'ordered', 'ordering',
        'serve', 'served', 'serving', 'ask', 'asked', 'asking',
        # Pronouns
        'i', 'me', 'my', 'you', 'your', 'he', 'him', 'his',
        'she', 'her', 'it', 'its', 'we', 'us', 'our', 'they', 'them',
        # Time units
        'min', 'mins', 'minute', 'minutes', 'hour', 'hours',
        # Adjectives (opinions, not aspects)
        'good', 'bad', 'great', 'nice', 'poor',
        # Generic
        'thing', 'things', 'stuff', 'way', 'time', 'place'
    }

    if cleaned in invalid_aspects:
        return False

    return True


def refine_aspect_term(aspect):
    """Clean and standardize aspect term."""
    if not aspect:
        return ""

    clean_aspect = aspect.strip()
    clean_aspect = re.sub(r'^(and|or|but)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^(the|a|an|this|that|my|our)\s+', '', clean_aspect, flags=re.IGNORECASE)
    clean_aspect = re.sub(r'^[-:;,.\'"]+\s*', '', clean_aspect)
    clean_aspect = re.sub(r'[-:;,.\'"]+$', '', clean_aspect)

    return clean_aspect.strip()


# ============================================
# SECTION 12: OPINION EXTRACTION (ENHANCED)
# ============================================

def check_sentence_completeness(opinion_words, tokens, current_end_idx, sentence_end):
    """
    Check if opinion forms a complete sentence/phrase.
    If incomplete, extend to next sentence boundary.

    ENHANCED: Better detection of incomplete phrases, especially for negative sentiments.

    Args:
        opinion_words: Current list of opinion words
        tokens: All tokens in the review
        current_end_idx: Current end position
        sentence_end: Maximum sentence boundary

    Returns:
        list: Extended opinion words if needed
    """
    if not opinion_words:
        return opinion_words

    # Join current words to check completeness
    opinion_text = ' '.join(opinion_words).lower().strip()

    # Minimum word count for meaningful opinion
    MIN_WORDS_FOR_COMPLETE = 3

    # If very short (1-2 words), always try to extend
    if len(opinion_words) < MIN_WORDS_FOR_COMPLETE:
        extended_words = list(opinion_words)
        for i in range(current_end_idx, min(sentence_end, current_end_idx + 10)):
            if i < len(tokens):
                word = tokens[i]
                if word in {'.', '!', '?'}:
                    break
                extended_words.append(word)
        return extended_words

    # Patterns indicating incomplete sentences
    incomplete_patterns = [
        r"(wasn|isn|aren|weren|didn|doesn|hasn|haven|hadn|wouldn|couldn|shouldn|won|can|don)\s*'\s*t\s*$",  # ends with contraction
        r"\s+(was|is|are|were|did|does|has|have|had|would|could|should|will|can|do)\s*$",  # ends with auxiliary verb
        r"\s+(he|she|they|it|we|i)\s+(said|told|asked)\s+.{0,20}$",  # incomplete quote
        r",\s*$",  # ends with comma
        r"\s+that\s*$",  # ends with "that"
        r"\s+which\s*$",  # ends with "which"
        r"\s+who\s*$",  # ends with "who"
        r"\s+when\s*$",  # ends with "when"
        r"\s+where\s*$",  # ends with "where"
        r"\s+if\s*$",  # ends with "if"
        r"\s+because\s*$",  # ends with "because"
        r"\s+although\s*$",  # ends with "although"
        r"\s+their\s*$",  # ends with possessive
        r"\s+his\s*$",
        r"\s+her\s*$",
        r"\s+our\s*$",
        r"\s+my\s*$",
        r"\s+but\s*$",  # ends with "but" - important for contrast!
        r"\s+and\s*$",  # ends with "and"
        r"\s+just\s*$",  # ends with "just"
        r"\s+very\s*$",  # ends with intensifier
        r"\s+really\s*$",
        r"\s+so\s*$",
    ]

    is_incomplete = False
    for pattern in incomplete_patterns:
        if re.search(pattern, opinion_text):
            is_incomplete = True
            break

    # Also check if last word is problematic
    if opinion_words:
        last_word = opinion_words[-1].lower().strip('.,!?;:\'"')
        incomplete_endings = {
            'was', 'is', 'are', 'were', 'did', 'does', 'has', 'have', 'had',
            'would', 'could', 'should', 'will', 'can', 'do', 'said', 'told',
            'their', 'his', 'her', 'our', 'my', 'your', 'its', 'the', 'a', 'an',
            "wasn", "isn", "aren", "weren", "didn", "doesn", "hasn",
            "haven", "hadn", "wouldn", "couldn", "shouldn", "won", "don",
            'but', 'and', 'or', 'just', 'very', 'really', 'so', 'she', 'he',
            'we', 'they', 'asking', 'saying', 'telling', 'for', 'with', 'to'
        }
        if last_word in incomplete_endings:
            is_incomplete = True

    # ENHANCED: Check if ends with a conjunction followed by beginning of clause
    # e.g., "nice but" should continue to get what comes after "but"
    if len(opinion_words) >= 2:
        second_last = opinion_words[-2].lower().strip('.,!?;:\'"') if len(opinion_words) >= 2 else ''
        last = opinion_words[-1].lower().strip('.,!?;:\'"')
        # "nice but" pattern - important for contrast
        if second_last in {'but', 'and', 'or', 'yet'} or last in {'but', 'and', 'or', 'yet'}:
            is_incomplete = True

    # If incomplete, extend to next sentence boundary
    if is_incomplete and current_end_idx < sentence_end:
        extended_words = list(opinion_words)
        for i in range(current_end_idx, min(sentence_end, current_end_idx + 15)):
            if i < len(tokens):
                word = tokens[i]
                if word in {'.', '!', '?'}:
                    break
                extended_words.append(word)
        return extended_words

    return opinion_words


def extract_complete_opinion_with_boundary(tokens, aspect_positions, aspect_text, all_aspect_positions):
    """
    Extract opinion associated with aspect, respecting boundaries.
    Enhanced to ensure complete sentences.

    Extraction Patterns (Priority Order):
        1. Pre-modifier: "delicious food" → opinion before aspect
        2. Be-adjective: "food is delicious" → opinion after be-verb
        3. Verb phrase: "food tastes great" → opinion in verb phrase
        4. Context search: Fallback for complex sentences

    Args:
        tokens: Tokenized review
        aspect_positions: Position indices of aspect
        aspect_text: Aspect text (for reference)
        all_aspect_positions: All aspect positions (for boundaries)

    Returns:
        tuple: (opinion_phrase: str, pattern_type: str)
    """
    if isinstance(aspect_positions, list) and len(aspect_positions) > 0:
        if isinstance(aspect_positions[0], list):
            start_idx = aspect_positions[0][0]
            end_idx = aspect_positions[0][-1]
        else:
            start_idx = aspect_positions[0]
            end_idx = aspect_positions[-1] if len(aspect_positions) > 1 else aspect_positions[0]
    else:
        return "", "none"

    sentence_start, sentence_end = find_sentence_boundaries(tokens, end_idx)
    prev_aspect_end = find_previous_aspect_end(start_idx, all_aspect_positions)
    next_aspect_start = find_next_aspect_position(end_idx, all_aspect_positions)

    # Relaxed boundary - allow extending to sentence end for completeness
    original_sentence_end = sentence_end
    if next_aspect_start is not None:
        max_search_end = max(next_aspect_start - 1, end_idx + 1)
        sentence_end = min(sentence_end, max_search_end)

    opinion_words = []
    pattern_type = "unknown"

    # PATTERN 1: Pre-modifier
    pre_modifiers = []
    search_start = max(sentence_start, start_idx - 4)

    if prev_aspect_end is not None:
        search_start = max(search_start, prev_aspect_end + 1)

    for i in range(search_start, start_idx):
        word = tokens[i]
        is_modifier, _ = is_adjective_or_adverb(word)

        if is_modifier:
            pre_modifiers.append(word)
        elif word.lower() in {',', 'and', 'with', 'or', 'but'}:
            pre_modifiers = []

    if pre_modifiers:
        opinion_words.extend(pre_modifiers)
        pattern_type = "pre_modifier"

    # PATTERN 2: Be-verb + Adjective
    next_idx = end_idx + 1
    found_be, skip_count, is_negative = detect_be_verb(tokens, next_idx)

    if found_be and not pre_modifiers:
        opinion_start = next_idx + skip_count
        opinion_end = min(opinion_start + 15, sentence_end)

        post_modifiers = []
        if is_negative:
            post_modifiers.append("not")

        # ENHANCED: Skip punctuation tokens at the start (including &)
        skip_punct = {',', ';', ':', '(', ')', '-', '"', "'", '&', '/', '\\'}

        for i in range(opinion_start, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            # Skip leading punctuation tokens
            if not post_modifiers and word.strip() in skip_punct:
                continue
            post_modifiers.append(word)

        if post_modifiers:
            # Check for completeness and extend if needed
            post_modifiers = check_sentence_completeness(
                post_modifiers, tokens, opinion_end, original_sentence_end
            )
            opinion_words.extend(post_modifiers)
            pattern_type = "be_adjective"

    # PATTERN 3: Verb Phrase
    elif next_idx < len(tokens) and not opinion_words:
        opinion_end = min(next_idx + 8, sentence_end)

        # ENHANCED: Skip punctuation tokens (including &)
        skip_punct = {',', ';', ':', '(', ')', '-', '"', "'", '&', '/', '\\'}

        temp_words = []
        for i in range(next_idx, opinion_end):
            word = tokens[i]
            if word in {'.', '!', '?', ';'}:
                break
            # Skip leading punctuation tokens
            if not temp_words and word.strip() in skip_punct:
                continue
            temp_words.append(word)

        if temp_words:
            # Check for completeness and extend if needed
            temp_words = check_sentence_completeness(
                temp_words, tokens, opinion_end, original_sentence_end
            )
            opinion_words.extend(temp_words)
            pattern_type = "verb_phrase"

    # PATTERN 4: Context Search
    if not opinion_words:
        search_start = end_idx + 1
        search_end = min(end_idx + 12, sentence_end)

        # ENHANCED: Skip punctuation tokens (including &)
        skip_punct = {',', ';', ':', '(', ')', '-', '"', "'", '&', '/', '\\'}

        for i in range(search_start, search_end):
            word = tokens[i]
            is_modifier, _ = is_adjective_or_adverb(word)

            if is_modifier:
                context_start = max(search_start, i - 2)
                context_end = min(i + 5, search_end)

                for j in range(context_start, context_end):
                    token_word = tokens[j]
                    if token_word not in {'.', '!', '?', ';'}:
                        # Skip leading punctuation tokens
                        if not opinion_words and token_word.strip() in skip_punct:
                            continue
                        opinion_words.append(token_word)

                # Check for completeness
                opinion_words = check_sentence_completeness(
                    opinion_words, tokens, context_end, original_sentence_end
                )
                pattern_type = "context_search"
                break

    opinion_phrase = ' '.join(opinion_words).strip()
    return opinion_phrase, pattern_type


# ============================================
# SECTION 13: OPINION FORMATTING (ENHANCED)
# ============================================

def format_opinion_for_display(opinion):
    """
    Format and clean extracted opinion for display.
    Enhanced to remove leading punctuation and ensure completeness.

    Processing Steps:
        1. Remove leading punctuation (comma, parenthesis, &, etc.)
        2. Remove leading conjunctions ("and", "or", "but")
        3. Remove leading function words
        4. Handle hyphenation
        5. Truncate at cutoff words
        6. Remove trailing garbage
        7. Enforce length limit (extended to 20 words)

    Args:
        opinion (str): Raw extracted opinion

    Returns:
        str: Cleaned and formatted opinion
    """
    if not opinion:
        return ""

    # ENHANCED: Remove leading punctuation with spaces (handles " , " or ", " or " ," or "&")
    # This pattern matches any combination of spaces and punctuation at the start
    opinion = re.sub(r'^[\s,;:\)\(\-\'\"\.\!\?&/\\]+', '', opinion)

    # Also handle cases where punctuation has spaces around it: " , just" -> "just"
    opinion = re.sub(r'^\s*[,;:\)\(\-&/\\]+\s*', '', opinion)

    # Remove leading conjunctions
    opinion = re.sub(r'^(and|or|but|so)\s+', '', opinion, flags=re.IGNORECASE)

    # Fix punctuation spacing
    opinion = re.sub(r'\s*,\s*,', ', ', opinion)
    opinion = re.sub(r'\s*-\s*', ' - ', opinion)

    # ENHANCED: Fix tokenization issues with contractions
    opinion = re.sub(r"\s+'\s*t\b", "'t", opinion)  # wasn ' t -> wasn't
    opinion = re.sub(r"\s+'\s*s\b", "'s", opinion)  # he ' s -> he's
    opinion = re.sub(r"\s+'\s*re\b", "'re", opinion)  # they ' re -> they're
    opinion = re.sub(r"\s+'\s*ve\b", "'ve", opinion)  # we ' ve -> we've
    opinion = re.sub(r"\s+'\s*ll\b", "'ll", opinion)  # we ' ll -> we'll
    opinion = re.sub(r"\s+'\s*d\b", "'d", opinion)  # he ' d -> he'd
    opinion = re.sub(r"\s+'\s*m\b", "'m", opinion)  # I ' m -> I'm

    # ENHANCED: Handle standalone ' s at the beginning -> it's
    opinion = re.sub(r"^'\s*s\s+", "it's ", opinion)  # ' s very nice -> it's very nice
    opinion = re.sub(r"^\s*'\s*s\s+", "it's ", opinion)  # ' s very nice -> it's very nice

    words = opinion.split()
    if not words:
        return ""

    # ENHANCED: First remove any leading punctuation tokens (with strip to handle spaces)
    # Including & symbol
    leading_punct = {',', ';', ':', ')', '(', '-', '.', "'", '"', '!', '?', '&', '/', '\\'}
    while words and words[0].strip() in leading_punct:
        words.pop(0)

    if not words:
        return ""

    # ENHANCED: Handle ' s or 's at beginning of words list -> it's
    # Case 1: First word is literally "'s" or "' s"
    if words and words[0].lower() in {"'s", "' s", "'s"}:
        words[0] = "it's"
    # Case 2: First word is just "s" (apostrophe was removed) followed by adjective/adverb
    elif words and words[0].lower() == "s" and len(words) > 1:
        # Check if second word looks like an adjective (very, really, quite, nice, good, etc.)
        likely_continuation = {'very', 'really', 'quite', 'so', 'pretty', 'nice', 'good',
                               'great', 'bad', 'terrible', 'delicious', 'amazing', 'okay',
                               'fine', 'alright', 'wonderful', 'awful', 'fantastic'}
        if words[1].lower() in likely_continuation or words[1].lower().endswith('ly'):
            words[0] = "it's"
    # Case 3: First two words are "'" and "s"
    elif len(words) >= 2 and words[0] == "'" and words[1].lower() == "s":
        words = ["it's"] + words[2:]

    # Remove leading fillers
    leading_fillers = {
        'a', 'an', 'the', 'this', 'that', 'also', 'too',
        'has', 'have', 'had', 'is', 'are', 'was', 'were',
        'it', 'itself', 'they', 'themselves', 'we', 'i',
        'and', 'or', 'but', 'so'
    }

    while words and words[0].lower().strip('.,;:') in leading_fillers:
        words.pop(0)

    # Check again for punctuation after removing fillers
    while words and words[0].strip() in leading_punct:
        words.pop(0)

    if not words:
        return ""

    # Truncate at cutoff words (but allow more context)
    cutoff_words = {
        'making', 'causing', 'forcing', 'leaving',
        'unless', 'except', 'despite', 'although',
    }

    truncated = []
    for i, word in enumerate(words):
        word_lower = word.lower().strip(',.')
        if word_lower in cutoff_words and i >= 4:  # Allow more words before cutoff
            break
        truncated.append(word)

    words = truncated

    # Remove trailing garbage
    trailing_garbage = {
        'and', 'or', 'with', 'for', 'to', 'at', 'in', 'on',
        'a', 'an', 'the', ',', '-',
        'is', 'are', 'was', 'were',
        'it', 'i', 'we', 'they'
    }

    while words:
        last_word = words[-1].lower().strip('.,!?;:')
        if last_word in trailing_garbage or words[-1].endswith('-'):
            words.pop()
        else:
            break

    result = ' '.join(words)

    # ENHANCED: Extended length limit to 20 words for completeness
    words = result.split()
    if len(words) > 20:
        words = words[:20]
        result = ' '.join(words)

    # Final cleanup
    result = result.strip().rstrip('.,;-\'"(')
    result = re.sub(r'^[\s,;:\)\(\-\'"\.]+', '', result)  # Remove leading punctuation again
    result = re.sub(r'^(and|or|but|so)\s+', '', result, flags=re.IGNORECASE)

    return result


# ============================================
# SECTION 14: SENTIMENT ANALYSIS
# ============================================

def analyze_sentiment_scores(text):
    """
    Analyze text sentiment using multiple tools.
    ENHANCED: Idiom-aware scoring - positive idioms get positive scores.

    Returns ensemble score combining:
        - VADER (weight: 0.4)
        - TextBlob (weight: 0.3)
        - Transformer (weight: 0.3)
        - Idiom override (when detected)

    Args:
        text (str): Text to analyze

    Returns:
        dict: Sentiment scores from all analyzers and ensemble
    """
    analyzers = get_sentiment_analyzers()
    results = {}

    if not text or len(text.strip()) < 2:
        return {
            'vader': None,
            'textblob': None,
            'transformer': None,
            'ensemble': {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}
        }

    # ENHANCED: Check for idioms first
    is_idiom, idiom_sentiment, idiom_matched = check_for_idiom(text.lower())

    # VADER
    if analyzers.get('vader'):
        try:
            vader_scores = analyzers['vader'].polarity_scores(text)
            results['vader'] = {'compound': vader_scores['compound']}
        except:
            results['vader'] = None
    else:
        results['vader'] = None

    # TextBlob
    if analyzers.get('textblob'):
        try:
            blob = analyzers['textblob'](text)
            results['textblob'] = {'polarity': blob.sentiment.polarity}
        except:
            results['textblob'] = None
    else:
        results['textblob'] = None

    # Transformer
    if analyzers.get('transformer'):
        try:
            truncated = text[:500] if len(text) > 500 else text
            trans_result = analyzers['transformer'](truncated)[0]
            normalized = trans_result['score'] if trans_result['label'] == 'POSITIVE' else -trans_result['score']
            results['transformer'] = {'normalized_score': normalized}
        except:
            results['transformer'] = None
    else:
        results['transformer'] = None

    # Calculate ensemble
    results['ensemble'] = calculate_ensemble_score(results)

    # ENHANCED: Override score for idioms
    # If we detected a positive idiom but got negative score, fix it
    if is_idiom:
        current_score = results['ensemble'].get('score', 0)
        if idiom_sentiment == 'Positive' and current_score < 0:
            # Override to positive score
            results['ensemble']['score'] = abs(current_score) if abs(current_score) > 0.3 else 0.6
            results['ensemble']['label'] = 'Positive'
            results['ensemble']['idiom_override'] = True
            results['ensemble']['idiom_matched'] = idiom_matched
        elif idiom_sentiment == 'Negative' and current_score > 0:
            # Override to negative score
            results['ensemble']['score'] = -abs(current_score) if abs(current_score) > 0.3 else -0.6
            results['ensemble']['label'] = 'Negative'
            results['ensemble']['idiom_override'] = True
            results['ensemble']['idiom_matched'] = idiom_matched

    return results


def calculate_ensemble_score(sentiment_results):
    """Calculate weighted ensemble sentiment score."""
    scores = []
    weights = []

    if sentiment_results.get('vader') and sentiment_results['vader'].get('compound') is not None:
        scores.append(sentiment_results['vader']['compound'])
        weights.append(0.4)

    if sentiment_results.get('textblob') and sentiment_results['textblob'].get('polarity') is not None:
        scores.append(sentiment_results['textblob']['polarity'])
        weights.append(0.3)

    if sentiment_results.get('transformer') and sentiment_results['transformer'].get('normalized_score') is not None:
        scores.append(sentiment_results['transformer']['normalized_score'])
        weights.append(0.3)

    if not scores:
        return {'score': 0.0, 'label': 'Neutral', 'confidence': 0.0}

    total_weight = sum(weights)
    normalized_weights = [w / total_weight for w in weights]
    ensemble_score = sum(s * w for s, w in zip(scores, normalized_weights))

    # Calculate confidence (agreement)
    if len(scores) > 1:
        variance = sum((s - ensemble_score) ** 2 for s in scores) / len(scores)
        confidence = max(0, 1 - variance)
    else:
        confidence = 0.7

    # Determine label
    if ensemble_score >= 0.05:
        label = 'Positive'
    elif ensemble_score <= -0.05:
        label = 'Negative'
    else:
        label = 'Neutral'

    return {
        'score': round(ensemble_score, 4),
        'label': label,
        'confidence': round(confidence, 4)
    }


# ============================================
# SECTION 15: ENHANCED SENTIMENT CORRECTION
# ============================================

# Strong sentiment indicators for sanity checking
OBVIOUS_NEGATIVE_INDICATORS = {
    'terrible', 'horrible', 'awful', 'worst', 'disgusting', 'nasty',
    'gross', 'pathetic', 'abysmal', 'dreadful', 'atrocious', 'appalling',
    'incompetent', 'useless', 'unacceptable', 'inexcusable', 'nightmare'
}

OBVIOUS_POSITIVE_INDICATORS = {
    'excellent', 'amazing', 'wonderful', 'fantastic', 'outstanding',
    'superb', 'incredible', 'phenomenal', 'magnificent', 'exceptional',
    'perfect', 'best', 'love', 'loved', 'brilliant', 'marvelous'
}


def sanity_check_sentiment(opinion, predicted_sentiment):
    """
    Sanity check to catch obvious sentiment misclassifications.

    Example:
        "TERRIBLE service" should NEVER be classified as Positive
        "EXCELLENT food" should NEVER be classified as Negative

    This is a safety net for when other correction logic fails.

    Args:
        opinion (str): Opinion text
        predicted_sentiment (str): Current sentiment prediction

    Returns:
        tuple: (corrected_sentiment, correction_reason) or (None, None) if no change
    """
    opinion_lower = opinion.lower()
    words = opinion_lower.split()

    # Check for obvious negative indicators
    for neg_word in OBVIOUS_NEGATIVE_INDICATORS:
        if neg_word in words or neg_word in opinion_lower:
            # Check if negated (e.g., "not terrible")
            neg_index = opinion_lower.find(neg_word)
            prefix = opinion_lower[max(0, neg_index-10):neg_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Positive':
                    return 'Negative', f'sanity_check_negative ({neg_word})'

    # Check for obvious positive indicators
    for pos_word in OBVIOUS_POSITIVE_INDICATORS:
        if pos_word in words or pos_word in opinion_lower:
            # Check if negated
            pos_index = opinion_lower.find(pos_word)
            prefix = opinion_lower[max(0, pos_index-10):pos_index]
            if not any(neg in prefix for neg in ['not ', "n't ", 'no ']):
                if predicted_sentiment == 'Negative':
                    return 'Positive', f'sanity_check_positive ({pos_word})'

    return None, None


def correct_sentiment_with_analysis(aspect, opinion, predicted_sentiment, sentiment_scores=None):
    """
    Correct sentiment prediction using rules and analysis.

    PRIORITY ORDER (Critical for accuracy):
        0. SANITY CHECK (FIRST) - Catch obvious misclassifications like "TERRIBLE" → Positive
        1. Idiom detection - "did not disappoint" → Positive
        2. Superlative patterns - "could not have been better" → Positive
        3. Negation + sentiment word (with safeguards for obvious sentiment)
        4. Score override (when highly confident)
        5. Keyword detection
        6. FINAL SANITY CHECK - Ensure obvious words are not misclassified

    Args:
        aspect (str): Aspect term
        opinion (str): Opinion text
        predicted_sentiment (str): Model's prediction
        sentiment_scores (dict): Sentiment analysis scores

    Returns:
        tuple: (corrected_sentiment, correction_reason)
    """
    opinion_lower = opinion.lower()

    # ========== PRIORITY 0: SANITY CHECK (FIRST) ==========
    # Catch obvious cases like "TERRIBLE" classified as Positive
    sanity_result, sanity_reason = sanity_check_sentiment(opinion, predicted_sentiment)
    if sanity_result:
        return sanity_result, sanity_reason

    # ========== PRIORITY 1: IDIOM DETECTION ==========
    is_idiom, idiom_sentiment, idiom_matched = check_for_idiom(opinion_lower)
    if is_idiom:
        if predicted_sentiment != idiom_sentiment:
            return idiom_sentiment, f'idiom_override ({idiom_matched})'
        return predicted_sentiment, None

    # ========== PRIORITY 2: SUPERLATIVE PATTERNS ==========
    superlative_patterns = [
        r"could(n't| not) have been (more )?(friendly|helpful|better|nicer|attentive|professional)",
        r"could(n't| not) ask for (more|better)",
        r"could(n't| not) be (happier|better)",
    ]
    for pattern in superlative_patterns:
        if re.search(pattern, opinion_lower):
            if predicted_sentiment != 'Positive':
                return 'Positive', 'superlative_positive'
            return predicted_sentiment, None

    # Get sentiment word lists
    positive_words = get_positive_sentiment_words()
    negative_words = get_negative_sentiment_words()

    # ========== PRIORITY 3: NEGATION LOGIC (ENHANCED) ==========
    has_negation, neg_type = detect_negation_in_opinion(opinion_lower)

    if has_negation and neg_type != 'idiom_detected':
        # ENHANCED: Check if there's an obvious negative indicator FIRST
        # If there's TERRIBLE/AWFUL/etc., don't apply double negation logic
        has_obvious_negative = any(neg in opinion_lower for neg in OBVIOUS_NEGATIVE_INDICATORS)
        has_obvious_positive = any(pos in opinion_lower for pos in OBVIOUS_POSITIVE_INDICATORS)

        if has_obvious_negative:
            # If obvious negative present, keep as negative regardless of negation
            return 'Negative', f'obvious_negative_override'

        if not has_obvious_negative:
            # Negating positive word → Negative
            for pos_word in positive_words:
                if pos_word in opinion_lower:
                    if "not only" not in opinion_lower:  # Exception
                        result_sentiment = 'Negative'
                        # Final sanity check
                        final_sanity, _ = sanity_check_sentiment(opinion, result_sentiment)
                        if final_sanity:
                            return final_sanity, f'negation_positive ({pos_word}) + sanity_override'
                        return result_sentiment, f'negation_positive ({pos_word})'

            # Negating negative word → Positive (double negative)
            # BUT only if no obvious negative indicators present
            for neg_word in negative_words:
                if neg_word in opinion_lower:
                    # Don't apply double negation if the word is an obvious negative indicator
                    if neg_word not in OBVIOUS_NEGATIVE_INDICATORS:
                        result_sentiment = 'Positive'
                        # Final sanity check
                        final_sanity, _ = sanity_check_sentiment(opinion, result_sentiment)
                        if final_sanity:
                            return final_sanity, f'double_negation ({neg_word}) + sanity_override'
                        return result_sentiment, f'double_negation ({neg_word})'

            # General negation with no clear word
            if sentiment_scores:
                score = sentiment_scores.get('ensemble', {}).get('score', 0)
                if score > 0.3:
                    return predicted_sentiment, None
            result_sentiment = 'Negative'
            # Final sanity check
            final_sanity, _ = sanity_check_sentiment(opinion, result_sentiment)
            if final_sanity:
                return final_sanity, f'negation_general ({neg_type}) + sanity_override'
            return result_sentiment, f'negation_general ({neg_type})'

    # ========== PRIORITY 4: SCORE OVERRIDE ==========
    if sentiment_scores:
        ensemble = sentiment_scores.get('ensemble', {})
        ensemble_score = ensemble.get('score', 0)
        confidence = ensemble.get('confidence', 0)

        if confidence >= SENTIMENT_CONFIDENCE_THRESHOLD:
            if predicted_sentiment == 'Positive' and ensemble_score < -SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(pw in opinion_lower for pw in positive_words):
                    result_sentiment = 'Negative'
                    # Final sanity check
                    final_sanity, _ = sanity_check_sentiment(opinion, result_sentiment)
                    if final_sanity:
                        return final_sanity, f'sentiment_override ({ensemble_score:.2f}) + sanity_override'
                    return result_sentiment, f'sentiment_override ({ensemble_score:.2f})'

            if predicted_sentiment == 'Negative' and ensemble_score > SENTIMENT_OVERRIDE_THRESHOLD:
                if not any(nw in opinion_lower for nw in negative_words):
                    result_sentiment = 'Positive'
                    # Final sanity check
                    final_sanity, _ = sanity_check_sentiment(opinion, result_sentiment)
                    if final_sanity:
                        return final_sanity, f'sentiment_override ({ensemble_score:.2f}) + sanity_override'
                    return result_sentiment, f'sentiment_override ({ensemble_score:.2f})'

    # ========== PRIORITY 5: KEYWORD DETECTION ==========
    for pos_word in positive_words:
        if pos_word in opinion_lower and predicted_sentiment != 'Positive':
            result_sentiment = 'Positive'
            # Final sanity check
            final_sanity, _ = sanity_check_sentiment(opinion, result_sentiment)
            if final_sanity:
                return final_sanity, f'positive_keyword ({pos_word}) + sanity_override'
            return result_sentiment, f'positive_keyword ({pos_word})'

    for neg_word in negative_words:
        if neg_word in opinion_lower and predicted_sentiment != 'Negative':
            result_sentiment = 'Negative'
            # Final sanity check
            final_sanity, _ = sanity_check_sentiment(opinion, result_sentiment)
            if final_sanity:
                return final_sanity, f'negative_keyword ({neg_word}) + sanity_override'
            return result_sentiment, f'negative_keyword ({neg_word})'

    # ========== FINAL SANITY CHECK ==========
    final_sanity, final_reason = sanity_check_sentiment(opinion, predicted_sentiment)
    if final_sanity:
        return final_sanity, final_reason

    return predicted_sentiment, None


# ============================================
# SECTION 16: CONFIDENCE-BASED FILTERING
# ============================================

def should_include_extraction(sentiment_scores, opinion, predicted_sentiment):
    """
    Determine if extraction should be included based on confidence.

    Exclusion Rules:
        1. |score| < MINIMUM_SENTIMENT_SCORE_THRESHOLD (unclear sentiment)
        2. confidence < MINIMUM_AGREEMENT_THRESHOLD (analyzers disagree)

    Exceptions:
        - Known idioms are always included
        - Clear sentiment words override low scores

    Args:
        sentiment_scores (dict): Sentiment analysis results
        opinion (str): Opinion text
        predicted_sentiment (str): Model prediction

    Returns:
        tuple: (include: bool, reason: str or None)
    """
    if not sentiment_scores:
        return True, None

    ensemble = sentiment_scores.get('ensemble', {})
    score = ensemble.get('score', 0)
    confidence = ensemble.get('confidence', 0)

    # Rule 1: Score too close to zero
    if abs(score) < MINIMUM_SENTIMENT_SCORE_THRESHOLD:
        # Exception: Idioms should be included
        is_idiom, _, _ = check_for_idiom(opinion.lower())
        if not is_idiom:
            return False, f'low_score (|{score:.2f}| < {MINIMUM_SENTIMENT_SCORE_THRESHOLD})'

    # Rule 2: Low agreement between analyzers
    if confidence < MINIMUM_AGREEMENT_THRESHOLD:
        # Exception: Clear sentiment words
        positive_words = get_positive_sentiment_words()
        negative_words = get_negative_sentiment_words()
        has_clear_word = any(w in opinion.lower() for w in positive_words | negative_words)
        if not has_clear_word:
            return False, f'low_agreement ({confidence:.2f})'

    return True, None


# ============================================
# SECTION 17: RESULT STRUCTURES
# ============================================

def create_aspect_opinion_pair(aspect, opinion, raw_opinion, sentiment,
                                original_sentiment, pattern, chunk_id,
                                sentiment_scores, correction_reason):
    """Create standardized result dictionary."""

    scores_dict = {
        'vader_compound': None,
        'textblob_polarity': None,
        'transformer_score': None,
        'ensemble_score': None,
        'ensemble_confidence': None
    }

    if sentiment_scores:
        if sentiment_scores.get('vader'):
            scores_dict['vader_compound'] = sentiment_scores['vader'].get('compound')
        if sentiment_scores.get('textblob'):
            scores_dict['textblob_polarity'] = sentiment_scores['textblob'].get('polarity')
        if sentiment_scores.get('transformer'):
            scores_dict['transformer_score'] = sentiment_scores['transformer'].get('normalized_score')
        if sentiment_scores.get('ensemble'):
            scores_dict['ensemble_score'] = sentiment_scores['ensemble'].get('score')
            scores_dict['ensemble_confidence'] = sentiment_scores['ensemble'].get('confidence')

    return {
        'aspect': aspect,
        'opinion': opinion,
        'raw_opinion': raw_opinion,
        'formatted': f"{aspect}: {opinion}",
        'pattern': pattern,
        'chunk': chunk_id,
        'original_sentiment': original_sentiment,
        'sentiment': sentiment,
        'correction_reason': correction_reason,
        'sentiment_scores': scores_dict
    }


# ============================================
# SECTION 18: FILE LOGGING
# ============================================

class FileLogger:
    """Logger for both console and file output."""

    def __init__(self, filename):
        self.filename = filename
        self.content = []

    def log(self, message=""):
        self.content.append(message)
        print(message)

    def save(self):
        with open(self.filename, 'w', encoding='utf-8') as f:
            f.write('\n'.join(self.content))
        print(f"\n✅ Results saved to: {self.filename}")


# ============================================
# SECTION 19: REVIEW CHUNKING
# ============================================

def split_long_review(text, max_words=MAX_WORDS_PER_CHUNK):
    """Split long reviews into sentence-preserving chunks."""
    sentences = text.replace('!', '.').replace('?', '.').split('.')

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        words = sentence.split()
        sentence_length = len(words)

        if sentence_length > max_words:
            if current_chunk:
                chunks.append('. '.join(current_chunk) + '.')
                current_chunk = []
                current_length = 0

            for i in range(0, sentence_length, max_words):
                sub = ' '.join(words[i:i + max_words])
                chunks.append(sub + '.')
            continue

        if current_length + sentence_length > max_words and current_chunk:
            chunks.append('. '.join(current_chunk) + '.')
            current_chunk = [sentence]
            current_length = sentence_length
        else:
            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append('. '.join(current_chunk) + '.')

    return chunks


# ============================================
# SECTION 20: MAIN ANALYSIS FUNCTION
# ============================================

def comprehensive_aspect_opinion_analysis(reviews, aspect_extractor,
                                          max_words=MAX_WORDS_PER_CHUNK,
                                          verbose=True, logger=None,
                                          use_sentiment_analysis=True,
                                          apply_confidence_filter=True):
    """
    Run complete ABSA analysis on restaurant reviews.

    Args:
        reviews: Iterable of review texts
        aspect_extractor: PyABSA aspect extractor
        max_words: Max words per chunk
        verbose: Enable detailed logging
        logger: FileLogger instance
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence-based filtering

    Returns:
        tuple: (results_list, statistics_dict)
    """
    if logger:
        logger.log("=" * 100)
        logger.log("ABSA Analysis System - V5 Enhanced Universal Edition")
        logger.log(f"Chunk size: {max_words} words")
        logger.log(f"Sentiment analysis: {'Enabled' if use_sentiment_analysis else 'Disabled'}")
        logger.log(f"Confidence filter: {'Enabled' if apply_confidence_filter else 'Disabled'}")
        logger.log(f"Min score threshold: {MINIMUM_SENTIMENT_SCORE_THRESHOLD}")
        logger.log(f"Analysis time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        logger.log("=" * 100)

    if use_sentiment_analysis:
        _ = get_sentiment_analyzers()

    all_results = []
    stats = {
        'total_aspects': 0,
        'filtered_invalid_aspects': 0,
        'filtered_invalid_opinions': 0,
        'filtered_low_confidence': 0,
        'sentiment_corrections': 0,
        'correction_reasons': Counter(),
        'idiom_detections': 0,
        'sanity_check_corrections': 0,  # NEW: Track sanity check fixes
    }

    for idx, review in enumerate(reviews, 1):
        words = review.split()
        word_count = len(words)

        if logger and verbose:
            logger.log(f"\n{'=' * 100}")
            logger.log(f"Review #{idx} ({word_count} words)")
            logger.log("=" * 100)

        # Chunking
        if word_count <= max_words:
            result = aspect_extractor.extract_aspect(
                inference_source=[review],
                pred_sentiment=True,
            )[0]
            analysis_results = [{'chunk_id': 1, 'result': result}]
        else:
            if logger and verbose:
                logger.log("📄 Splitting long review into chunks...")

            chunks = split_long_review(review, max_words=max_words)

            if logger and verbose:
                logger.log(f"   Split into {len(chunks)} chunks")

            chunk_results = aspect_extractor.extract_aspect(
                inference_source=chunks,
                pred_sentiment=True,
            )

            analysis_results = [
                {'chunk_id': i + 1, 'result': r}
                for i, r in enumerate(chunk_results)
            ]

        # Process extractions
        aspect_opinion_pairs = []
        filtered_aspect_count = 0
        filtered_opinion_count = 0
        filtered_confidence_count = 0

        for chunk_data in analysis_results:
            result = chunk_data['result']

            tokens = result.get('tokens', [])
            aspects = result.get('aspect', [])
            positions = result.get('position', [])
            sentiments = result.get('sentiment', [])

            for aspect, pos, original_sentiment in zip(aspects, positions, sentiments):

                # Validate aspect
                aspect = refine_aspect_term(aspect)
                if not is_valid_aspect(aspect):
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Extract opinion
                raw_opinion, pattern = extract_complete_opinion_with_boundary(
                    tokens, [pos], aspect, positions
                )

                if not raw_opinion or raw_opinion.strip() == "":
                    filtered_aspect_count += 1
                    stats['filtered_invalid_aspects'] += 1
                    continue

                # Format and validate opinion
                display_opinion = format_opinion_for_display(raw_opinion)

                if not display_opinion or not is_valid_opinion(display_opinion):
                    filtered_opinion_count += 1
                    stats['filtered_invalid_opinions'] += 1
                    continue

                # Sentiment analysis
                sentiment_scores = None
                if use_sentiment_analysis:
                    sentiment_scores = analyze_sentiment_scores(display_opinion)

                # Confidence filtering
                if apply_confidence_filter and sentiment_scores:
                    should_include, _ = should_include_extraction(
                        sentiment_scores, display_opinion, original_sentiment
                    )
                    if not should_include:
                        filtered_confidence_count += 1
                        stats['filtered_low_confidence'] += 1
                        continue

                # Sentiment correction
                corrected_sentiment, correction_reason = correct_sentiment_with_analysis(
                    aspect, raw_opinion, original_sentiment, sentiment_scores
                )

                if corrected_sentiment != original_sentiment:
                    stats['sentiment_corrections'] += 1
                    if correction_reason:
                        stats['correction_reasons'][correction_reason] += 1
                        if 'idiom' in correction_reason:
                            stats['idiom_detections'] += 1
                        if 'sanity_check' in correction_reason:
                            stats['sanity_check_corrections'] += 1

                stats['total_aspects'] += 1

                # Create result
                pair = create_aspect_opinion_pair(
                    aspect=aspect,
                    opinion=display_opinion,
                    raw_opinion=raw_opinion,
                    sentiment=corrected_sentiment,
                    original_sentiment=original_sentiment,
                    pattern=pattern,
                    chunk_id=chunk_data['chunk_id'] if len(analysis_results) > 1 else None,
                    sentiment_scores=sentiment_scores,
                    correction_reason=correction_reason
                )
                aspect_opinion_pairs.append(pair)

        # ENHANCED: Log results with wider columns and full content
        if logger and verbose:
            logger.log(f"\n🎯 Extraction Results:")
            # ENHANCED: Wider opinion column (60 chars instead of 35)
            header = f"{'#':<3} {'Aspect':<20} {'Opinion':<60} {'Orig':<10} {'Final':<10} {'Score':<8}"
            logger.log(header)
            logger.log("-" * 115)

            if aspect_opinion_pairs:
                for i, pair in enumerate(aspect_opinion_pairs, 1):
                    opinion_text = pair['opinion']

                    scores = pair['sentiment_scores']
                    score_str = f"{scores['ensemble_score']:+.2f}" if scores.get('ensemble_score') else "N/A"

                    final = pair['sentiment']
                    if pair['correction_reason']:
                        final += "*"

                    # ENHANCED: Show full opinion with hyphen continuation if needed
                    if len(opinion_text) > 60:
                        # First line with hyphen at end to indicate continuation
                        opinion_line1 = opinion_text[:59] + "-"
                        logger.log(f"{i:<3} {pair['aspect']:<20} {opinion_line1:<60} "
                                  f"{pair['original_sentiment']:<10} {final:<10} {score_str:<8}")

                        # Additional lines for remaining opinion text (with hyphen continuation)
                        remaining = opinion_text[59:]
                        while remaining:
                            if len(remaining) > 60:
                                chunk = remaining[:59] + "-"
                                remaining = remaining[59:]
                            else:
                                chunk = remaining
                                remaining = ""
                            logger.log(f"{'':3} {'':20} {chunk:<60}")
                    else:
                        logger.log(f"{i:<3} {pair['aspect']:<20} {opinion_text:<60} "
                                  f"{pair['original_sentiment']:<10} {final:<10} {score_str:<8}")

                total_filtered = filtered_aspect_count + filtered_opinion_count + filtered_confidence_count
                if total_filtered > 0:
                    logger.log(f"\n   ℹ️ Filtered: {filtered_aspect_count} aspects, "
                              f"{filtered_opinion_count} opinions, "
                              f"{filtered_confidence_count} low confidence")
            else:
                logger.log("   ⚠️ No valid aspect-opinion pairs found")

        all_results.append({
            'review_id': idx,
            'word_count': word_count,
            'was_chunked': word_count > max_words,
            'chunk_count': len(analysis_results),
            'pairs': aspect_opinion_pairs,
        })

    return all_results, stats


# ============================================
# SECTION 21: ENHANCED REPORT GENERATION
# ============================================

def select_representative_opinions(pairs, n=5, target_sentiment=None):
    """
    Select high-quality representative opinions.
    ENHANCED: Now returns 5 opinions by default (increased from 3).

    Selection Criteria:
        1. Sentiment matches target (CRITICAL: positive section shows only positive opinions)
        2. High absolute sentiment score (clear sentiment)
        3. Sufficient length (not fragments)
        4. Passes completeness validation
        5. Diverse (not similar to already selected)
        6. No negative words in positive section (and vice versa)

    Args:
        pairs: List of aspect-opinion pairs
        n: Number of opinions to select (default: 5)
        target_sentiment: 'Positive' or 'Negative' - MUST match this sentiment

    Returns:
        list: Selected representative opinions
    """
    # CRITICAL FIX: Filter by target sentiment FIRST
    if target_sentiment:
        pairs = [p for p in pairs if p.get('sentiment') == target_sentiment]

    # Sort by score magnitude (clearest sentiment first)
    sorted_pairs = sorted(
        pairs,
        key=lambda p: abs(p['sentiment_scores'].get('ensemble_score', 0) if p['sentiment_scores'] else 0),
        reverse=True
    )

    # Define words that indicate opposite sentiment
    strong_negative_words = {
        'terrible', 'horrible', 'awful', 'worst', 'bad', 'poor', 'disgusting',
        'rude', 'slow', 'cold', 'bland', 'disappointing', 'mediocre', 'nasty',
        'gross', 'dirty', 'stale', 'overpriced', 'incompetent', 'never'
    }
    strong_positive_words = {
        'great', 'excellent', 'amazing', 'wonderful', 'fantastic', 'perfect',
        'delicious', 'friendly', 'awesome', 'best', 'love', 'outstanding',
        'superb', 'incredible', 'fresh', 'tasty', 'beautiful', 'attentive'
    }

    selected = []
    seen_stems = set()

    for pair in sorted_pairs:
        opinion = pair['opinion']
        opinion_lower = opinion.lower()

        # CRITICAL: Skip if opinion contains strong opposite-sentiment words
        if target_sentiment == 'Positive':
            # For positive section, skip opinions with strong negative words
            if any(neg_word in opinion_lower for neg_word in strong_negative_words):
                continue
        elif target_sentiment == 'Negative':
            # For negative section, verify it actually sounds negative
            # (optional: could skip if too many positive words)
            pass

        # Quality checks
        if len(opinion.split()) < 2:
            continue

        if opinion_lower.startswith(('and ', 'or ', 'but ')):
            continue

        is_valid, _ = validate_opinion_completeness(opinion)
        if not is_valid:
            continue

        # Diversity check (avoid similar opinions)
        stem = ' '.join(opinion_lower.split()[:3])
        if stem in seen_stems:
            continue

        seen_stems.add(stem)
        selected.append(pair)

        if len(selected) >= n:
            break

    return selected


def generate_management_report(results, stats, logger=None):
    """
    Generate enhanced management report with multiple representative opinions.
    ENHANCED: Shows 5 examples per aspect (increased from 3) with full content.

    Report Sections:
        1. Overall Statistics
        2. Correction Breakdown
        3. Competitive Advantages (with 5 example opinions each)
        4. Areas for Improvement (with 5 example complaints each)
        5. Recommendations
        6. Quality Metrics
    """
    def log(msg=""):
        if logger:
            logger.log(msg)
        else:
            print(msg)

    log("\n" + "=" * 100)
    log("MANAGEMENT ANALYSIS REPORT")
    log("Universal Restaurant Review Analysis")
    log("=" * 100)

    # Aggregate by sentiment
    positive_pairs = []
    negative_pairs = []

    for result in results:
        for pair in result['pairs']:
            if pair['sentiment'] == 'Positive':
                positive_pairs.append(pair)
            else:
                negative_pairs.append(pair)

    total_pairs = len(positive_pairs) + len(negative_pairs)

    # ===== STATISTICS =====
    log(f"\n📊 OVERALL STATISTICS")
    log(f"   • Total reviews analyzed: {len(results)}")
    log(f"   • Total aspects extracted: {total_pairs}")
    log(f"   • Invalid aspects filtered: {stats['filtered_invalid_aspects']}")
    log(f"   • Invalid opinions filtered: {stats['filtered_invalid_opinions']}")
    log(f"   • Low confidence filtered: {stats.get('filtered_low_confidence', 0)}")

    if stats['sentiment_corrections'] > 0:
        log(f"   • Sentiments corrected: {stats['sentiment_corrections']}")
        log(f"   • Idiom detections: {stats.get('idiom_detections', 0)}")
        log(f"   • Sanity check fixes: {stats.get('sanity_check_corrections', 0)}")

    if total_pairs > 0:
        pos_pct = len(positive_pairs) / total_pairs * 100
        neg_pct = len(negative_pairs) / total_pairs * 100
        log(f"   • Positive mentions: {len(positive_pairs)} ({pos_pct:.1f}%)")
        log(f"   • Negative mentions: {len(negative_pairs)} ({neg_pct:.1f}%)")

    # ===== CORRECTION BREAKDOWN =====
    if stats['correction_reasons']:
        log(f"\n📈 CORRECTION BREAKDOWN")
        for reason, count in stats['correction_reasons'].most_common(10):
            log(f"   • {reason}: {count}")

    # Aggregate by aspect
    positive_aspects = {}
    for pair in positive_pairs:
        key = pair['aspect'].lower()
        if key not in positive_aspects:
            positive_aspects[key] = []
        positive_aspects[key].append(pair)

    negative_aspects = {}
    for pair in negative_pairs:
        key = pair['aspect'].lower()
        if key not in negative_aspects:
            negative_aspects[key] = []
        negative_aspects[key].append(pair)

    # ===== COMPETITIVE ADVANTAGES =====
    if positive_aspects:
        log(f"\n✅ COMPETITIVE ADVANTAGES (Top Positive Aspects)")
        log("-" * 100)

        sorted_pos = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_pos[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # ENHANCED: 5 representative opinions - MUST be positive sentiment
            examples = select_representative_opinions(pairs, n=5, target_sentiment='Positive')
            for ex in examples:
                # ENHANCED: Show full opinion text on single line (no truncation)
                opinion_text = ex['opinion']
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0

                # Display full content on one line
                log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid positive examples found, note it
            if not examples:
                log(f'   • [No clear positive examples available]')

    # ===== AREAS FOR IMPROVEMENT =====
    if negative_aspects:
        log(f"\n⚠️ AREAS FOR IMPROVEMENT (Top Negative Aspects)")
        log("-" * 100)

        sorted_neg = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)

        for i, (aspect, pairs) in enumerate(sorted_neg[:10], 1):
            count = len(pairs)
            scores = [p['sentiment_scores']['ensemble_score'] for p in pairs
                      if p['sentiment_scores'].get('ensemble_score')]
            avg_score = sum(scores) / len(scores) if scores else 0

            log(f"\n{i}. {aspect.upper()} ({count} mentions, avg: {avg_score:+.2f})")

            # ENHANCED: 5 representative opinions - MUST be negative sentiment
            examples = select_representative_opinions(pairs, n=5, target_sentiment='Negative')
            for ex in examples:
                # ENHANCED: Show full opinion text on single line (no truncation)
                opinion_text = ex['opinion']
                ex_score = ex['sentiment_scores'].get('ensemble_score', 0) if ex['sentiment_scores'] else 0

                # Display full content on one line
                log(f'   • ({ex_score:+.2f}) "{opinion_text}"')

            # If no valid negative examples found, note it
            if not examples:
                log(f'   • [No clear negative examples available]')

    # ===== RECOMMENDATIONS =====
    log(f"\n💡 RECOMMENDATIONS")

    if negative_aspects:
        worst = sorted(negative_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   1. PRIORITY: Address '{worst[0]}' ({len(worst[1])} negative mentions)")

    if positive_aspects:
        best = sorted(positive_aspects.items(), key=lambda x: len(x[1]), reverse=True)[0]
        log(f"   2. LEVERAGE: Promote '{best[0]}' ({len(best[1])} positive mentions)")

    log(f"   3. MONITOR: Track sentiment trends over time")
    log(f"   4. INVESTIGATE: Review corrected sentiments for accuracy")

    # ===== QUALITY METRICS =====
    log(f"\n📉 EXTRACTION QUALITY METRICS")
    total_reviews = len(results)
    reviews_with_pairs = sum(1 for r in results if len(r['pairs']) > 0)
    avg_pairs = sum(len(r['pairs']) for r in results) / total_reviews if total_reviews > 0 else 0

    log(f"   • Extraction success rate: {reviews_with_pairs}/{total_reviews} ({reviews_with_pairs/total_reviews*100:.1f}%)")
    log(f"   • Average aspects per review: {avg_pairs:.2f}")

    total_filtered = (stats['filtered_invalid_aspects'] +
                      stats['filtered_invalid_opinions'] +
                      stats.get('filtered_low_confidence', 0))
    total_extracted = total_pairs + total_filtered
    if total_extracted > 0:
        quality_rate = total_pairs / total_extracted * 100
        log(f"   • Quality pass rate: {total_pairs}/{total_extracted} ({quality_rate:.1f}%)")


# ============================================
# SECTION 22: DATA EXPORT
# ============================================

def export_results_to_dataframe(results):
    """Export results to pandas DataFrame."""
    import pandas as pd

    rows = []
    for result in results:
        review_id = result['review_id']
        for pair in result['pairs']:
            scores = pair.get('sentiment_scores', {})
            rows.append({
                'review_id': review_id,
                'aspect': pair['aspect'],
                'opinion': pair['opinion'],
                'original_sentiment': pair['original_sentiment'],
                'corrected_sentiment': pair['sentiment'],
                'correction_reason': pair.get('correction_reason'),
                'ensemble_score': scores.get('ensemble_score'),
                'ensemble_confidence': scores.get('ensemble_confidence')
            })

    return pd.DataFrame(rows)


# ============================================
# SECTION 23: MAIN EXECUTION
# ============================================

def run_analysis(reviews, aspect_extractor,
                 output_file="absa_results_v5_universal.txt",
                 use_sentiment_analysis=True,
                 apply_confidence_filter=True):
    """
    Run complete ABSA analysis pipeline.

    Args:
        reviews: Review texts (list or Series)
        aspect_extractor: PyABSA extractor
        output_file: Output log file path
        use_sentiment_analysis: Enable sentiment scoring
        apply_confidence_filter: Enable confidence filtering

    Returns:
        tuple: (results, stats, dataframe)
    """
    logger = FileLogger(output_file)

    logger.log("=" * 100)
    logger.log("ABSA ANALYSIS SYSTEM - V5 ENHANCED UNIVERSAL EDITION")
    logger.log(f"Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    logger.log(f"Reviews to analyze: {len(reviews)}")
    logger.log("=" * 100)

    results, stats = comprehensive_aspect_opinion_analysis(
        reviews,
        aspect_extractor,
        max_words=MAX_WORDS_PER_CHUNK,
        verbose=True,
        logger=logger,
        use_sentiment_analysis=use_sentiment_analysis,
        apply_confidence_filter=apply_confidence_filter
    )

    logger.log("\n" + "=" * 100)
    logger.log("✅ ANALYSIS COMPLETE")
    logger.log("=" * 100)

    generate_management_report(results, stats, logger)
    logger.save()

    df_results = export_results_to_dataframe(results)

    return results, stats, df_results


if __name__ == "__main__":
    print("=" * 70)
    print("ABSA Analysis System - V5 Enhanced Universal Edition")
    print("=" * 70)
    print("\nDesigned for UNIVERSAL restaurant review analysis:")
    print("  • All restaurant types (fast food to fine dining)")
    print("  • All cuisines (American, Asian, Mexican, etc.)")
    print("  • All service models (dine-in, takeout, delivery)")
    print("\nKey Enhancements:")
    print("  1. Idiom/Slang Dictionary (~80 expressions)")
    print("  2. Confidence-Based Filtering")
    print("  3. Improved Opinion Boundary Detection")
    print("  4. Enhanced Management Report (5 examples per aspect)")
    print("  5. Complete sentence extraction")
    print("  6. Removed leading punctuation (comma, parenthesis)")
    print("\nUsage:")
    print("  results, stats, df = run_analysis(reviews, aspect_extractor)")


In [ ]:
results, stats, df = run_analysis(test_reviews, aspect_extractor)

In [ ]:
還沒  先不要做程式  我的想法是 最近的LLM模型是不是可以有句子的相似性判斷 若相似性高表示再討論同一個屬性的內容 這樣是不是較佳 我們一個一個討論 先Q1確定好再討論Q2

In [ ]:
# =============================================================================
# 餐廳評論完整分析系統 - Colab 一站式執行版
# =============================================================================
#
# 使用方式：
# 1. 在 Colab 中執行此程式碼
# 2. 上傳您的原始評論資料 (Excel/CSV)
# 3. 上傳 V3 ABSA 分析結果
# 4. 執行分析並下載報告
#
# =============================================================================

# 安裝必要套件
# !pip install pandas openpyxl sentence-transformers scikit-learn plotly

import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 第一部分：配置與常數定義
# =============================================================================

# 顧客旅程階段（按實際體驗時間順序）
JOURNEY_STAGES = {
    1: {'id': 'visit_motivation', 'name': '1. 來店動機',
        'desc': '顧客選擇原因',
        'keywords': ['recommendation', 'reviews', 'rating', 'yelp', 'google',
                     'heard', 'friend', 'event', 'astros', 'game', 'downtown', 'hotel']},
    2: {'id': 'arrival_waiting', 'name': '2. 到達與等待',
        'desc': '停車、排隊、等位',
        'keywords': ['parking', 'park', 'wait', 'waiting', 'line', 'queue',
                     'reservation', 'walk-in', 'busy', 'crowded']},
    3: {'id': 'first_impression', 'name': '3. 第一印象',
        'desc': '環境、氛圍、裝潢',
        'keywords': ['atmosphere', 'vibe', 'ambiance', 'decor', 'decoration',
                     'interior', 'music', 'noise', 'loud', 'quiet', 'lighting', 'space']},
    4: {'id': 'seating_host', 'name': '4. 接待入座',
        'desc': '接待人員、座位安排',
        'keywords': ['host', 'hostess', 'greeter', 'seated', 'seating',
                     'table', 'booth', 'patio']},
    5: {'id': 'ordering_experience', 'name': '5. 點餐體驗',
        'desc': '菜單、點餐、推薦',
        'keywords': ['menu', 'order', 'ordering', 'recommend', 'specials', 'options']},
    6: {'id': 'wait_for_food', 'name': '6. 等待上菜',
        'desc': '上菜速度',
        'keywords': ['wait for food', 'took long', 'timing', 'came out', 'arrived',
                     'forever', 'minutes']},
    7: {'id': 'food_quality', 'name': '7. 食物品質',
        'desc': '整體食物評價',
        'keywords': ['food', 'meal', 'dish', 'taste', 'flavor', 'texture',
                     'quality', 'fresh', 'cooked', 'seasoning', 'portion']},
    8: {'id': 'specific_dishes', 'name': '8. 具體菜品',
        'desc': '各道菜的評價',
        'keywords': ['chicken', 'waffles', 'brisket', 'nachos', 'burger',
                     'wings', 'fries', 'tacos', 'sandwich', 'salad', 'soup',
                     'steak', 'ribs', 'pork', 'bacon', 'eggs', 'pancakes',
                     'mac', 'cheese', 'grits', 'bread', 'dessert', 'pie']},
    9: {'id': 'beverages', 'name': '9. 飲品',
        'desc': '調酒、飲料',
        'keywords': ['drink', 'drinks', 'cocktail', 'beer', 'wine', 'moonshine',
                     'margarita', 'mimosa', 'bloody mary', 'coffee', 'tea',
                     'happy hour']},
    10: {'id': 'service_during_meal', 'name': '10. 用餐服務',
         'desc': '服務態度、服務品質',
         'keywords': ['server', 'waiter', 'waitress', 'service', 'staff',
                      'attentive', 'friendly', 'rude', 'helpful', 'manager', 'bartender']},
    11: {'id': 'checkout', 'name': '11. 結帳體驗',
         'desc': '結帳、帳單',
         'keywords': ['bill', 'check', 'pay', 'payment', 'tip', 'charge']},
    12: {'id': 'value_perception', 'name': '12. 價值感知',
         'desc': '性價比',
         'keywords': ['price', 'prices', 'cost', 'value', 'worth', 'money',
                      'expensive', 'cheap', 'affordable', 'reasonable', 'overpriced']},
    13: {'id': 'overall_impression', 'name': '13. 整體印象',
         'desc': '總體評價、是否回訪',
         'keywords': ['overall', 'experience', 'return', 'come back', 'again',
                      'recommend', 'definitely', 'never', 'visit']}
}

# 服務人員名字
STAFF_NAMES = {'joel', 'paul', 'kaitlyn', 'kaitlin', 'christina', 'natalie',
               'melanie', 'sara', 'morgan', 'fatima', 'veronica', 'andy',
               'nick', 'alyssa', 'janelle', 'kiara', 'braelyn', 'gabby',
               'adela', 'ray', 'kristina', 'tina', 'cameron', 'zoe',
               'greg', 'steven', 'jordan'}

# 屬性語義聚類映射
SEMANTIC_CLUSTERS = {
    'server': ['server', 'waiter', 'waitress', 'waitstaff'],
    'staff': ['staff', 'employee', 'team', 'crew'],
    'manager': ['manager', 'gm', 'general manager', 'supervisor'],
    'bartender': ['bartender', 'barman', 'barmaid'],
    'host': ['host', 'hostess', 'greeter'],
    'food': ['food', 'meal', 'dish', 'cuisine'],
    'chicken_and_waffles': ['chicken and waffles', 'waffles', 'chicken waffles',
                            'waffle', 'hot chicken', 'nashville hot chicken'],
    'brisket': ['brisket', 'beef brisket', 'smoked brisket'],
    'nachos': ['nachos', 'nacho', 'brisket nachos'],
    'wings': ['wings', 'chicken wings', 'wing'],
    'burger': ['burger', 'hamburger', 'cheeseburger'],
    'fries': ['fries', 'french fries', 'fry', 'sweet potato fries'],
    'tacos': ['tacos', 'taco', 'pulled pork tacos'],
    'mac_and_cheese': ['mac', 'mac and cheese', 'macaroni'],
    'ribs': ['ribs', 'rib', 'bbq ribs'],
    'sandwich': ['sandwich', 'sandwiches', 'grilled cheese'],
    'drinks': ['drinks', 'drink', 'beverages'],
    'cocktails': ['cocktail', 'cocktails', 'mixed drink'],
    'beer': ['beer', 'beers', 'draft', 'ale'],
    'moonshine': ['moonshine', 'shine'],
    'mimosa': ['mimosa', 'mimosas'],
    'margarita': ['margarita', 'margaritas'],
    'atmosphere': ['atmosphere', 'vibe', 'ambiance', 'ambience'],
    'decor': ['decor', 'decoration', 'interior'],
    'seating': ['seating', 'seat', 'seats', 'booth', 'table', 'tables'],
    'service': ['service'],
    'wait_time': ['wait', 'waiting', 'wait time', 'waited'],
    'price': ['price', 'prices', 'cost', 'pricing'],
    'value': ['value', 'worth', 'deal'],
    'cleanliness': ['clean', 'cleanliness', 'dirty', 'sticky'],
    'parking': ['parking', 'park', 'valet'],
    'location': ['location', 'place', 'spot', 'restaurant', 'bar']
}


# =============================================================================
# 第二部分：時間解析函數
# =============================================================================

def parse_relative_time(time_str, reference_date=None):
    """解析相對時間字串"""
    if reference_date is None:
        reference_date = datetime(2026, 1, 15)

    if pd.isna(time_str) or not time_str:
        return None

    time_str = str(time_str).lower().strip()
    time_str = re.sub(r'^edited\s+', '', time_str)

    patterns = [
        (r'(\d+)\s*years?\s*ago', 'years'),
        (r'(\d+)\s*months?\s*ago', 'months'),
        (r'(\d+)\s*weeks?\s*ago', 'weeks'),
        (r'(\d+)\s*days?\s*ago', 'days'),
        (r'(\d+)\s*hours?\s*ago', 'hours'),
        (r'a\s+year\s*ago', 'a_year'),
        (r'a\s+month\s*ago', 'a_month'),
        (r'a\s+week\s*ago', 'a_week'),
        (r'a\s+day\s*ago', 'a_day'),
    ]

    for pattern, unit in patterns:
        match = re.search(pattern, time_str)
        if match:
            if unit.startswith('a_'):
                unit_type = unit.replace('a_', '')
                if unit_type == 'year':
                    return reference_date - relativedelta(years=1)
                elif unit_type == 'month':
                    return reference_date - relativedelta(months=1)
                elif unit_type == 'week':
                    return reference_date - timedelta(weeks=1)
                elif unit_type == 'day':
                    return reference_date - timedelta(days=1)
            else:
                num = int(match.group(1))
                if unit == 'years':
                    return reference_date - relativedelta(years=num)
                elif unit == 'months':
                    return reference_date - relativedelta(months=num)
                elif unit == 'weeks':
                    return reference_date - timedelta(weeks=num)
                elif unit == 'days':
                    return reference_date - timedelta(days=num)
                elif unit == 'hours':
                    return reference_date - timedelta(hours=num)

    return None


def categorize_time_period(review_date):
    """分類時間區間"""
    if review_date is None:
        return 'Unknown'

    now = datetime(2026, 1, 15)
    days = (now - review_date).days

    if days <= 30: return '近1個月'
    elif days <= 90: return '近3個月'
    elif days <= 180: return '近6個月'
    elif days <= 365: return '近1年'
    elif days <= 730: return '1-2年前'
    elif days <= 1095: return '2-3年前'
    elif days <= 1825: return '3-5年前'
    else: return '5年以上'


# =============================================================================
# 第三部分：屬性聚類與旅程分類
# =============================================================================

def standardize_aspect(aspect):
    """標準化屬性名稱"""
    aspect_lower = aspect.lower().strip()

    for canonical, synonyms in SEMANTIC_CLUSTERS.items():
        for synonym in synonyms:
            if synonym in aspect_lower or aspect_lower in synonym:
                return canonical

    return aspect_lower


def classify_journey_stage(aspect, opinion):
    """分類到顧客旅程階段"""
    aspect_lower = aspect.lower()
    opinion_lower = opinion.lower()
    combined = f"{aspect_lower} {opinion_lower}"

    # 優先檢查具體菜品
    for keyword in JOURNEY_STAGES[8]['keywords']:
        if keyword in aspect_lower:
            return 8

    # 檢查飲品
    for keyword in JOURNEY_STAGES[9]['keywords']:
        if keyword in aspect_lower:
            return 9

    # 檢查服務人員
    for keyword in JOURNEY_STAGES[10]['keywords']:
        if keyword in aspect_lower:
            return 10

    # 檢查服務人員名字
    for name in STAFF_NAMES:
        if name in combined:
            return 10

    # 檢查其他階段
    for stage_num, config in JOURNEY_STAGES.items():
        for keyword in config['keywords']:
            if keyword in aspect_lower or keyword in opinion_lower:
                return stage_num

    return 7  # 默認食物品質


def apply_bert_clustering(aspects, use_bert=False):
    """
    應用 BERT 語義聚類（可選）

    如果 use_bert=True，需要安裝 sentence-transformers
    """
    if not use_bert:
        return {aspect: standardize_aspect(aspect) for aspect in aspects}

    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.cluster import AgglomerativeClustering

        print("   載入 BERT 模型...")
        model = SentenceTransformer('all-MiniLM-L6-v2')

        print("   編碼屬性...")
        embeddings = model.encode(list(aspects))

        print("   執行聚類...")
        clustering = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=0.5,
            linkage='ward'
        )
        labels = clustering.fit_predict(embeddings)

        # 為每個聚類選擇代表詞
        cluster_mapping = {}
        for cluster_id in set(labels):
            cluster_aspects = [aspects[i] for i, l in enumerate(labels) if l == cluster_id]
            # 選擇最常見的作為代表
            representative = Counter(cluster_aspects).most_common(1)[0][0]
            for asp in cluster_aspects:
                cluster_mapping[asp] = representative

        return cluster_mapping

    except ImportError:
        print("   ⚠️ sentence-transformers 未安裝，使用規則式聚類")
        return {aspect: standardize_aspect(aspect) for aspect in aspects}


# =============================================================================
# 第四部分：資料處理主函數
# =============================================================================

def parse_v3_results(file_path):
    """解析 V3 ABSA 結果"""
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    review_pattern = r'評論 #(\d+) \((\d+) 字\)'
    pair_pattern = r'(\d+)\.\s+([^:]+):\s+(.+?)\s+(😊|😞)'

    pairs = []
    current_review_id = None

    for line in content.split('\n'):
        review_match = re.search(review_pattern, line)
        if review_match:
            current_review_id = int(review_match.group(1))
            continue

        pair_match = re.search(pair_pattern, line.strip())
        if pair_match and current_review_id:
            pairs.append({
                'review_id': current_review_id,
                'aspect': pair_match.group(2).strip(),
                'opinion': pair_match.group(3).strip(),
                'sentiment': 'Positive' if pair_match.group(4) == '😊' else 'Negative'
            })

    return pd.DataFrame(pairs)


def process_reviews_data(df_reviews, scrape_date_col='scrape_date',
                         relative_time_col='review_time_relative'):
    """處理評論時間資料"""

    df = df_reviews.copy()

    # 解析 scrape_date
    if scrape_date_col in df.columns:
        df['scrape_date_parsed'] = pd.to_datetime(df[scrape_date_col])
    else:
        df['scrape_date_parsed'] = datetime(2026, 1, 15)

    # 解析相對時間
    if relative_time_col in df.columns:
        df['review_date'] = df.apply(
            lambda row: parse_relative_time(
                row[relative_time_col],
                row['scrape_date_parsed'] if pd.notna(row.get('scrape_date_parsed')) else datetime(2026, 1, 15)
            ),
            axis=1
        )
        df['time_period'] = df['review_date'].apply(categorize_time_period)
        df['review_year'] = df['review_date'].apply(lambda x: x.year if x else None)

    return df


def integrate_and_process(df_pairs, df_reviews=None, use_bert=False):
    """整合並處理所有資料"""

    print("=" * 60)
    print("資料整合與處理")
    print("=" * 60)

    # 1. 清理屬性
    print("\n[1/5] 清理屬性名稱...")
    df_pairs['aspect_cleaned'] = df_pairs['aspect'].apply(
        lambda x: re.sub(r'^(and|or|the|a|an)\s+', '', x.strip(), flags=re.IGNORECASE)
    )

    # 2. 語義聚類
    print("[2/5] 屬性語義聚類...")
    unique_aspects = df_pairs['aspect_cleaned'].unique().tolist()
    cluster_mapping = apply_bert_clustering(unique_aspects, use_bert=use_bert)
    df_pairs['aspect_clustered'] = df_pairs['aspect_cleaned'].map(cluster_mapping)

    # 3. 顧客旅程分類
    print("[3/5] 顧客旅程分類...")
    df_pairs['journey_stage'] = df_pairs.apply(
        lambda row: classify_journey_stage(row['aspect_cleaned'], row['opinion']),
        axis=1
    )
    df_pairs['journey_name'] = df_pairs['journey_stage'].map(
        lambda x: JOURNEY_STAGES.get(x, {}).get('name', f'階段{x}')
    )

    # 4. 整合評論元資料
    print("[4/5] 整合評論元資料...")
    if df_reviews is not None:
        df_reviews_processed = process_reviews_data(df_reviews)
        df_merged = df_pairs.merge(
            df_reviews_processed,
            on='review_id',
            how='left'
        )
    else:
        df_merged = df_pairs.copy()

    # 5. 去重
    print("[5/5] 去重處理...")
    original_count = len(df_merged)
    df_merged = df_merged.drop_duplicates(
        subset=['review_id', 'aspect_clustered', 'sentiment'],
        keep='first'
    )
    print(f"   去重: {original_count} -> {len(df_merged)} 筆")

    return df_merged


# =============================================================================
# 第五部分：深度分析
# =============================================================================

def analyze_journey_satisfaction(df):
    """分析各旅程階段滿意度"""
    results = []

    for stage_num in range(1, 14):
        stage_df = df[df['journey_stage'] == stage_num]
        if len(stage_df) == 0:
            continue

        pos = len(stage_df[stage_df['sentiment'] == 'Positive'])
        neg = len(stage_df[stage_df['sentiment'] == 'Negative'])
        total = len(stage_df)

        # 找出該階段主要屬性
        top_pos = stage_df[stage_df['sentiment'] == 'Positive']['aspect_clustered'].value_counts().head(3)
        top_neg = stage_df[stage_df['sentiment'] == 'Negative']['aspect_clustered'].value_counts().head(3)

        results.append({
            'stage_num': stage_num,
            'stage_name': JOURNEY_STAGES[stage_num]['name'],
            'stage_desc': JOURNEY_STAGES[stage_num]['desc'],
            'total_mentions': total,
            'positive_count': pos,
            'negative_count': neg,
            'satisfaction_rate': pos / total * 100,
            'top_positive_aspects': top_pos.to_dict(),
            'top_negative_aspects': top_neg.to_dict()
        })

    return results


def compare_ratings(df, high_rating=5, low_rating=1):
    """比較不同評分的旅程差異"""
    if 'rating' not in df.columns:
        return None

    comparison = {}

    for rating in [high_rating, low_rating]:
        rating_df = df[df['rating'] == rating]
        if len(rating_df) == 0:
            continue

        journey_breakdown = {}
        for stage_num in range(1, 14):
            stage_df = rating_df[rating_df['journey_stage'] == stage_num]
            if len(stage_df) > 0:
                pos = len(stage_df[stage_df['sentiment'] == 'Positive'])
                journey_breakdown[stage_num] = {
                    'stage_name': JOURNEY_STAGES[stage_num]['name'],
                    'total': len(stage_df),
                    'positive': pos,
                    'satisfaction_rate': pos / len(stage_df) * 100
                }

        comparison[rating] = {
            'review_count': rating_df['review_id'].nunique(),
            'mention_count': len(rating_df),
            'overall_positive_rate': len(rating_df[rating_df['sentiment'] == 'Positive']) / len(rating_df) * 100,
            'journey_breakdown': journey_breakdown
        }

    # 計算差異
    gaps = []
    if high_rating in comparison and low_rating in comparison:
        for stage_num in range(1, 14):
            high_data = comparison[high_rating]['journey_breakdown'].get(stage_num)
            low_data = comparison[low_rating]['journey_breakdown'].get(stage_num)

            if high_data and low_data:
                gap = high_data['satisfaction_rate'] - low_data['satisfaction_rate']
                gaps.append({
                    'stage_num': stage_num,
                    'stage_name': JOURNEY_STAGES[stage_num]['name'],
                    'high_rating_satisfaction': high_data['satisfaction_rate'],
                    'low_rating_satisfaction': low_data['satisfaction_rate'],
                    'gap': gap
                })

    comparison['gaps'] = sorted(gaps, key=lambda x: abs(x['gap']), reverse=True)

    return comparison


def analyze_time_trends(df):
    """分析時間趨勢"""
    if 'review_year' not in df.columns or df['review_year'].isna().all():
        return None

    df_with_year = df[df['review_year'].notna()].copy()

    yearly = df_with_year.groupby('review_year').agg({
        'review_id': 'nunique',
        'sentiment': lambda x: (x == 'Positive').sum() / len(x) * 100
    }).reset_index()
    yearly.columns = ['year', 'review_count', 'positive_rate']

    return yearly.to_dict('records')


# =============================================================================
# 第六部分：報告生成
# =============================================================================

def generate_excel_report(df, journey_analysis, comparison, time_trends, output_file):
    """生成完整 Excel 報告"""

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

        # Sheet 1: 原始資料
        export_cols = ['review_id', 'aspect_cleaned', 'aspect_clustered', 'opinion',
                      'sentiment', 'journey_stage', 'journey_name']
        optional_cols = ['rating', 'review_date', 'time_period', 'review_url',
                        'review_text', 'has_photos']

        for col in optional_cols:
            if col in df.columns:
                export_cols.append(col)

        df_export = df[[c for c in export_cols if c in df.columns]].copy()
        df_export.to_excel(writer, sheet_name='原始資料', index=False)

        # Sheet 2: 顧客旅程統計
        journey_df = pd.DataFrame(journey_analysis)
        journey_df = journey_df[[
            'stage_num', 'stage_name', 'stage_desc', 'total_mentions',
            'positive_count', 'negative_count', 'satisfaction_rate'
        ]]
        journey_df.columns = ['旅程順序', '階段名稱', '說明', '提及次數',
                             '正面評價', '負面評價', '滿意度(%)']
        journey_df['滿意度(%)'] = journey_df['滿意度(%)'].round(1)
        journey_df.to_excel(writer, sheet_name='顧客旅程統計', index=False)

        # Sheet 3: 屬性統計
        aspect_stats = df.groupby('aspect_clustered').agg({
            'review_id': 'count',
            'sentiment': lambda x: (x == 'Positive').sum()
        }).reset_index()
        aspect_stats.columns = ['屬性', '總提及', '正面']
        aspect_stats['負面'] = aspect_stats['總提及'] - aspect_stats['正面']
        aspect_stats['滿意度(%)'] = (aspect_stats['正面'] / aspect_stats['總提及'] * 100).round(1)
        aspect_stats = aspect_stats.sort_values('總提及', ascending=False)
        aspect_stats.to_excel(writer, sheet_name='屬性統計', index=False)

        # Sheet 4: 評分對比
        if comparison and 'gaps' in comparison:
            gaps_df = pd.DataFrame(comparison['gaps'])
            if len(gaps_df) > 0:
                gaps_df.columns = ['旅程階段', '階段名稱', '5星滿意度(%)',
                                  '1星滿意度(%)', '差異(%)']
                gaps_df = gaps_df.round(1)
                gaps_df.to_excel(writer, sheet_name='評分對比', index=False)

        # Sheet 5: 5星評論
        if 'rating' in df.columns:
            df_5star = df[df['rating'] == 5][['review_id', 'aspect_clustered',
                                              'opinion', 'journey_name']]
            df_5star.columns = ['評論ID', '屬性', '意見', '旅程階段']
            df_5star.to_excel(writer, sheet_name='5星評論', index=False)

            # Sheet 6: 1星評論
            df_1star = df[df['rating'] == 1][['review_id', 'aspect_clustered',
                                              'opinion', 'journey_name']]
            df_1star.columns = ['評論ID', '屬性', '意見', '旅程階段']
            df_1star.to_excel(writer, sheet_name='1星評論', index=False)

        # Sheet 7: 時間趨勢
        if time_trends:
            trends_df = pd.DataFrame(time_trends)
            trends_df.columns = ['年份', '評論數', '正面率(%)']
            trends_df['正面率(%)'] = trends_df['正面率(%)'].round(1)
            trends_df.to_excel(writer, sheet_name='年度趨勢', index=False)

        # Sheet 8: 負面評價詳情
        df_negative = df[df['sentiment'] == 'Negative'].copy()
        neg_cols = ['review_id', 'aspect_clustered', 'opinion', 'journey_name']
        if 'rating' in df_negative.columns:
            neg_cols.append('rating')
        if 'review_url' in df_negative.columns:
            neg_cols.append('review_url')
        df_negative = df_negative[[c for c in neg_cols if c in df_negative.columns]]
        df_negative.to_excel(writer, sheet_name='負面評價詳情', index=False)

        # Sheet 9: 摘要統計
        summary = {
            '指標': ['總評論數', '有效屬性-意見對', '正面評價數', '負面評價數',
                    '正面評價比例(%)', '平均每則評論屬性數'],
            '數值': [
                df['review_id'].nunique(),
                len(df),
                len(df[df['sentiment'] == 'Positive']),
                len(df[df['sentiment'] == 'Negative']),
                round(len(df[df['sentiment'] == 'Positive']) / len(df) * 100, 1),
                round(len(df) / df['review_id'].nunique(), 2)
            ]
        }
        pd.DataFrame(summary).to_excel(writer, sheet_name='摘要統計', index=False)

    print(f"✅ Excel 報告已保存: {output_file}")


def generate_text_report(df, journey_analysis, comparison, output_file):
    """生成文字報告"""

    report = []

    def add(text=""):
        report.append(text)

    add("=" * 80)
    add("餐廳評論深度洞察分析報告")
    add(f"分析時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    add("=" * 80)

    # 執行摘要
    add("\n" + "=" * 80)
    add("📊 執行摘要")
    add("=" * 80)

    total_reviews = df['review_id'].nunique()
    total_mentions = len(df)
    pos_count = len(df[df['sentiment'] == 'Positive'])
    pos_rate = pos_count / total_mentions * 100

    add(f"\n總評論數: {total_reviews}")
    add(f"有效屬性-意見對: {total_mentions}")
    add(f"正面評價: {pos_count} ({pos_rate:.1f}%)")
    add(f"負面評價: {total_mentions - pos_count} ({100-pos_rate:.1f}%)")

    if 'rating' in df.columns:
        avg_rating = df.groupby('review_id')['rating'].first().mean()
        add(f"平均評分: {avg_rating:.2f}/5.0")

    # 顧客旅程分析
    add("\n" + "=" * 80)
    add("🚶 顧客旅程分析 (按實際體驗順序)")
    add("=" * 80)

    for stage in journey_analysis:
        sat_rate = stage['satisfaction_rate']
        status = "✅" if sat_rate >= 70 else ("⚠️" if sat_rate >= 50 else "🚨")

        add(f"\n{status} {stage['stage_name']}")
        add(f"   {stage['stage_desc']}")
        add(f"   提及: {stage['total_mentions']} | 正面: {stage['positive_count']} | "
            f"負面: {stage['negative_count']} | 滿意度: {sat_rate:.1f}%")

        if stage['top_positive_aspects']:
            add(f"   ✅ 正面焦點: {', '.join(stage['top_positive_aspects'].keys())}")
        if stage['top_negative_aspects']:
            add(f"   ⚠️ 負面焦點: {', '.join(stage['top_negative_aspects'].keys())}")

    # 5星 vs 1星對比
    if comparison and 'gaps' in comparison:
        add("\n" + "=" * 80)
        add("⭐ 5星 vs 1星 關鍵差異")
        add("=" * 80)

        add(f"\n{'旅程階段':<20} {'5星':<10} {'1星':<10} {'差異':<10}")
        add("-" * 50)

        for gap in comparison['gaps'][:10]:
            add(f"{gap['stage_name']:<20} "
                f"{gap['high_rating_satisfaction']:.1f}%{'':<5} "
                f"{gap['low_rating_satisfaction']:.1f}%{'':<5} "
                f"{gap['gap']:+.1f}%")

    # 行動建議
    add("\n" + "=" * 80)
    add("💡 行動建議")
    add("=" * 80)

    # 找出最需改進的階段
    worst_stages = sorted(journey_analysis, key=lambda x: x['satisfaction_rate'])[:3]

    add("\n🔴 優先改進:")
    for i, stage in enumerate(worst_stages, 1):
        add(f"   {i}. {stage['stage_name']} (滿意度 {stage['satisfaction_rate']:.1f}%)")
        if stage['top_negative_aspects']:
            top_issue = list(stage['top_negative_aspects'].keys())[0]
            add(f"      主要問題: {top_issue}")

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report))

    print(f"✅ 文字報告已保存: {output_file}")

    return '\n'.join(report)


# =============================================================================
# 第七部分：主程式
# =============================================================================

def run_full_analysis(v3_result_file, reviews_file=None, output_prefix="restaurant_analysis",
                      use_bert=False, high_rating=5, low_rating=1):
    """
    執行完整分析

    Parameters:
    -----------
    v3_result_file : str
        V3 ABSA 分析結果檔案路徑
    reviews_file : str, optional
        原始評論資料檔案路徑 (Excel 或 CSV)
    output_prefix : str
        輸出檔案前綴
    use_bert : bool
        是否使用 BERT 進行語義聚類
    high_rating : int
        高分組評分 (用於對比分析)
    low_rating : int
        低分組評分 (用於對比分析)

    Returns:
    --------
    df : DataFrame
        處理後的完整資料
    results : dict
        分析結果
    """

    print("=" * 60)
    print("餐廳評論完整分析系統")
    print("=" * 60)

    # 1. 解析 V3 結果
    print("\n[1/6] 解析 V3 ABSA 結果...")
    df_pairs = parse_v3_results(v3_result_file)
    print(f"   解析完成: {len(df_pairs)} 筆屬性-意見對")

    # 2. 載入評論元資料
    print("\n[2/6] 載入評論元資料...")
    if reviews_file:
        if reviews_file.endswith('.xlsx'):
            df_reviews = pd.read_excel(reviews_file)
        else:
            df_reviews = pd.read_csv(reviews_file)
        print(f"   載入完成: {len(df_reviews)} 則評論")

        print(f"   載入完成: {len(df_reviews)} 則評論")
    else:
        df_reviews = None
        print("   ⚠️ 未提供評論元資料，將使用有限分析")

    # 3. 整合與處理
    print("\n[3/6] 整合與處理資料...")
    df = integrate_and_process(df_pairs, df_reviews, use_bert=use_bert)

    # 4. 顧客旅程分析
    print("\n[4/6] 顧客旅程分析...")
    journey_analysis = analyze_journey_satisfaction(df)

    # 5. 評分對比分析
    print("\n[5/6] 評分對比分析...")
    comparison = compare_ratings(df, high_rating, low_rating)

    # 6. 時間趨勢分析
    print("\n[6/6] 時間趨勢分析...")
    time_trends = analyze_time_trends(df)

    # 生成報告
    print("\n" + "=" * 60)
    print("生成報告...")
    print("=" * 60)

    excel_file = f"{output_prefix}_report.xlsx"
    text_file = f"{output_prefix}_report.txt"

    generate_excel_report(df, journey_analysis, comparison, time_trends, excel_file)
    generate_text_report(df, journey_analysis, comparison, text_file)

    # 彙總結果
    results = {
        'journey_analysis': journey_analysis,
        'comparison': comparison,
        'time_trends': time_trends,
        'summary': {
            'total_reviews': df['review_id'].nunique(),
            'total_mentions': len(df),
            'positive_count': len(df[df['sentiment'] == 'Positive']),
            'negative_count': len(df[df['sentiment'] == 'Negative']),
            'positive_rate': len(df[df['sentiment'] == 'Positive']) / len(df) * 100
        }
    }

    print("\n" + "=" * 60)
    print("✅ 分析完成!")
    print("=" * 60)
    print(f"\n📊 輸出檔案:")
    print(f"   • {excel_file}")
    print(f"   • {text_file}")

    return df, results


# =============================================================================
# 執行範例
# =============================================================================

if __name__ == "__main__":

    # 配置檔案路徑
    V3_RESULT_FILE = "absa_analysis_results_v3.txt"
    REVIEWS_FILE = "selected_restaurant_data.xlsx"  # 如果有評論元資料，設定檔案路徑
    OUTPUT_PREFIX = "restaurant_analysis"

    # 執行分析
    df, results = run_full_analysis(
        v3_result_file=V3_RESULT_FILE,
        reviews_file=REVIEWS_FILE,
        output_prefix=OUTPUT_PREFIX,
        use_bert=False,  # 設為 True 啟用 BERT 聚類
        high_rating=5,
        low_rating=1
    )

    # 顯示快速摘要
    print("\n" + "=" * 60)
    print("📊 快速摘要")
    print("=" * 60)

    summary = results['summary']
    print(f"總評論數: {summary['total_reviews']}")
    print(f"有效屬性-意見對: {summary['total_mentions']}")
    print(f"正面評價: {summary['positive_count']} ({summary['positive_rate']:.1f}%)")
    print(f"負面評價: {summary['negative_count']} ({100-summary['positive_rate']:.1f}%)")

    print("\n🚶 顧客旅程滿意度排行:")
    journey = sorted(results['journey_analysis'], key=lambda x: x['satisfaction_rate'], reverse=True)
    for stage in journey[:5]:
        print(f"   {stage['stage_name']}: {stage['satisfaction_rate']:.1f}%")

    if results['comparison'] and 'gaps' in results['comparison']:
        print("\n⭐ 5星vs1星最大差異:")
        for gap in results['comparison']['gaps'][:3]:
            print(f"   {gap['stage_name']}: 差異 {gap['gap']:+.1f}%")

In [ ]:
aspect_extractor

In [ ]:
# 執行全面的方面級意見分析
# 確保 test_reviews 和 aspect_extractor 已經存在
if 'test_reviews' in locals() and test_reviews and 'aspect_extractor' in locals():
    print("開始對實際店家評論進行全面方面級情感分析...")
    print("=" * 80)

    # 使用之前定義的 comprehensive_aspect_opinion_analysis_complete 函數
    # verbose=True 將顯示詳細的處理過程和結果
    real_reviews_analysis_results = comprehensive_aspect_opinion_analysis_complete(
        test_reviews,
        aspect_extractor,
        max_words=80, # 每段評論的最大字數，可根據需求調整
        verbose=True
    )

    print("\n" + "=" * 80)
    print("✅ 實際評論分析完成！")
    print("=" * 80)

    # 生成管理報告
    generate_management_report_complete(real_reviews_analysis_results)

else:
    print("錯誤: 'test_reviews' 或 'aspect_extractor' 變數未定義或為空。請確保之前的儲存格已執行。")



In [ ]:
# 執行全面的方面級意見分析
# 確保 test_reviews 和 aspect_extractor 已經存在
if 'test_reviews' in locals() and test_reviews and 'aspect_extractor' in locals():
    print("開始對實際店家評論進行全面方面級情感分析...")
    print("=" * 80)

    # 使用之前定義的 comprehensive_aspect_opinion_analysis_complete 函數
    # verbose=True 將顯示詳細的處理過程和結果
    real_reviews_analysis_results = comprehensive_aspect_opinion_analysis_complete(
        test_reviews,
        aspect_extractor,
        max_words=80, # 每段評論的最大字數，可根據需求調整
        verbose=True
    )

    print("\n" + "=" * 80)
    print("✅ 實際評論分析完成！")
    print("=" * 80)

    # 生成管理報告
    generate_management_report_complete(real_reviews_analysis_results)

else:
    print("錯誤: 'test_reviews' 或 'aspect_extractor' 變數未定義或為空。請確保之前的儲存格已執行。")

In [ ]:
test_reviews[0]

In [ ]:
# 確保 df_selected_data 和 aspect_extractor 已經定義
if 'df_selected_data' in locals() and 'aspect_extractor' in locals():
    perform_sentiment_analysis_for_score_range(df_selected_data.copy(), [1,2,3,4,5], aspect_extractor)
else:
    print("錯誤: 'df_selected_data' 或 'aspect_extractor' 未定義。請確保之前的儲存格已執行。")